# Shopify Pipeline v9

**v9 changes:** Fixed CSS framework · EPROLO variants · Trust signals · Font-size guard · Mobile validation · Smart interlinking

## Cell 1 — Config + Upload

In [1]:
# ╔══════════════════════════════════════════════════════╗
# ║         CONFIG                                       ║
# ╚══════════════════════════════════════════════════════╝

import os
from pathlib import Path

# Auto-load from Colab Secrets (one-time setup) — falls through to os.environ if not in Colab.
# Add 7 keys ONCE in Colab → 🔑 panel → enable Notebook access. After that, Cell 1 just works.
try:
    from google.colab import userdata
    for _k in ["ANTHROPIC_API_KEY", "SHOPIFY_STORE", "SHOPIFY_CLIENT_ID", "SHOPIFY_CLIENT_SECRET",
               "DATAFORSEO_LOGIN", "DATAFORSEO_PASSWORD"]:
        if not os.environ.get(_k):
            try: os.environ[_k] = userdata.get(_k)
            except Exception: pass  # secret missing or no notebook access — handled below
except ImportError:
    pass  # not running in Colab

ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]
SHOPIFY_STORE = os.environ["SHOPIFY_STORE"]
SHOPIFY_CLIENT_ID = os.environ["SHOPIFY_CLIENT_ID"]
SHOPIFY_CLIENT_SECRET = os.environ["SHOPIFY_CLIENT_SECRET"]
DATAFORSEO_LOGIN = os.environ["DATAFORSEO_LOGIN"]
DATAFORSEO_PASSWORD = os.environ["DATAFORSEO_PASSWORD"]

LOCATION_CODE = 2840
LANGUAGE_CODE = 'en'
MAX_KD = 50  # WANELO.com = strong dropped domain, can target higher KD
MIN_VOLUME = 50
TOTAL_BUDGET = 15.00
AUTO_CONFIRM_UNDER = 20.0
STAGE_EXPAND_TOP = 30
TOP_K_PER_PRODUCT = 25
RECALL_K_BASE = 100
RERANKER_THRESHOLD = -3

LANGUAGE = "English"
MODEL_VISION = "claude-haiku-4-5-20251001"
MODEL_WRITER = "claude-sonnet-4-6"
MODEL_DESIGNER = "claude-sonnet-4-6"  # writes JSON content (cheaper, structural).
# Designer provider switch (A/B-validated). 'deepseek' = ~20x cheaper Designer step.
DESIGNER_PROVIDER = os.environ.get('DESIGNER_PROVIDER', 'anthropic').strip().lower()
DEEPSEEK_MODEL    = os.environ.get('DEEPSEEK_MODEL', 'deepseek-v4-pro')
DEEPSEEK_API_KEY  = os.environ.get('DEEPSEEK_API_KEY', '')

# === Strategy + Designer agents ===
# MODEL_STRATEGY: Opus 4.7 — marketing positioning (called once per product, cached in DB).
# MODEL_DESIGNER (defined above): Sonnet 4.6 — writes content JSON + photo briefs in ONE call.
MODEL_STRATEGY        = "claude-opus-4-8"

# ╔══════════════════════════════════════════════════════╗
# ║   💰 BILLING DASHBOARDS — check before every run      ║
# ╠══════════════════════════════════════════════════════╣
# ║ Anthropic (Claude):   https://console.anthropic.com/settings/billing
# ║ DataForSEO (SEO):     https://app.dataforseo.com/billing
# ╚══════════════════════════════════════════════════════╝
print("\n💰 BILLING — пополни ДО прогона, чтобы пайплайн не остановился посередине:")
print("   Anthropic:  https://console.anthropic.com/settings/billing")
print("   DataForSEO: https://app.dataforseo.com/billing\n")


# === Источник `taxonomy.json` (raw URL — фетчится в рантайме) ===
# После merge feature-ветки в main — поменять на "main".
GITHUB_BRANCH = "claude/enrich-shopify-taxonomy-kMPmA"  # после merge feature → main: смени на "main"

#@title Database reset switch { display-mode: "form" }
#@markdown Поставь галочку **RESET_DB** чтобы wipe pipeline.db и upload свежие CSV/Excel.
#@markdown Без галочки — пайплайн продолжит с существующей DB (resumable).
RESET_DB = False  #@param {type:"boolean"}
GITHUB_REPO   = "invme2/tags-recomendations"
# Если репо приватный — поставь токен в окружение: os.environ["GITHUB_TOKEN"]="ghp_..."
# Если репо публичный (Settings → Change visibility → Public) — токен не нужен.


# === HTML clean-output rules — добавляются ко всем prompt'ам, генерящим HTML/текст ===
# Просим модель тихо вычистить невидимые символы (zero-width, BOM, directional, PUA)
# и не вставлять AI-водяные знаки / скрытый SEO-стаффинг.
HTML_CLEAN_OUTPUT_RULES = """

CLEAN-OUTPUT RULES (apply silently before delivering):

Silently remove these invisible/control characters if any were generated:
- U+200B Zero Width Space
- U+200C Zero Width Non-Joiner
- U+200D Zero Width Joiner
- U+FEFF BOM / Zero Width No-Break Space
- U+2060 Word Joiner
- U+00AD Soft Hyphen
- U+034F Combining Grapheme Joiner
- U+061C Arabic Letter Mark
- U+180E Mongolian Vowel Separator
- U+200E Left-To-Right Mark
- U+200F Right-To-Left Mark
- U+202A–U+202E directional embedding / override marks
- U+2066–U+2069 isolate marks
- U+E000–U+F8FF Private Use Area characters
- Variation selectors used as hidden marks
- Any other invisible or suspicious control characters

NEVER include in output:
- AI notes about the generation process
- Prompt fragments from the user request
- Debug comments
- Hidden HTML comments containing generation context
- Generation metadata
- Attributes like data-ai, data-generated, data-watermark, data-model
- Hidden SEO text or invisible keyword stuffing
- Off-screen keyword blocks (position:absolute; left:-9999px; and similar)
- Transparent text (color matching background)
- Zero-size text (font-size:0)
- Mixed-script homoglyphs in English words (Cyrillic/Greek lookalikes inside Latin text)
"""

# ═══ v9.2 NEW CONSTANTS ═══
STORE_VENDOR = "wanelo"
STORE_DOMAIN = "wanelo.com"  # Dropped domain with DA 60+
DEFAULT_INVENTORY = 999
MARGIN_ALERT_MIN = 5.00  # Flag products where sell price < $5
PUBLISH_STATUS = "ACTIVE"  # Full automation — products go live immediately
# Retail-pricing multipliers — env-tunable for different price tiers / categories.
# RETAIL_MARKUP=10 means a $5 cost product sells at ~$50 (cost × 10, .90 psych).
# COMPARE_AT_MARKUP=15 means the strike-through price shows ~$80 (cost × 15) so
# buyer perceives a 30%+ "deal" — anchor effect lifts cold-traffic CVR.

def _safe_env_float(key, default):
    """Read float from env. On parse failure print a warning and return
    default — protects unattended runs from a typo'd Colab Secret."""
    raw = os.environ.get(key)
    if raw is None or raw == '':
        return float(default)
    try:
        return float(raw)
    except (ValueError, TypeError):
        print(f'  ⚠ env {key}={raw!r} is not a number, using default {default}')
        return float(default)

def _safe_env_int(key, default):
    """Read int from env. On parse failure print a warning and return default."""
    raw = os.environ.get(key)
    if raw is None or raw == '':
        return int(default)
    try:
        return int(raw)
    except (ValueError, TypeError):
        print(f'  ⚠ env {key}={raw!r} is not an integer, using default {default}')
        return int(default)

RETAIL_MARKUP      = _safe_env_float('RETAIL_MARKUP', 10.0)
COMPARE_AT_MARKUP  = _safe_env_float('COMPARE_AT_MARKUP', 15.0)
# Per-product concurrency for the Cell 6 loop. Default 1 = serial (no
# behaviour change). With WAL SQLite + Shopify rate limit + AsyncAnthropic
# (true async, not sync-in-async), 3-5 is a safe ceiling for big batches.
# >5 risks Shopify 429 + Anthropic concurrent limits.
BATCH_CONCURRENCY  = _safe_env_int('BATCH_CONCURRENCY', 3)
# Hard cap on total Anthropic spend for this batch. Protects against
# runaway cost (e.g. a 100k-row CSV typo, runaway retries). Default $1000
# is generous for a 4-5K product batch at ~$0.20/product. Pipeline stops
# accepting new AI calls once breached; products beyond the cap end up
# status='error' with a clear message.
MAX_ANTHROPIC_BUDGET = _safe_env_float('MAX_ANTHROPIC_BUDGET', 1000.0)
# DataForSEO spend cap — was previously hardcoded TOTAL_BUDGET = 15.00 at
# the top of Cell 1 (still set for backwards-compat). Now env-tunable via
# MAX_DATAFORSEO_BUDGET. CostTracker reads TOTAL_BUDGET below, so we
# reassign here to pick up env override before tracker is created.
TOTAL_BUDGET = _safe_env_float('MAX_DATAFORSEO_BUDGET', TOTAL_BUDGET)

# ═══ TAXONOMY CONFIG (v3.2) ═══
# Set this to your raw taxonomy.json URL (e.g. GitHub raw)
TAXONOMY_URL = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{GITHUB_BRANCH}/taxonomy/taxonomy.json"
TAXONOMY_REFRESH = False     # True = always re-download; False = use cache after first fetch
TAXONOMY_APPROVED_ONLY = False  # True = use only V3.1 status=approved; False = include V3.2 drafts
TAGS_PER_PRODUCT_MIN = 4    # min total tags per product (cluster + persona + intent + demo)
TAGS_PER_PRODUCT_MAX = 9
TAXONOMY_SHORTLIST_K = 40   # how many candidate clusters to include in Stage 2 prompt


SHIPPING_RETURNS_HTML = """
<div class="wa-section wa-shipping-block">
  <div class="wa-trust" style="border-top:1px solid #e5e7eb;padding-top:1.5rem">
    <div class="wa-trust-badge"><span class="icon">🚚</span> Free Shipping</div>
    <div class="wa-trust-badge"><span class="icon">📦</span> Ships in 1-3 Business Days</div>
    <div class="wa-trust-badge"><span class="icon">🔄</span> 30-Day Easy Returns</div>
    <div class="wa-trust-badge"><span class="icon">🛡️</span> Buyer Protection</div>
    <div class="wa-trust-badge"><span class="icon">💳</span> Secure Checkout</div>
  </div>
  <details class="wa-faq" style="margin-top:0.5rem">
    <summary>Shipping & Returns Policy</summary>
    <div class="wa-faq-body">
      <p><strong>Shipping:</strong> We offer free standard shipping on all orders. Orders are processed within 1-2 business days and typically arrive within 7-15 business days depending on your location.</p>
      <p><strong>Returns:</strong> Not satisfied? Return any unused item within 30 days of delivery for a full refund. Simply contact our support team to initiate your return.</p>
      <p><strong>Buyer Protection:</strong> Every purchase is covered by our buyer protection guarantee. If your item arrives damaged or not as described, we will replace it or refund you in full.</p>
    </div>
  </details>
  <div id="judgeme_product_reviews" class="wa-section" style="margin-top:1.5rem">
    <!-- Judge.me widget renders here automatically -->
  </div>
</div>
"""


BRAND_BLACKLIST = [
    'cerave','crest','neutrogena','maybelline','nyx','loreal','revlon',
    'clinique','olay','dove','nivea','garnier','covergirl','sephora',
    'ulta','mac ','benefit','tarte','fenty','rare beauty','elf ',
    'colgate','sensodyne','oral-b','philips','braun','waterpik',
    'listerine','colourpop','morphe','anastasia','urban decay',
    'too faced','nars','bobbi brown','estee lauder','lancome',
    'victoria secret','bath body works','the ordinary','cetaphil',
    'marvis','la roche','bioderma','vichy','avene','kiehl','persmax','cutex',
    'aveeno','eucerin','pantene','amika','lonris','mario badescu',
    'makeup by mario','cardi b','pete davidson','dr.','dr ',"l'oreal",
    'sally hansen','opi ','essie ','zoya ','gelish','orly ',
    'bath and body works','bath & body works','bath and body',
    'dyson','airwrap','air wrap',
    'victoria secret','calvin klein','ralph lauren',
]
IRRELEVANT_PATTERNS = [
    'near me','close to me','around me','in my area',
    'salon','clinic','spa near','parlor','parlour','appointment','booking',
    'dermatologist','surgeon','doctor','botox','filler injection',
    'hair transplant','laser treatment','laser removal','laser hair',
    'tattoo removal','tattoo artist','tattoo shop',
    'nail salon','nail tech near','manicure near','pedicure near',
    'costume makeup','halloween costume','cosplay makeup',
    'amazon','walmart','target ',
    'baby shower','baby clothes','baby registry','baby names',
    'shower curtain','shower head','shower door','shower rod',
    'bath bomb recipe','bath bomb mold diy',
    'body works sale','body works coupon',
    'hair dryer','blow dryer','flat iron','curling iron',
]

# ════ INSTALL ════
!pip install anthropic sentence-transformers requests openpyxl pandas tqdm httpx playwright aiohttp -q
!pip install faiss-gpu -q 2>/dev/null || pip install faiss-cpu -q
!playwright install chromium 2>/dev/null
!playwright install-deps 2>/dev/null

# Fix async for Colab — MUST run before any await
print('async: patched')

# ════ DRIVE + DB ════
from google.colab import drive, files as colab_files
drive.mount('/content/drive')

import os, sys, sqlite3, json, time, re, base64, shutil
os.environ['PYTHONIOENCODING'] = 'utf-8'
try: sys.stdout.reconfigure(encoding='utf-8')
except: pass

PROJECT_DIR = '/content/drive/MyDrive/shopify_pipeline'
OUTPUT_DIR = f'{PROJECT_DIR}/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

DB_PATH = f'{PROJECT_DIR}/pipeline.db'
# Local DB (fast) + Drive backup (survives restart)
DB_LOCAL = '/content/pipeline.db'
if os.path.exists(DB_PATH) and not os.path.exists(DB_LOCAL):
    shutil.copy(DB_PATH, DB_LOCAL)
    print(f'Restored DB from Drive')
db = sqlite3.connect(DB_LOCAL)
db.execute("PRAGMA journal_mode=WAL")
db.row_factory = sqlite3.Row

db.executescript("""
CREATE TABLE IF NOT EXISTS products (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL, eprolo_url TEXT NOT NULL,
    status TEXT DEFAULT 'pending', error_msg TEXT,
    scrape_json TEXT, image_urls_json TEXT,
    stage1_json TEXT, stage2_json TEXT, final_html TEXT,
    seo_title TEXT, seo_description TEXT, url_handle TEXT,
    shopify_product_id TEXT, product_tags TEXT,
    variants_json TEXT, cost_usd REAL DEFAULT 0, updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
CREATE TABLE IF NOT EXISTS collections (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    original_name TEXT, seo_title TEXT NOT NULL,
    seo_handle TEXT, full_url TEXT,
    meta_title TEXT, meta_description TEXT,
    shopify_collection_id TEXT
);
CREATE TABLE IF NOT EXISTS product_collections (
    product_id INTEGER, collection_id INTEGER, UNIQUE(product_id, collection_id)
);
CREATE TABLE IF NOT EXISTS seo_state (key TEXT PRIMARY KEY, value TEXT);
CREATE TABLE IF NOT EXISTS seo_keywords (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    keyword TEXT NOT NULL,
    collection_name TEXT,
    volume INTEGER DEFAULT 0,
    kd INTEGER DEFAULT 0,
    cpc REAL DEFAULT 0,
    competition TEXT,
    UNIQUE(keyword, collection_name)
);
""")
for col in ['shopify_product_id','seo_title','seo_description','url_handle','product_tags','variants_json','seo_keywords','primary_collection','extra_css','strategy_json','assets_json']:
    try: db.execute(f'ALTER TABLE products ADD COLUMN {col} TEXT')
    except: pass
try: db.execute('ALTER TABLE collections ADD COLUMN shopify_collection_id TEXT')
except: pass
try: db.execute('ALTER TABLE collections ADD COLUMN total_volume INTEGER DEFAULT 0')
except: pass
try: db.execute('ALTER TABLE collections ADD COLUMN top_keywords TEXT DEFAULT ""')
except: pass
# Idempotency guard for batch CSV imports: same EPROLO URL cannot be inserted twice.
# Coupled with INSERT OR IGNORE below — re-sending a CSV that overlaps a previous
# batch silently skips duplicates instead of erroring out the import.
try: db.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_products_eprolo_url ON products(eprolo_url) WHERE eprolo_url != ''")
except: pass
db.commit()

# ════ CHECK STATE OR UPLOAD ════
n_prods = db.execute('SELECT COUNT(*) FROM products').fetchone()[0]
n_colls = db.execute('SELECT COUNT(*) FROM collections').fetchone()[0]
n_done = db.execute("SELECT COUNT(*) FROM products WHERE status='done'").fetchone()[0]
SEO_SKIP = False
DESIGN_TEMPLATE = ""
CATEGORY = "Unknown"
PRODUCTS = []

if RESET_DB and n_prods > 0:
    print(f'DB: {n_prods} products ({n_done} done), {n_colls} collections')
    print('⚠ RESET_DB=True — wiping all data...')
    db.execute("DELETE FROM products"); db.execute("DELETE FROM collections")
    db.execute("DELETE FROM product_collections"); db.execute("DELETE FROM seo_state")
    try: db.execute("DELETE FROM seo_keywords")
    except: pass
    db.commit(); n_prods = 0; n_colls = 0
    print('✓ Database wiped clean! Upload widget will appear below — выбери CSV/Excel.')
elif RESET_DB and n_prods == 0:
    print('RESET_DB=True (DB уже пустая) — upload widget ниже.')
elif n_prods > 0:
    PRODUCTS = [r[0] for r in db.execute('SELECT name FROM products ORDER BY id').fetchall()]
    row = db.execute("SELECT value FROM seo_state WHERE key='category'").fetchone()
    CATEGORY = row[0] if row else 'Unknown'
    tmpl = db.execute("SELECT value FROM seo_state WHERE key='design_template'").fetchone()
    DESIGN_TEMPLATE = tmpl[0] if tmpl else ""
    SEO_SKIP = n_colls > 0
    print(f'✓ Continuing with {n_prods} products ({n_done} done) — RESET_DB=False (поставь галочку чтобы перезалить).')
else:
    print('DB пустая — upload widget ниже.')

# Runner-injected CSV: if a local runner has set WANELO_RUNNER_CSV to an
# existing path, we merge it on every run (deduped by UNIQUE index above)
# without requiring RESET_DB. Lets operator drip-feed batches.
_runner_csv = os.environ.get('WANELO_RUNNER_CSV', '').strip()
_runner_xlsx = os.environ.get('WANELO_RUNNER_XLSX', '').strip()
if _runner_csv and not Path(_runner_csv).exists():
    print(f'⚠ WANELO_RUNNER_CSV={_runner_csv!r} does not exist — falling back to upload widget')
    _runner_csv = ''
_force_merge = bool(_runner_csv)

if n_prods == 0 or _force_merge:
    if _runner_csv:
        print(f'\nRunner-injected CSV: {_runner_csv}')
        if _runner_xlsx and Path(_runner_xlsx).exists():
            print(f'Runner-injected XLSX: {_runner_xlsx}')
        # Build same shape as colab_files.upload() returns
        uploaded = {Path(_runner_csv).name: Path(_runner_csv).read_bytes()}
        if _runner_xlsx and Path(_runner_xlsx).exists():
            uploaded[Path(_runner_xlsx).name] = Path(_runner_xlsx).read_bytes()
    else:
        print('\nUpload all files at once (Ctrl+click to select multiple):')
        print('  - CSV with products (required)')
        print('  - SEO Excel .xlsx (optional)')
        print('  - HTML template .html (optional)')
        uploaded = colab_files.upload()
    fnames = list(uploaded.keys())
    print(f'Files: {fnames}')

    # Filter out non-data files
    fnames = [f for f in fnames if not f.endswith('.ipynb')]
    csv_f = next((f for f in fnames if f.endswith('.csv') or f.endswith('.txt')), None)
    seo_f = next((f for f in fnames if f.endswith('.xlsx')), None)
    html_f = next((f for f in fnames if f.endswith('.html') or f.endswith('.htm')), None)

    if html_f:
        with open(html_f, 'r', encoding='utf-8') as fh: DESIGN_TEMPLATE = fh.read()
        db.execute("INSERT OR REPLACE INTO seo_state (key,value) VALUES ('design_template',?)", (DESIGN_TEMPLATE,))
        print(f'Template: {html_f} ({len(DESIGN_TEMPLATE)} chars)')

    if csv_f:
        # Auto-detect delimiter: ; (typical EPROLO export) or , or tab
        with open(csv_f, 'r', encoding='utf-8-sig') as fh:
            raw_lines = [l.rstrip('\r\n') for l in fh if l.strip()]
        if raw_lines:
            first = raw_lines[0]
            # Choose delimiter by count in first line: prefer ; if present, else , else tab
            if ';' in first:
                delim = ';'
            elif '\t' in first:
                delim = '\t'
            elif ',' in first:
                delim = ','
            else:
                delim = ';'  # fallback (single-column CSV)
            print(f'CSV delimiter detected: {delim!r}')
            # Skip header if first line looks like one (no http/https in either column)
            if delim in first and 'http' not in first.lower():
                print(f'  Skipping header line: {first[:80]!r}')
                raw_lines = raw_lines[1:]
        else:
            delim = ';'
        # Auto-detect category from CSV filename
        CATEGORY = csv_f.rsplit('.', 1)[0].strip()
        for suffix in ['_products', '_items', ' products', ' items', '(1)', '(2)', '(3)']:
            CATEGORY = CATEGORY.replace(suffix, '').strip()
        print(f'Category (from filename): {CATEGORY}')
        db.execute("INSERT OR REPLACE INTO seo_state (key,value) VALUES ('category',?)", (CATEGORY,))
        # Use proper csv module to handle quoted fields with commas inside
        import csv as _csv
        from io import StringIO
        reader = _csv.reader(StringIO('\n'.join(raw_lines)), delimiter=delim)
        skipped = 0
        for row in reader:
            if not row: continue
            name = (row[0] or '').strip()
            url  = (row[1].strip() if len(row) > 1 else '')
            if not name:
                skipped += 1; continue
            try:
                # OR IGNORE: when operator sends a CSV that overlaps an earlier batch
                # the duplicate eprolo_url is silently skipped (UNIQUE index above).
                # cur.rowcount tells us whether the row was actually new vs ignored.
                cur = db.execute('INSERT OR IGNORE INTO products (name,eprolo_url) VALUES (?,?)', (name, url))
                if cur.rowcount > 0:
                    PRODUCTS.append(name)
                else:
                    skipped += 1
            except Exception as _e:
                skipped += 1
        db.commit()
        print(f'{len(PRODUCTS)} products loaded ({skipped} skipped — empty/duplicates)')
    else:
        print('ERROR: No CSV file found!')

    if seo_f:
        import openpyxl
        wb = openpyxl.load_workbook(seo_f, read_only=True)

        # ── Sheet: Shopify Collections (+ volume, top keywords) ──
        cc = 0
        for row in wb['Shopify Collections'].iter_rows(min_row=2, values_only=True):
            st = (row[0] or '')   # SEO Title (H1)
            h = (row[1] or '')    # URL Handle
            mt = (row[2] or '')   # Meta Title
            md_val = (row[3] or '')  # Meta Description
            orig = (row[4] or '')    # Original name
            total_vol = int(row[7] or 0) if len(row) > 7 else 0    # Total Volume
            top_kw = (row[8] or '') if len(row) > 8 else ''         # Top Keywords
            try:
                db.execute("""INSERT INTO collections
                    (original_name,seo_title,seo_handle,full_url,meta_title,meta_description,total_volume,top_keywords)
                    VALUES (?,?,?,?,?,?,?,?)""",
                    (orig, st, h, f'/collections/{h}' if h else '', mt, md_val, total_vol, top_kw))
                cc += 1
            except: pass
        print(f'  Collections: {cc} (with volume + top keywords)')

        # ── Sheet: Product Keywords (+ SEO keywords per product) ──
        lc = 0; kw_saved = 0
        for row in wb['Product Keywords'].iter_rows(min_row=2, values_only=True):
            pn = (row[0] or '')   # Product name
            ct = (row[1] or '')   # Primary Collection
            kw_text = (row[3] or '') if len(row) > 3 else ''  # Keywords (pipe-separated)
            if not pn or not ct: continue
            # Fuzzy match: first 40 chars of product name (CSV name may differ slightly from Excel)
            pr = db.execute('SELECT id FROM products WHERE name=?', (pn,)).fetchone()
            if not pr:
                pr = db.execute('SELECT id FROM products WHERE name LIKE ?', (pn[:40] + '%',)).fetchone()
            if not pr:
                # Last resort: match by any word overlap
                words = [w for w in pn.split()[:3] if len(w) > 3]
                for w in words:
                    pr = db.execute('SELECT id FROM products WHERE name LIKE ?', ('%' + w + '%',)).fetchone()
                    if pr: break
            cr = db.execute('SELECT id FROM collections WHERE seo_title=?', (ct,)).fetchone()
            if not cr:
                cr = db.execute('SELECT id FROM collections WHERE seo_title LIKE ?', ('%' + ct[:30] + '%',)).fetchone()
            if pr and cr:
                pass  # Will link below
            elif not pr:
                if lc == 0: print(f'    ⚠ Product not found in DB: {pn[:50]}')
            elif not cr:
                if lc == 0: print(f'    ⚠ Collection not found: {ct[:50]}')
            if pr and cr:
                try:
                    db.execute('INSERT OR IGNORE INTO product_collections VALUES (?,?)', (pr['id'], cr['id']))
                    db.execute('UPDATE products SET primary_collection=? WHERE id=?', (ct, pr['id']))
                    lc += 1
                except: pass
            if pr and kw_text:
                db.execute('UPDATE products SET seo_keywords=? WHERE id=?', (kw_text, pr['id']))
                kw_saved += 1
        print(f'  Product links: {lc} | Keywords saved: {kw_saved}')

        # ── Sheet: Winners KD<30 (all filtered keywords with volume/CPC) ──
        wc = 0
        if 'Winners KD<30' in wb.sheetnames:
            for row in wb['Winners KD<30'].iter_rows(min_row=2, values_only=True):
                kw = (row[0] or '').strip()
                coll = (row[1] or '').strip()
                vol = int(row[2] or 0) if row[2] else 0
                kd = int(row[3] or 0) if row[3] else 0
                cpc = float(row[4] or 0) if row[4] else 0
                comp = str(row[5] or '') if len(row) > 5 else ''
                if kw:
                    try:
                        db.execute('INSERT OR REPLACE INTO seo_keywords (keyword,collection_name,volume,kd,cpc,competition) VALUES (?,?,?,?,?,?)',
                            (kw, coll, vol, kd, cpc, comp))
                        wc += 1
                    except: pass
            print(f'  Winner keywords: {wc} (KD<30, with volume + CPC)')

        db.commit(); wb.close()
        db.execute("INSERT OR REPLACE INTO seo_state (key,value) VALUES ('seo_complete','1')")
        db.commit()
        SEO_SKIP = True

        # Summary
        total_vol = db.execute('SELECT SUM(total_volume) FROM collections').fetchone()[0] or 0
        avg_kw = db.execute('SELECT AVG(volume) FROM seo_keywords').fetchone()[0] or 0
        print(f'  Total search volume: {total_vol:,} | Avg keyword volume: {avg_kw:,.0f}')
    else:
        SEO_SKIP = False

db.commit()
safe_cat = CATEGORY.replace(' & ','_').replace(' ','_')
print(f'\n{"="*50}')
print(f'  Category:    {CATEGORY}')
print(f'  Products:    {db.execute("SELECT COUNT(*) FROM products").fetchone()[0]}')
print(f'  Collections: {db.execute("SELECT COUNT(*) FROM collections").fetchone()[0]}')
print(f'  SEO_SKIP:    {SEO_SKIP}')
print(f'{"="*50}')


# Backup DB to Drive
shutil.copy(DB_LOCAL, DB_PATH)
print(f'DB backed up to Drive')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 478.8/478.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 72.4 MB/s eta 0:00:00
Chrome for Testing 145.0.7632.6 (playwright chromium v1208) downloaded to /root/.cache/ms-playwright/chromium-1208
FFmpeg (playwright ffmpeg v1011) downloaded to /root/.cache/ms-playwright/ffmpeg-1011
Chrome Headless Shell 145.0.7632.6 (playwright chromium-headless-shell v1208) downloaded to /root/.cache/ms-playwright/chromium_headless_shell-1208
Installing dependencies...
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-upd

Saving Bath & Shower.csv to Bath & Shower.csv
Saving Bath_Shower_SEO_Keywords (1).xlsx to Bath_Shower_SEO_Keywords (1).xlsx
Files: ['Bath & Shower.csv', 'Bath_Shower_SEO_Keywords (1).xlsx']
Category (from filename): Bath & Shower
26 products
  Collections: 20 (with volume + top keywords)
  Product links: 0 | Keywords saved: 26
  Total search volume: 2,312,500 | Avg keyword volume: 0

  Category:    Bath & Shower
  Products:    26
  Collections: 20
  SEO_SKIP:    True
DB backed up to Drive


## Cell 2 — Helpers

In [2]:
import requests, base64, time, json, re
import asyncio


import pandas as pd
import numpy as np
from collections import Counter
from tqdm.auto import tqdm

class CostTracker:
    PRICES={'search_volume':0.075,'keywords_for_keywords':0.075,'keyword_suggestions':0.075,'related_keywords':0.075,'keyword_ideas':0.075,'bulk_keyword_difficulty':0.075}
    def __init__(s,b): s.budget=b; s.spent=0.0; s.log=[]
    def charge(s,ep,d=''): c=s.PRICES.get(ep,0.075); s.spent+=c; s.log.append((ep,c,d))
    def can_afford(s,ep): return (s.spent+s.PRICES.get(ep,0.075))<=s.budget
    def remaining(s): return s.budget-s.spent
    def summary(s):
        by={};
        for ep,c,_ in s.log: by[ep]=by.get(ep,0)+c
        print('--- Cost ---')
        for ep,t in sorted(by.items()): print('  '+ep+': $'+str(round(t,3)))
        print('  TOTAL: $'+str(round(s.spent,3))+' | LEFT: $'+str(round(s.remaining(),3)))

cost_tracker = CostTracker(TOTAL_BUDGET)
auth=base64.b64encode((DATAFORSEO_LOGIN+':'+DATAFORSEO_PASSWORD).encode()).decode()
API_HEADERS={'Authorization':'Basic '+auth,'Content-Type':'application/json'}
r=requests.get('https://api.dataforseo.com/v3/appendix/user_data',headers=API_HEADERS,timeout=10)
bal=r.json()['tasks'][0]['result'][0].get('money',{}).get('balance',0)
print('DataForSEO balance: $'+str(round(bal,2)))

def is_brand(kw):
    k=kw.lower()
    for b in BRAND_BLACKLIST:
        if b in k: return True
    return False

def is_irrelevant(kw):
    """v6.1: Filter local/service/medical/irrelevant queries"""
    k = kw.lower()
    for pat in IRRELEVANT_PATTERNS:
        if pat in k: return True
    return False

def auto_confirm(est_cost, desc):
    print(desc+' | Est: $'+str(round(est_cost,2))+' | Budget left: $'+str(round(cost_tracker.remaining(),2)))
    if est_cost <= AUTO_CONFIRM_UNDER:
        print('>>> Auto-confirmed'); return True
    return input('Continue? (y/n): ').lower()=='y'
print('Helpers OK')

import anthropic, sys, time

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY, max_retries=5, default_headers={"anthropic-beta": "extended-cache-ttl-2025-04-11"})  # default 2 — bump for 10K-product unattended runs
# Async client for the per-product loop (Cell 6). Sync `client` above blocks
# the event loop when called from async context — under BATCH_CONCURRENCY>1
# this serialises all AI calls and defeats concurrency. AsyncAnthropic does
# real async I/O so gather() actually parallelises Vision+Strategy+Designer
# requests across products.
client_async = anthropic.AsyncAnthropic(api_key=ANTHROPIC_API_KEY, max_retries=5, default_headers={"anthropic-beta": "extended-cache-ttl-2025-04-11"})

# ── Batch-level cost + cache tracker ──
# All AI calls in Cell 6 accumulate into this dict; end-of-batch summary
# prints cache hit ratio (so we can VERIFY prompt caching is actually
# working — silent breakage of cache_control would otherwise look fine
# but cost ~10× more).
_anthropic_cost_tracker = {
    'spent_usd':       0.0,
    'cache_read_in':   0,   # tokens read from cache (cheap, 0.08/M)
    'cache_create_in': 0,   # tokens written to cache (full price)
    'fresh_in':        0,   # tokens not cached (full price)
    'output_tokens':   0,
    'calls':           0,
    'by_model':        {},  # {model: {fresh,read,create,output,calls,spent}} per-model split
}
_usage_sidecar_loaded = [False]  # one-shot guard: merge prior restart-segment totals once

class _AnthropicBudgetExceeded(Exception):
    """Raised when cumulative AI spend exceeds MAX_ANTHROPIC_BUDGET.
    Caught by Cell 6 per-product try/except → product status='error'
    with explicit budget message; batch continues for remaining products
    but every subsequent AI call also fails fast."""
    pass


def _usage_sidecar_path():
    """JSON sidecar next to THIS run's DB — survives babysitter restarts so the
    cache-hit ratio reflects the whole job, not a single resumed segment."""
    try:
        _base = PROJECT_DIR
    except NameError:
        _base = '.'
    return os.path.join(_base, 'anthropic_usage.json')


_USAGE_KEYS = ('spent_usd', 'cache_read_in', 'cache_create_in', 'fresh_in', 'output_tokens', 'calls')


def _load_usage_once():
    """Merge any prior segments' totals from the sidecar into the live counter,
    exactly once per process. Makes restart-accumulated totals job-wide."""
    if _usage_sidecar_loaded[0]:
        return
    _usage_sidecar_loaded[0] = True
    try:
        _p = _usage_sidecar_path()
        if os.path.exists(_p):
            with open(_p, encoding='utf-8') as _f:
                _prior = json.load(_f)
            for _k in _USAGE_KEYS:
                _anthropic_cost_tracker[_k] += _prior.get(_k, 0) or 0
            for _m, _mv in (_prior.get('by_model') or {}).items():
                _cur = _anthropic_cost_tracker['by_model'].setdefault(
                    _m, {'fresh': 0, 'read': 0, 'create': 0, 'output': 0, 'calls': 0, 'spent': 0.0})
                for _kk in _cur:
                    _cur[_kk] += _mv.get(_kk, 0) or 0
    except Exception as _e_lu:
        print(f'    (usage sidecar load skipped: {str(_e_lu)[:60]})')


def _persist_usage():
    """Write the cumulative counter to the sidecar (atomic-ish: tmp + replace)."""
    try:
        _p = _usage_sidecar_path()
        _tmp = _p + '.tmp'
        with open(_tmp, 'w', encoding='utf-8') as _f:
            json.dump(_anthropic_cost_tracker, _f, ensure_ascii=False)
        os.replace(_tmp, _p)
    except Exception:
        pass


def _track_anthropic_response(resp, cost_usd):
    """Bump the global counter from an Anthropic Messages response, with per-model
    split, and persist to a per-run JSON sidecar (restart-safe, job-wide totals).
    resp.usage is the truth-source for token counts incl. cache fields."""
    _load_usage_once()
    _anthropic_cost_tracker['spent_usd'] += cost_usd
    _anthropic_cost_tracker['calls'] += 1
    u = getattr(resp, 'usage', None)
    _model = getattr(resp, 'model', None) or 'unknown'
    if u is not None:
        _fin = int(getattr(u, 'input_tokens', 0) or 0)
        _crd = int(getattr(u, 'cache_read_input_tokens', 0) or 0)
        _ccr = int(getattr(u, 'cache_creation_input_tokens', 0) or 0)
        _out = int(getattr(u, 'output_tokens', 0) or 0)
        _anthropic_cost_tracker['fresh_in']        += _fin
        _anthropic_cost_tracker['cache_read_in']   += _crd
        _anthropic_cost_tracker['cache_create_in'] += _ccr
        _anthropic_cost_tracker['output_tokens']   += _out
        _bm = _anthropic_cost_tracker['by_model'].setdefault(
            _model, {'fresh': 0, 'read': 0, 'create': 0, 'output': 0, 'calls': 0, 'spent': 0.0})
        _bm['fresh'] += _fin; _bm['read'] += _crd; _bm['create'] += _ccr
        _bm['output'] += _out; _bm['calls'] += 1; _bm['spent'] += cost_usd
    _persist_usage()


# ─── DeepSeek Designer provider (flagged: DESIGNER_PROVIDER=deepseek) ───
_DEEPSEEK_DESIGNER_NUDGE = (
    "\n\n## PROVIDER NOTE — follow strictly:\n"
    "- Put <em>...</em> around 1-2 emotional words in EVERY heading (h1/h2/h4). "
    "It powers the italic serif styling — never omit it.\n"
    "- AIM HIGH on modules: FILL 18-25 of the available sections whenever the "
    "product supports them. Do NOT be conservative — a richer page converts better.\n"
    "- PHOTO BRIEFS are high-stakes: write concept + edit_instructions like an e-commerce "
    "art director; every brief names a LIGHT background explicitly; the 5 carousel slots "
    "must each answer one buyer question.\n"
)

class _ShimUsage:
    def __init__(self, i, o):
        self.input_tokens = int(i or 0); self.output_tokens = int(o or 0)

class _ShimResp:
    def __init__(self, i, o):
        self.usage = _ShimUsage(i, o)

async def _deepseek_designer(system_text, user_text, max_tokens=16000):
    """Call DeepSeek (OpenAI-compatible, JSON mode) off-thread so the event
    loop is not blocked. Returns (raw_json_text, shim_response_with_usage)."""
    import asyncio as _aio
    def _do():
        return requests.post(
            "https://api.deepseek.com/chat/completions",
            headers={"Authorization": "Bearer " + DEEPSEEK_API_KEY,
                     "Content-Type": "application/json"},
            json={"model": DEEPSEEK_MODEL,
                  "messages": [{"role": "system", "content": system_text + _DEEPSEEK_DESIGNER_NUDGE},
                               {"role": "user", "content": user_text}],
                  "max_tokens": max_tokens, "temperature": 0.7,
                  "response_format": {"type": "json_object"}},
            timeout=1800)
    _r = await _aio.to_thread(_do)
    _j = _r.json()
    if "choices" not in _j:
        raise RuntimeError("DeepSeek error: " + str(_j)[:200])
    _txt = _j["choices"][0]["message"]["content"].strip()
    _u = _j.get("usage", {}) or {}
    return _txt, _ShimResp(_u.get("prompt_tokens", 0), _u.get("completion_tokens", 0))


# ── Per-product photo styling theme (variety for photo_briefs; all LIGHT) ──
_PHOTO_SURFACES = [
    "white carrara marble", "pale honey-oak wood", "cream washed linen",
    "soft pink terrazzo", "frosted pale-blue glass", "warm sand travertine stone",
    "matte ivory ceramic tile", "light poured concrete", "brushed champagne metal",
    "a blush-to-cream paper gradient", "pale sage-painted plaster", "light woven rattan",
    "pale grey limewash wall", "soft white boucle fabric", "light beech butcher-block",
    "pale pink quartz", "cream marble with soft grey veining", "raw oat-toned canvas",
    "light birch plywood", "soft lilac matte acrylic", "pale celadon ceramic",
    "warm white plaster", "light cork board", "pastel-mint glass",
    "sun-bleached driftwood", "ivory ribbed glass", "pale apricot suede", "white pebbled leather",
]
_PHOTO_PROPS = [
    "a sprig of fresh eucalyptus", "a few dried wildflowers", "halved citrus and herbs",
    "smooth river pebbles", "a neatly folded waffle towel", "scattered water droplets",
    "a trailing silk ribbon", "a single fresh bloom", "soft monstera-leaf shadows",
    "pastel geometric blocks", "loose flower petals", "a small potted succulent",
    "a sprig of lavender", "fresh mint leaves", "a curl of orange peel",
    "soft cotton flowers", "a few pale seashells", "a glass of still water with tiny bubbles",
    "dried palm fronds", "a loosely draped linen napkin", "fresh chamomile flowers",
    "a smooth ceramic dish", "soft tulle gauze", "a sprig of rosemary",
]
_PHOTO_PALETTES = [
    "warm cream and sage green", "blush pink and ivory", "pale sky-blue and white",
    "soft peach and sand", "dusty lavender and grey", "fresh mint and cream",
    "terracotta and natural linen", "butter-yellow and white", "soft coral and bone",
    "powder-blue and oat", "sage and warm taupe", "apricot and cream",
    "lilac and soft grey", "seafoam and ivory", "dusty rose and beige", "pale gold and white",
]
_PHOTO_LIGHT = [
    "soft morning window light", "bright clean diffused daylight", "warm golden-hour glow",
    "airy overcast softbox light", "gentle dappled sunlight through leaves",
    "even bright editorial daylight", "soft backlit halo glow", "fresh cool north-window light",
    "warm afternoon side-light", "bright high-key wash", "gentle sunrise pastel light",
    "soft diffused skylight from above",
]
_PHOTO_ANGLES = [
    "straight-on eye-level", "a 45-degree three-quarter view", "a crisp top-down flat-lay",
    "a slightly low hero angle", "a floating gently-tilted angle", "a close macro detail angle",
    "a high three-quarter overhead", "an off-center editorial angle",
    "a slight dutch-tilt dynamic angle", "a centered symmetrical hero angle",
]
_PHOTO_COMPOSITION = [
    "minimalist with generous negative space", "a styled flat-lay arrangement",
    "the product as a bold single hero", "an editorial off-center layout",
    "a tight macro crop", "a relaxed lifestyle vignette",
    "a clean symmetrical centered shot", "a layered depth arrangement with soft foreground",
]

def _photo_theme(seed_str):
    import hashlib as _hh
    _d = _hh.md5((seed_str or "x").encode("utf-8")).hexdigest()
    def _pk(seq, a, b):
        return seq[int(_d[a:b], 16) % len(seq)]
    _s    = _pk(_PHOTO_SURFACES,    0, 6)
    _p    = _pk(_PHOTO_PROPS,       6, 12)
    _pal  = _pk(_PHOTO_PALETTES,    12, 18)
    _l    = _pk(_PHOTO_LIGHT,       18, 23)
    _a    = _pk(_PHOTO_ANGLES,      23, 27)
    _comp = _pk(_PHOTO_COMPOSITION, 27, 32)
    return ("surface/backdrop: " + _s + "; props: " + _p + "; palette accents: " + _pal
            + "; light mood: " + _l + "; composition: " + _comp + "; hero angle: " + _a)


def _check_anthropic_budget():
    """Raise _AnthropicBudgetExceeded if cumulative spend has exceeded
    the env-configured ceiling. Call before each AI request."""
    if _anthropic_cost_tracker['spent_usd'] >= MAX_ANTHROPIC_BUDGET:
        raise _AnthropicBudgetExceeded(
            f"Anthropic spend ${_anthropic_cost_tracker['spent_usd']:.2f} "
            f"≥ cap ${MAX_ANTHROPIC_BUDGET:.2f} — skipping AI calls."
        )


def _sanitize_for_prompt(s, max_len=2000):
    """Strip control chars + cap length on any text from external sources
    (EPROLO scrape) before feeding into an Anthropic prompt. Defends
    against accidental or malicious prompt-injection payloads inside
    product titles / descriptions / specs in a 10K batch."""
    if not s:
        return ''
    if not isinstance(s, str):
        s = str(s)
    # Strip ASCII control chars (except common whitespace) and zero-width
    # exotics that could hide instructions.
    out = []
    for ch in s:
        cp = ord(ch)
        if cp < 32 and ch not in ('\n', '\t', ' '):
            continue
        if cp in (0x200B, 0x200C, 0x200D, 0x2060, 0xFEFF):  # zero-width
            continue
        out.append(ch)
    cleaned = ''.join(out)
    # Hard cap — a 30-page injection payload doesn't help us scrape SEO.
    if len(cleaned) > max_len:
        cleaned = cleaned[:max_len] + '...'
    return cleaned

# === OpenAI client (Phase 2 image generation) ===

MODEL = "claude-opus-4-8"

print(f'Testing Claude {MODEL}...', end=' ', flush=True)
try:
    resp = client.messages.create(
        model=MODEL, max_tokens=20, timeout=30.0,
        messages=[{"role": "user", "content": "Say OK"}])
    print(f'✓ OK! Response: "{resp.content[0].text.strip()[:30]}"')
except Exception as e:
    print(f'✗ Error: {str(e)[:200]}')
    print('Falling back to claude-sonnet-4-6...')
    MODEL = "claude-sonnet-4-6"
    try:
        resp = client.messages.create(
            model=MODEL, max_tokens=20, timeout=30.0,
            messages=[{"role": "user", "content": "Say OK"}])
        print(f'✓ Sonnet OK!')
    except Exception as e2:
        print(f'✗ Sonnet also failed: {str(e2)[:100]}')

print(f'\n{"="*50}')
print(f'  >>> SELECTED MODEL: {MODEL}')
print(f'{"="*50}')

# ═══ JSON parser ═══
def parse_json(text):
    for prefix in ['```json','```']:
        if text.startswith(prefix): text=text[len(prefix):]
    if text.endswith('```'): text=text[:-3]
    text=text.strip()
    if '[' not in text: return []
    text=text[text.index('['):]
    text=re.sub(r',\s*]',']',text); text=re.sub(r',\s*}','}',text)
    try: return json.loads(text)
    except: pass
    last=text.rfind('}')
    if last>0:
        try: return json.loads(re.sub(r',\s*]',']',text[:last+1]+']'))
        except: pass
    items=[]
    for m in re.finditer(r'\{[^{}]+\}',text):
        try: items.append(json.loads(m.group()))
        except: pass
    return items

# ═══ Claude caller with retries ═══
def call_claude(prompt, temp=0.8, retries=4):
    for a in range(retries):
        try:
            # Opus 4.7+ deprecated the `temperature` parameter - passing it
            # returns 400 invalid_request_error. Sonnet/Haiku still accept it.
            _ckw = {
                "model": MODEL, "max_tokens": 8192,
                "messages": [{"role": "user", "content": prompt}],
            }
            if "opus" not in MODEL.lower():
                _ckw["temperature"] = temp
            resp = client.messages.create(timeout=90.0, **_ckw)
            text = resp.content[0].text
            r = parse_json(text)
            if r: return r
            if a < retries-1:
                print(f'[empty JSON, retry {a+1}/{retries}]', end=' ', flush=True)
                time.sleep(5)
            else: return []
        except anthropic.RateLimitError:
            w = 30*(a+1)
            print(f'[Anthropic rate limit, wait {w}s]', end=' ', flush=True)
            time.sleep(w)
        except anthropic.APIStatusError as e:
            if e.status_code in (401, 403):
                # auth-errors не лечатся ретраем — fail-fast с actionable хинтом
                _hint = "check ANTHROPIC_API_KEY in env" if e.status_code == 401 else "API key valid but no access (model permission?)"
                print(f'[Anthropic AUTH {e.status_code} — {_hint}]', flush=True)
                return []
            if e.status_code == 529 or 'overloaded' in str(e).lower():
                w = 20*(a+1)
                print(f'[Anthropic overloaded (529), wait {w}s]', end=' ', flush=True)
                time.sleep(w)
            elif a < retries-1:
                print(f'[Anthropic {e.status_code}, retry {a+1}/{retries}]', end=' ', flush=True)
                time.sleep(10)
            else:
                print(f'[Anthropic FAILED ({e.status_code}): {str(e)[:80]}]', flush=True)
                return []
        except Exception as e:
            if a < retries-1:
                print(f'[Anthropic err: {str(e)[:50]}, retry {a+1}/{retries}]', end=' ', flush=True)
                time.sleep(10)
            else:
                print(f'[Anthropic FAILED: {str(e)[:80]}]', flush=True)
                return []
    return []

def chunk_list(lst,n):
    for i in range(0,len(lst),n): yield lst[i:i+n]


# ════ DB HELPERS ════
def db_update_status(product_id, status, **kwargs):
    sets = ['status=?', 'updated_at=CURRENT_TIMESTAMP']
    vals = [status]
    for k, v in kwargs.items():
        sets.append(f'{k}=?'); vals.append(v)
    vals.append(product_id)
    db.execute(f'UPDATE products SET {",".join(sets)} WHERE id=?', vals)
    db.commit()

def get_product_collections(product_id):
    rows = db.execute("""SELECT c.seo_title, c.full_url, c.shopify_collection_id FROM collections c
        JOIN product_collections pc ON c.id = pc.collection_id WHERE pc.product_id = ?""", (product_id,)).fetchall()
    # sqlite3.Row has no .get() — use [key] with None fallback (the column may be NULL).
    return [(r['seo_title'], r['full_url'], r['shopify_collection_id'] or '') for r in rows]

# ════ SHOPIFY ════
import httpx, asyncio, aiohttp
from playwright.async_api import async_playwright

_shop_domain = SHOPIFY_STORE.replace(".myshopify.com","")
SHOPIFY_TOKEN = None
try:
    _r = httpx.post(f"https://{_shop_domain}.myshopify.com/admin/oauth/access_token",
        data={"grant_type":"client_credentials","client_id":SHOPIFY_CLIENT_ID,"client_secret":SHOPIFY_CLIENT_SECRET},
        headers={"Content-Type":"application/x-www-form-urlencoded"}, timeout=30)
    if _r.status_code == 200: SHOPIFY_TOKEN = _r.json()["access_token"]; print(f"Shopify token OK")
    else: print(f"Shopify token FAILED: {_r.status_code}")
except Exception as e: print(f"Shopify: {e}")

SHOP_GQL = f"https://{_shop_domain}.myshopify.com/admin/api/2025-04/graphql.json"
SHOP_HEADERS = {"X-Shopify-Access-Token": SHOPIFY_TOKEN or "", "Content-Type": "application/json"}

async def shopify_gql(query, variables=None, retries=3):
    """Shopify GraphQL with full retry coverage:
       429 throttling, 5xx errors, network exceptions (timeout/connect/read).
       Network exception → exponential backoff. 4xx (non-429) → return parsed
       error JSON (caller decides). For 10K-product unattended runs."""
    if not SHOPIFY_TOKEN: return None
    async with httpx.AsyncClient(timeout=60) as c:
        for attempt in range(retries):
            try:
                r = await c.post(SHOP_GQL, headers=SHOP_HEADERS, json={"query": query, "variables": variables or {}})
            except (httpx.NetworkError, httpx.TimeoutException, httpx.HTTPError) as _net_e:
                if attempt < retries - 1:
                    _wait = 2 ** (attempt + 1)  # 2s, 4s, 8s
                    print(f'    [Shopify network err: {type(_net_e).__name__}, retry in {_wait}s]', flush=True)
                    await asyncio.sleep(_wait)
                    continue
                else:
                    print(f'    [Shopify network FAIL after {retries} attempts: {str(_net_e)[:80]}]', flush=True)
                    return None
            if r.status_code == 429:
                wait = int(r.headers.get('Retry-After', 2 * (attempt + 1)))
                print(f'    [Shopify 429, wait {wait}s]', end=' ', flush=True)
                await asyncio.sleep(wait)
                continue
            if r.status_code == 200:
                data = r.json()
                # Check for throttled error in response
                errs = data.get('errors', [])
                if errs and any('throttled' in str(e).lower() for e in errs):
                    await asyncio.sleep(2 * (attempt + 1))
                    continue
                return data
            # Other error
            if attempt < retries - 1:
                await asyncio.sleep(2 ** attempt)
            else:
                print(f'    [Shopify {r.status_code}]', flush=True)
                return r.json() if r.status_code < 500 else None
    return None

async def shopify_create_collection(title, handle, seo_title, seo_desc):
    # v9.3: Rich collection description for SEO
    # Get top keywords + volume from DB if available
    coll_row = db.execute('SELECT top_keywords, total_volume FROM collections WHERE seo_title=?', (title,)).fetchone()
    top_kw = coll_row['top_keywords'] if coll_row and coll_row['top_keywords'] else ''
    tot_vol = coll_row['total_volume'] if coll_row and coll_row['total_volume'] else 0
    coll_html = generate_collection_html(title, seo_desc, top_kw, tot_vol)
    r = await shopify_gql("mutation($i:CollectionInput!){collectionCreate(input:$i){collection{id}userErrors{field message}}}",
        {"i":{"title":title,"handle":handle,"descriptionHtml":coll_html,"seo":{"title":seo_title or title,"description":seo_desc or ""}}})
    coll_id = r['data']['collectionCreate']['collection']['id'] if r and r.get('data',{}).get('collectionCreate',{}).get('collection') else None
    # Publish to Online Store + Shop — without this storefront 404s on /collections/<handle>
    # (collectionCreate alone doesn't attach to any sales channel — same as products).
    if coll_id:
        try:
            pub_ids = await shopify_get_storefront_publication_ids()
            if pub_ids:
                inputs = [{'publicationId': pid} for pid in pub_ids]
                pr = await shopify_gql("""mutation($id:ID!,$input:[PublicationInput!]!){
                    publishablePublish(id:$id, input:$input){
                        userErrors{field message}
                        publishable{availablePublicationsCount{count}}}}""",
                    {'id': coll_id, 'input': inputs})
                _errs = (((pr or {}).get('data') or {}).get('publishablePublish') or {}).get('userErrors') or []
                if _errs:
                    print(f'    collection publish err: {_errs[:2]}')
        except Exception as _e_cp:
            print(f'    collection publish failed: {str(_e_cp)[:80]}')
    return coll_id


def generate_collection_html(title, meta_desc, top_keywords="", total_volume=0):
    """Generate rich SEO collection description using Claude with keyword data."""
    try:
        kw_instruction = ""
        if top_keywords:
            kw_instruction = f"\nCandidate SEO keywords (volume-ranked, NOT curated — may include off-topic or brand terms). Weave in ONLY those that genuinely match the product type of THIS collection; IGNORE off-topic, wrong-category, or brand keywords: {top_keywords}\n"
        if total_volume > 0:
            kw_instruction += f"This category has {total_volume:,} monthly searches — write for this audience.\n"

        prompt = f"""Write a short, compelling collection page description for an online store.

Collection: {title}
Meta description: {meta_desc}
{kw_instruction}

Rules:
- 3-4 sentences, 80-120 words
- Mention the collection name naturally for SEO
- Include a benefit or reason to browse
- End with a soft CTA like "Find your perfect..." or "Explore our curated..."
- Return ONLY the HTML, no preamble
- Use <p> tags, one paragraph
- Casual, warm tone — not corporate
- Include 1-2 relevant long-tail phrases naturally

Example output:
<p>Discover our handpicked Teeth Whitening collection — everything you need for a brighter, more confident smile at home. From professional-grade whitening kits to gentle daily treatments, each product is selected for real results without the dentist price tag. Whether you're prepping for a big event or building a daily routine, you'll find options for every sensitivity level. Find your perfect whitening match today.</p>""" + HTML_CLEAN_OUTPUT_RULES

        resp = client.messages.create(model=MODEL_VISION, max_tokens=300, temperature=0.7,
            timeout=30.0,
            messages=[{"role":"user","content":prompt}])
        html = resp.content[0].text.strip()
        # Clean any markdown wrapping
        if html.startswith("```"): html = html.split("\n", 1)[-1].rsplit("\n", 1)[0]
        if '<p>' in html: return html
    except Exception as e:
        print(f'    Collection desc generation failed: {str(e)[:60]}')
    return f"<p>Shop our {title} collection — carefully curated for quality and value. Browse the full range and find exactly what you're looking for.</p>"

async def shopify_find_product(title):
    r = await shopify_gql('query($q:String!){products(first:1,query:$q){nodes{id}}}', {"q":f'title:"{title}"'})
    nodes = r.get('data',{}).get('products',{}).get('nodes',[]) if r else []
    return nodes[0]['id'] if nodes else None

async def shopify_create_product(title, body_html, handle, seo_title, seo_desc, tags=None, product_type="", vendor=""):
    inp = {"title":title,"descriptionHtml":body_html,"handle":handle,"status":PUBLISH_STATUS,
           "seo":{"title":seo_title,"description":seo_desc},"tags":tags or []}
    if product_type: inp["productType"] = product_type
    if vendor: inp["vendor"] = vendor
    r = await shopify_gql("mutation($p:ProductCreateInput!){productCreate(product:$p){product{id}userErrors{field message}}}",
        {"p": inp})
    return r['data']['productCreate']['product']['id'] if r and r.get('data',{}).get('productCreate',{}).get('product') else None


# ════ v10.0: AUTO-PUBLISH to storefront sales channels ════
# Channels we always want a new product visible on. Skip POS / Inbox /
# Microsoft Copilot — those are not customer-facing storefronts.
_STOREFRONT_PUBLICATIONS = ['Online Store', 'Shop']
_PUB_CACHE = None  # module-level cache: {name: gid}

async def shopify_get_storefront_publication_ids():
    """Cached lookup of Publication GIDs for the channels in
    _STOREFRONT_PUBLICATIONS. One GraphQL call per Python process."""
    global _PUB_CACHE
    if _PUB_CACHE is not None:
        return [_PUB_CACHE[n] for n in _STOREFRONT_PUBLICATIONS if n in _PUB_CACHE]
    if not SHOPIFY_TOKEN: return []
    r = await shopify_gql('query{publications(first:20){nodes{id name}}}')
    nodes = (((r or {}).get('data') or {}).get('publications') or {}).get('nodes') or []
    _PUB_CACHE = {n['name']: n['id'] for n in nodes}
    return [_PUB_CACHE[n] for n in _STOREFRONT_PUBLICATIONS if n in _PUB_CACHE]

async def shopify_publish_to_storefronts(product_id):
    """Publish a product to Online Store + Shop sales channels.

    Without this call, productCreate(status=ACTIVE) creates the product
    in admin but it's NOT attached to any sales channel — storefront
    returns 404. Idempotent: re-publishing an already-published product
    is a no-op (Shopify silently succeeds).
    """
    if not SHOPIFY_TOKEN: return 0
    pub_ids = await shopify_get_storefront_publication_ids()
    if not pub_ids: return 0
    inputs = [{'publicationId': pid} for pid in pub_ids]
    r = await shopify_gql("""mutation($id:ID!,$input:[PublicationInput!]!){
        publishablePublish(id:$id, input:$input){
            userErrors{field message}
            publishable{availablePublicationsCount{count}}}}""",
        {'id': product_id, 'input': inputs})
    errs = (((r or {}).get('data') or {}).get('publishablePublish') or {}).get('userErrors') or []
    if errs:
        print(f'    publish err: {errs[:2]}')
        return 0
    return len(pub_ids)

async def shopify_update_product(pid, body_html, seo_title, seo_desc, handle=None):
    inp = {"id":pid,"descriptionHtml":body_html,"seo":{"title":seo_title,"description":seo_desc}}
    if handle: inp["handle"] = handle
    r = await shopify_gql("mutation($p:ProductUpdateInput!){productUpdate(product:$p){product{id}userErrors{field message}}}",{"p":inp})
    return r['data']['productUpdate']['product']['id'] if r and r.get('data',{}).get('productUpdate',{}).get('product') else None

async def shopify_add_to_collections(product_id, collection_ids):
    for cid in collection_ids:
        if cid: await shopify_gql("mutation($id:ID!,$pids:[ID!]!){collectionAddProducts(id:$id,productIds:$pids){collection{id}}}",{"id":cid,"pids":[product_id]})


# ════ SCRAPE EPROLO (v9: + variants, shared-browser opt) ════

# Module-level shared Playwright state. Browser launch costs ~1.5s; for
# 10K-product batches this adds ~4h of pure launch overhead if we spin
# one per scrape. Lazy-init on first call, reused thereafter. Operator
# (or end of Cell 6) can call close_shared_browser_ctx() to release.
_PW_STATE = {"pw": None, "browser": None, "ctx": None}

# Photo-pack funnel order — used by Step 4.5 to sort briefs before naming
# files. Module-level so it isn't rebuilt on every product iteration; also
# easier for tests to import + reason about.
_PHOTO_FUNNEL_ORDER = [
    'carousel-hero', 'carousel-lifestyle', 'carousel-in-use',
    'carousel-detail', 'carousel-scale',
    'inline-hero', 'inline-story-1', 'inline-story-2', 'inline-story-3',
    'inline-feature', 'inline-cta',
]
_PHOTO_FUNNEL_IDX = {s: i for i, s in enumerate(_PHOTO_FUNNEL_ORDER)}


class _PhotoPackSkipped(Exception):
    """Sentinel: graceful skip mid-build in Step 4.5 (e.g. all downloads
    failed). Caught separately from real errors so logs stay clean.
    Module-level so the class identity is stable across resume / retry."""
    pass

# ── Per-task stdout buffer for concurrent product processing ──
# Under asyncio.gather with N>1, raw print() calls from different tasks
# interleave into unreadable output. ContextVar lets each task target its
# own io.StringIO buffer; on task completion the buffer is flushed atomically
# to real stdout, keeping each product's log block intact.
import sys as _sys_buf, io as _io_buf
from contextvars import ContextVar as _ContextVar
_TASK_LOG_BUF: _ContextVar = _ContextVar('_TASK_LOG_BUF', default=None)
_real_stdout = _sys_buf.stdout

class _BufferedStdout:
    def write(self, s):
        buf = _TASK_LOG_BUF.get()
        if buf is None:
            _real_stdout.write(s)
        else:
            buf.write(s)
    def flush(self):
        buf = _TASK_LOG_BUF.get()
        if buf is None:
            _real_stdout.flush()
        else:
            buf.flush()

# Only swap stdout when concurrency > 1; otherwise zero overhead.
def _enable_task_buffering():
    """Install the buffering stdout shim. Idempotent."""
    if not isinstance(_sys_buf.stdout, _BufferedStdout):
        _sys_buf.stdout = _BufferedStdout()

def _disable_task_buffering():
    """Restore real stdout. Idempotent."""
    _sys_buf.stdout = _real_stdout

async def get_shared_browser_ctx():
    """Lazy-init shared Playwright context. Returns a BrowserContext that
    can spawn new_page() per scrape. Idempotent — repeated calls return
    the same ctx until close_shared_browser_ctx() is invoked."""
    if _PW_STATE["ctx"] is None:
        _PW_STATE["pw"] = await async_playwright().start()
        _PW_STATE["browser"] = await _PW_STATE["pw"].chromium.launch(headless=True)
        # Bug A: load EPROLO storage_state if EPROLO_STATE_FILE is set and
        # points at an existing file. Without it EPROLO redirects product
        # URLs to /sign-up or /home for unauthenticated sessions.
        _ctx_kwargs = {"user_agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
        _state_file = os.environ.get("EPROLO_STATE_FILE", "")
        if _state_file and os.path.exists(_state_file):
            _ctx_kwargs["storage_state"] = _state_file
            print(f"  ⚙ EPROLO session loaded from {_state_file}")
        elif _state_file:
            print(f"  ⚠ EPROLO_STATE_FILE={_state_file} doesn't exist - Bug A risk")
        else:
            print(f"  ⚠ EPROLO_STATE_FILE not set - Bug A risk (unauthenticated session)")
        _PW_STATE["ctx"] = await _PW_STATE["browser"].new_context(**_ctx_kwargs)
    return _PW_STATE["ctx"]

async def close_shared_browser_ctx():
    """Tear down shared Playwright state — call at end of Cell 6 to free
    Chromium resources. Safe to call when not initialised (no-op)."""
    try:
        if _PW_STATE.get("browser"):
            await _PW_STATE["browser"].close()
        if _PW_STATE.get("pw"):
            await _PW_STATE["pw"].stop()
    except Exception as _e:
        print(f'  (shared browser teardown noisy: {str(_e)[:80]})')
    finally:
        _PW_STATE["pw"] = None
        _PW_STATE["browser"] = None
        _PW_STATE["ctx"] = None


async def scrape_eprolo(url, ctx=None):
    """Scrape EPROLO product page. If ctx is provided, reuses that shared
    Playwright context (saves ~1.5s/call). If None, falls back to the
    module-level shared ctx (lazy-init via get_shared_browser_ctx).

    DB cache short-circuit: if `db` (Cell 1) has a non-empty `scrape_json`
    for this URL, return it directly without opening Chromium. Used by
    pipeline/tools/prescrape_eprolo.py to work around the ipykernel
    SelectorEventLoop + Windows asyncio.subprocess incompatibility
    (NotImplementedError on async_playwright().start())."""
    try:
        _cached_row = db.execute(
            "SELECT scrape_json FROM products WHERE eprolo_url=? "
            "AND scrape_json IS NOT NULL AND scrape_json != ''",
            (url,),
        ).fetchone()
        if _cached_row and _cached_row[0]:
            _cached = json.loads(_cached_row[0])
            if _cached.get("title"):
                print(f"  scrape cache hit for {url[:70]}")
                return _cached
    except NameError:
        pass  # db not defined (e.g. unit test context)
    except Exception as _cache_err:
        print(f"  [scrape cache lookup skipped: {_cache_err}]")
    result = {"title":"","description":"","top_image_urls":[],"desc_image_urls":[],"cost_price":0,"variants":[],"specs":{},"video_urls":[],"shipping_time":"","weight_g":0,"size_chart":[]}
    if ctx is None:
        ctx = await get_shared_browser_ctx()
    page = await ctx.new_page()
    try:
        await page.goto(url, wait_until="networkidle", timeout=60000)
        # Bug A defense-in-depth: detect redirect (product removed, session
        # expired, etc) and bail with empty title so validate_scrape() flags it.
        _final_url = page.url or ""
        if "/app/product/" not in _final_url:
            result["_redirect_target"] = _final_url
            print(f'    ⚠ EPROLO redirect: {url[:60]} -> {_final_url[:60]}')
            await page.close()
            return result
        await page.wait_for_timeout(5000)
        try:
            el = await page.query_selector("h1")
            if el: result["title"] = (await el.inner_text()).strip()
        except: pass
        if not result["title"]: result["title"] = await page.title()
        html = await page.content()

        # ── v10: Scrape variants from EPROLO Element-UI table ──
        # EPROLO structure (verified 2026-05-28):
        #   .el-form-item              → one per option axis (color/size/NET WT)
        #     .el-form-item__label     → axis name
        #     .el-table__row           → one per variant value
        #       td[0]                  → variant value (e.g. "1box(1 and 3)")
        #       td[1]                  → "USD X.XX" — per-variant cost
        #       td[2-4]                → stock / shipping / weight
        # If variant value is "default" + only 1 row → single-variant, skip.
        variants_raw = []
        try:
            form_items = await page.query_selector_all('.el-form-item')
            for fi in form_items:
                lbl_el = await fi.query_selector('.el-form-item__label')
                if not lbl_el: continue
                label_text = (await lbl_el.inner_text()).strip().rstrip(':')
                if not label_text: continue
                rows = await fi.query_selector_all('.el-table__row')
                if not rows: continue
                values = []
                value_prices = {}
                for row in rows:
                    cells = await row.query_selector_all('td')
                    if not cells: continue
                    name_el = cells[0]
                    name = (await name_el.inner_text()).strip()
                    if not name or len(name) > 60: continue
                    if name in values: continue
                    values.append(name)
                    # Try cell[1] for "USD X.XX" price
                    if len(cells) > 1:
                        ptxt = (await cells[1].inner_text()).strip()
                        pm = re.search(r"(?:USD|\$)\s*(\d+(?:\.\d+)?)", ptxt)
                        if pm:
                            try:
                                pv = float(pm.group(1))
                                if 0.5 <= pv <= 5000:
                                    value_prices[name] = pv
                            except ValueError: pass
                # Skip ANY single-value axis — Shopify shouldn't have option
                # picker with only 1 choice (no actual variant for customer).
                if len(values) <= 1:
                    continue
                if values:
                    entry = {"option_name": label_text, "values": values}
                    if value_prices:
                        entry["_value_prices"] = value_prices
                    variants_raw.append(entry)
                    # NOTE: per-variant photo capture lives in prescrape_eprolo.py
                    # (sync click loop). The async notebook path skips it to keep
                    # the event loop responsive; prescrape runs first in prod so
                    # _value_photos is already populated by the time we get here.

            # Method 1 fallback: generic option selectors (older EPROLO pages)
            if not variants_raw:
                option_groups = await page.query_selector_all('[class*="option"], [class*="variant"], [class*="sku-select"], [class*="spec-item"], [class*="attribute"]')
                for grp in option_groups:
                    label_el = await grp.query_selector('label, [class*="label"], [class*="name"], [class*="title"], dt')
                    items = await grp.query_selector_all('button, li, option, [class*="item"], [class*="value"], dd, span[class*="val"]')
                    if not items or len(items) < 1: continue
                    label_text = ""
                    if label_el:
                        label_text = (await label_el.inner_text()).strip().rstrip(':')
                    values = []
                    for item in items:
                        val = (await item.inner_text()).strip()
                        if val and len(val) < 40 and val not in values:
                            values.append(val)
                    if values and label_text:
                        variants_raw.append({"option_name": label_text, "values": values})

            # Method 2: Parse from JSON-LD or embedded data
            if not variants_raw:
                for pattern in [r'"variants"\s*:\s*\[([^\]]{10,})\]', r'"options"\s*:\s*\[([^\]]{10,})\]',
                                r'"skus"\s*:\s*\[([^\]]{10,})\]']:
                    m = re.search(pattern, html)
                    if m:
                        try:
                            raw_list = json.loads('[' + m.group(1) + ']')
                            opt_names = set()
                            for v in raw_list:
                                if isinstance(v, dict):
                                    for k in v:
                                        kl = k.lower()
                                        if any(w in kl for w in ['color','colour','size','style','option','type','flavor','scent','shade','material']):
                                            opt_names.add(k)
                            for opt in opt_names:
                                vals = list(dict.fromkeys([str(v.get(opt,'')) for v in raw_list if isinstance(v,dict) and v.get(opt)]))
                                if vals: variants_raw.append({"option_name": opt.replace('_',' ').title(), "values": vals})
                        except: pass
                        break

            # Method 3: Color swatches
            if not any(v['option_name'].lower() in ('color','colour','shade') for v in variants_raw):
                swatches = await page.query_selector_all('[class*="swatch"], [class*="color"] img, [class*="colour"] img')
                colors = []
                for sw in swatches:
                    alt = await sw.get_attribute('alt') or await sw.get_attribute('title') or ''
                    alt = alt.strip()
                    if alt and len(alt) < 30 and alt not in colors: colors.append(alt)
                if len(colors) >= 2:
                    variants_raw.append({"option_name": "Color", "values": colors})

        except Exception as ve:
            print(f'    Variant scrape warning: {str(ve)[:80]}')

        result["variants"] = variants_raw[:3]  # Shopify max 3 options

        # Try to extract per-variant prices from page data
        try:
            variant_prices = {}
            for pat in [r'"variants"\s*:\s*\[([^\]]+)\]', r'"skus"\s*:\s*\[([^\]]+)\]']:
                pm = re.search(pat, html)
                if pm:
                    try:
                        vlist = json.loads('[' + pm.group(1) + ']')
                        for vi in vlist:
                            if isinstance(vi, dict):
                                vp = vi.get('price') or vi.get('cost') or vi.get('salePrice')
                                vn = vi.get('name') or vi.get('title') or vi.get('option1','')
                                if vp and vn:
                                    try: variant_prices[str(vn)] = float(str(vp).replace(',','.'))
                                    except: pass
                    except: pass
                    break
            if variant_prices:
                result["variant_prices"] = variant_prices
        except: pass

        # ── Auto-detect product image URLs ──
        # EPROLO serves photos from multiple regional CDNs (oss-accelerate,
        # oss-us-west-1, oss-eu-central-1). Substring match catches all
        # regions — was hardcoded oss-accelerate only (Crest 3D White bug:
        # served from oss-us-west-1 → 0 photos got through).
        all_urls = re.findall(r'https?://[^\s"\'<>\)]+\.(?:jpg|jpeg|png|webp)', html)
        all_urls = list(dict.fromkeys(all_urls))
        product_imgs = [u for u in all_urls if "shopifyfile." in u and "aliyuncs.com/attached" in u]

        boundary = None
        for pat in [r'(?i)>\s*Description\s*<', r'(?i)description-content',
                    r'(?i)product-description', r'(?i)tab.*description']:
            m = re.search(pat, html)
            if m: boundary = m.start(); break

        if boundary:
            top = []; desc = []
            for u in product_imgs:
                pos = html.find(u)
                if pos >= 0 and pos < boundary: top.append(u)
                else: desc.append(u)
            desc = [u for u in desc if u not in top]
            chunk = re.sub(r'<[^>]+>',' ',html[boundary:boundary+5000])
            result["description"] = re.sub(r'\s+',' ',chunk).strip()[:2000]
        else:
            top = product_imgs[:5]; desc = product_imgs[5:]

        # Dedup — EPROLO occasionally returns the same photo URL 2-3 times.
        # Otherwise carousel + photo_pack ZIP would contain duplicates.
        # dict.fromkeys preserves insertion order, drops duplicates.
        result["top_image_urls"]  = list(dict.fromkeys(top))
        result["desc_image_urls"] = list(dict.fromkeys(desc))

        # ── Parse price ──
        try:
            for sel in ['[class*="price"]', '[class*="Price"]', '.product-price', '.sale-price', 'span.price', '[class*="cost"]']:
                price_el = await page.query_selector(sel)
                if price_el:
                    pt = (await price_el.inner_text()).strip()
                    pm = re.search(r'[\d]+[.,]?\d*', pt.replace(',','.'))
                    if pm and float(pm.group()) > 0:
                        result["cost_price"] = float(pm.group()); break
        except: pass
        if result["cost_price"] == 0:
            for pp in [r'(?:USD|\$)\s*([\d]+\.?\d*)', r'"cost"\s*:\s*"?([\d.]+)', r'"price"\s*:\s*"?([\d.]+)']:
                pm = re.search(pp, html)
                if pm:
                    try: result["cost_price"] = float(pm.group(1))
                    except: pass
                    if result["cost_price"] > 0: break


        # ── v9.2: Specs table ──
        try:
            spec_rows = await page.query_selector_all('[class*="spec"] tr, [class*="detail"] tr, [class*="param"] tr, [class*="attribute"] tr, [class*="info-item"], [class*="product-prop"]')
            specs = {}
            for row in spec_rows[:30]:
                cells = await row.query_selector_all('td, th, span, div')
                texts = []
                for cell in cells:
                    t = (await cell.inner_text()).strip()
                    if t and len(t) < 100: texts.append(t)
                if len(texts) >= 2:
                    key = texts[0].rstrip(':').strip()
                    val = texts[1].strip()
                    if key and val and len(key) < 50:
                        specs[key] = val
            result["specs"] = specs
            # Extract weight from specs
            for k, v in specs.items():
                kl = k.lower()
                if any(w in kl for w in ['weight', 'net weight', 'gross weight', 'waga']):
                    wm = re.search(r'([\d.]+)\s*(?:g|gram)', v.lower())
                    if wm: result["weight_g"] = float(wm.group(1)); break
                    wm = re.search(r'([\d.]+)\s*(?:kg)', v.lower())
                    if wm: result["weight_g"] = float(wm.group(1)) * 1000; break
                    wm = re.search(r'([\d.]+)\s*(?:oz)', v.lower())
                    if wm: result["weight_g"] = float(wm.group(1)) * 28.35; break
        except: pass

        # ── v9.2: Video URLs ──
        try:
            videos = []
            for tag in await page.query_selector_all('video source, video[src], [class*="video"] iframe'):
                src = await tag.get_attribute('src') or ''
                if src and 'http' in src and src not in videos:
                    videos.append(src)
            # Also check for video in HTML
            for pat in [r'<video[^>]*src=["\'](https?://[^"\']+)["\'"]', r'<source[^>]*src=["\'](https?://[^"\']+\.mp4)["\'"]']:
                for m in re.finditer(pat, html):
                    if m.group(1) not in videos: videos.append(m.group(1))
            result["video_urls"] = videos[:3]
        except: pass

        # ── v9.2: Shipping time ──
        try:
            for sel in ['[class*="shipping"]', '[class*="delivery"]', '[class*="dispatch"]']:
                el = await page.query_selector(sel)
                if el:
                    txt = (await el.inner_text()).strip()
                    tm = re.search(r'(\d+[-–]\d+)\s*(?:days|business)', txt, re.I)
                    if tm: result["shipping_time"] = tm.group(0); break
        except: pass

        # ── v9.2: Size chart ──
        try:
            size_rows = await page.query_selector_all('[class*="size"] table tr, [class*="measurement"] tr, [class*="chart"] tr')
            chart = []
            for row in size_rows[:20]:
                cells = await row.query_selector_all('td, th')
                texts = [(await c.inner_text()).strip() for c in cells]
                if texts and any(t for t in texts):
                    chart.append(texts)
            result["size_chart"] = chart
        except: pass
    finally:
        try:
            await page.close()
        except Exception:
            pass
    var_summary = ', '.join([f'{v["option_name"]}({len(v["values"])})' for v in result["variants"]]) or 'none'
    extras = []
    if result["specs"]: extras.append(f'{len(result["specs"])} specs')
    if result["video_urls"]: extras.append(f'{len(result["video_urls"])} videos')
    if result["weight_g"]: extras.append(f'{result["weight_g"]:.0f}g')
    if result["size_chart"]: extras.append('size chart')
    extra_str = ' | ' + ', '.join(extras) if extras else ''
    print(f'  Scrape: {len(result["top_image_urls"])} top + {len(result["desc_image_urls"])} desc | ${result["cost_price"]:.2f} | variants: {var_summary}{extra_str}')
    return result


# Минимальная ширина изображения. Меньше — почти наверняка иконка / бейдж / разделитель.
MIN_IMAGE_WIDTH = 150

async def dl_images(urls, label):
    # Whitelist by substring across regional EPROLO CDNs (see scrape_eprolo).
    urls = [u for u in urls if "shopifyfile." in u and "aliyuncs.com/attached" in u]
    imgs = []
    skipped_small = 0
    async with httpx.AsyncClient(timeout=30, follow_redirects=True) as c:
        for i, u in enumerate(urls):
            try:
                r = await c.get(u)
                if r.status_code!=200 or len(r.content)>5*1024*1024: continue
                data = r.content
                if data[:4]==b'RIFF' and data[8:12]==b'WEBP': mt="image/webp"
                elif data[:8]==b'\x89PNG\r\n\x1a\n': mt="image/png"
                elif data[:4] in (b'GIF8',): continue
                else: mt="image/jpeg"
                # Width filter via PIL: skip icons / small badges
                try:
                    from PIL import Image
                    import io as _io
                    _img = Image.open(_io.BytesIO(data))
                    if _img.width < MIN_IMAGE_WIDTH:
                        skipped_small += 1
                        continue
                except Exception:
                    pass  # if PIL fails to decode — accept anyway
                imgs.append({"index":len(imgs)+1,"url":u,"media_type":mt,"base64":base64.standard_b64encode(r.content).decode("ascii"),"size_kb":len(r.content)/1024})
            except Exception: pass
    if skipped_small:
        print(f"    [dl_images] skipped {skipped_small} images < {MIN_IMAGE_WIDTH}px (icons/badges)")
    return imgs


async def upload_to_shopify(img_list, label="IMG", alt_texts=None):
    if not SHOPIFY_TOKEN:
        return [{"index":img['index'],"url":img['url'],"cdn":False} for img in img_list]
    alt_map = alt_texts or {}
    results = []
    async with httpx.AsyncClient(timeout=60) as gql:
        for img in img_list:
            idx = img['index']; raw = base64.standard_b64decode(img['base64'])
            ext = 'png' if img['media_type']=='image/png' else 'webp' if img['media_type']=='image/webp' else 'jpg'
            fname = f"funnel_{label.lower()}_{idx}.{ext}"
            alt = alt_map.get(idx, f"{label} image {idx}")
            try:
                r1 = await gql.post(SHOP_GQL, headers=SHOP_HEADERS, json={"query":"mutation($i:[StagedUploadInput!]!){stagedUploadsCreate(input:$i){stagedTargets{url parameters{name value}resourceUrl}userErrors{field message}}}","variables":{"i":[{"resource":"FILE","filename":fname,"mimeType":img['media_type'],"fileSize":str(len(raw)),"httpMethod":"POST"}]}})
                targets = r1.json().get('data',{}).get('stagedUploadsCreate',{}).get('stagedTargets',[])
                if not targets: results.append({"index":idx,"url":img['url'],"cdn":False}); continue
                t = targets[0]; params = {p['name']:p['value'] for p in t['parameters']}
                form = aiohttp.FormData()
                for k,v in params.items(): form.add_field(k,v)
                form.add_field('file', raw, filename=fname, content_type=img['media_type'])
                async with aiohttp.ClientSession() as sess:
                    async with sess.post(t['url'], data=form) as resp:
                        if resp.status not in (200,201,204): results.append({"index":idx,"url":img['url'],"cdn":False}); continue
                r2 = await gql.post(SHOP_GQL, headers=SHOP_HEADERS, json={"query":"mutation($f:[FileCreateInput!]!){fileCreate(files:$f){files{id}userErrors{field message}}}","variables":{"f":[{"originalSource":t['resourceUrl'],"contentType":"IMAGE","alt":alt}]}})
                files_made = r2.json().get('data',{}).get('fileCreate',{}).get('files',[])
                if not files_made: results.append({"index":idx,"url":img['url'],"cdn":False}); continue
                file_id = files_made[0]['id']; cdn_url = None
                for _ in range(12):
                    await asyncio.sleep(2)
                    r3 = await gql.post(SHOP_GQL, headers=SHOP_HEADERS, json={"query":'query($id:ID!){node(id:$id){...on MediaImage{image{url}fileStatus}}}', "variables":{"id":file_id}})
                    node = r3.json().get('data',{}).get('node',{}) or {}
                    if node.get('fileStatus')=='READY' and node.get('image',{}).get('url'): cdn_url = node['image']['url']; break
                    elif node.get('fileStatus')=='FAILED': break
                results.append({"index":idx,"url":cdn_url or img['url'],"cdn":bool(cdn_url)})
            except: results.append({"index":idx,"url":img['url'],"cdn":False})
    return results


# ── Upload arbitrary file (e.g. ZIP) to Shopify Files via stagedUploads + fileCreate(contentType:FILE) ──
async def shopify_upload_file(file_bytes: bytes, filename: str, mime_type: str = 'application/zip', alt: str = '') -> dict:
    """Upload a non-image file to Shopify Files. Returns {url, size_kb} or {} on failure.

    Used for the photo_pack ZIP archive (per-product bundle of EPROLO photos +
    ChatGPT-edit prompt.txt). User downloads the ZIP, edits photos in ChatGPT-UI
    (which preserves style across multi-image conversation, unlike API edit),
    then re-uploads results manually to Shopify product images."""
    if not SHOPIFY_TOKEN: return {}
    try:
        async with httpx.AsyncClient(timeout=60) as gql:
            r1 = await gql.post(SHOP_GQL, headers=SHOP_HEADERS,
                json={"query":"mutation($i:[StagedUploadInput!]!){stagedUploadsCreate(input:$i){stagedTargets{url parameters{name value}resourceUrl}userErrors{field message}}}",
                      "variables":{"i":[{"resource":"FILE","filename":filename,"mimeType":mime_type,"fileSize":str(len(file_bytes)),"httpMethod":"POST"}]}})
            targets = r1.json().get('data',{}).get('stagedUploadsCreate',{}).get('stagedTargets',[])
            if not targets: return {}
            t = targets[0]; params = {p['name']:p['value'] for p in t['parameters']}
            form = aiohttp.FormData()
            for k,v in params.items(): form.add_field(k,v)
            form.add_field('file', file_bytes, filename=filename, content_type=mime_type)
            async with aiohttp.ClientSession() as sess:
                async with sess.post(t['url'], data=form) as resp:
                    if resp.status not in (200,201,204): return {}
            r2 = await gql.post(SHOP_GQL, headers=SHOP_HEADERS,
                json={"query":"mutation($f:[FileCreateInput!]!){fileCreate(files:$f){files{id ... on GenericFile{url originalFileSize}}userErrors{field message}}}",
                      "variables":{"f":[{"originalSource":t['resourceUrl'],"contentType":"FILE","alt":alt or filename}]}})
            files_made = r2.json().get('data',{}).get('fileCreate',{}).get('files',[])
            if not files_made: return {}
            file_id = files_made[0].get('id')
            cdn_url = files_made[0].get('url')
            # GenericFile.url may be deferred; poll if not ready
            for _ in range(12):
                if cdn_url: break
                await asyncio.sleep(2)
                r3 = await gql.post(SHOP_GQL, headers=SHOP_HEADERS,
                    json={"query":'query($id:ID!){node(id:$id){...on GenericFile{url fileStatus}}}', "variables":{"id":file_id}})
                node = r3.json().get('data',{}).get('node',{}) or {}
                if node.get('fileStatus') == 'READY' and node.get('url'):
                    cdn_url = node['url']; break
                if node.get('fileStatus') == 'FAILED': return {}
        return {"url": cdn_url or "", "size_kb": round(len(file_bytes) / 1024, 1)}
    except Exception:
        return {}



# ── Pricing ──
def calc_price(cost):
    """cost × RETAIL_MARKUP, psychological .90 price. Min $7.90 to ensure
    margin after Shopify+payment fees. Multiplier tunable via env var
    RETAIL_MARKUP (default 10.0)."""
    raw = cost * RETAIL_MARKUP
    if raw < 10: price = round(raw) - 0.10
    elif raw < 30: price = (round(raw / 5) * 5) - 0.10
    elif raw < 100: price = (round(raw / 10) * 10) - 0.10
    else: price = (round(raw / 10) * 10) - 0.10
    return max(7.90, round(price, 2))

def calc_compare_price(cost):
    """cost × COMPARE_AT_MARKUP, rounded for compare_at (shows strikethrough).
    Multiplier tunable via env var COMPARE_AT_MARKUP (default 15.0). Anchor
    effect: a $20 price next to a $32 strike-through reads as a 38% deal."""
    raw = cost * COMPARE_AT_MARKUP
    if raw < 20: return float(max(15, round(raw / 5) * 5))
    return float(round(raw / 10) * 10)

# ── Attach images to product (gallery) ──
async def shopify_attach_media(product_id, image_urls, alt_map=None):
    """Attach images to product gallery with SEO alt text from vision.

    IDEMPOTENT: checks how many media the product already has and skips if
    >= len(image_urls). Prevents image duplication on Step 5 retry after
    partial-success crash (e.g. metafields pushed but next API call fails)."""
    if not image_urls or not SHOPIFY_TOKEN: return 0
    alt_map = alt_map or {}
    media = [{"originalSource": url, "mediaContentType": "IMAGE", "alt": alt_map.get(i+1, alt_map.get(url, f"Product image {i+1}"))}
             for i, url in enumerate(image_urls) if url.startswith('http')]
    if not media: return 0
    # Idempotency check — query existing media count
    try:
        _check = await shopify_gql(
            'query($id:ID!){product(id:$id){media(first:50){nodes{id}}}}',
            {"id": product_id})
        _existing = ((_check or {}).get('data',{}) or {}).get('product',{}).get('media',{}).get('nodes',[])
        if len(_existing) >= len(media):
            print(f'    [attach_media: product already has {len(_existing)} images >= {len(media)} requested — skipped to prevent duplicates]')
            return len(_existing)
    except Exception:
        pass  # if check fails, proceed with attach (worst case = duplicates, better than no images)
    r = await shopify_gql("""mutation($pid:ID!,$m:[CreateMediaInput!]!){
        productCreateMedia(productId:$pid,media:$m){
            media{id status} mediaUserErrors{field message}
        }}""", {"pid": product_id, "m": media})
    if r and r.get('data',{}).get('productCreateMedia',{}).get('media'):
        return len(r['data']['productCreateMedia']['media'])
    errs = r.get('data',{}).get('productCreateMedia',{}).get('mediaUserErrors',[]) if r else []
    if errs: print(f'    Media errors: {errs[:2]}')
    return 0


# ── Attach a hosted MP4 video to the product carousel ──
async def shopify_attach_video_from_url(product_id, mp4_url, alt="Product video"):
    """Download an EPROLO-hosted mp4 and attach it to the Shopify product
    media carousel. Shopify VIDEO media needs a staged-upload resourceUrl
    (it must HOST the file — a direct external mp4 URL is rejected as
    "Invalid video url"). Flow:
       download bytes → stagedUploadsCreate(VIDEO) → POST → productCreateMedia(VIDEO)
    Returns True on success."""
    if not SHOPIFY_TOKEN or not mp4_url:
        return False
    try:
        # 1. Download mp4 bytes (strip ?x-oss-process query)
        _clean = mp4_url.split('?')[0]
        async with httpx.AsyncClient(timeout=120, follow_redirects=True) as _c:
            _vr = await _c.get(_clean)
            if _vr.status_code != 200 or len(_vr.content) < 1000:
                print(f'    video download failed: HTTP {_vr.status_code}')
                return False
            _vbytes = _vr.content
        if len(_vbytes) > 100 * 1024 * 1024:
            print(f'    video too large ({len(_vbytes)//1024//1024} MB > 100 MB) — skip')
            return False
        _fname = _clean.rsplit('/', 1)[-1] or 'product-video.mp4'
        # 2. stagedUploadsCreate for VIDEO
        _r1 = await shopify_gql(
            "mutation($i:[StagedUploadInput!]!){stagedUploadsCreate(input:$i){"
            "stagedTargets{url resourceUrl parameters{name value}} userErrors{field message}}}",
            {"i": [{"resource": "VIDEO", "filename": _fname, "mimeType": "video/mp4",
                    "fileSize": str(len(_vbytes)), "httpMethod": "POST"}]})
        _targets = (((_r1 or {}).get('data') or {}).get('stagedUploadsCreate') or {}).get('stagedTargets') or []
        if not _targets:
            print(f'    video stagedUploadsCreate failed: {(((_r1 or {}).get("data") or {}).get("stagedUploadsCreate") or {}).get("userErrors")}')
            return False
        _t = _targets[0]
        _params = {p['name']: p['value'] for p in _t['parameters']}
        # 3. POST bytes to staged target
        _form = aiohttp.FormData()
        for _k, _v in _params.items():
            _form.add_field(_k, _v)
        _form.add_field('file', _vbytes, filename=_fname, content_type='video/mp4')
        async with aiohttp.ClientSession() as _sess:
            async with _sess.post(_t['url'], data=_form) as _resp:
                if _resp.status not in (200, 201, 204):
                    print(f'    video staged POST failed: {_resp.status}')
                    return False
        # 4. productCreateMedia VIDEO from resourceUrl
        _r2 = await shopify_gql(
            "mutation($pid:ID!,$m:[CreateMediaInput!]!){productCreateMedia(productId:$pid,media:$m){"
            "media{id status} mediaUserErrors{field message}}}",
            {"pid": product_id, "m": [{"originalSource": _t['resourceUrl'],
                                       "mediaContentType": "VIDEO", "alt": alt}]})
        _errs = (((_r2 or {}).get('data') or {}).get('productCreateMedia') or {}).get('mediaUserErrors') or []
        if _errs:
            print(f'    productCreateMedia(VIDEO) errors: {_errs[:2]}')
            return False
        return True
    except Exception as _e_vid:
        print(f'    video attach failed: {str(_e_vid)[:100]}')
        return False

# ── Set variant price ──
async def shopify_set_price(product_id, price, compare_at):
    """Set price + compare_at on the default variant."""
    # Get default variant ID
    r = await shopify_gql('query($id:ID!){product(id:$id){variants(first:1){nodes{id}}}}',
        {"id": product_id})
    variants = r.get('data',{}).get('product',{}).get('variants',{}).get('nodes',[]) if r else []
    if not variants: return False
    vid = variants[0]['id']
    r2 = await shopify_gql("""mutation($pid:ID!,$v:[ProductVariantsBulkInput!]!){
        productVariantsBulkUpdate(productId:$pid,variants:$v){
            productVariants{id price compareAtPrice} userErrors{field message}
        }}""", {"pid": product_id, "v": [{"id": vid, "price": str(price), "compareAtPrice": str(compare_at)}]})
    ok = r2.get('data',{}).get('productVariantsBulkUpdate',{}).get('productVariants') if r2 else None
    return bool(ok)


# ── Set product metafield ──
async def shopify_set_metafield(product_id, namespace, key, value, mtype="multi_line_text_field"):
    """Set a metafield on a product."""
    r = await shopify_gql("""mutation($m:[MetafieldsSetInput!]!){
        metafieldsSet(metafields:$m){metafields{id} userErrors{field message}}
    }""", {"m": [{"ownerId": product_id, "namespace": namespace, "key": key, "type": mtype, "value": value}]})
    if r and r.get('data',{}).get('metafieldsSet',{}).get('metafields'):
        return True
    errs = r.get('data',{}).get('metafieldsSet',{}).get('userErrors',[]) if r else []
    if errs: print(f'    Metafield error: {errs[:2]}')
    return False


# ── Idempotent metafield definition ensure (Admin UI requires definition to display field) ──
async def shopify_ensure_metafield_definition(namespace, key, name, type_name="multi_line_text_field",
                                                owner_type="PRODUCT", description="", pin=True,
                                                visible_to_storefront=True):
    """Создаёт metafield definition если её нет. Idempotent — TAKEN ошибки игнорируем.

    visible_to_storefront=False сетит access.storefront=NONE — поле НЕ
    отдаётся через Storefront API и НЕ вытягивается ботами. Используется
    для admin-only утилит (photo_pack ZIP URL)."""
    if not SHOPIFY_TOKEN: return None
    # Сначала проверяем что уже есть
    q = """query($namespace:String!,$key:String!,$ownerType:MetafieldOwnerType!){
        metafieldDefinitions(first:1, namespace:$namespace, key:$key, ownerType:$ownerType){
            nodes{id name namespace key type{name}}
        }
    }"""
    r = await shopify_gql(q, {"namespace": namespace, "key": key, "ownerType": owner_type})
    existing = ((r or {}).get("data",{}) or {}).get("metafieldDefinitions",{}).get("nodes",[])
    if existing:
        return existing[0]["id"]
    # Создаём
    m = """mutation($d:MetafieldDefinitionInput!){
        metafieldDefinitionCreate(definition:$d){
            createdDefinition{id} userErrors{field message code}
        }
    }"""
    inp = {"name": name, "namespace": namespace, "key": key,
           "type": type_name, "ownerType": owner_type, "description": description, "pin": pin,
           "access": {"storefront": ("PUBLIC_READ" if visible_to_storefront else "NONE")}}
    r = await shopify_gql(m, {"d": inp})
    data = ((r or {}).get("data",{}) or {}).get("metafieldDefinitionCreate",{}) or {}
    if data.get("createdDefinition"):
        return data["createdDefinition"]["id"]
    errs = data.get("userErrors", []) or []
    # TAKEN = уже есть (race-condition или из прошлого запуска) — это OK
    if any((e.get("code") or "").upper() == "TAKEN" for e in errs):
        return None
    # PINNED_LIMIT_REACHED — Shopify caps at 20 pinned defs per ownerType.
    # Definition itself is still useful unpinned (operator can manually pin
    # later), so retry without pin=True.
    if any((e.get("code") or "").upper() == "PINNED_LIMIT_REACHED" for e in errs):
        inp_unpinned = dict(inp); inp_unpinned["pin"] = False
        r2 = await shopify_gql(m, {"d": inp_unpinned})
        d2 = ((r2 or {}).get("data",{}) or {}).get("metafieldDefinitionCreate",{}) or {}
        if d2.get("createdDefinition"):
            print(f"    Metafield definition created UNPINNED ({namespace}.{key}) — pin limit reached")
            return d2["createdDefinition"]["id"]
        e2 = d2.get("userErrors", []) or []
        if e2 and not any((e.get("code") or "").upper() == "TAKEN" for e in e2):
            print(f"    Metafield definition error ({namespace}.{key}) [unpinned retry]: {e2[:2]}")
        return None
    if errs:
        print(f"    Metafield definition error ({namespace}.{key}): {errs[:2]}")
    return None


# ════ v9.3: GOOGLE — ADDITIONAL SCHEMA BUILDERS ════

# ════ v9.3: GOOGLE MERCHANT — EXTENDED METAFIELDS ════

# ════ v9.3: SITEMAP PING ════
# ════ v9.4: KEYWORD SCORING — low KD first ════
# ════ v9.5: QUALITY ASSURANCE ════

print('All helpers OK')


# ════ v9: CREATE SHOPIFY VARIANTS ════
async def shopify_create_variants(product_id, variants_data, base_cost):
    """Create product options and variant combinations in Shopify."""
    if not variants_data or not SHOPIFY_TOKEN: return 0
    options = variants_data[:3]
    for opt in options:
        opt['values'] = opt['values'][:100]

    option_inputs = [{"name": opt["option_name"], "values": [{"name": v} for v in opt["values"]]} for opt in options]
    # variantStrategy:CREATE — Shopify default LEAVE_AS_IS only renames the
    # single standalone variant; CREATE materialises one variant per value
    # (cartesian product for multi-axis). Without this, a 6-color product
    # gets just 1 variant.
    r = await shopify_gql("""mutation($pid:ID!,$opts:[OptionCreateInput!]!){
        productOptionsCreate(productId:$pid, options:$opts, variantStrategy:CREATE){
            product{id options{id name optionValues{name}} variantsCount{count}} userErrors{field message}
        }}""", {"pid": product_id, "opts": option_inputs})
    errs = r.get('data',{}).get('productOptionsCreate',{}).get('userErrors',[]) if r else []
    if errs:
        print(f'    Option errors: {[e["message"] for e in errs[:3]]}')

    await asyncio.sleep(1)
    r2 = await shopify_gql('query($id:ID!){product(id:$id){variants(first:100){nodes{id title selectedOptions{name value}}}}}',
        {"id": product_id})
    all_vars = r2.get('data',{}).get('product',{}).get('variants',{}).get('nodes',[]) if r2 else []
    if all_vars and base_cost > 0:
        batch = []
        _vprices = variants_data[0].get('_variant_prices', {}) if variants_data else {}
        for v in all_vars:
            vt = v.get("title", "")
            # Use per-variant cost from EPROLO if available
            _vc = None
            for vname, vprice in _vprices.items():
                if vname.lower() in vt.lower() or vt.lower() in vname.lower():
                    _vc = vprice; break
            if _vc and _vc > 0:
                p, cp = calc_price(_vc), calc_compare_price(_vc)
            else:
                p, cp = calc_variant_price(base_cost, vt)
            batch.append({"id": v["id"], "price": str(p), "compareAtPrice": str(cp)})
        for i in range(0, len(batch), 50):
            chunk = batch[i:i+50]
            await shopify_gql("""mutation($pid:ID!,$v:[ProductVariantsBulkInput!]!){
                productVariantsBulkUpdate(productId:$pid,variants:$v){
                    productVariants{id} userErrors{field message}
                }}""", {"pid": product_id, "v": chunk})
            await asyncio.sleep(0.3)

    # ── Per-variant featured photos ──
    # If the scrape captured _value_photos {value_name: url} for the first
    # axis, upload each + attach to the matching variant so the carousel
    # image switches when the customer picks that option.
    _vphotos = {}
    for opt in options:
        for _vn, _vurl in (opt.get('_value_photos') or {}).items():
            _vphotos[_vn.lower()] = _vurl
    if _vphotos and len(set(_vphotos.values())) > 1 and all_vars:  # BUGFIX 2026-06-05: skip variant images when all values share ONE photo (quantity/size) — else gallery re-renders on select -> layout-shift flicker
        # refetch with selectedOptions to match value names
        _rv = await shopify_gql(
            'query($id:ID!){product(id:$id){variants(first:100){nodes{id image{id} selectedOptions{value}}}}}',
            {"id": product_id})
        _vnodes = (((_rv or {}).get('data') or {}).get('product') or {}).get('variants', {}).get('nodes', [])
        _url_to_media = {}
        _assigned = 0
        for _v in _vnodes:
            if _v.get('image'):
                continue  # already has a featured image
            _vals = [so.get('value', '') for so in (_v.get('selectedOptions') or [])]
            _match = " ".join(_vals).lower()
            _vurl = None
            for _vn, _u in _vphotos.items():
                if _vn and (_vn in _match or _match in _vn):
                    _vurl = _u
                    break
            if not _vurl:
                continue
            # upload media (poll READY) once per distinct url
            if _vurl not in _url_to_media:
                _rm = await shopify_gql(
                    "mutation($pid:ID!,$m:[CreateMediaInput!]!){productCreateMedia(productId:$pid,media:$m){"
                    "media{id status} mediaUserErrors{field message}}}",
                    {"pid": product_id, "m": [{"originalSource": _vurl,
                                               "mediaContentType": "IMAGE", "alt": "variant photo"}]})
                _md = (((_rm or {}).get('data') or {}).get('productCreateMedia') or {}).get('media') or []
                if not _md:
                    continue
                _mid = _md[0]['id']
                # poll READY (variants reject non-ready media)
                for _ in range(15):
                    await asyncio.sleep(1.5)
                    _pr = await shopify_gql('query($id:ID!){node(id:$id){... on MediaImage{status}}}', {"id": _mid})
                    _st = (((_pr or {}).get('data') or {}).get('node') or {}).get('status')
                    if _st == 'READY': break
                    if _st == 'FAILED': _mid = None; break
                if not _mid:
                    continue
                _url_to_media[_vurl] = _mid
            _ra = await shopify_gql(
                "mutation($pid:ID!,$vm:[ProductVariantAppendMediaInput!]!){"
                "productVariantAppendMedia(productId:$pid,variantMedia:$vm){userErrors{field message}}}",
                {"pid": product_id, "vm": [{"variantId": _v['id'], "mediaIds": [_url_to_media[_vurl]]}]})
            _ae = (((_ra or {}).get('data') or {}).get('productVariantAppendMedia') or {}).get('userErrors') or []
            if not any('already' not in (e.get('message') or '').lower() for e in _ae):
                _assigned += 1
            await asyncio.sleep(0.3)
        if _assigned:
            print(f'    Variant photos: {_assigned} assigned')
    return len(all_vars)


# ════ v9: HTML POST-PROCESSOR ════
# ════ v9: SMART INTERLINKING ════
def get_smart_interlinks(product_id, product_name, all_shopify_collections):
    """Return relevant collection links with keyword-rich anchor text options."""
    own_colls = get_product_collections(product_id)
    own_handles = set()
    links = []

    for title, url, sid in own_colls:
        if not sid: continue
        handle = url.replace('/collections/','') if url else ''
        own_handles.add(handle)
        # Get keyword anchors from DB
        anchors = _get_collection_anchors(title)
        links.append({"title": title, "handle": handle, "relevance": "own", "anchors": anchors})

    # Find related collections by keyword overlap
    name_words = set(product_name.lower().split())
    stop_words = {'the','a','an','for','and','or','with','in','of','to','set','kit','pro','new','best','2','3','4','5','pack','pcs','pc'}
    name_words -= stop_words
    scored = []
    for c in all_shopify_collections:
        h = c.get('handle','')
        if h in own_handles: continue
        title_words = set(c.get('title','').lower().split())
        handle_words = set(h.replace('-',' ').lower().split())
        overlap = len(name_words & (title_words | handle_words))
        if overlap > 0:
            scored.append((overlap, c))

    scored.sort(key=lambda x: -x[0])
    for score, c in scored[:4]:
        coll_title = c.get('title','')
        anchors = _get_collection_anchors(coll_title)
        links.append({"title": coll_title, "handle": c["handle"], "relevance": "related", "anchors": anchors})

    return links[:6]


def _get_collection_anchors(collection_title):
    """Get 3-5 keyword-rich anchor text options for a collection from DB."""
    anchors = []

    # Source 1: top_keywords from collections table
    row = db.execute('SELECT top_keywords FROM collections WHERE seo_title=?', (collection_title,)).fetchone()
    if row and row['top_keywords']:
        kws = [k.strip() for k in row['top_keywords'].split(',')]
        # Pick short ones (2-4 words) — best for anchor text
        for kw in kws:
            word_count = len(kw.split())
            if 2 <= word_count <= 4 and kw not in anchors:
                anchors.append(kw)
            if len(anchors) >= 3: break

    # Source 2: top volume keywords from seo_keywords table
    if len(anchors) < 3:
        rows = db.execute(
            'SELECT keyword FROM seo_keywords WHERE collection_name=? AND length(keyword) - length(replace(keyword, " ", "")) BETWEEN 1 AND 3 ORDER BY volume DESC LIMIT 5',
            (collection_title,)).fetchall()
        for r in rows:
            kw = r['keyword']
            if kw not in anchors:
                anchors.append(kw)
            if len(anchors) >= 5: break

    # Fallback: derive from collection title
    if not anchors:
        # "Luxury Soap Gift Sets & Hand Cream Collections" → "soap gift sets", "hand cream"
        title_lower = collection_title.lower()
        for remove in ['luxury','premium','best','top','&','collections','collection','for','-','daily','natural','essential','essentials']:
            title_lower = title_lower.replace(remove, ' ')
        parts = [p.strip() for p in title_lower.split() if len(p.strip()) > 2]
        if len(parts) >= 2:
            anchors.append(' '.join(parts[:3]))

    return anchors[:5]


def _compute_collection_related(coll_rows, k=6, cap=8):
    """For each collection, build a list of `k` related collections.

    Scoring: title_word_overlap × 3 + keyword_overlap × 2 + handle_word_overlap × 1.

    Reciprocity: if A links to B, B also links to A (within `cap` capacity —
    replaces the lowest-scored entry in B's list if full). Symmetric graph
    is critical for SEO authority flow.

    Hub detection: top ~15% by in-degree are flagged `is_hub=True`. Hubs sit
    first in each related list and get distinctive anchor ("Shop all <title>").

    Anchor variety: cycles through {exact title, short keyword, alt keyword,
    generic "explore X"} so we don't over-optimize a single anchor text
    (Google penalty signal).

    coll_rows: sqlite3.Row iterable, must have seo_handle / seo_title /
               original_name / top_keywords / total_volume.
    Returns: dict mapping handle → [{handle, title, anchor, score, is_hub}].
    """
    import re as _re_cr
    _stop_cr = {'the','a','an','for','and','or','with','in','of','to','set','kit',
                'pro','new','best','collection','collections','our','your','from','by'}

    cdata = {}
    for r in coll_rows:
        try:
            handle = r['seo_handle'] or ''
        except (IndexError, KeyError):
            continue
        if not handle:
            continue
        try:
            title = r['seo_title'] or r['original_name'] or handle
        except (IndexError, KeyError):
            title = handle
        try:
            kws_raw = (r['top_keywords'] or '').split(',')
        except (IndexError, KeyError):
            kws_raw = []
        try:
            volume = int(r['total_volume'] or 0)
        except (IndexError, KeyError, TypeError):
            volume = 0
        try:
            shop_id = r['shopify_collection_id']
        except (IndexError, KeyError):
            shop_id = None
        def _norm_word(w):
            # Crude singular/plural normalisation — strip trailing 's' on 4+ char
            # words so 'fans' and 'fan', 'bottles' and 'bottle' match. Skips
            # words like 'lens' (where 's' is structural). Good enough for
            # interlinking signal.
            w = w.strip()
            if len(w) >= 4 and w.endswith('s') and not w.endswith('ss'):
                return w[:-1]
            return w
        kws = set()
        for kw in kws_raw:
            kw = _re_cr.sub(r'[^a-z0-9 ]', '', kw.lower().strip())
            # Split keyword phrases into individual words — phrase-level
            # intersection misses 'fan' shared between 'cooling fan' and
            # 'neck fan'. Word-level intersection captures the topical link.
            for word in kw.split():
                word = _norm_word(word)
                if len(word) > 2 and word not in _stop_cr:
                    kws.add(word)
        title_words = {_norm_word(w) for w in _re_cr.findall(r'[a-z0-9]+', title.lower())} - _stop_cr
        handle_words = {_norm_word(w) for w in handle.lower().split('-')} - _stop_cr
        cdata[handle] = {
            'handle': handle, 'title': title, 'kws': kws,
            'title_words': title_words, 'handle_words': handle_words,
            'volume': volume, 'shopify_id': shop_id,
        }

    handles = list(cdata.keys())
    raw = {h: [] for h in handles}
    for i, a in enumerate(handles):
        for b in handles[i+1:]:
            da, db_ = cdata[a], cdata[b]
            kw_o = len(da['kws'] & db_['kws'])
            tw_o = len(da['title_words'] & db_['title_words'])
            hw_o = len(da['handle_words'] & db_['handle_words'])
            score = tw_o * 3 + kw_o * 2 + hw_o * 1
            if score <= 0:
                continue
            raw[a].append((score, b))
            raw[b].append((score, a))

    # Top-k per collection (NATURAL adjacency, before reciprocity).
    # Hub detection runs on this — after symmetric closure in-degrees
    # converge across the graph and the hub signal disappears. We need
    # to capture which collections naturally attract MORE inbound links.
    adj_natural = {}
    for h in handles:
        raw[h].sort(reverse=True)
        adj_natural[h] = {b: s for s, b in raw[h][:k]}

    # Hub detection: top ~15% by SCORE-WEIGHTED in-degree on natural adj.
    # Weighted (not just count) so a node attracting strong-score links
    # ranks higher than one attracting many weak-score links.
    in_deg_w = {h: 0 for h in handles}
    for a in handles:
        for b, sc in adj_natural[a].items():
            in_deg_w[b] += sc
    sorted_deg = sorted(in_deg_w.values(), reverse=True)
    cut_idx = max(0, len(handles) // 7 - 1)
    hub_thresh = sorted_deg[cut_idx] if sorted_deg else 999
    hubs = {h for h, d in in_deg_w.items() if d >= hub_thresh and d > 0}

    # Symmetric closure (reciprocity) — final adjacency for output.
    adj = {h: dict(adj_natural[h]) for h in handles}
    for a in handles:
        for b in list(adj[a].keys()):
            if a in adj[b]:
                continue
            if len(adj[b]) < cap:
                adj[b][a] = adj[a][b]
            else:
                low_h = min(adj[b].items(), key=lambda x: x[1])[0]
                if adj[a][b] > adj[b][low_h]:
                    del adj[b][low_h]
                    adj[b][a] = adj[a][b]

    # Build output with anchor variety + hub flag
    out = {}
    for a in handles:
        sorted_rel = sorted(adj[a].items(),
                            key=lambda x: (0 if x[0] in hubs else 1, -x[1]))
        items = []
        for idx, (b, score) in enumerate(sorted_rel):
            db_data = cdata[b]
            short_kws = sorted(
                [k for k in db_data['kws'] if 2 <= len(k.split()) <= 4],
                key=len,
            )
            if idx == 0 and b in hubs:
                anchor = f"Shop all {db_data['title']}"
            elif idx % 4 == 0:
                anchor = db_data['title']
            elif idx % 4 == 1 and short_kws:
                anchor = short_kws[0]
            elif idx % 4 == 2 and len(short_kws) > 1:
                anchor = short_kws[1]
            else:
                anchor = f"explore {db_data['title'].lower()}"
            items.append({
                'handle': b,
                'title': db_data['title'],
                'anchor': anchor,
                'score': score,
                'is_hub': b in hubs,
            })
        out[a] = items
    return out


# ════ v10.0: ALWAYS-IN-STOCK (dropshipping mode) ════
async def shopify_set_inventory(product_id, quantity=999):
    """Make every variant ALWAYS sellable (no 'Sold out' state).

    Dropshipping reality: we don't hold stock — EPROLO fulfils on order.
    So per-variant we:
      1. inventoryItemUpdate tracked=False    — Shopify treats as
         unlimited; storefront never shows Sold-out badge.
      2. productVariantsBulkUpdate inventoryPolicy=CONTINUE — belt &
         suspenders. If the operator later flips tracked=True manually,
         purchases still go through at 0 stock instead of blocking.

    Drops the v9.2 inventoryActivate + inventorySetQuantities calls —
    those only matter for TRACKED inventory, which we don't want.

    `quantity` arg kept for signature back-compat but is now ignored.
    """
    if not SHOPIFY_TOKEN: return False
    r = await shopify_gql('query($id:ID!){product(id:$id){variants(first:100){nodes{id inventoryItem{id}}}}}',
        {"id": product_id})
    variants = r.get('data',{}).get('product',{}).get('variants',{}).get('nodes',[]) if r else []
    if not variants:
        return 0
    count = 0
    # 1) Per-variant: tracked=False
    for v in variants:
        inv_id = v.get('inventoryItem',{}).get('id')
        if not inv_id: continue
        _r_track = await shopify_gql("""mutation($id:ID!,$input:InventoryItemInput!){
            inventoryItemUpdate(id:$id,input:$input){inventoryItem{id tracked} userErrors{field message}}}""",
            {"id": inv_id, "input": {"tracked": False}})
        _e = (((_r_track or {}).get('data') or {}).get('inventoryItemUpdate') or {}).get('userErrors') or []
        if _e: print(f'    inventory untrack err: {_e[:2]}')
        else: count += 1
        await asyncio.sleep(0.1)
    # 2) Bulk: inventoryPolicy=CONTINUE on all variants (up to 250/call)
    _pol_batch = [{"id": v["id"], "inventoryPolicy": "CONTINUE"} for v in variants if v.get('id')]
    for i in range(0, len(_pol_batch), 250):
        _r_pol = await shopify_gql("""mutation($pid:ID!,$v:[ProductVariantsBulkInput!]!){
            productVariantsBulkUpdate(productId:$pid, variants:$v){
                productVariants{id inventoryPolicy} userErrors{field message}}}""",
            {"pid": product_id, "v": _pol_batch[i:i+250]})
        _e = (((_r_pol or {}).get('data') or {}).get('productVariantsBulkUpdate') or {}).get('userErrors') or []
        if _e: print(f'    inventory policy err: {_e[:2]}')
        await asyncio.sleep(0.2)
    return count


# ════ v9.2: SET PRODUCT WEIGHT ════
async def shopify_set_weight(product_id, weight_grams):
    """Set weight on all variants for shipping calculator."""
    if not SHOPIFY_TOKEN or weight_grams <= 0: return False
    r = await shopify_gql('query($id:ID!){product(id:$id){variants(first:100){nodes{id}}}}',
        {"id": product_id})
    variants = r.get('data',{}).get('product',{}).get('variants',{}).get('nodes',[]) if r else []
    if not variants: return False
    batch = [{"id":v["id"],"weight":weight_grams,"weightUnit":"GRAMS"} for v in variants]
    for i in range(0, len(batch), 50):
        await shopify_gql("""mutation($pid:ID!,$v:[ProductVariantsBulkInput!]!){
            productVariantsBulkUpdate(productId:$pid,variants:$v){
                productVariants{id} userErrors{field message}}}""",
            {"pid": product_id, "v": batch[i:i+50]})
        await asyncio.sleep(0.3)
    return True


# ════ v9.2: GOOGLE SHOPPING METAFIELDS ════
async def shopify_set_google_shopping(product_id, product_type_category, condition="new", age_group="adult"):
    """Set Google Shopping metafields for Merchant Center feed."""
    if not SHOPIFY_TOKEN: return False
    metafields = [
        {"ownerId":product_id,"namespace":"google","key":"product_category","type":"single_line_text_field","value":product_type_category},
        {"ownerId":product_id,"namespace":"google","key":"condition","type":"single_line_text_field","value":condition},
        {"ownerId":product_id,"namespace":"google","key":"age_group","type":"single_line_text_field","value":age_group},
    ]
    r = await shopify_gql("mutation($m:[MetafieldsSetInput!]!){metafieldsSet(metafields:$m){metafields{id} userErrors{field message}}}",
        {"m": metafields})
    return bool(r and r.get('data',{}).get('metafieldsSet',{}).get('metafields'))


# ════ v9.2: SEO IMAGE FILENAME ════
# ════ v9.2: SCHEMA.ORG JSON-LD ════
# ════ v9.2: BUNDLE DETECTION ════
def detect_bundle(title):
    """Detect if product is a bundle/set and return multiplier."""
    t = title.lower()
    for pat in [r'set\s+of\s+(\d+)', r'(\d+)\s*(?:pcs|pieces|pack|count)', r'(\d+)\s*in\s*1']:
        m = re.search(pat, t)
        if m:
            n = int(m.group(1))
            if 2 <= n <= 20:
                return n
    return 1


# ════ v9.2: TIERED VARIANT PRICING ════
def calc_variant_price(base_cost, variant_title):
    """Adjust price based on variant size/type. Bigger = more expensive."""
    vt = variant_title.lower()
    multiplier = 1.0
    # Size tiers
    if any(s in vt for s in ['xxl', '2xl', '3xl', 'xxxl']): multiplier = 1.3
    elif any(s in vt for s in ['xl', 'extra large']): multiplier = 1.2
    elif any(s in vt for s in ['large', ' l ', ' l,']): multiplier = 1.1
    # Volume tiers
    vm = re.search(r'(\d+)\s*(?:ml|oz|g)', vt)
    if vm:
        vol = float(vm.group(1))
        if 'oz' in vt: vol *= 30
        if 'g' in vt and vol < 10: vol *= 30  # probably oz mislabeled
        if vol > 100: multiplier = max(multiplier, 1.2)
        if vol > 200: multiplier = max(multiplier, 1.4)
    # Premium materials/colors
    if any(s in vt for s in ['gold', 'premium', 'pro', 'deluxe']): multiplier = max(multiplier, 1.15)
    adjusted_cost = base_cost * multiplier
    return calc_price(adjusted_cost), calc_compare_price(adjusted_cost)



# ════ v9.3: GOOGLE — ADDITIONAL SCHEMA BUILDERS ════

# ════ v9.3: GOOGLE MERCHANT — EXTENDED METAFIELDS ════

async def shopify_set_merchant_extended(product_id, stage1_data, variants_data):
    """Set extended Google Merchant Center metafields from vision + variant data."""
    if not SHOPIFY_TOKEN: return 0
    metafields = []

    # Gender from vision target audience
    target = stage1_data.get('conversion', {}).get('target_audience', '').lower()
    gender = "unisex"
    if any(w in target for w in ['women', 'female', 'girl', 'her', 'ladies']): gender = "female"
    elif any(w in target for w in ['men', 'male', 'boy', 'his', 'guys']): gender = "male"
    metafields.append({"ownerId":product_id,"namespace":"google","key":"gender","type":"single_line_text_field","value":gender})

    # Material from vision sensory
    material = stage1_data.get('sensory', {}).get('material', '')
    if material:
        metafields.append({"ownerId":product_id,"namespace":"google","key":"material","type":"single_line_text_field","value":material})

    # Color from variants or vision
    colors = stage1_data.get('sensory', {}).get('color_names', [])
    if not colors and variants_data:
        for v in variants_data:
            if v.get('option_name','').lower() in ('color','colour','shade'):
                colors = v['values'][:5]
                break
    if colors:
        metafields.append({"ownerId":product_id,"namespace":"google","key":"color","type":"single_line_text_field","value":", ".join(colors[:3])})

    # Size from variants
    for v in (variants_data or []):
        if v.get('option_name','').lower() in ('size','sizes'):
            metafields.append({"ownerId":product_id,"namespace":"google","key":"size","type":"single_line_text_field","value":", ".join(v['values'][:5])})
            break

    # Item group ID (links variants together)
    metafields.append({"ownerId":product_id,"namespace":"google","key":"custom_label_0","type":"single_line_text_field","value":stage1_data.get('category','General')})

    # Product highlights from vision
    highlights = []
    hero = stage1_data.get('conversion', {}).get('hero_claim', '')
    if hero: highlights.append(hero)
    for claim in stage1_data.get('packaging_text', {}).get('claims', [])[:4]:
        if claim not in highlights: highlights.append(claim)
    for feat in stage1_data.get('physical', {}).get('key_features', [])[:3]:
        if feat not in highlights: highlights.append(feat)
    if highlights:
        metafields.append({"ownerId":product_id,"namespace":"google","key":"custom_label_1","type":"single_line_text_field","value":" | ".join(highlights[:5])})

    if not metafields: return 0
    r = await shopify_gql("mutation($m:[MetafieldsSetInput!]!){metafieldsSet(metafields:$m){metafields{id} userErrors{field message}}}",
        {"m": metafields})
    ok = r.get('data',{}).get('metafieldsSet',{}).get('metafields',[]) if r else []
    return len(ok)


# ════ v9.3: SITEMAP PING ════
def ping_google_sitemap(store_domain):
    """Notify Google of updated sitemap after batch publish."""
    try:
        sitemap_url = f"https://{store_domain}/sitemap.xml"
        r = requests.get(f"https://www.google.com/ping?sitemap={sitemap_url}", timeout=10)
        return r.status_code == 200
    except:
        return False



# ════ v9.4: KEYWORD SCORING — low KD first ════
def kw_score(volume, kd, cpc):
    """Score keywords for STRONG dropped domain (WANELO.com DA 60+).
    KD penalty is linear, not quadratic — we CAN rank for KD 30-40.
    KD 15 keeps 70%, KD 30 keeps 40%, KD 45 keeps 10%."""
    kd = min(kd, 50)
    kd_factor = max(0, (50 - kd) / 50)  # Linear: KD 0=1.0, KD 25=0.5, KD 50=0.0
    cpc_factor = 1 + min(cpc, 5)
    return volume * kd_factor * cpc_factor


def tier_keywords(keywords_with_data):
    """Split keywords into tiers by KD for STRONG domain.
    Tier 1 (KD 0-20): Primary targets → H1, H2, first paragraphs — will rank in weeks
    Tier 2 (KD 21-35): Achievable → body text, alt text, FAQ — will rank in 1-3 months
    Tier 3 (KD 36-50): Stretch targets → mentioned naturally — will rank in 3-6 months"""
    t1, t2, t3 = [], [], []
    for kw_data in keywords_with_data:
        kd = kw_data.get('kd', 50)
        if kd <= 20:
            t1.append(kw_data)
        elif kd <= 35:
            t2.append(kw_data)
        else:
            t3.append(kw_data)
    for t in [t1, t2, t3]:
        t.sort(key=lambda x: -kw_score(x.get('volume',0), x.get('kd',0), x.get('cpc',0)))
    return t1, t2, t3



# ════ v9.5: QUALITY ASSURANCE ════

def validate_scrape(product_data):
    """Check scrape result is usable before continuing.

    Hard-fail tells: empty title OR title matches EPROLO marketing/auth pages
    (when scrape lands on signup/home instead of the actual product page —
    usually because the session is not logged in or the product URL changed).
    Soft warnings: missing images / price.
    """
    issues = []
    t = (product_data.get('title') or '').strip()
    if not t or len(t) < 5:
        issues.append("empty/short title")
    else:
        # Known EPROLO non-product page titles. If we land on any of these,
        # the scrape is junk — refuse to create a Shopify product from it.
        _tl = t.lower()
        _marketing_tells = (
            'eprolo -',
            'eprolo-',
            'sign up',
            'sign in',
            'log in',
            'login',
            'dropshipping supply',
            'all-in-one dropshipping',
        )
        if any(tell in _tl for tell in _marketing_tells):
            issues.append("marketing_page (EPROLO redirect — not a product)")
    if not product_data.get('top_image_urls') and not product_data.get('desc_image_urls'):
        issues.append("no images found")
    if product_data.get('cost_price', 0) <= 0:
        issues.append("no price found")
    return issues


print('All helpers OK (v9.2: variants, sanitizer, smart interlinks)')


# ── Auto-ensure metafield definitions (so Admin UI shows fields with stored data) ──
# Без definition Admin не отображает поле, даже если данные записаны через metafieldsSet.
# Idempotent — повторные вызовы безопасны.
if SHOPIFY_TOKEN:
    # All 9 product-page sections as JSON metafields. Theme reads them via
    # snippets/wanelo-*.liquid (one snippet per metafield).
    _wanelo_metafields = [
        ('hero',            'Wanelo · Hero',            'Hero section: kicker, h1, lead, image_url, image_alt'),
        ('videos',          'Wanelo · Videos',          'Horizontal-scroll row of vertical 9:16 short videos (YouTube Shorts / TikTok / Reels) shown ABOVE the long-form description, always visible without Show-full click. {head:{kicker,h2}, items:[{platform, video_id, title}]}'),
        ('story',           'Wanelo · Story',           'Story chapters: head + 1-3 chapter cards'),
        ('features',        'Wanelo · Features',        'Features grid: head + 3 cards (glyph, meta, h4, p)'),
        ('stats',           'Wanelo · Stats',           'Stats grid: head + 4 cells (kicker, value, label)'),
        ('reviews',         'Wanelo · Reviews',         'Reviews wall: head + 9-12 mini-reviews (3-col animated)'),
        ('faq',             'Wanelo · FAQ',             'FAQ accordion: head + 4-7 q/a pairs (also emits FAQPage JSON-LD)'),
        ('cta',             'Wanelo · CTA',             'Call-to-action: kicker, h2, p, button_label'),
        ('palette',         'Wanelo · Palette',         'Per-product palette: brand_1/2/3/soft/deep (5 hex)'),
        ('interlinks',      'Wanelo · Interlinks',      'Collection interlinks: head + items[handle, anchor]'),
        ('how_to',          'Wanelo · How To',          'Steps: head + 3-7 ordered steps {n, h4, p}. For rituals/setup/recipes.'),
        ('specs',           'Wanelo · Specs',           'Specs table: head + groups[{label, value}]. For tech/appliances/jewelry.'),
        ('whats_included',  'Wanelo · Whats Included',  'What\'s in the box: head + items[{glyph, name, qty, note}]. For bundles/kits/gifts.'),
        ('ingredients',     'Wanelo · Ingredients',     'Ingredient chips: head + items[{name, role, badge}]. For skincare/food/supplements.'),
        ('timeline',        'Wanelo · Timeline',        'Before/after milestones: head + items[{when, h4, p}]. For transformation products.'),
        ('trust',           'Wanelo · Trust',           'Certifications + press: head + certs[{label}] + press[{label}]. For regulated/premium.'),
        ('compare',         'Wanelo · Compare',         'vs alternatives matrix: head + columns[{name, badge}] + rows[{label, values[]}]. For research-buy/upgrade.'),
        ('size_guide',      'Wanelo · Size Guide',      'Sizing chart: head + labels[] + rows[{label, values[], unit}]. For clothing/jewelry/pet/baby.'),
        ('care',            'Wanelo · Care',            'Care instructions: head + items[{glyph, label, note}]. For fashion/jewelry/textiles/durables.'),
        ('dimensions',      'Wanelo · Dimensions',      'Physical dimensions: head + items[{axis, value}] + scale_note. For decor/furniture/lighting.'),
        ('variants',        'Wanelo · Variants',        'Color/pattern swatches: head + items[{name, color, available}]. For fashion/accessories/decor.'),
        ('gift_options',    'Wanelo · Gift Options',    'Gift wrap + occasions: head + wrap{available,price,options[]} + card{available,examples[]} + occasions[]. For gift intent.'),
        # ─── Round 5 additions ───
        ('safety',           'Wanelo · Safety',           'Warnings, age rating, contraindications, safety certs. FILL for beauty devices / supplements / kids toys / appliances. {head, age_rating, warnings[{label, detail}], contraindications[], safety_certs[{label, sub}]}'),
        ('protocol',         'Wanelo · Protocol',         'Ongoing usage rhythm — distinct from how_to (setup) and timeline (transformation). FILL for beauty devices / supplements / fitness / wellness rituals. {head, schedule[{phase, frequency, duration, intensity, note}], total_commitment}'),
        ('target_profile',   'Wanelo · Target Profile',   'Who is this for. {head, ideal_for[{glyph, label, detail}], not_ideal_for[], fitness_level, skin_type, age_range}'),
        ('video_demo',       'Wanelo · Video Demo',       'Embedded product video. {head, video{source, id_or_url, duration_seconds, thumbnail_url}, chapters[{time, label}]}'),
        ('compatibility',    'Wanelo · Compatibility',    'Works-with matrix. {head, platforms[{name, version, logo_hint}], accessories[{name, sku, fit_note}], incompatibility_warnings[]}'),
        ('assembly',         'Wanelo · Assembly',         'Setup time + tools. {head, time_minutes, difficulty, person_count, tools_required[{name, included, optional}], warnings[]}'),
        # ─── Round 6 additions ───
        ('sustainability',     'Wanelo · Sustainability',     'Eco/ethical credentials. Distinct from trust (awards) and safety (warnings). {head, certs[{label, since, verifier}], materials[{name, percent, note}], carbon_footprint_g, recycling}'),
        ('brand_story',        'Wanelo · Brand Story',        'COMPANY narrative — origin, founder, values. Distinct from story (PRODUCT narrative). {head, founded{year, city, founder}, mission, values[{label, detail}], founder_quote{text, author}, founder_image_url}'),
        ('nutrition_facts',    'Wanelo · Nutrition Facts',    'FDA-style nutrition label box. {head, serving_size, servings_per_container, calories, macros[{label, value, dv_percent}], highlights[], allergens[]}'),
        ('app_showcase',       'Wanelo · App Showcase',       'Companion app screens + features + store badges. Distinct from compatibility (platform support). {head, app_name, store_links[{store, url}], screens[{caption, image_url}], features[]}'),
        ('subscription_refill','Wanelo · Subscription Refill','Subscribe & Save messaging for consumables. VISUAL only — actual cart logic is operator (Recharge/Skio/Shopify Subscriptions). {head, lasts_days, default_refill_interval_days, discount_percent, perks[], savings_per_year_usd, best_for_buyer_text}'),
        ('clinical_evidence',  'Wanelo · Clinical Evidence',  'Scientific backing for claims. Distinct from trust (press/awards) and safety (warnings). {head, claims[{stat, claim, study_size, study_type, study_year}], mechanism, disclaimer}'),
        ('room_placement',     'Wanelo · Room Placement',     'Visual scale + room-fit for furniture / decor / lighting. Distinct from dimensions (numbers only). {head, best_for_rooms[], scale_photos[{image_url, caption}], pairs_well_with[]}'),
        ('lifestyle_gallery',  'Wanelo · Lifestyle Gallery',  'UGC mood grid for aspirational positioning. Distinct from story images (narrative-anchored). {head, items[{image_url, credit, caption, tall}], footer_text}'),
        # ─── Admin-only ───
        ('research_refs',   'Wanelo · Research Refs',   'Backed-by-science card with links to real peer-reviewed research (PubMed / AAD / NIH / etc) on the product topic. Auto-generated by Opus Strategy per category. {head:{kicker,h2}, items:[{title,source,year,url}]}'),
        ('unboxing_journey','Wanelo · Unboxing Journey','4-5 steps the customer experiences opening the package. Conversion-focused anticipation block. {head:{kicker,h2}, steps:[{step_number,title,description}]}'),
        ('progress_milestones','Wanelo · Progress Milestones','Day 1 / Week 1 / Week 4 / Week 8 expected results. Manages expectations to reduce returns. {head:{kicker,h2}, milestones:[{timeframe,label,expected_result}]}'),
        ('mistake_warnings','Wanelo · Mistake Warnings','Common-mistakes block with why + correct alternative. Positions brand as caring expert. {head:{kicker,h2}, warnings:[{mistake,why_problem,instead_do}]}'),
        ('use_scenarios',   'Wanelo · Use Scenarios',   '3-5 distinct usage contexts (morning/date/travel). Expands perceived value. {head:{kicker,h2}, scenarios:[{scenario,when,how_to_use,key_benefit}]}'),
        ('rating',          'Wanelo · Rating',          'Star rating + review count near product title. Manual or review-sync. {stars, count, label}'),
        ('trust_strip',     'Wanelo · Trust Strip',     'Horizontal icon strip above long-form. Manual edit per product. {items:[{icon,label}]}'),
        ('photo_pack',      'Wanelo · Photo Pack',      'Admin-only download link (type: url). One click in Admin → opens ZIP with EPROLO photos + prompt.txt for ChatGPT-UI edit workflow. NEVER exposed to storefront.'),
        ('source',          'Wanelo · Source',          'Admin-only EPROLO origin record: {platform, url, scraped_at, title, description, cost_price_usd, image_urls{top[], desc[]}}. Click url to open original product page on EPROLO.'),
        ('source_url',      'Wanelo · Source URL',      'Admin-only one-click EPROLO link (type: url). Same URL that lives inside source.url JSON, surfaced as a clickable button in Admin so operator never has to expand the JSON.'),
        ('designer_prompt', 'Wanelo · Designer Prompt', 'Admin-only text of the per-product photo-editor prompt (type: multi_line_text_field). Identical to prompt.txt inside the photo_pack ZIP — surfaced here so operator can copy-paste into ChatGPT-UI without downloading the archive.'),
    ]
    # Admin-only metafields: NOT exposed via Storefront API. Used for utility
    # data the customer should never see (photo_pack ZIP URL, EPROLO source provenance)
    _admin_only = {'photo_pack', 'source', 'source_url', 'designer_prompt'}
    # Per-key type override. photo_pack is type 'url' so Shopify Admin renders it
    # as a one-click clickable download button. All other metafields are JSON
    # objects consumed by Liquid snippets.
    _type_overrides = {'photo_pack': 'url', 'source_url': 'url', 'designer_prompt': 'multi_line_text_field'}
    for _k, _name, _desc in _wanelo_metafields:
        await shopify_ensure_metafield_definition(
            namespace='custom',
            key=_k,
            name=_name,
            type_name=_type_overrides.get(_k, 'json'),
            description=_desc,
            visible_to_storefront=(_k not in _admin_only),
        )
    print(f'Metafield definitions ensured: {len(_wanelo_metafields)} (35 storefront + 4 admin-only: photo_pack ZIP + source JSON + source_url + designer_prompt)')

    # Build curated allowlist from what actually exists in Shopify now.
    # Cell 6 will filter _mf_inputs by this so we never pollute the store
    # with metafields whose definition hit PINNED_LIMIT_REACHED.
    _KNOWN_CUSTOM_KEYS = set()
    try:
        _q_known = await shopify_gql('{ metafieldDefinitions(first: 250, ownerType: PRODUCT, namespace: "custom") { nodes { key } } }')
        if _q_known and _q_known.get('data'):
            _KNOWN_CUSTOM_KEYS = {
                n['key'] for n in _q_known['data']['metafieldDefinitions']['nodes']
            }
    except Exception as _e_known:
        print(f'  curated keys fetch failed: {str(_e_known)[:80]}')
    print(f'Curated custom keys live in Shopify: {len(_KNOWN_CUSTOM_KEYS)} ({sorted(_KNOWN_CUSTOM_KEYS)[:6]}...)')

    # Collection-level metafields — separate owner_type=COLLECTION.
    # related_collections drives cross-collection SEO interlinking (silo
    # structure approximation). Theme renders via wanelo-collection-related
    # section. visible_to_storefront=True so theme reads it.
    _wanelo_coll_metafields = [
        ('related_collections', 'Wanelo · Related Collections',
         'JSON array of related collection chips [{handle, title, anchor, score, is_hub}]. Computed post-creation in Cell 4 with reciprocity + hub detection. Theme renders chip-row via wanelo-collection-related section.'),
    ]
    for _k, _name, _desc in _wanelo_coll_metafields:
        await shopify_ensure_metafield_definition(
            namespace='custom', key=_k, name=_name, type_name='json',
            description=_desc, owner_type='COLLECTION',
            visible_to_storefront=True,
        )
    print(f'Collection metafield definitions ensured: {len(_wanelo_coll_metafields)} (related_collections for SEO silo)')


DataForSEO balance: $123.14
Helpers OK
Testing Claude claude-opus-4-6... ✓ OK! Response: "OK"

  >>> SELECTED MODEL: claude-opus-4-6
Shopify token OK
All helpers OK
All helpers OK (v9.2: variants, sanitizer, smart interlinks)


## Cell 3 — SEO Pipeline (auto-skip if Excel loaded)

In [3]:
# ════ SEO PIPELINE ════
_seo_done = db.execute("SELECT value FROM seo_state WHERE key='seo_complete'").fetchone()
_skip = bool(_seo_done or SEO_SKIP)

if _skip:
    print("SEO: SKIPPED (data already in DB)")
    print(f"  Collections: {db.execute('SELECT COUNT(*) FROM collections').fetchone()[0]}")

if not _skip:
    print("SEO: RUNNING...")

    # ─── RESUME CHECKPOINT LAYER (don't re-burn API on restart) ───
    import hashlib as _hl
    db.execute("CREATE TABLE IF NOT EXISTS seo_state (key TEXT PRIMARY KEY, value TEXT)")
    db.execute("CREATE TABLE IF NOT EXISTS seo_kw_cache (ckey TEXT PRIMARY KEY, payload TEXT)")
    db.commit()
    def _seo_ckey(ep, items):
        _h = _hl.sha1((str(ep) + '|' + '|'.join(sorted(map(str, items)))).encode('utf-8')).hexdigest()
        return str(ep) + ':' + _h
    def _seo_cache_get(k):
        _row = db.execute("SELECT payload FROM seo_kw_cache WHERE ckey=?", (k,)).fetchone()
        return json.loads(_row[0]) if _row else None
    def _seo_cache_put(k, payload):
        db.execute("INSERT OR REPLACE INTO seo_kw_cache (ckey,payload) VALUES (?,?)", (k, json.dumps(payload)))
        db.commit()
    def _seo_persist_spend():
        try:
            db.execute("INSERT OR REPLACE INTO seo_state (key,value) VALUES ('dfs_spent',?)", (str(cost_tracker.spent),))
            db.commit()
        except Exception:
            pass
    try:
        _dfs_prev = db.execute("SELECT value FROM seo_state WHERE key='dfs_spent'").fetchone()
        if _dfs_prev:
            cost_tracker.spent = float(_dfs_prev[0])
            print(f"  [resume] DataForSEO prior spend restored: ${cost_tracker.spent:.2f}")
    except Exception:
        pass
    _seo_cache_n = db.execute("SELECT COUNT(*) FROM seo_kw_cache").fetchone()[0]
    if _seo_cache_n:
        print(f"  [resume] {_seo_cache_n} cached API batches will be reused (no re-charge)")

    print('\n' + '='*50)
    print('STAGE 1: Categorizing products...')
    print(f'Products: {len(PRODUCTS)} | Batches: {(len(PRODUCTS)+14)//15}')
    print('='*50)

    CAT_PROMPT='Shopify store manager. Categorize each product into a collection.\n''Rules:\n- 1-3 word collection names (product CATEGORY not product name)\n''- 8-25 collections total\n- Similar products same collection\n''- Be specific: "Eyeshadow" not "Makeup", "Teeth Whitening" not "Beauty"\n\n''Products:\nPROD_PH\n\nJSON: [{"p":"exact product name","c":"Collection"}]'

    product_coll_map={}
    total_batches = (len(PRODUCTS)+14)//15
    t_start = time.time()

    for bi,batch in enumerate(chunk_list(PRODUCTS,15)):
        elapsed = time.time() - t_start
        print(f'\n  Batch {bi+1}/{total_batches} ({len(batch)} prods) [{elapsed:.0f}s]', end=' ', flush=True)
        prompt=CAT_PROMPT.replace('PROD_PH','\n'.join([str(i+1)+'. '+p for i,p in enumerate(batch)]))
        _ckc = _seo_ckey('cat1', batch)
        result = _seo_cache_get(_ckc)
        if result is None:
            result=call_claude(prompt,0.3)
            if result: _seo_cache_put(_ckc, result)
        matched=0
        for item in result:
            if not isinstance(item,dict): continue
            pn=item.get('p',''); co=item.get('c','Uncategorized')
            for product in batch:
                if product not in product_coll_map:
                    if pn.lower()[:20] in product.lower() or product.lower()[:20] in pn.lower():
                        product_coll_map[product]=co; matched+=1; break
        for product in batch:
            if product not in product_coll_map: product_coll_map[product]='Uncategorized'
        print(f'→ {matched}/{len(batch)}', flush=True)
        time.sleep(4)

    coll_products={}
    for p,c in product_coll_map.items():
        if c not in coll_products: coll_products[c]=[]
        coll_products[c].append(p)

    print(f'\n✓ Stage 1 done in {time.time()-t_start:.0f}s')
    print(f'Collections ({len(coll_products)}):')
    for c in sorted(coll_products,key=lambda x:len(coll_products[x]),reverse=True):
        print(f'  {c}: {len(coll_products[c])}')

    SEED_PROMPT = '''You are an SEO keyword expert.
    I need SHORT SEED keywords (1-3 words) for a Google Ads keyword research tool.
    These must be GENERIC product category terms that MILLIONS of people search.

    GOOD seeds: "eyeliner", "teeth whitening", "body glitter", "matte lipstick", "cleansing oil"
    BAD seeds: "buy waterproof eyeliner online", "best anti aging firming serum for wrinkles"

    Rules:
    - 1-3 words MAX per seed
    - Must be a real product category people search on Google
    - NO brand names
    - Generate 5-8 seeds per collection
    - Include the collection name itself as a seed

    Category: CAT_PH
    Collections and their products:
    DATA_PH

    JSON: [{"seed":"short keyword","c":"Collection Name"}]'''

    all_seeds = []
    coll_names = [c for c in coll_products if c != 'Uncategorized']
    total_batches = (len(coll_names)+7)//8
    t_start = time.time()
    print(f'\nSTAGE 2: Extracting seeds from {len(coll_names)} collections ({total_batches} batches)')

    for bi, batch in enumerate(chunk_list(coll_names, 8)):
        elapsed = time.time() - t_start
        print(f'  Batch {bi+1}/{total_batches} [{elapsed:.0f}s]', end=' ', flush=True)
        data = ''
        for c in batch:
            prods = coll_products[c][:5]
            data += '\n'+c+': ' + ', '.join([p[:40] for p in prods]) + '\n'
        prompt = SEED_PROMPT.replace('CAT_PH', CATEGORY).replace('DATA_PH', data)
        _cks = _seo_ckey('seed1', batch)
        result = _seo_cache_get(_cks)
        if result is None:
            result = call_claude(prompt, 0.5)
            if result: _seo_cache_put(_cks, result)
        cnt = 0
        for item in result:
            if isinstance(item, dict) and item.get('seed'):
                seed = item['seed'].lower().strip()
                coll = item.get('c', '')
                if len(seed) > 2 and len(seed.split()) <= 4 and not is_brand(seed):
                    all_seeds.append({'seed': seed, 'collection': coll})
                    cnt += 1
        print(f'→ +{cnt} seeds', flush=True)
        time.sleep(4)

    # Deduplicate seeds
    seen = set()
    unique_seeds = []
    for s in all_seeds:
        if s['seed'] not in seen:
            seen.add(s['seed']); unique_seeds.append(s)

    print(f'\n✓ Seeds: {len(unique_seeds)} (from {len(all_seeds)} raw, {len(all_seeds)-len(unique_seeds)} deduped)')
    for s in unique_seeds[:20]: print(f'  [{s["collection"]}] {s["seed"]}')

    seed_list = [s['seed'] for s in unique_seeds]
    seed_coll_map = {s['seed']: s['collection'] for s in unique_seeds}

    # ═══ PART A: keywords_for_keywords (Google Ads) ═══
    seed_batches = list(chunk_list(seed_list, 20))
    ideas_batches = list(chunk_list(seed_list, 20))

    total_calls = len(seed_batches) + len(ideas_batches)
    est = total_calls * 0.075
    if not auto_confirm(est, 'Stage 3: '+str(len(seed_list))+' seeds → '
        +str(len(seed_batches))+' keywords_for_keywords + '
        +str(len(ideas_batches))+' keyword_ideas = '+str(total_calls)+' calls'):
        raise SystemExit('Cancelled')

    dataforseo_kws = []
    existing_kws = set()
    filtered_irrelevant = 0  # v6.1 counter

    # v6.2: track which seed produced each keyword
    kw_seed_source = {}  # keyword -> seed that found it

    def add_kw(kw, vol, cpc, comp, comp_idx, src, seed_batch=None):
        global filtered_irrelevant
        kw = (kw or '').lower().strip()
        if not kw or len(kw) < 4 or kw in existing_kws: return 0
        if is_brand(kw): return 0
        if is_irrelevant(kw):  # v6.1: filter irrelevant
            filtered_irrelevant += 1; return 0
        dataforseo_kws.append({
            'keyword': kw, 'search_volume': vol or 0,
            'cpc': cpc or 0, 'competition': comp or '',
            'competition_index': comp_idx or 0, 'source': src,
        })
        existing_kws.add(kw)
        # v6.2: remember which seed batch found this keyword
        if seed_batch:
            for seed in seed_batch:
                if seed.lower() in kw or kw in seed.lower() or len(set(seed.lower().split()) & set(kw.split())) > 0:
                    kw_seed_source[kw] = seed
                    break
            if kw not in kw_seed_source and seed_batch:
                kw_seed_source[kw] = seed_batch[0]  # default to first seed in batch
        return 1


    # ═══ v6.1: Retry wrapper for DataForSEO API ═══
    def api_post_with_retry(url, payload, max_retries=3, timeout=120):
        """POST with retry on timeout/connection errors. Identifies service via URL host."""
        # Идентификация сервиса для лога: host из URL (api.dataforseo.com → DataForSEO).
        _service = "DataForSEO" if "dataforseo" in url else "API"
        for attempt in range(max_retries):
            try:
                r = requests.post(url, headers=API_HEADERS, json=payload, timeout=timeout)
                if r.status_code == 401:
                    print(f'[{_service} AUTH 401 — check DATAFORSEO_LOGIN/DATAFORSEO_PASSWORD env]', flush=True)
                    return None
                if r.status_code in (402, 403):
                    # 402 = insufficient balance, 403 = forbidden
                    _kind = "balance" if r.status_code == 402 else "forbidden"
                    print(f'[{_service} {r.status_code} ({_kind}): {r.text[:120]}]', flush=True)
                    return None
                return r
            except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
                wait = 15 * (attempt + 1)
                if attempt < max_retries - 1:
                    print(f'[{_service} timeout, retry {attempt+1}/{max_retries} in {wait}s]', end=' ', flush=True)
                    time.sleep(wait)
                else:
                    print(f'[{_service} FAILED after {max_retries} retries: {str(e)[:60]}]', flush=True)
                    return None
        return None

    print('─── Part A: keywords_for_keywords (Google Ads) ───')
    for bi, batch in enumerate(tqdm(seed_batches, desc='keywords_for_keywords')):
        try:
            _cka = _seo_ckey('keywords_for_keywords', batch)
            data = _seo_cache_get(_cka)
            if data is None:
                if not cost_tracker.can_afford('keywords_for_keywords'):
                    print('Budget limit!'); break
                r = api_post_with_retry(
                    'https://api.dataforseo.com/v3/keywords_data/google_ads/keywords_for_keywords/live',
                    [{'keywords': batch, 'location_code': LOCATION_CODE,
                      'language_code': LANGUAGE_CODE, 'sort_by': 'search_volume',
                      'include_adult_keywords': False}])
                if r is None: continue
                cost_tracker.charge('keywords_for_keywords', str(len(batch))+' seeds')
                data = r.json()
                _seo_cache_put(_cka, data); _seo_persist_spend()

            if bi == 0:
                print('\n[DEBUG] status:', data.get('status_code'), data.get('status_message'))
                if data.get('tasks'):
                    t0 = data['tasks'][0]
                    print('[DEBUG] task:', t0.get('status_code'), '| result_count:', t0.get('result_count'))
                    rr = t0.get('result') or []
                    if rr: print('[DEBUG] first:', rr[0].get('keyword','?'), 'vol:', rr[0].get('search_volume','?'))

            if data.get('status_code') == 20000:
                cnt = 0
                for task in data.get('tasks', []):
                    for item in (task.get('result') or []):
                        if not isinstance(item, dict): continue
                        cnt += add_kw(item.get('keyword'), item.get('search_volume'),
                            item.get('cpc'), item.get('competition'),
                            item.get('competition_index'), 'gads', seed_batch=batch)
                if bi < 3: print(f'  Batch {bi+1}: +{cnt}')
        except Exception as e:
            print(f'Error: '+str(e)[:100])
        time.sleep(2)

    print(f'After keywords_for_keywords: {len(dataforseo_kws)} keywords (filtered {filtered_irrelevant} irrelevant)')

    # ═══ PART B: keyword_ideas (DataForSEO Labs) ═══
    print('\n─── Part B: keyword_ideas (DataForSEO Labs SERP DB) ───')
    for bi, batch in enumerate(tqdm(ideas_batches, desc='keyword_ideas')):
        try:
            _ckb = _seo_ckey('keyword_ideas', batch)
            data = _seo_cache_get(_ckb)
            if data is None:
                if not cost_tracker.can_afford('keyword_ideas'):
                    print('Budget limit!'); break
                r = api_post_with_retry(
                    'https://api.dataforseo.com/v3/dataforseo_labs/google/keyword_ideas/live',
                    [{'keywords': batch, 'location_code': LOCATION_CODE,
                      'language_code': LANGUAGE_CODE, 'include_serp_info': False,
                      'limit': 700, 'filters': [['keyword_info.search_volume', '>', MIN_VOLUME - 1]],
                      'order_by': ['keyword_info.search_volume,desc']}])
                if r is None: continue
                cost_tracker.charge('keyword_ideas', str(len(batch))+' seeds')
                data = r.json()
                _seo_cache_put(_ckb, data); _seo_persist_spend()

            if bi == 0:
                print('\n[DEBUG] status:', data.get('status_code'), data.get('status_message'))
                if data.get('tasks'):
                    t0 = data['tasks'][0]
                    print('[DEBUG] task:', t0.get('status_code'), '| result_count:', t0.get('result_count'))
                    rr = (t0.get('result') or [None])[0]
                    if rr and isinstance(rr, dict):
                        print('[DEBUG] total_count:', rr.get('total_count'), '| items_count:', rr.get('items_count'))
                        items = rr.get('items') or []
                        if items:
                            item0 = items[0]
                            print('[DEBUG] item keys:', list(item0.keys())[:10])
                            # Try both structures
                            kd0 = item0.get('keyword_data') or item0
                            ki0 = kd0.get('keyword_info') or kd0
                            kw0 = kd0.get('keyword') or item0.get('keyword') or '?'
                            vol0 = ki0.get('search_volume') or item0.get('search_volume') or '?'
                            print('[DEBUG] first:', kw0, 'vol:', vol0)

            if data.get('status_code') == 20000:
                cnt = 0
                for task in data.get('tasks', []):
                    for res in (task.get('result') or []):
                        if not isinstance(res, dict): continue
                        for item in (res.get('items') or []):
                            if not isinstance(item, dict): continue
                            # Handle both structures: {keyword_data:{keyword, keyword_info:{...}}} and {keyword, keyword_info:{...}}
                            kd = item.get('keyword_data') or item
                            if not isinstance(kd, dict): continue
                            ki = kd.get('keyword_info') or kd
                            if not isinstance(ki, dict): continue
                            kw_name = kd.get('keyword') or item.get('keyword')
                            cnt += add_kw(
                                kw_name, ki.get('search_volume'),
                                ki.get('cpc'), ki.get('competition'),
                                ki.get('competition_index'), 'ideas', seed_batch=batch
                            )
                if bi < 3: print(f'  Batch {bi+1}: +{cnt}')
        except Exception as e:
            print(f'Error: '+str(e)[:100])
        time.sleep(0.5)

    print(f'\nTotal REAL keywords: {len(dataforseo_kws)}')
    print(f'Filtered as irrelevant: {filtered_irrelevant}')

    # ═══ Build DataFrame ═══
    df_real = pd.DataFrame(dataforseo_kws)
    if len(df_real) > 0:
        df_real = df_real.drop_duplicates(subset=['keyword'], keep='first')
        df_real = df_real.sort_values('search_volume', ascending=False).reset_index(drop=True)

    print('\n' + '='*50)
    print('  STAGE 3 RESULTS')
    print('='*50)
    print('Total REAL keywords: '+str(len(df_real)))
    if len(df_real) > 0:
        n_gads = len(df_real[df_real['source']=='gads'])
        n_ideas = len(df_real[df_real['source']=='ideas'])
        print(f'  From keywords_for_keywords: {n_gads}')
        print(f'  From keyword_ideas:         {n_ideas}')
        print(f'  With volume > 0:            {len(df_real[df_real["search_volume"]>0])}')
        print(f'  With volume >= {MIN_VOLUME}:          {len(df_real[df_real["search_volume"]>=MIN_VOLUME])}')
        print(f'  Filtered irrelevant:        {filtered_irrelevant}')
        print('\nTop 25 keywords:')
        print(df_real[['keyword','search_volume','cpc','competition','source']].head(25).to_string(index=False))
    cost_tracker.summary()

    # Build DataFrame
    df_real = pd.DataFrame(dataforseo_kws)
    if len(df_real) > 0:
        df_real['keyword'] = df_real['keyword'].str.lower().str.strip()
        df_real = df_real.drop_duplicates(subset=['keyword'], keep='first')
    else:
        print('WARNING: DataForSEO returned 0 keywords!')
        df_real = pd.DataFrame(columns=['keyword','search_volume','cpc','competition','competition_index','source'])
    print(f'df_real: {len(df_real)} keywords')


    import os, warnings
    os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
    os.environ['TOKENIZERS_PARALLELISM'] = 'false'
    warnings.filterwarnings('ignore')

    import torch, faiss, numpy as np, time
    from sentence_transformers import SentenceTransformer

    t0 = time.time()
    print('STAGE 4: Seed-based keyword mapping + product ranking')

    # ═══ GPU ═══
    USE_GPU = torch.cuda.is_available()
    DEVICE = 'cuda' if USE_GPU else 'cpu'
    EMBED_BATCH = 512 if USE_GPU else 128
    if USE_GPU:
        gpu = torch.cuda.get_device_properties(0)
        print(f'  GPU: {gpu.name} ({gpu.total_memory/1e9:.1f} GB)')

    # ═══ df_all ═══
    df_all = df_real.copy()
    df_all = df_all[df_all['keyword'].notna() & (df_all['keyword'] != '')].reset_index(drop=True)

    def smart_kd_quick(row):
        ci = row.get('competition_index', 50) or 50
        vol = row.get('search_volume', 0) or 0
        cpc = row.get('cpc', 0) or 0
        kd = ci * 0.5
        if vol > 100000: kd += 15
        elif vol > 10000: kd += 8
        elif vol > 1000: kd += 3
        if cpc > 5: kd += 10
        elif cpc > 2: kd += 5
        return min(max(int(kd), 0), 100)

    if len(df_all) == 0:
        print('\n⚠ No keywords from DataForSEO — will use product names as basic keywords')
        # Create minimal df_all so remaining code doesn't crash
        basic_kws = []
        for product in PRODUCTS:
            coll = product_coll_map.get(product, 'Uncategorized')
            # Use product name words as basic keywords
            words = product.lower().split()[:4]
            kw = ' '.join(words)
            basic_kws.append({'keyword': kw, 'search_volume': 100, 'cpc': 0.5,
                'competition': 'LOW', 'competition_index': 10, 'source': 'fallback',
                'keyword_difficulty': 5})
        df_all = pd.DataFrame(basic_kws)
        print(f'  Created {len(basic_kws)} fallback keywords from product names')

    if 'keyword_difficulty' not in df_all.columns:
        df_all['keyword_difficulty'] = df_all.apply(smart_kd_quick, axis=1)
    MIN_VOLUME = 100
    vol_map = dict(zip(df_all['keyword'], df_all['search_volume']))
    print(f'  Total keywords: {len(df_all):,}')

    # ═══ STEP 1: Build keyword → collection from seed lineage ═══
    # kw_seed_source: keyword → seed (from Cell 17)
    # seed_coll_map: seed → collection (from Cell 17)
    # Chain: keyword → seed → collection

    kw_to_collection_via_seed = {}  # keyword → collection name
    collection_keywords = {}  # collection → set of keywords

    mapped = 0
    unmapped = 0
    for kw, seed in kw_seed_source.items():
        coll = seed_coll_map.get(seed)
        if coll:
            kw_to_collection_via_seed[kw] = coll
            collection_keywords.setdefault(coll, set()).add(kw)
            mapped += 1
        else:
            unmapped += 1

    print(f'  Seed lineage: {mapped:,} keywords mapped, {unmapped:,} unmapped')
    print(f'  Collections from seeds: {len(collection_keywords)}')

    # Show coverage
    for coll in sorted(collection_keywords, key=lambda c: len(collection_keywords[c]), reverse=True)[:10]:
        n = len(collection_keywords[coll])
        vol_sum = sum(vol_map.get(kw, 0) for kw in collection_keywords[coll])
        print(f'    {n:>5} kws | {vol_sum:>10,} vol | {coll}')

    # ═══ STEP 2: Rank keywords per product using FAISS ═══
    print(f'\n  Loading embedding model...')
    embed_model = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)

    # Embed products
    prod_embs = embed_model.encode(PRODUCTS, show_progress_bar=False,
                                    batch_size=EMBED_BATCH, normalize_embeddings=True)
    prod_embs = np.array(prod_embs).astype('float32')
    d = prod_embs.shape[1]

    validated_product_kws = {}
    products_assigned = 0

    for product in PRODUCTS:
        coll = product_coll_map.get(product, 'Uncategorized')
        # Get all keywords in this product's collection (from seed lineage)
        coll_kws = list(collection_keywords.get(coll, set()))

        if not coll_kws:
            validated_product_kws[product] = []
            continue

        # Filter to keywords with volume
        coll_kws_vol = [(kw, vol_map.get(kw, 0)) for kw in coll_kws if vol_map.get(kw, 0) >= MIN_VOLUME]
        if not coll_kws_vol:
            validated_product_kws[product] = []
            continue

        # Embed collection keywords
        kw_texts = [kw for kw, _ in coll_kws_vol]
        kw_embs = embed_model.encode(kw_texts, show_progress_bar=False,
                                      batch_size=EMBED_BATCH, normalize_embeddings=True)
        kw_embs = np.array(kw_embs).astype('float32')

        # FAISS: rank by similarity to THIS product
        pi = PRODUCTS.index(product)
        prod_emb = prod_embs[pi:pi+1]

        kw_idx = faiss.IndexFlatIP(d)
        kw_idx.add(kw_embs)
        n_results = min(30, len(kw_texts))
        scores, indices = kw_idx.search(prod_emb, n_results)

        # Take top ranked keywords
        product_kws = []
        for j in range(n_results):
            idx = indices[0][j]
            if 0 <= idx < len(kw_texts):
                product_kws.append(kw_texts[idx])

        validated_product_kws[product] = product_kws[:25]
        if product_kws:
            products_assigned += 1
            # v9.3: Save SEO keywords to DB
            kw_str = ' | '.join(product_kws[:15])
            if kw_str:
                db.execute('UPDATE products SET seo_keywords=? WHERE name=?', (kw_str, product))

    print(f'\n  Products with keywords: {products_assigned}/{len(PRODUCTS)}')
    total_pairs = sum(len(v) for v in validated_product_kws.values())
    print(f'  Total product-keyword pairs: {total_pairs:,}')

    # ═══ STEP 3: Build kw_to_collections (ONLY from seed lineage) ═══
    kw_to_collections = {}
    for kw, coll in kw_to_collection_via_seed.items():
        kw_to_collections[kw] = {coll}

    # final_product_colls from Stage 1 (CLEAN)
    final_product_colls = {}
    for product in PRODUCTS:
        coll = product_coll_map.get(product, 'Uncategorized')
        final_product_colls[product] = [coll]

    all_validated = set()
    for kws in validated_product_kws.values():
        all_validated.update(kws)

    # ═══ Samples ═══
    print(f'\n  Keywords with collection: {len(kw_to_collections):,}')
    print(f'\n  Samples:')
    for p in PRODUCTS[:8]:
        kws = validated_product_kws.get(p, [])
        coll = product_coll_map.get(p, '?')
        kw_sample = ', '.join(kws[:5])
        print(f'    {p[:50]}')
        print(f'      [{coll}] {len(kws)} kws: {kw_sample}')

    print(f'\n  Stage 4: {time.time()-t0:.1f}s')


    print('STAGE 5: Skipped (collections from Stage 1, no clustering)')
    # Build final_coll_products from Stage 1 categorization
    final_coll_products = {}
    for p, colls in final_product_colls.items():
        for c in colls:
            final_coll_products.setdefault(c, []).append(p)

    # Remove Uncategorized if empty or useless
    if 'Uncategorized' in final_coll_products:
        if len(final_coll_products['Uncategorized']) == 0:
            del final_coll_products['Uncategorized']

    # No irrelevant clusters in this approach
    irrelevant_clusters = set()

    print(f'  Collections: {len(final_coll_products)}')
    for c in sorted(final_coll_products, key=lambda x: len(final_coll_products[x]), reverse=True)[:15]:
        print(f'    {len(final_coll_products[c]):>3} products | {c}')


    # ═══ Stage 6: Get real KD + Expand top winners ═══
    print('STAGE 6: Real KD + Expansion')
    t_start = time.time()

    # df_all already created in Stage 4 with smart_kd estimates
    # Now get REAL KD from DataForSEO (more accurate)
    vol_map = dict(zip(df_all['keyword'], df_all['search_volume']))
    comp_map = dict(zip(df_all['keyword'], df_all['competition']))
    ci_map = dict(zip(df_all['keyword'], df_all['competition_index']))
    cpc_map = dict(zip(df_all['keyword'], df_all['cpc']))

    has_vol = df_all[df_all['search_volume'] >= MIN_VOLUME]['keyword'].tolist()
    kd_map = {}
    if has_vol:
        print('Getting KD for '+str(len(has_vol))+' keywords...')
        for chunk in [has_vol[i:i+1000] for i in range(0, len(has_vol), 1000)]:
            try:
                _ckk = _seo_ckey('bulk_keyword_difficulty', chunk)
                data = _seo_cache_get(_ckk)
                if data is None:
                    if not cost_tracker.can_afford('bulk_keyword_difficulty'):
                        break
                    r = api_post_with_retry('https://api.dataforseo.com/v3/dataforseo_labs/google/bulk_keyword_difficulty/live',
                        [{'keywords': chunk, 'location_code': LOCATION_CODE, 'language_code': LANGUAGE_CODE}])
                    if r is None: continue
                    cost_tracker.charge('bulk_keyword_difficulty', str(len(chunk))+' kws')
                    data = r.json()
                    _seo_cache_put(_ckk, data); _seo_persist_spend()
                if data.get('status_code') == 20000:
                    for task in data.get('tasks', []):
                        for res in (task.get('result') or []):
                            if not res or not isinstance(res, dict): continue
                            for item in (res.get('items') or []):
                                if not item or not isinstance(item, dict): continue
                                kw = (item.get('keyword','') or '').lower()
                                kd = item.get('keyword_difficulty')
                                if kw and kd and kd > 0: kd_map[kw] = kd
            except: pass
            time.sleep(1)
        print('Bulk KD > 0: '+str(len(kd_map)))

    def smart_kd(row):
        kw = row.get('keyword', ''); bulk = kd_map.get(kw, 0)
        if bulk > 0: return bulk
        comp = row.get('competition', '') or ''; ci = row.get('competition_index', 0) or 0; vol = row.get('search_volume', 0) or 0
        if comp == 'HIGH':
            if vol >= 50000: return max(50, int(ci*0.8))
            if vol >= 10000: return max(35, int(ci*0.6))
            return max(25, int(ci*0.5))
        elif comp == 'MEDIUM': return max(15, int(ci*0.4))
        elif comp == 'LOW': return max(5, int(ci*0.2))
        if vol >= 10000: return 25
        if vol >= 1000: return 15
        return 10

    df_all['keyword_difficulty'] = df_all.apply(smart_kd, axis=1)
    winners = df_all[(df_all['search_volume'] >= MIN_VOLUME) & (df_all['keyword_difficulty'] <= MAX_KD)].sort_values('search_volume', ascending=False)
    print('\nWith volume >= '+str(MIN_VOLUME)+': '+str(len(has_vol)))
    print('Winners (KD<='+str(MAX_KD)+'): '+str(len(winners)))
    if len(winners) > 0:
        print(winners[['keyword','search_volume','keyword_difficulty','cpc','competition']].head(20).to_string(index=False))

    # ═══ Expand ═══
    def dedup_seeds(kl, mx):
        s = []
        for kw in kl:
            d = False
            for ex in s:
                if kw in ex or ex in kw: d = True; break
                w1, w2 = set(kw.split()), set(ex.split())
                if len(w1&w2) >= 2 and len(w1&w2)/max(len(w1), len(w2)) > 0.6: d = True; break
            if not d: s.append(kw)
            if len(s) >= mx: break
        return s

    sp = []
    if len(winners) > 0:
        sp.extend(winners.head(STAGE_EXPAND_TOP*2)['keyword'].tolist())
    seeds = dedup_seeds(list(dict.fromkeys(sp)), STAGE_EXPAND_TOP)
    est = len(seeds) * 2 * 0.075

    expanded_kws = []
    expanded_irr = 0
    if seeds and auto_confirm(est, 'Expand: '+str(len(seeds))+' seeds x 2 endpoints'):
        existing = set(df_all['keyword'].tolist())
        for seed in tqdm(seeds, desc='Expanding'):
            for ep in ['keyword_suggestions', 'related_keywords']:
                try:
                    fp = 'keyword_info' if ep == 'keyword_suggestions' else 'keyword_data.keyword_info'
                    _cke = _seo_ckey(ep, [seed])
                    data = _seo_cache_get(_cke)
                    if data is None:
                        if not cost_tracker.can_afford(ep): break
                        r = api_post_with_retry('https://api.dataforseo.com/v3/dataforseo_labs/google/'+ep+'/live',
                            [{'keyword': seed, 'location_code': LOCATION_CODE, 'language_code': LANGUAGE_CODE,
                            'include_seed_keyword': True, 'limit': 80,
                            'filters': [[fp+'.search_volume', '>', MIN_VOLUME-1]],
                            'order_by': [fp+'.search_volume,desc']}], timeout=60)
                        if r is None: continue
                        cost_tracker.charge(ep, seed)
                        data = r.json()
                        _seo_cache_put(_cke, data); _seo_persist_spend()
                    if data.get('status_code') == 20000:
                        for task in data.get('tasks', []):
                            for res in (task.get('result') or []):
                                if not res or not isinstance(res, dict): continue
                                for item in (res.get('items') or []):
                                    if not item or not isinstance(item, dict): continue
                                    kd = item.get('keyword_data') or {}
                                    if not isinstance(kd, dict): continue
                                    ki = kd.get('keyword_info') or {}
                                    kw = (kd.get('keyword','') or '').lower().strip()
                                    if not kw or kw in existing: continue
                                    if is_brand(kw) or is_irrelevant(kw):
                                        expanded_irr += 1; continue
                                    expanded_kws.append({'keyword': kw, 'search_volume': ki.get('search_volume',0) or 0,
                                        'keyword_difficulty': ki.get('keyword_difficulty',0) or 0,
                                        'cpc': ki.get('cpc',0) or 0, 'competition': ki.get('competition','') or '', 'source': ep})
                                    existing.add(kw)
                except: pass
            time.sleep(0.5)
        print('Expanded: +'+str(len(expanded_kws))+' (filtered '+str(expanded_irr)+' irrelevant/brand)')
    cost_tracker.summary()


    print('STAGE 7: Merge expanded keywords')
    t_start = time.time()

    # Merge expanded into df_all (for Winners/All Keywords sheets)
    if expanded_kws:
        df_exp = pd.DataFrame(expanded_kws)
        df_all = pd.concat([df_all, df_exp], ignore_index=True)
        df_all['keyword'] = df_all['keyword'].str.lower().str.strip()
        df_all = df_all.sort_values('search_volume', ascending=False).drop_duplicates(subset=['keyword'], keep='first')
        df_all = df_all[df_all['keyword'].notna() & (df_all['keyword'] != '')]
        if df_all['keyword_difficulty'].isna().any():
            df_all['keyword_difficulty'] = df_all.apply(smart_kd, axis=1)
        print(f'  df_all: {len(df_all):,} keywords')

        # Add expanded keywords to kw_to_collections via seed lineage ONLY
        # (expanded keywords don't have seed lineage — they came from winner seeds)
        # So we assign them to collection IF their expansion seed was from that collection
        # We DON'T add them to product keyword lists (to keep products clean)
        print(f'  Expanded keywords added to df_all only (product lists unchanged)')

    # Update vol_map
    vol_map = dict(zip(df_all['keyword'], df_all['search_volume']))

    total_kw = sum(len(v) for v in validated_product_kws.values())
    print(f'  Product-keyword pairs: {total_kw:,}')
    print(f'  Stage 7: {time.time()-t_start:.1f}s')


    print('STAGE 8: Collection assembly (using Stage 1 categories)')
    t_start = time.time()

    # Per product: top keywords sorted by volume
    product_top_kws = {}
    for product in PRODUCTS:
        kws = validated_product_kws.get(product, [])
        kws_with_vol = [(k, vol_map.get(k, 0)) for k in kws if vol_map.get(k, 0) >= MIN_VOLUME]
        kws_with_vol.sort(key=lambda x: x[1], reverse=True)
        product_top_kws[product] = kws_with_vol[:10]

    # final_product_colls already set in Stage 4 (from Stage 1 clean categories)
    # final_coll_products already built in Stage 5

    # Rebuild kw_to_collections to ensure consistency
    kw_to_collections = {}
    for product, kws in validated_product_kws.items():
        coll = product_coll_map.get(product, 'Uncategorized')
        for kw in kws:
            kw_to_collections.setdefault(kw, set()).add(coll)

    print(f'  Collections: {len(final_coll_products)}')
    print(f'  Products: {len(PRODUCTS)}')
    print(f'  Keywords with collection: {len(kw_to_collections):,}')
    total_matched = sum(len(v) for v in validated_product_kws.values())
    print(f'  Product-keyword pairs: {total_matched:,}')

    # Show collection summary
    for c in sorted(final_coll_products, key=lambda x: len(final_coll_products[x]), reverse=True)[:15]:
        n_prods = len(final_coll_products[c])
        coll_kw_count = sum(len(validated_product_kws.get(p, [])) for p in final_coll_products[c])
        print(f'    {n_prods:>3} prods | {coll_kw_count:>4} kws | {c}')

    print(f'  Stage 8: {time.time()-t_start:.1f}s')


    print('STAGE 9: SEO titles & meta')
    t_start = time.time()

    # Build collection_info from product keywords
    collection_info = {}
    for coll, prods in final_coll_products.items():
        coll_kws = set()
        for p in prods:
            coll_kws.update(validated_product_kws.get(p, []))

        kw_df = df_all[(df_all['keyword'].isin(coll_kws)) & (df_all['search_volume'] >= MIN_VOLUME)]
        # NO word-matching fallback — only use validated product keywords
        # (word-matching caused "clown makeup" in "Makeup" collection)
        top_kws = kw_df.nlargest(10, 'search_volume')['keyword'].tolist() if len(kw_df) > 0 else []
        collection_info[coll] = {
            'keywords': top_kws,
            'total_volume': int(kw_df['search_volume'].sum()) if len(kw_df) > 0 else 0,
            'n_keywords': len(kw_df),
            'n_products': len(prods),
        }

    # Fill collection column in df_all
    print('Mapping keywords to collections...')
    def get_kw_collection(kw):
        colls = kw_to_collections.get(kw, set())
        real_colls = {c for c in colls if c != 'IRRELEVANT'}
        if real_colls:
            return ' | '.join(sorted(real_colls)[:3])
        return ''

    df_all['collection'] = df_all['keyword'].apply(get_kw_collection)
    filled = (df_all['collection'] != '').sum()
    print(f'  Keywords with collection: {filled}/{len(df_all)} ({filled/max(len(df_all),1)*100:.1f}%)')

    # Generate SEO titles via Claude
    print('\nGenerating SEO titles...')
    seo_collections = {}
    coll_names = list(collection_info.keys())
    for bi in range(0, len(coll_names), 15):
        batch = coll_names[bi:bi+15]
        batch_data = ''
        for c in batch:
            info = collection_info[c]
            batch_data += '\nCollection: ' + c + ' (' + str(info['n_products']) + ' products)'
            if info['keywords']: batch_data += '\nTop kw: ' + ', '.join(info['keywords'][:8])
            batch_data += '\nVolume: ' + str(info['total_volume']) + '\n---'
        prompt = ('Shopify SEO. For each collection create:\n'
            '1. t: SEO H1 title (50-70 chars) with top keywords naturally\n'
            '2. h: URL slug (lowercase-hyphens)\n'
            '3. mt: Meta title (55-60 chars), primary keyword first\n'
            '4. md: Meta description (140-155 chars) with call to action\n'
            'CRITICAL: keyword lists are volume-ranked, NOT curated — they may include off-topic, wrong-category, or brand keywords. Use ONLY keywords that genuinely match the product type of each collection; IGNORE the rest. Never put a brand name or unrelated product category in the title or meta.\n'
            'DATA:\n' + batch_data + '\n'
            'JSON: [{"o":"original","t":"title","h":"handle","mt":"meta","md":"desc"}]')
        try:
            r = call_claude(prompt, 0.7)
            for item in r:
                if isinstance(item, dict) and item.get('o'):
                    seo_collections[item['o']] = {
                        'seo_title': item.get('t', ''),
                        'seo_handle': item.get('h', ''),
                        'meta_title': item.get('mt', ''),
                        'meta_description': item.get('md', '')
                    }
            print(f'  Batch: +{len(r)}', flush=True)
        except Exception as e:
            print(f'  error: {str(e)[:50]}', flush=True)
        time.sleep(3)

    for c in coll_names:
        if c not in seo_collections:
            seo_collections[c] = {
                'seo_title': c,
                'seo_handle': c.lower().replace(' ', '-').replace("'", ""),
                'meta_title': c,
                'meta_description': 'Shop our ' + c + ' collection.'
            }
    print(f'SEO data for {len(seo_collections)} collections')
    print(f'Stage 9: {time.time()-t_start:.1f}s')


    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill
    from google.colab import files as colab_files

    def get_seo(orig):
        if orig in seo_collections: return seo_collections[orig].get('seo_title', orig)
        return orig

    # ═══ v6.1: Map collection column using seo_collections ═══
    def get_seo_collection(kw):
        colls = kw_to_collections.get(kw, set())
        real_colls = {c for c in colls if c != 'IRRELEVANT'}
        if real_colls:
            seo = [get_seo(c) for c in sorted(real_colls)[:3]]
            return ' | '.join(seo)
        return ''

    df_all['seo_collection'] = df_all['keyword'].apply(get_seo_collection)
    # v6.2: exclude keywords marked IRRELEVANT by Claude
    irrelevant_kws = {kw for kw, colls in kw_to_collections.items() if colls == {'IRRELEVANT'}}
    print(f'Excluding {len(irrelevant_kws)} IRRELEVANT keywords from winners')
    df_all = df_all[~df_all['keyword'].isin(irrelevant_kws)]


    df_win = df_all[(df_all['keyword_difficulty']<=MAX_KD)&(df_all['search_volume']>=MIN_VOLUME)].sort_values('search_volume',ascending=False).reset_index(drop=True)

    # Stats
    win_with_coll = (df_win['seo_collection'] != '').sum()
    print(f'Winners: {len(df_win)} | with collection: {win_with_coll} ({win_with_coll/max(len(df_win),1)*100:.1f}%)')

    # Build lookup sets once (lowercase)
    all_kw_lower = set(df_all['keyword'].str.lower().tolist())
    win_kw_lower = set(df_win['keyword'].str.lower().tolist()) if len(df_win) > 0 else set()

    # DEBUG: Check validated_product_kws health
    total_vpk = sum(len(v) for v in validated_product_kws.values())
    empty_vpk = sum(1 for v in validated_product_kws.values() if not v)
    print(f'DEBUG: validated_product_kws: {total_vpk} total pairs, {empty_vpk} empty products')
    if total_vpk > 0:
        sample_p = [p for p in PRODUCTS if validated_product_kws.get(p, [])][:3]
        for p in sample_p:
            kws = validated_product_kws[p][:3]
            in_all = sum(1 for k in kws if k.lower() in all_kw_lower)
            in_win = sum(1 for k in kws if k.lower() in win_kw_lower)
            print(f'  {p[:40]}: {len(validated_product_kws[p])} kws, {in_all}/3 in df_all, {in_win}/3 in winners')
            print(f'    Sample: {kws}')

    product_rows = []
    for product in PRODUCTS:
        valid_kws = validated_product_kws.get(product, [])

        # Sort by volume (highest first), keep all
        kws_sorted = sorted(valid_kws, key=lambda k: vol_map.get(k, 0), reverse=True)

        colls = final_product_colls.get(product, ['Uncategorized'])
        seo_colls = [get_seo(c) for c in colls]
        product_rows.append({
            'product': product,
            'primary_collection': seo_colls[0] if seo_colls else 'Uncategorized',
            'all_collections': ' | '.join(seo_colls),
            'keywords': ' | '.join(kws_sorted[:15]),
            'keyword_count': len(kws_sorted),
        })
    df_products = pd.DataFrame(product_rows)

    # Stats
    kw_filled = sum(1 for r in product_rows if r['keyword_count'] > 0)
    print(f'Products with keywords: {kw_filled}/{len(PRODUCTS)}')
    if kw_filled < len(PRODUCTS):
        empty_prods = [r['product'][:50] for r in product_rows if r['keyword_count'] == 0]
        print(f'  Empty: {", ".join(empty_prods[:5])}')

    safe_cat = CATEGORY.replace(' & ','_').replace(' ','_')
    out = f'{safe_cat}_SEO_Keywords.xlsx'
    wb = Workbook()
    hf=Font(bold=True,size=11,color='FFFFFF')
    hb=PatternFill('solid',fgColor='1F2937')
    gf=PatternFill('solid',fgColor='D1FAE5')
    yf=PatternFill('solid',fgColor='FEF3C7')
    rf=PatternFill('solid',fgColor='FEE2E2')

    # Sheet 1: Product Keywords
    ws1=wb.active; ws1.title='Product Keywords'
    for ci,(h,w) in enumerate([('Product',55),('Primary Collection',40),('All Collections',60),('Keywords',100),('# KW',8)],1):
        c=ws1.cell(1,ci,value=h); c.font=hf; c.fill=hb
    ws1.column_dimensions['A'].width=55; ws1.column_dimensions['B'].width=40
    ws1.column_dimensions['C'].width=60; ws1.column_dimensions['D'].width=100; ws1.column_dimensions['E'].width=8
    for idx,row in df_products.iterrows():
        r=idx+2
        ws1.cell(r,1,value=row['product']); ws1.cell(r,2,value=row['primary_collection'])
        ws1.cell(r,3,value=row['all_collections']); ws1.cell(r,4,value=row['keywords']); ws1.cell(r,5,value=row['keyword_count'])
    ws1.freeze_panes='A2'; ws1.auto_filter.ref='A1:E'+str(len(df_products)+1)

    # Sheet 2: Shopify Collections
    ws_c=wb.create_sheet('Shopify Collections')
    ch=['SEO Title (H1)','URL Handle','Meta Title','Meta Description','Original','# Products','# KW','Total Volume','Top Keywords']
    for ci,h in enumerate(ch,1): c=ws_c.cell(1,ci,value=h); c.font=hf; c.fill=hb
    ws_c.column_dimensions['A'].width=55; ws_c.column_dimensions['B'].width=35
    ws_c.column_dimensions['C'].width=55; ws_c.column_dimensions['D'].width=70; ws_c.column_dimensions['I'].width=80
    ri=2
    for orig in sorted(collection_info,key=lambda x:collection_info[x]['n_products'],reverse=True):
        info=collection_info[orig]
        data=seo_collections.get(orig,{})
        ws_c.cell(ri,1,value=data.get('seo_title',orig)); ws_c.cell(ri,2,value=data.get('seo_handle',''))
        ws_c.cell(ri,3,value=data.get('meta_title','')); ws_c.cell(ri,4,value=data.get('meta_description',''))
        ws_c.cell(ri,5,value=orig); ws_c.cell(ri,6,value=info['n_products'])
        ws_c.cell(ri,7,value=info['n_keywords']); ws_c.cell(ri,8,value=info['total_volume'])
        ws_c.cell(ri,9,value=', '.join(info['keywords'][:8]))
        ri+=1
    ws_c.freeze_panes='A2'

    # Sheet 3: Winners
    ws2=wb.create_sheet('Winners KD<'+str(MAX_KD))
    wc=[('Keyword',50),('Collection',35),('Volume',12),('KD',8),('CPC',10),('Competition',14)]
    # Ensure seo_collection column exists
    if 'seo_collection' not in df_all.columns:
        if 'collection' in df_all.columns:
            df_all['seo_collection'] = df_all['collection']
        else:
            df_all['seo_collection'] = df_all['keyword'].apply(get_seo_collection)
    wk=['keyword','seo_collection','search_volume','keyword_difficulty','cpc','competition']
    for ci,(h,w) in enumerate(wc,1): c=ws2.cell(1,ci,value=h); c.font=hf; c.fill=hb
    ws2.column_dimensions['A'].width=50; ws2.column_dimensions['B'].width=35
    for idx,row in df_win.iterrows():
        r=idx+2
        for ci,key in enumerate(wk,1): v=row.get(key,''); ws2.cell(r,ci,value='' if pd.isna(v) else v)
        kd=row.get('keyword_difficulty',0) or 0
        kc=ws2.cell(r,4)
        if kd<=10: kc.fill=gf
        elif kd<=20: kc.fill=yf
        elif kd<=30: kc.fill=rf
    ws2.freeze_panes='A2'; ws2.auto_filter.ref='A1:F'+str(len(df_win)+1)

    # Sheet 4: All Keywords
    ws3=wb.create_sheet('All Keywords')
    for ci,(h,w) in enumerate(wc,1): c=ws3.cell(1,ci,value=h); c.font=hf; c.fill=hb
    ws3.column_dimensions['A'].width=50; ws3.column_dimensions['B'].width=35
    for idx,row in df_all.iterrows():
        r=idx+2
        for ci,key in enumerate(wk,1): v=row.get(key,''); ws3.cell(r,ci,value='' if pd.isna(v) else v)
    ws3.freeze_panes='A2'

    # Sheet 5: Cost
    ws5=wb.create_sheet('Cost Log')
    for ci,h in enumerate(['Endpoint','Cost','Details'],1): c=ws5.cell(1,ci,value=h); c.font=hf; c.fill=hb
    for i,(ep,cost,det) in enumerate(cost_tracker.log,2):
        ws5.cell(i,1,value=ep); ws5.cell(i,2,value=round(cost,4)); ws5.cell(i,3,value=det)
    tr=len(cost_tracker.log)+3
    ws5.cell(tr,1,value='TOTAL').font=Font(bold=True)
    ws5.cell(tr,2,value=round(cost_tracker.spent,4)).font=Font(bold=True)

    wb.save(out)
    print('\n'+'='*60)
    print('  SEO Ultimate v6.3 Results')
    print('='*60)
    print('  Total keywords:     '+str(len(df_all)))
    print('  With volume:        '+str(len(df_all[df_all["search_volume"]>0])))
    print('  Winners (KD<'+str(MAX_KD)+'):  '+str(len(df_win)))
    print('  Winners w/coll:     '+str(win_with_coll)+' ('+str(round(win_with_coll/max(len(df_win),1)*100,1))+'%)')
    print('  KW matched to prod: '+str(len(kw_to_collections)))
    print('  Collections:        '+str(ri-2))
    print('  Products:           '+str(len(df_products)))
    print('  COST:               $'+str(round(cost_tracker.spent,2)))
    print('='*60)
    shutil.copy(out, f'{PROJECT_DIR}/{out}')
    colab_files.download(out)

    # Save to DB
    for orig, data in seo_collections.items():
        h = data.get('seo_handle', orig.lower().replace(' ','-'))
        try: db.execute("INSERT INTO collections (original_name,seo_title,seo_handle,full_url,meta_title,meta_description) VALUES (?,?,?,?,?,?)",
            (orig, data.get('seo_title',''), h, f'/collections/{h}', data.get('meta_title',''), data.get('meta_description','')))
        except: pass
    for pn in PRODUCTS:
        pr = db.execute('SELECT id FROM products WHERE name=?', (pn,)).fetchone()
        if not pr: continue
        for cn in final_product_colls.get(pn, []):
            cr = db.execute('SELECT id FROM collections WHERE original_name=?', (cn,)).fetchone()
            if cr:
                try: db.execute('INSERT OR IGNORE INTO product_collections VALUES (?,?)', (pr['id'], cr['id']))
                except: pass
    db.execute("INSERT OR REPLACE INTO seo_state (key,value) VALUES ('seo_complete','1')")
    db.commit()
    print(f'Saved {db.execute("SELECT COUNT(*) FROM collections").fetchone()[0]} collections to DB')

    print("SEO COMPLETE!")


SEO: SKIPPED (data already in DB)
  Collections: 20


## Cell 4 — Collections → Shopify

In [4]:
# ════ Collections → Shopify ════


import shutil

# ── Automatic PARENT CATEGORY assignment (mirrors
# pipeline/tools/assign_parent_categories.py) ──────────────────────────────
# Every collection the pipeline creates gets custom.parent_category set by the
# same keyword logic the bulk tool uses, so the homepage category hubs and the
# footer All-collections directory (both rendered from the live `collections`
# object) stay fully automatic — no hand-maintained link lists, ever.
_PARENT_FALLBACK = "More"
_PARENT_RULES = [
    ("Baby & Kids", ["baby", "infant", "newborn", "toddler", "nursery", "diaper",
        "stroller", "potty", "feeding", "pregnancy", "maternity", "kids", "kid ",
        "kid's", "boys", "boy's", "girls", "girl's", "childrens", "children's",
        "teen", "young adult", "school supplies", "classroom", "parenting"]),
    ("Foot & Leg Care", ["foot care", "foot health", "foot scrub", "foot massag",
        "foot detox", "detox foot", "foot pad", "foot patch", "callus",
        "pedicure", "feet", "leg care", "leg massag", "leg slimming", "calf",
        "thigh", "compression sock", "compression", "open toe", "open-toe"]),
    ("Joint & Muscle", ["joint", "arthritis", "knee", "muscle", "back pain",
        "neck pain", "shoulder pain", "sciatica", "tendon", "cramp",
        "bee venom", "pain relief", "pain patch", "relief patch", "heating pad"]),
    ("Massage & Relaxation", ["massage", "massager", "massagers", "massage gun",
        "massage tool", "relaxation", "acupressure", "shiatsu", "foam roller",
        "spa"]),
    ("Skin Care", ["skin care", "skincare", "facial", "face care", "face wash",
        "face mask", "moisturizer", "moisturizing", "moisturiz", "hydrating",
        "serum", "essence", "anti-aging", "anti aging", "wrinkle", "acne",
        "pimple", "blemish", "dark spot", "discoloration", "whitening",
        "brightening", "lightening", "scar", "wart", "mole", "skin tag",
        "cream", "lotion", "ointment", "balm", "vision care", "cleansing oil",
        "eye care", "skin treatment", "itch relief", "daily routine",
        "natural & organic", "natural organic"]),
    ("Bath & Body", ["bath", "shower", "soap", "body wash", "body scrub",
        "body care", "body lotion", "body oil", "body shaping", "body sculpting",
        "exfoliat", "scrubber", "scrub brush", "towel", "loofah", "bath bomb",
        "deodorant"]),
    ("Hair Care", ["hair", "scalp", "shampoo", "conditioner", "bonnet",
        "hair towel", "hair dry", "hair cap", "wig", "headband", "hair accessor"]),
    ("Makeup & Nails", ["makeup", "cosmetic", "lipstick", "lip gloss",
        "lip color", "lip care", "eyeliner", "eyeshadow", "mascara", "eyelash",
        "lash", "brow ", "concealer", "foundation", "palette", "nail",
        "manicure", "tattoo"]),
    ("Oral Care", ["oral care", "oral hygiene", "toothbrush", "toothpaste",
        "teeth", "tooth ", "dental", "denture", "tongue cleaner", "floss",
        "mouthwash"]),
    ("Intimate & Wellness", ["intimate", "feminine", "vaginal", "ph balanced",
        "sexual wellness", "libido", "testosterone", "male enhancement",
        "prostate", "breast", "butt care", "butt enhancement"]),
    ("Sleep & Wellness", ["sleep", "insomnia", "snore", "snoring",
        "anti-snoring", "wellness", "relaxation product", "aromatherapy",
        "stress relief", "anxiety", "meditation", "essential oil"]),
    ("Patches & Pads", ["patch", "patches", "kinoki", "lifewave", "transdermal"]),
    ("Supplements & Vitamins", ["supplement", "vitamin", "gummies", "gummy",
        "probiotic", "collagen", "magnesium", "multivitamin", "electrolyte",
        "dietary", "mineral", "capsule", "tablet", "pills", "pill", "drops",
        "tincture", "slimming tea", "detox tea", "slimming", "weight loss",
        "fat burn", "tea", "coffee"]),
    ("Health Devices", ["monitor", "blood pressure", "glucose", "thermometer",
        "test kit", "medical supplies", "medical supply", "first aid",
        "bandage", "wound", "light therapy", "red light", "infrared",
        "ear care", "ear wax", "nasal", "nose care", "throat",
        "health monitor", "wellness monitor", "personal care electronics",
        "oral care electronics", "health device"]),
    ("Men's", ["men's", "mens ", "shaving", "razor", "men "]),
    ("Women's Fashion", ["women's", "womens", "ladies", "beachwear", "swimwear",
        "bikini", "lingerie", "dress", "skirt", "blouse", "denim", "women "]),
    ("Clothing & Shoes", ["clothing", "apparel", "shoes", "sneakers", "boots",
        "sandals", "loafers", "pumps", "mules", "clogs", "athletic", "t-shirt",
        "shirt", "jacket", "coat", "pants", "uniform", "costume", "ties",
        "belt"]),
    ("Bags & Jewelry", ["bag", "bags", "luggage", "backpack", "wallet", "purse",
        "handbag", "tote", "crossbody", "clutch", "jewelry", "jewellery",
        "ring ", "necklace", "earring", "bracelet", "watch", "smartwatch",
        "eyewear", "sunglasses", "umbrella", "keyring", "keychain"]),
    ("Home & Kitchen", ["furniture", "home", "kitchen", "cookware", "bakeware",
        "drinkware", "glassware", "appliance", "vacuum", "laundry", "cleaning",
        "household", "decor", "wall art", "storage", "organization", "lighting",
        "light bulb", "bedroom", "living room", "patio", "bathroom",
        "nursery decor", "candle", "refrigerator", "fridge", "fan ", "heater",
        "air conditioner", "pest", "garden", "planter", "plant ", "plants",
        "seeds", "greenhouse", "outdoor holiday", "canopy", "gazebo", "bedding",
        "pond", "air quality", "holiday & party", "party supplies",
        "anti vibration", "led strip"]),
    ("Pet Supplies", ["pet", "dog ", "cat ", "puppy", "kitten", "bird",
        "fish ", "reptile", "horse", "small animal", "aquarium"]),
    ("Tech & Electronics", ["phone", "cell phone", "charger", "power bank",
        "cable", "adapter", "smart home", "smart device", "smart light", "wifi",
        "networking", "doorbell", "camera", "surveillance", "headphone",
        "earbud", "speaker", "electronic", "gadget", "computer", "laptop",
        "tablet ", "tech"]),
    ("Music & Instruments", ["musical instrument", "guitar", "drum",
        "percussion", "keyboard", "midi", "microphone", "stringed", "brass",
        "woodwind", "dj equipment", "studio recording", "instrument"]),
    ("Office & School", ["office", "school supplies", "writing", "stationery",
        "desk", "paper", "label", "tape", "adhesive", "greeting card",
        "gift wrap", "craft", "crafting", "beading", "packing", "shipping"]),
    ("Books & Media", ["book", "books", "media", "literature", "fiction",
        "comic", "graphic novel", "cookbook", "reference", "education",
        "teaching", "audiobook", "magazine"]),
    ("Sports & Outdoors", ["sports", "outdoor", "fitness", "exercise",
        "camping", "hiking", "fishing", "cycling", "bike", "pool", "hot tub",
        "adventure"]),
    ("Tools & Auto", ["hand tool", "power tool", "tools", "welding",
        "soldering", "automotive", "auto ", "car ", "oils & fluids",
        "maintenance", "repair", "hardware", "lighter", "utility tool",
        "electrical", "plumbing", "fastener", "building supplies", "industrial",
        "abrasive", "generator", "battery", "batteries", "power strip", "plug",
        "outlet", "3d printer", "material handling", "lab ", "scientific",
        "safety", "personal protective", "test, measure", "snow removal"]),
    ("Food & Grocery", ["food", "grocery", "snack", "beverage", "wine making",
        "coffee bean"]),
]

def _parent_for(title, handle):
    hay = f"{title or ''} {handle or ''}".lower().replace("-", " ")
    for parent, needles in _PARENT_RULES:
        for n in needles:
            if n in hay:
                return parent
    return _PARENT_FALLBACK

async def _set_parent_category(coll_gid, title, handle):
    """Idempotently write custom.parent_category for a collection gid. No-op if
    SHOPIFY_TOKEN is missing or the gid is falsy."""
    if not coll_gid or not SHOPIFY_TOKEN:
        return None
    parent = _parent_for(title, handle)
    try:
        r = await shopify_gql(
            "mutation($m:[MetafieldsSetInput!]!){metafieldsSet(metafields:$m)"
            "{metafields{id} userErrors{field message code}}}",
            {"m": [{"ownerId": coll_gid, "namespace": "custom",
                    "key": "parent_category", "type": "single_line_text_field",
                    "value": parent}]})
        errs = (((r or {}).get("data", {}) or {}).get("metafieldsSet", {}) or {}).get("userErrors", [])
        if errs:
            print(f"  ⚠ parent_category {handle}: {errs[:1]}")
        return parent
    except Exception as _e_pc:
        print(f"  ⚠ parent_category write failed for {handle}: {str(_e_pc)[:80]}")
        return None

# ── Auto-check: load Excel if no collections in DB ──
n_colls = db.execute("SELECT COUNT(*) FROM collections").fetchone()[0]
if n_colls == 0:
    print("⚠ No collections in DB! Loading from Excel...")
    from google.colab import files as colab_files
    import openpyxl
    print("Upload SEO Excel (.xlsx):")
    up = colab_files.upload()
    seo_f = list(up.keys())[0]
    wb = openpyxl.load_workbook(seo_f, read_only=True)
    cc = 0
    for row in wb['Shopify Collections'].iter_rows(min_row=2, values_only=True):
        st, h, mt, md_val, orig = (row[0] or ''), (row[1] or ''), (row[2] or ''), (row[3] or ''), (row[4] or '')
        try:
            db.execute("INSERT INTO collections (original_name,seo_title,seo_handle,full_url,meta_title,meta_description) VALUES (?,?,?,?,?,?)",
                (orig, st, h, f'/collections/{h}' if h else '', mt, md_val))
            cc += 1
        except: pass
    lc = 0
    for row in wb['Product Keywords'].iter_rows(min_row=2, values_only=True):
        pn, ct = (row[0] or ''), (row[1] or '')
        if not pn or not ct: continue
        pr = db.execute('SELECT id FROM products WHERE name=?', (pn,)).fetchone()
        cr = db.execute('SELECT id FROM collections WHERE seo_title=?', (ct,)).fetchone()
        if pr and cr:
            try: db.execute('INSERT OR IGNORE INTO product_collections VALUES (?,?)', (pr['id'], cr['id'])); lc += 1
            except: pass
    db.commit(); wb.close()
    print(f'{cc} collections, {lc} product links loaded')

# ── Link to Shopify ──
colls = db.execute("SELECT * FROM collections WHERE shopify_collection_id IS NULL").fetchall()
total_colls = db.execute("SELECT COUNT(*) FROM collections").fetchone()[0]

if not colls:
    print(f'All {total_colls} collections linked to Shopify.')
else:
    print(f'Linking {len(colls)} collections to Shopify...')
    found = 0; created = 0
    for c in colls:
        handle = c['seo_handle']
        title = c['seo_title']
        r = await shopify_gql('query($h:String!){collectionByHandle(handle:$h){id title}}', {"h": handle})
        coll = r.get('data',{}).get('collectionByHandle') if r else None
        if coll:
            db.execute('UPDATE collections SET shopify_collection_id=? WHERE id=?', (coll['id'], c['id']))
            db.commit(); found += 1
            await _set_parent_category(coll['id'], title, handle)
            print(f'  FOUND: {title[:45]}')
        else:
            cid = await shopify_create_collection(title, handle, c['meta_title'], c['meta_description'])
            if cid:
                db.execute('UPDATE collections SET shopify_collection_id=? WHERE id=?', (cid, c['id']))
                db.commit(); created += 1
                _pc = await _set_parent_category(cid, title, handle)
                print(f'  CREATED: {title[:45]}  [parent: {_pc}]')
            else:
                print(f'  FAIL: {title[:45]}')
        await asyncio.sleep(0.3)
    linked = db.execute("SELECT COUNT(*) FROM collections WHERE shopify_collection_id IS NOT NULL").fetchone()[0]
    print(f'\nResult: {linked}/{total_colls} (found {found}, created {created})')

# ── Cross-collection SEO interlinking ──
# After all collections are in Shopify, compute a related-collections graph
# with reciprocity (A↔B) and hub detection (high in-degree nodes). Result is
# written to custom.related_collections per collection, AND a 1-line inline-
# mention paragraph is appended to each collection's bodyHtml (with varied
# anchor text on 2 top related). This fills the "collection → collection"
# SEO gap — each category page now links laterally instead of being a dead end.
if SHOPIFY_TOKEN:
    _ci_rows = db.execute(
        "SELECT id, original_name, seo_title, seo_handle, shopify_collection_id, "
        "meta_title, meta_description, top_keywords, total_volume "
        "FROM collections WHERE shopify_collection_id IS NOT NULL"
    ).fetchall()
    if len(_ci_rows) >= 2:
        print(f'\nCross-collection interlinking: computing graph over {len(_ci_rows)} collections...')
        _related_map = _compute_collection_related(_ci_rows, k=6, cap=8)
        _n_mf = 0
        _n_inline = 0
        _n_skip = 0
        _hub_count = sum(1 for items in _related_map.values()
                         for it in items[:1] if it.get('is_hub'))
        for _ci_c in _ci_rows:
            _ci_handle = _ci_c['seo_handle']
            _ci_sid = _ci_c['shopify_collection_id']
            _ci_related = _related_map.get(_ci_handle, [])
            if not _ci_related or not _ci_sid:
                _n_skip += 1
                continue
            # Write custom.related_collections metafield
            try:
                _ci_mq = ('mutation($m:[MetafieldsSetInput!]!){metafieldsSet(metafields:$m)'
                          '{metafields{id} userErrors{field message code}}}')
                _ci_r = await shopify_gql(_ci_mq, {"m": [{
                    "ownerId": _ci_sid,
                    "namespace": "custom",
                    "key": "related_collections",
                    "type": "json",
                    "value": json.dumps(_ci_related, ensure_ascii=False),
                }]})
                _ci_errs = (((_ci_r or {}).get('data', {}) or {})
                            .get('metafieldsSet', {}) or {}).get('userErrors', [])
                if _ci_errs:
                    print(f'  \u26a0 mf {_ci_handle}: {_ci_errs[:1]}')
                else:
                    _n_mf += 1
            except Exception as _e_mf:
                print(f'  \u26a0 related_collections write failed for {_ci_handle}: {str(_e_mf)[:80]}')
                continue
            # Append inline-mention paragraph (idempotent — check sentinel class)
            _ci_top2 = _ci_related[:2]
            if len(_ci_top2) >= 2:
                _ci_inline = ('<p class="wanelo-coll-also">Browse related: '
                              f'<a href="/collections/{_ci_top2[0]["handle"]}">{_ci_top2[0]["anchor"]}</a> &middot; '
                              f'<a href="/collections/{_ci_top2[1]["handle"]}">{_ci_top2[1]["anchor"]}</a>.</p>')
                try:
                    _r_body = await shopify_gql(
                        'query($id:ID!){collection(id:$id){descriptionHtml}}', {"id": _ci_sid})
                    _existing = (((_r_body or {}).get('data', {}) or {})
                                 .get('collection', {}) or {}).get('descriptionHtml', '') or ''
                    if 'wanelo-coll-also' not in _existing:
                        _new_body = _existing + '\n' + _ci_inline
                        await shopify_gql(
                            'mutation($i:CollectionInput!){collectionUpdate(input:$i){collection{id} userErrors{field message}}}',
                            {"i": {"id": _ci_sid, "descriptionHtml": _new_body}})
                        _n_inline += 1
                except Exception as _e_inline:
                    print(f'  \u26a0 inline-mention append failed for {_ci_handle}: {str(_e_inline)[:80]}')
            await asyncio.sleep(0.2)  # rate limit
        print(f'  Cross-link: {_n_mf} metafields written, {_n_inline} inline-mentions appended, {_n_skip} skipped')
        print(f'  Hubs detected (high in-degree nodes): {_hub_count}')

# Verify product-collection links
pc = db.execute("SELECT COUNT(*) FROM product_collections").fetchone()[0]
print(f'Product-collection links: {pc}')
shutil.copy(DB_LOCAL, DB_PATH)


Linking 20 collections to Shopify...
  FOUND: Body Scrubbers – Exfoliating Gloves & Dry Ski
  FOUND: Handmade Moisturising Soap – Nourishing Hand 
  FOUND: Baby Bath Essentials – Gentle Shampoo & Showe
  FOUND: Silicone Body Brushes – Gentle Face & Body Sc
  FOUND: Hair Drying Caps – Quick Dry Microfiber Towel
  FOUND: Natural Body Scrubs – Honey, Almond & Strawbe
  FOUND: Scalp Brushes – Shampoo Scrubber for a Deep C
  FOUND: Premium Bath Towels – Luxury Spa-Quality Show
  FOUND: Bath Bombs – Fizzy Bubble Bath Products for R
  FOUND: Baby Oral Care – Soft Bristle Infant & Newbor
  FOUND: Electric Toothbrushes – Sonic Powered for a D
  FOUND: Satin Hair Bonnets – Sleep Caps for Natural &
  FOUND: Shower Foot Scrubbers – Brushes & Cleaners fo
  FOUND: Pedicure Tools – Professional Callus Removers
  FOUND: Eyelash Tools – Tweezers, Curlers & Lash Appl
  FOUND: Anti Vibration Pads for Washer & Washing Mach
  FOUND: Spa Headbands & Bandana Headbands for Skincar
  FOUND: Odor Removers for L

'/content/drive/MyDrive/shopify_pipeline/pipeline.db'

## Cell 5 — CSS Framework (v9)

In [5]:
# === STRATEGY AGENT prompt (Phase 1 — runs BEFORE content designer) ===
STRATEGY_SYSTEM_PROMPT = """You are a senior direct-response copywriter and brand strategist with 15 years of experience launching e-commerce products. Your job: define the MARKETING STRATEGY for this product BEFORE any copy or design is generated.

Two principles:
1. Every product solves a SPECIFIC PAIN. Identify it precisely.
2. Buyers don't buy products — they buy TRANSFORMATIONS. Articulate before-state vs. after-state.

OUTPUT — single JSON object, no preamble, no markdown fences. Begin with {.

{
  "seo_title": "Storefront product title + <title> tag, 50-60 chars. See TITLE/META/HERO RULES below.",
  "seo_description": "Google SERP meta description, 140-155 chars. See RULES below.",
  "hero": {"kicker": "2-3 word product-specific tease", "h1": "the ONE promise = selling_idea, with <em>...</em> on 1-2 emotional words", "lead": "1-2 sentences echoing the transformation (before to after)"},
  "positioning": "Who buys this in one specific sentence (named persona, not 'everyone'). E.g. 'Cat owners 28-45 with 1-2 indoor cats who feel guilty their pet is bored all day.'",
  "pain_point": "The acute, visceral problem. Concrete, real, felt daily.",
  "current_solutions": "What they currently buy/do that fails them, and why.",
  "transformation": {
    "before": "Their day-to-day pain in 1 sentence",
    "after":  "Their day-to-day after this product in 1 sentence"
  },
  "key_emotion": "ONE word: relief / pride / belonging / safety / curiosity / nostalgia / guilt-removal / desire / etc.",
  "selling_idea": "ONE sentence: the central concept that frames the entire product page. E.g. 'Twenty minutes of stalking equals an evening of peace.'",
  "buying_trigger": "The exact moment/feeling that pushes them to add to cart.",
  "key_objections": ["Top 3 reasons they MIGHT NOT buy"],
  "narrative_arc": [
    "Beat 1: empathize with the pain",
    "Beat 2: tension — what happens if pain continues",
    "Beat 3: discovery — product introduced",
    "Beat 4: proof — why it works",
    "Beat 5: transformation — life after",
    "Beat 6: urgency — why now",
    "Beat 7: resolution — buyer feels safe to commit"
  ],
  "differentiator": "What makes THIS product the clear choice vs alternatives. ONE sentence.",
  "voice": "Tone of voice: 'warm-confidant' | 'witty-irreverent' | 'clinical-precise' | 'editorial-thoughtful' | 'aspirational-luxury' | 'down-to-earth-honest'. Pick ONE.",
  "research_refs": {
    "head": {
      "kicker": "BACKED BY SCIENCE",
      "h2": "Published research on [product topic]"
    },
    "items": [
      {"title": "<paper title>", "source": "<journal name>", "year": <int or null>, "url": "<verified URL>"}
    ]
  },
    "unboxing_journey": {
    "head": {"kicker": "WHAT YOU'LL OPEN", "h2": "..."},
    "steps": [
      {"step_number": 1, "title": "...", "description": "20-40 words sensory description of what they see/feel at this step"}
    ]
  },
  "progress_milestones": {
    "head": {"kicker": "YOUR JOURNEY", "h2": "..."},
    "milestones": [
      {"timeframe": "Day 1 | Week 1 | Week 4 | Week 8 OR First Use | 3 Uses | 30 Days | 90 Days", "label": "4-6 word headline", "expected_result": "20-30 words HONEST — no invented percentages"}
    ]
  },
  "mistake_warnings": {
    "head": {"kicker": "DON'T DO THIS", "h2": "..."},
    "warnings": [
      {"mistake": "one short wrong-action sentence", "why_problem": "factual 15-25 words", "instead_do": "correct action 10-20 words"}
    ]
  },
  "use_scenarios": {
    "head": {"kicker": "WHEN TO REACH FOR IT", "h2": "..."},
    "scenarios": [
      {"scenario": "short context name", "when": "one-line timing/context", "how_to_use": "15-25 word micro-protocol for THIS context", "key_benefit": "outcome for this scenario"}
    ]
  },
"pinterest_pins": [
    {
      "variation_id": "A",
      "variation_intent": "what hypothesis this pin tests — DIFFERENT per variation",
      "category_profile": "beauty|supplements|fitness|kitchen|fashion|decor|tech|baby|pet|other",
      "format": "before_after_mechanism|comparison_chart|decoded_science|listicle_insider|flatlay_editorial|process_steps",
      "hook_headline": "5-8 word save-worthy headline",
      "hook_subhead": "8-12 word italic-serif subtitle",
      "demonstration": {
        "type": "before_after|mechanism_diagram|stat_comparison|process_steps|ingredient_breakdown|size_guide|comparison_table",
        "details_for_image": "very specific visual instructions",
        "labels": ["3-5 short overlay labels"]
      },
      "stats_to_use": [{"label": "...", "value": "...", "context": "...", "source": "..."}],
      "mood_palette": ["bg color", "text color", "accent color"],
      "pin_voice": "investigative|personal_blog|magazine_editorial|scientific_explainer|listicle",
      "brand_attribution": "wanelo.com · saved Nk"
    }
  ]
}

TITLE / META / HERO RULES (these become the customer-facing storefront title, the
Google snippet, and the page hero — craft them like a senior marketplace SEO):
- seo_title: <recognizable product-type noun> + <strongest benefit>. Primary keyword
  in the first ~5 words (it is ALSO the <title> tag). Title Case, 50-60 chars, UNIQUE.
  FORBIDDEN: ALL-CAPS, '2024/2025 New', 'Hot Sale', emoji, spec dumps, keyword stuffing.
  Do NOT append the store/brand name (the storefront adds it).
- seo_description: primary keyword + core benefit in the FIRST ~120 chars (mobile +
  Google bolds query matches = higher CTR); active voice; one proof/curiosity hook;
  soft pull ('See why...'); UNIQUE; no price, no shipping, no exclamation marks.
- hero.h1: express selling_idea as ONE promise (the transformation/outcome, not a spec),
  with <em>...</em> on 1-2 emotional words. hero.lead: 1-2 sentences, before to after.

CRITICAL RULES:
- Be HONEST. Generic dropship gear gets honest positioning, not fake premium.
- Be SPECIFIC. Named persona, not demographic bracket.
- Pull facts from VISION ANALYSIS — never invent specs.
- Think first about who has the SHARPEST pain that this product solves. That's the buyer.
- Output ONLY the JSON. No prose, no markdown.

RESEARCH_REFS — 3-4 REAL peer-reviewed links per product (NOT fake press
mentions). FTC-safe because links go to scientific literature, not endorsements
of THIS product. Picks per category:

  beauty/skincare/cosmetic → PubMed Central (PMC) papers on the active
    ingredient or mechanism. e.g. microneedling: PMC4976400. e.g. retinol:
    PMC3673342. Also: aad.org cosmetic pages, Mayo Clinic, Cleveland Clinic.
  supplements/wellness/vitamins → Examine.com summary, NIH Office of Dietary
    Supplements (ods.od.nih.gov), Linus Pauling Institute Micronutrient Info.
  fitness/equipment → NSCA/ACSM journals, PubMed sports medicine RCTs,
    Harvard T.H. Chan exercise epidemiology.
  kitchen/appliances → USDA Food Safety, Harvard Nutrition Source, Cook's
    Illustrated technique articles.
  fashion → material-science abstracts (Nature Materials, PubMed), Tide /
    Wool / Leather care council guides.
  decor/home/lighting → IES lighting guides, color-psychology research.
  tech/gadgets → IEEE Xplore, manufacturer spec sheets, Wirecutter testing
    methodology pages.
  baby/kids → aap.org (American Academy of Pediatrics), CDC parenting,
    CPSC safety standards.
  pet → ASPCA, AVMA care guides, veterinary medicine PubMed.

CRITICAL RESEARCH_REFS RULES:
  • URLs MUST be REAL and VERIFIED. If unsure, OMIT the item rather than
    invent. Better 2 verified than 4 broken/fake — Liquid renders only
    items with a URL.
  • Use ONLY well-known stable sources (PMC, AAD, NIH, Mayo, AAP, IEEE).
    NO blog spam / Medium / random domains.
  • Item shape: {title, source: "<journal/site name>", year: int|null, url}
  • head.h2 mentions the product topic (e.g. "Published research on
    micro-needling", "Studies on collagen peptides").
  • head.kicker stays "BACKED BY SCIENCE" or "REFERENCES" — short caps
    authority signal.
  • If product category is "other" or you have no high-confidence sources:
    emit research_refs with EMPTY items[] — snippet hides on storefront.

PINTEREST_PINS — 3 distinct save-worthy pin variations, NOT direct ads:

GOAL: Reader saves the pin even if they never buy. Pin gives free value
(research, comparison, education). Brand link is whispered at the bottom.

CATEGORY → FORMAT mapping (pick exactly ONE format based on the product's
CATEGORY field, then design the demonstration to match):

  beauty/skincare/cosmetic → "before_after_mechanism"
    Required: 2 oval vignettes (week 1 vs week 8 of the relevant body part)
    + anatomical/mechanism diagram showing HOW the product affects the body
    (skin cross-section + needle depth + ingredient flow, etc.).

  supplements/wellness/vitamins → "decoded_science"
    Required: in-body timeline of effects (day 1 → day 30 milestones)
    + ingredient transparency table (active dose + form + bioavailability).

  fitness/equipment/yoga → "process_steps" OR "comparison_chart"
    Required: correct form vs wrong form OR muscle activation diagram OR
    rep-count/weight progression chart over 30 days.

  kitchen/appliances/cookware → "process_steps"
    Required: recipe-card aesthetic with 4-dish gallery
    + "what you can make" + time saved vs traditional method.

  fashion/clothing/accessories → "comparison_chart"
    Required: 4 outfit variations grid OR styling guide by body type/occasion.

  decor/home/lighting → "before_after_mechanism"
    Required: room before vs after photos + scale/material specifics
    (dimensions, what surfaces it works on, etc.).

  tech/gadgets/electronics → "comparison_chart"
    Required: spec table vs the leading competitor + workflow before/after
    (a task that took 30min now takes 3min).

  baby/kids/toys → "listicle_insider"
    Required: age-milestone alignment + safety badges row
    + 3-5 numbered tips parents wish they knew.

  pet/animal → "process_steps"
    Required: routine timeline (week 1-4) + breed/size guide
    + visible effect (coat shine / behavior / cleanliness).

HOOK HEADLINE RULES:
  - First 3-5 words MUST stop scroll. Insider / investigative / personal angle.
  - GOOD examples (steal these patterns):
      "I tested this for 8 weeks"
      "The 7 things estheticians won't tell you"
      "What $400 facials actually do"
      "The depth that changes everything"
      "Decoded: my 5-step night routine"
      "30 days. tracked everything."
  - BAD (Pinterest flags as sales):
      "Best skincare 2026"
      "Buy the kit today!"
      "50% off this week"
      "Limited time offer"

STATS RULES (most violated, double-check):
  - Use ONLY stats from VISION packaging_text.claims OR product.description.
  - NEVER invent percentages. If no number exists, use generic supportable
    language instead: "targets fine lines" not "47% reduction in fine lines".
  - Source citation is OPTIONAL — if you have one, cite softly
    ("from independent consumer study") otherwise OMIT the source field.
  - Stats array CAN be empty if no real numbers exist.

VOICE matching:
  beauty/wellness → "personal_blog" or "magazine_editorial"
  fitness/tech → "investigative" or "scientific_explainer"
  fashion/decor → "magazine_editorial"
  baby/pet → "personal_blog"
  kitchen → "magazine_editorial" or "listicle"

MOOD PALETTE — pick 3 colors max (background + primary text + one accent):
  beauty: ["cream", "deep navy", "sage green"] or ["dusty pink", "warm charcoal", "terracotta"]
  fitness: ["charcoal", "off-white", "neon coral"]
  kitchen: ["warm wood beige", "deep brown", "burnt orange"]
  fashion: ["off-white", "deep navy", "ONE bold accent from product"]
  decor: ["muted sage", "warm charcoal", "soft brass"]
  tech: ["dark gray", "off-white", "electric blue"]
  baby: ["soft cream", "muted gray", "dusty rose"]
  pet: ["warm golden cream", "earthy brown", "soft caramel"]

BRAND ATTRIBUTION:
  Always bottom 5-8% of pin only. Whispered tone. Format examples:
    "find this on wanelo.com"
    "wanelo.com · saved 2.3k times"
    "wanelo.com/[product-handle]"
  NEVER big logos. NEVER "BUY NOW" buttons. The pin sells through
  aesthetic + implicit recommendation, not direct pitch.""" + HTML_CLEAN_OUTPUT_RULES


# === VISION SYSTEM PROMPT (Haiku 4.5 — runs Step 3 per product) ===
# Static schema lives here as a system prompt so cache_control:ephemeral
# kicks in. Without this it sat in user content with interpolated title,
# making every call a fresh input — cost + latency hit on 10K batches.
VISION_SYSTEM_PROMPT = """You analyse product images and return ONE strictly-valid JSON object describing what you see and what helps sell the product. The product title will be given in the user message. Return ONLY JSON (no markdown fences, no prose). Begin with { end with }.

Schema (fill every field; use empty string or empty array if not visible):
{
  "accent": "#hex pastel color from product",
  "accent_soft": "#hex 10% opacity version",
  "bg": "#FFFFFF",
  "palette": {
    "brand_1": "#hex primary warm accent (e.g. from packaging hero color)",
    "brand_2": "#hex secondary cool accent (complementary to brand_1)",
    "brand_3": "#hex tertiary accent (analogous to brand_1)",
    "brand_soft": "#hex 10% lightened brand_1 (for soft backgrounds)",
    "brand_deep": "#hex 20% darkened brand_1 (for emphasis)"
  },
  "category": "product category",
  "product_type": "simple|complex|technical|fashion|decorative",
  "product_summary": "2-3 engaging sentences about what this product does",
  "needs_steps": true,
  "packaging_text": {
    "volume": "size/weight if visible (e.g. 30ml, 50g) or empty",
    "claims": ["exact claims visible: 24h hydration, SPF 50, etc"],
    "ingredients_visible": ["key ingredients readable on label"],
    "certifications": ["logos/text: cruelty-free, vegan, organic, etc"],
    "instructions_visible": "usage directions if readable, else empty"
  },
  "sensory": {
    "texture": "matte|glossy|shimmer|cream|gel|powder|liquid|foam|oil",
    "material": "glass|plastic|metal|wood|silicone|ceramic|fabric or empty",
    "finish_description": "short vivid phrase: frosted glass with gold pump",
    "color_names": ["descriptive names: dusty rose, champagne gold"],
    "perceived_quality": "budget|mid-range|premium|luxury"
  },
  "physical": {
    "size_impression": "travel-size|compact|standard|full-size|jumbo",
    "key_features": ["visible features: pump, magnetic closure, built-in mirror"],
    "whats_included": ["all visible items: main product, brush, case"],
    "applicator_type": "brush|sponge|dropper|pump|spray|tube|jar or empty"
  },
  "conversion": {
    "hero_claim": "single strongest selling proposition visible or inferred",
    "pain_points_solved": ["specific problems: uneven skin, dry lips"],
    "visible_results": "describe any before/after or visible effect",
    "premium_signals": ["what makes it look premium: glass bottle, gold accents"],
    "target_audience": "specific: young women into K-beauty, men with beard care",
    "usage_scenario": "when/where: daily morning routine, date night, travel",
    "emotional_tone": "luxury|clinical|fun|natural|professional|playful"
  },
  "images": [
    {"index": 1, "role": "hero|feature|detail|lifestyle|ingredient|before_after|packaging|skip", "desc": "ALT text", "shows": "what this image proves about the product"}
  ]
}
"""



CSS Framework: 4013 chars
CSS Mode: inline
Trust signals prompt: ready
Cell 5 OK


## Cell 5.5 — Taxonomy Loader (v3.2)

Loads master taxonomy from Git URL, builds FAISS index for semantic shortlist, exposes `get_cluster_shortlist()` and `validate_tags()` to the product loop.

In [ ]:
# ════ TAXONOMY LOADER (v3.2) ════
# Loads master taxonomy from Git URL, indexes in DB + FAISS.
# Provides: get_cluster_shortlist(), validate_tags(), TAX_PERSONAS, TAX_INTENTS

import json, os, requests
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# ── Fetch from URL with local cache ──
TAX_LOCAL = f"{PROJECT_DIR}/taxonomy.json"
_gh_headers = {}
if os.environ.get("GITHUB_TOKEN"):
    _gh_headers["Authorization"] = f"Bearer {os.environ['GITHUB_TOKEN']}"
if TAXONOMY_REFRESH or not os.path.exists(TAX_LOCAL):
    print(f"Fetching taxonomy from {TAXONOMY_URL}...")
    r = requests.get(TAXONOMY_URL, headers=_gh_headers, timeout=30)
    if r.status_code == 404:
        raise RuntimeError(
            f"taxonomy.json не найден. Проверь:\n"
            f"  - GITHUB_BRANCH ({GITHUB_BRANCH!r}) — ветка существует и содержит taxonomy/taxonomy.json?\n"
            f"  - Репо приватный? Поставь GITHUB_TOKEN в env (или сделай репо public).\n"
            f"URL: {TAXONOMY_URL}"
        )
    r.raise_for_status()
    with open(TAX_LOCAL, "w", encoding="utf-8") as f:
        f.write(r.text)
    print(f"  Saved to {TAX_LOCAL}")
else:
    print(f"Using cached taxonomy: {TAX_LOCAL}")

with open(TAX_LOCAL, encoding="utf-8") as f:
    TAXONOMY = json.load(f)

print(f"Taxonomy v{TAXONOMY['version']}: "
      f"{TAXONOMY['stats']['total_clusters']} clusters, "
      f"{TAXONOMY['stats']['personas']} personas, "
      f"{TAXONOMY['stats']['intents']} intents")

# ── Filter approved + (optionally) drafts ──
_status_filter = {"approved"} if TAXONOMY_APPROVED_ONLY else {"approved", "draft"}
_clusters = [c for c in TAXONOMY["clusters"] if c["status"] in _status_filter]
print(f"Active clusters: {len(_clusters)} (status: {sorted(_status_filter)})")

# ── Index in DB ──
db.executescript("""
CREATE TABLE IF NOT EXISTS taxonomy_clusters (
    tag TEXT PRIMARY KEY,
    section_id TEXT, section TEXT, title_en TEXT, title_ru TEXT,
    typical_products TEXT, personas TEXT, intents TEXT,
    primary_gender TEXT, ages TEXT, status TEXT, data_json TEXT
);
CREATE INDEX IF NOT EXISTS idx_tax_section ON taxonomy_clusters(section_id);
CREATE INDEX IF NOT EXISTS idx_tax_status ON taxonomy_clusters(status);
""")
db.execute("DELETE FROM taxonomy_clusters")
for c in _clusters:
    demos = c.get("demos") or {}
    db.execute("""INSERT INTO taxonomy_clusters
        (tag, section_id, section, title_en, title_ru, typical_products,
         personas, intents, primary_gender, ages, status, data_json)
        VALUES (?,?,?,?,?,?,?,?,?,?,?,?)""",
        (c["tag"], c["section_id"], c["section"], c["title_en"], c.get("title_ru",""),
         json.dumps(c.get("typical_products",[]), ensure_ascii=False),
         json.dumps(c.get("personas",[])),
         json.dumps(c.get("intents",[])),
         demos.get("primary_gender",""),
         json.dumps(demos.get("ages",[])),
         c["status"],
         json.dumps(c, ensure_ascii=False)))
db.commit()

# ── FAISS index for semantic shortlist ──
print("Building FAISS index...")
_embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
_texts = [c["embed_text"] for c in _clusters]
_vectors = _embedder.encode(_texts, show_progress_bar=False, normalize_embeddings=True).astype("float32")
_faiss_index = faiss.IndexFlatIP(_vectors.shape[1])
_faiss_index.add(_vectors)

TAX_TAGS_LIST = [c["tag"] for c in _clusters]
TAX_BY_TAG = {c["tag"]: c for c in _clusters}
TAX_PERSONAS = [p["tag"] for p in TAXONOMY["personas"]]
TAX_INTENTS = [i["tag"] for i in TAXONOMY["intents"]]
TAX_GENDERS = [d["tag"] for d in TAXONOMY["demos"] if d["type"] == "gender"]
TAX_AGES = [d["tag"] for d in TAXONOMY["demos"] if d["type"] == "age"]
TAX_VALID_TAGS = set(TAX_TAGS_LIST) | set(TAX_PERSONAS) | set(TAX_INTENTS) | set(TAX_GENDERS) | set(TAX_AGES)

print(f"FAISS index: {len(_clusters)} vectors")

# ── Helper: shortlist for a product ──
def get_cluster_shortlist(product_summary, category, top_k=40):
    """Return top-K candidate clusters by semantic similarity to product."""
    query = f"{category} {product_summary}".strip() or "product"
    qv = _embedder.encode([query], normalize_embeddings=True).astype("float32")
    D, I = _faiss_index.search(qv, min(top_k, len(_clusters)))
    return [TAX_BY_TAG[TAX_TAGS_LIST[i]] for i in I[0]]

def format_shortlist_for_prompt(shortlist):
    """Format shortlist as compact text for Stage 2 prompt."""
    lines = ["AVAILABLE CLUSTERS (pick 2-4 from this list ONLY — never invent):"]
    for c in shortlist:
        prods = ", ".join(c.get("typical_products", [])[:4])
        lines.append(f"  {c['tag']} — {c['title_en']} (e.g. {prods})")
    lines.append("")
    lines.append(f"PERSONAS (pick 1-2): {', '.join(TAX_PERSONAS)}")
    lines.append(f"INTENTS (pick 1-2): {', '.join(TAX_INTENTS)}")
    lines.append(f"DEMO GENDER (pick exactly 1): {', '.join(TAX_GENDERS)}")
    lines.append(f"DEMO AGES (pick 1-2): {', '.join(TAX_AGES)}")
    return "\n".join(lines)

def validate_tags(picked_tags):
    """Drop tags not in master taxonomy. Returns (valid_tags, dropped_tags)."""
    valid = [t for t in picked_tags if t in TAX_VALID_TAGS]
    dropped = [t for t in picked_tags if t not in TAX_VALID_TAGS]
    return valid, dropped

print("✓ Taxonomy loader ready: get_cluster_shortlist, validate_tags, format_shortlist_for_prompt")


## Cell 6 — Product Loop v9 (resumable)

In [ ]:
# ════ PRODUCT LOOP v9 ════


import shutil

# ── Dynamic schema awareness across the batch ──
# When Designer emits a NEW key (not in hardcoded schema) for product N,
# we cache its sample value so product N+1's Designer sees it as an
# "additional available key" with shape reference. Persisted to disk so
# new runs immediately know about previously-discovered keys.

_DYNAMIC_SCHEMAS_PATH = os.path.join(os.path.dirname(DB_LOCAL), 'dynamic_schemas.json')
_AUDIT_LOG_PATH = os.path.join(os.path.dirname(DB_LOCAL), 'metafield_audit.jsonl')
def _load_dynamic_schemas() -> dict:
    """Bootstrap from disk. Empty dict on first run."""
    try:
        if os.path.exists(_DYNAMIC_SCHEMAS_PATH):
            with open(_DYNAMIC_SCHEMAS_PATH, 'r', encoding='utf-8') as f:
                return json.load(f)
    except Exception as _e:
        print(f'  dynamic_schemas.json load failed: {_e} (starting empty)')
    return {}

def _save_dynamic_schemas(d: dict) -> None:
    """Persist after each product. Atomic write via tmp+rename."""
    try:
        _tmp = _DYNAMIC_SCHEMAS_PATH + '.tmp'
        with open(_tmp, 'w', encoding='utf-8') as f:
            json.dump(d, f, ensure_ascii=False, indent=2)
        os.replace(_tmp, _DYNAMIC_SCHEMAS_PATH)
    except Exception as _e:
        print(f'  dynamic_schemas.json save failed: {_e}')

def _audit_log(record: dict) -> None:
    """Append-only audit record for any auto-created metafield event.
    Use cases: operator greps to find candidates for dedicated snippets,
    or to spot drift / spam. JSONL so it's tail-friendly."""
    try:
        record = {**record, 'ts': __import__('datetime').datetime.utcnow().isoformat() + 'Z'}
        with open(_AUDIT_LOG_PATH, 'a', encoding='utf-8') as f:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')
    except Exception:
        pass  # audit log is best-effort, never blocks pipeline

def _shape_fingerprint(value) -> tuple:
    """Stable fingerprint of a JSON value's shape (not content). Used to
    detect drift: if same key gets different shape across products, log warn
    and keep the cached/canonical version."""
    if isinstance(value, dict):
        return ('dict', tuple(sorted(value.keys())))
    if isinstance(value, list):
        if not value:
            return ('list', 'empty')
        # Fingerprint based on first item's shape (representative)
        return ('list', _shape_fingerprint(value[0]))
    return (type(value).__name__,)

_DYNAMIC_DESIGNER_SCHEMAS = _load_dynamic_schemas()
if _DYNAMIC_DESIGNER_SCHEMAS:
    print(f'Dynamic schemas bootstrapped: {len(_DYNAMIC_DESIGNER_SCHEMAS)} keys from previous runs: {list(_DYNAMIC_DESIGNER_SCHEMAS.keys())[:8]}')

# Hardcoded schema keys Designer already knows from its system prompt.
# Anything NOT in this set + isinstance(dict, list) + non-empty → "new key".
_DESIGNER_HARDCODED_KEYS = {
    'hero', 'story', 'features', 'stats', 'reviews', 'faq', 'cta', 'palette',
    'interlinks', 'how_to', 'specs', 'whats_included', 'ingredients',
    'timeline', 'trust', 'compare', 'size_guide', 'care', 'dimensions',
    'variants', 'gift_options', 'safety', 'protocol', 'target_profile',
    'video_demo', 'compatibility', 'assembly', 'brand_story',
    'lifestyle_gallery', 'nutrition_facts', 'clinical_evidence',
    'app_showcase', 'room_placement', 'subscription_refill',
    'sustainability', 'unboxing_journey', 'progress_milestones',
    'mistake_warnings', 'use_scenarios',
    # internal output keys (not metafields)
    'meta', 'photo_briefs', 'visual_style',
}

# ── Fetch ALL collections from Shopify for interlinking ──
# Paginate through all (was capped at first 100 — missed 4× catalog).
# Include productsCount so we can filter out empty collections — linking
# to /collections/X where X has 0 products = user lands on blank page.
ALL_SHOP_COLLECTIONS = []
_total_seen = 0
_cursor = None
try:
    while True:
        _r = await shopify_gql(
            'query($c:String){collections(first:100, after:$c){'
            'pageInfo{hasNextPage endCursor} '
            'nodes{id title handle productsCount{count}}}}',
            {'c': _cursor},
        )
        _page = ((_r or {}).get('data') or {}).get('collections') or {}
        _nodes = _page.get('nodes') or []
        _total_seen += len(_nodes)
        # Keep only collections with >= 1 product (empty ones = 404-page UX)
        for _n in _nodes:
            if (_n.get('productsCount') or {}).get('count', 0) >= 1:
                ALL_SHOP_COLLECTIONS.append(_n)
        if not (_page.get('pageInfo') or {}).get('hasNextPage'):
            break
        _cursor = _page['pageInfo']['endCursor']
    _empty_n = _total_seen - len(ALL_SHOP_COLLECTIONS)
    print(f'Shopify collections for interlinking: {len(ALL_SHOP_COLLECTIONS)} non-empty (filtered {_empty_n} empty out of {_total_seen} total)')
except Exception as _e_coll:
    print(f'Could not fetch Shopify collections: {str(_e_coll)[:80]}')

# ── v9.1: Build rich vision context for writer ──
def _build_vision_context(s1):
    """Format all vision data into a structured prompt section for the writer."""
    lines = []

    # Packaging text (read from labels)
    pkg = s1.get('packaging_text', {})
    if pkg.get('volume'):
        lines.append(f"VOLUME: {pkg['volume']}")
    if pkg.get('claims'):
        lines.append(f"CLAIMS ON PACKAGING: {', '.join(pkg['claims'])}")
    if pkg.get('ingredients_visible'):
        lines.append(f"KEY INGREDIENTS (visible on label): {', '.join(pkg['ingredients_visible'])}")
    if pkg.get('certifications'):
        lines.append(f"CERTIFICATIONS: {', '.join(pkg['certifications'])}")
    if pkg.get('instructions_visible'):
        lines.append(f"USAGE INSTRUCTIONS: {pkg['instructions_visible']}")

    # Sensory details
    sns = s1.get('sensory', {})
    sensory_parts = []
    if sns.get('texture'): sensory_parts.append(f"texture: {sns['texture']}")
    if sns.get('material'): sensory_parts.append(f"material: {sns['material']}")
    if sns.get('finish_description'): sensory_parts.append(sns['finish_description'])
    if sensory_parts:
        lines.append(f"LOOK & FEEL: {', '.join(sensory_parts)}")
    if sns.get('color_names'):
        lines.append(f"COLORS: {', '.join(sns['color_names'])}")
    if sns.get('perceived_quality') and sns['perceived_quality'] != 'mid-range':
        lines.append(f"PERCEIVED QUALITY: {sns['perceived_quality']}")

    # Physical details
    phys = s1.get('physical', {})
    if phys.get('key_features'):
        lines.append(f"KEY FEATURES: {', '.join(phys['key_features'])}")
    if phys.get('whats_included') and len(phys['whats_included']) > 1:
        lines.append(f"WHATS INCLUDED: {', '.join(phys['whats_included'])}")
    if phys.get('applicator_type'):
        lines.append(f"APPLICATOR: {phys['applicator_type']}")
    if phys.get('size_impression') and phys['size_impression'] != 'standard':
        lines.append(f"SIZE: {phys['size_impression']}")

    # Conversion signals
    conv = s1.get('conversion', {})
    if conv.get('hero_claim'):
        lines.append(f"HERO CLAIM (use as headline or first line): {conv['hero_claim']}")
    if conv.get('pain_points_solved'):
        lines.append(f"SOLVES THESE PROBLEMS: {', '.join(conv['pain_points_solved'])}")
    if conv.get('visible_results'):
        lines.append(f"VISIBLE RESULTS: {conv['visible_results']}")
    if conv.get('premium_signals'):
        lines.append(f"PREMIUM SIGNALS: {', '.join(conv['premium_signals'])}")
    if conv.get('target_audience'):
        lines.append(f"TARGET AUDIENCE: {conv['target_audience']}")
    if conv.get('usage_scenario'):
        lines.append(f"USAGE SCENARIO: {conv['usage_scenario']}")
    if conv.get('emotional_tone'):
        lines.append(f"TONE: {conv['emotional_tone']}")

    # Conversion hints
    vol_text = pkg.get('volume', '')
    if vol_text:
        ml_m = re.search(r'(\d+)\s*ml', vol_text)
        oz_m = re.search(r'(\d+)\s*(?:fl\s*)?oz', vol_text)
        if ml_m:
            ml = int(ml_m.group(1))
            lines.append(f"LONGEVITY HINT: ~{max(30, ml * 2)} days at daily use ({ml}ml)")
        elif oz_m:
            oz = int(oz_m.group(1))
            lines.append(f"LONGEVITY HINT: ~{max(30, oz * 60)} days at daily use ({oz}oz)")
    items = phys.get('whats_included', [])
    if len(items) >= 2:
        lines.append(f"VALUE HINT: {len(items)} items — emphasize as great value")
    faq_hints = []
    if conv.get('premium_signals'): faq_hints.append("price: premium materials")
    if conv.get('target_audience'): faq_hints.append(f"audience: {conv['target_audience'][:30]}")
    if pkg.get('instructions_visible'): faq_hints.append("usage: from packaging")
    if faq_hints:
        lines.append("FAQ HINTS: " + " | ".join(faq_hints))
    return '\n'.join(lines) if lines else 'No additional vision data available.'





# Auto-recover errored products: demote each 'error' row to the stage AFTER
# its last successful step (based on which JSON columns are filled). Without
# this, errored rows would loop forever in 'error' state because no per-stage
# branch in _process_one matches `pstatus == 'error'`.
_demote_queries = [
    ("UPDATE products SET status='pending',         error_msg=NULL WHERE status='error' AND scrape_json     IS NULL",
     'no scrape yet'),
    ("UPDATE products SET status='scraped',         error_msg=NULL WHERE status='error' AND scrape_json     IS NOT NULL AND image_urls_json IS NULL",
     'scrape OK, images pending'),
    ("UPDATE products SET status='images_uploaded', error_msg=NULL WHERE status='error' AND image_urls_json IS NOT NULL AND stage1_json     IS NULL",
     'images OK, vision pending'),
    ("UPDATE products SET status='vision_done',     error_msg=NULL WHERE status='error' AND stage1_json     IS NOT NULL AND strategy_json   IS NULL",
     'vision OK, strategy pending'),
    ("UPDATE products SET status='strategy_done',   error_msg=NULL WHERE status='error' AND strategy_json   IS NOT NULL AND final_html      IS NULL",
     'strategy OK, designer pending'),
    ("UPDATE products SET status='html_ready',      error_msg=NULL WHERE status='error' AND final_html      IS NOT NULL AND shopify_product_id IS NULL",
     'designer OK, photo_pack + Shopify pending'),
    ("UPDATE products SET status='pack_done',       error_msg=NULL WHERE status='error' AND final_html      IS NOT NULL AND shopify_product_id IS NOT NULL",
     'Shopify push partial, retry metafields'),
]
_demoted_total = 0
for _q, _label in _demote_queries:
    _r = db.execute(_q)
    if _r.rowcount > 0:
        print(f'  ↻ Auto-demoted {_r.rowcount} errored products ({_label}) for retry')
        _demoted_total += _r.rowcount
db.commit()
if _demoted_total:
    print(f'  ↻ Total auto-demoted: {_demoted_total}\n')

pending = db.execute("SELECT * FROM products WHERE status != 'done' ORDER BY id").fetchall()
total = db.execute('SELECT COUNT(*) FROM products').fetchone()[0]
done_count = total - len(pending)

print(f'\n{"="*60}')
print(f'  {total} total | {done_count} done | {len(pending)} remaining')
print(f'{"="*60}')

async def _process_one(pi, prod):
    """Per-product pipeline (Steps 1-5). Extracted from the for-loop so we
    can either await it serially OR feed it to asyncio.gather() under a
    Semaphore for parallel batches. Top-level `continue` statements that
    used to skip to the next product are now `return`."""
    pid, pname, purl, pstatus = prod['id'], prod['name'], prod['eprolo_url'], prod['status']
    print(f'\n{"─"*60}')
    print(f'[{done_count+pi+1}/{total}] {pname[:60]}')
    print(f'  Status: {pstatus}')

    if not purl:
        db_update_status(pid, 'error', error_msg='No EPROLO URL'); return

    if prod['shopify_product_id'] and pstatus == 'done':
        print('  Already in Shopify, skipping.'); return

    try:
        # ═══ STEP 1: Scrape (v9: + variants) ═══
        if pstatus == 'pending':
            print('  → Scraping...')
            # Retry scrape up to 3x with exponential backoff (playwright timeouts,
            # EPROLO 503, network glitches happen ~1-3% of products in big batches).
            product = None
            for _scrape_attempt in range(3):
                try:
                    product = await scrape_eprolo(purl)
                    if product and product.get('title'):
                        break  # success
                    if _scrape_attempt < 2:
                        _wait = 5 * (_scrape_attempt + 1)
                        print(f'    [scrape attempt {_scrape_attempt+1}/3 returned empty title, retry in {_wait}s]')
                        await asyncio.sleep(_wait)
                except Exception as _e_scrape:
                    if _scrape_attempt < 2:
                        _wait = 10 * (_scrape_attempt + 1)
                        print(f'    [scrape attempt {_scrape_attempt+1}/3 failed: {str(_e_scrape)[:80]} — retry in {_wait}s]')
                        await asyncio.sleep(_wait)
                    else:
                        raise  # 3rd failure → outer try/except marks product as error
            if not product:
                product = {"title": "", "description": "", "top_image_urls": [], "desc_image_urls": [],
                           "cost_price": 0, "variants": [], "specs": {}}
            print(f'    Title: {product["title"][:60]}')
            scrape_issues = validate_scrape(product)
            if scrape_issues:
                print(f'    ⚠ Scrape issues: {", ".join(scrape_issues)}')
                # Hard-fail tells — refuse to push junk to Shopify.
                _hard = ('empty/short title',)
                _hard_match = any(h in scrape_issues for h in _hard) or                     any('marketing_page' in s for s in scrape_issues)
                if _hard_match:
                    db_update_status(pid, 'error', error_msg=f'Scrape failed: {", ".join(scrape_issues)}')
                    return
            db_update_status(pid, 'scraped',
                scrape_json=json.dumps(product, ensure_ascii=False),
                variants_json=json.dumps(product.get("variants",[]), ensure_ascii=False))
            pstatus = 'scraped'

        # ═══ STEP 2: Images ═══
        if pstatus == 'scraped':
            product = json.loads(db.execute('SELECT scrape_json FROM products WHERE id=?',(pid,)).fetchone()[0])
            print('  → Images...')
            top_images = await dl_images(product["top_image_urls"], "TOP")
            desc_images = await dl_images(product["desc_image_urls"], "DESC")
            if SHOPIFY_TOKEN:
                top_cdn = await upload_to_shopify(top_images, "TOP")
                desc_cdn = await upload_to_shopify(desc_images, "DESC")
                for item in top_cdn:
                    if item['cdn']:
                        for img in top_images:
                            if img['index']==item['index']: img['url']=item['url']
                for item in desc_cdn:
                    if item['cdn']:
                        for img in desc_images:
                            if img['index']==item['index']: img['url']=item['url']
                cdn_ok = sum(1 for x in top_cdn+desc_cdn if x['cdn'])
                print(f'    CDN: {cdn_ok}/{len(top_cdn)+len(desc_cdn)}')
            img_data = {
                "top":[{"index":i["index"],"url":i["url"],"media_type":i["media_type"],"size_kb":i["size_kb"]} for i in top_images],
                "desc":[{"index":i["index"],"url":i["url"],"media_type":i["media_type"],"size_kb":i["size_kb"]} for i in desc_images],
                "top_b64":[{"index":i["index"],"base64":i["base64"],"media_type":i["media_type"]} for i in top_images],
                "desc_b64":[{"index":i["index"],"base64":i["base64"],"media_type":i["media_type"]} for i in desc_images]}
            db_update_status(pid, 'images_uploaded', image_urls_json=json.dumps(img_data, ensure_ascii=False))
            pstatus = 'images_uploaded'

        # ═══ STEP 3: Vision ═══
        if pstatus == 'images_uploaded':
            _check_anthropic_budget()
            product = json.loads(db.execute('SELECT scrape_json FROM products WHERE id=?',(pid,)).fetchone()[0])
            img_data = json.loads(db.execute('SELECT image_urls_json FROM products WHERE id=?',(pid,)).fetchone()[0])
            print('  → Vision...')
            vb = []
            _top_imgs = img_data.get('top_b64', [])[:4]
            _desc_imgs = img_data.get('desc_b64', [])[:4]
            if _top_imgs:
                vb.append({"type":"text","text":"GALLERY IMAGES:"})
                for img in _top_imgs:
                    vb.append({"type":"image","source":{"type":"base64","media_type":img["media_type"],"data":img["base64"]}})
            if _desc_imgs:
                vb.append({"type":"text","text":"DESCRIPTION IMAGES:"})
                for img in _desc_imgs:
                    vb.append({"type":"text","text":f"IMG_{img['index']}:"})
                    vb.append({"type":"image","source":{"type":"base64","media_type":img["media_type"],"data":img["base64"]}})
            # Static schema lives in VISION_SYSTEM_PROMPT (Cell 5) with
            # cache_control. User content holds ONLY product-specific bits.
            safe_title = _sanitize_for_prompt(product.get('title') or '', max_len=300)
            vb.append({"type":"text","text":f"Product: {safe_title}\n\nAnalyse the images above and return the JSON described in the system prompt."})

            if not _top_imgs and not _desc_imgs:
                print('    No images — using default vision')
                stage1 = {"accent":"#C4B09A","accent_soft":"#F5F0EA","bg":"#FFFFFF",
                    "category":product['title'].split()[0],"product_type":"simple",
                    "product_summary":product.get('description','')[:200],"needs_steps":False,
                    "packaging_text":{"volume":"","claims":[],"ingredients_visible":[],"certifications":[],"instructions_visible":""},
                    "sensory":{"texture":"","material":"","finish_description":"","color_names":[],"perceived_quality":"mid-range"},
                    "physical":{"size_impression":"standard","key_features":[],"whats_included":[],"applicator_type":""},
                    "conversion":{"hero_claim":"","pain_points_solved":[],"visible_results":"","premium_signals":[],"target_audience":"","usage_scenario":"","emotional_tone":""},
                    "images":[]}
                cost1 = 0
            else:
                _vm = MODEL_VISION
                r1 = await client_async.messages.create(model=_vm, max_tokens=2000, timeout=90.0,
                    system=[{"type":"text","text":VISION_SYSTEM_PROMPT,"cache_control":{"type":"ephemeral","ttl":"1h"}}],
                    messages=[{"role":"user","content":vb}])
                raw = r1.content[0].text.strip()
                if raw.startswith("```"): raw = raw.split("\n", 1)[-1].rsplit("\n", 1)[0]
                try:
                    stage1 = json.loads(raw)
                    pkg = stage1.get('packaging_text', {})
                    if _vm == MODEL_VISION and not pkg.get('claims') and not pkg.get('ingredients_visible') and len(_top_imgs) <= 5:
                        print(f'    Haiku found no packaging text, retrying Sonnet...')
                        _vm = MODEL_WRITER
                        r1 = await client_async.messages.create(model=_vm, max_tokens=2000, timeout=90.0,
                            system=[{"type":"text","text":VISION_SYSTEM_PROMPT,"cache_control":{"type":"ephemeral","ttl":"1h"}}],
                            messages=[{"role":"user","content":vb}])
                        raw = r1.content[0].text.strip()
                        if raw.startswith("```"): raw = raw.split("\n", 1)[-1].rsplit("\n", 1)[0]
                        try: stage1 = json.loads(raw)
                        except: pass
                except json.JSONDecodeError:
                    print(f'    Vision JSON failed, using defaults')
                    stage1 = {"accent":"#C4B09A","accent_soft":"#F5F0EA","bg":"#FFFFFF",
                        "category":"Product","product_type":"simple",
                        "product_summary":"","needs_steps":False,"images":[]}
                if 'sonnet' in str(_vm).lower():
                    cost1 = r1.usage.input_tokens*3/1e6 + r1.usage.output_tokens*15/1e6
                else:
                    cost1 = r1.usage.input_tokens*0.80/1e6 + r1.usage.output_tokens*4/1e6
                _track_anthropic_response(r1, cost1)

            pkg_claims = len(stage1.get('packaging_text',{}).get('claims',[]))
            ingr_count = len(stage1.get('packaging_text',{}).get('ingredients_visible',[]))
            hero = stage1.get('conversion',{}).get('hero_claim','')[:40]
            print(f'    OK ${cost1:.4f} | {stage1.get("product_type")} | {len(stage1.get("images",[]))} imgs | claims:{pkg_claims} ingr:{ingr_count}')
            if hero: print(f'    Hero: {hero}')
            db_update_status(pid, 'vision_done', stage1_json=json.dumps(stage1, ensure_ascii=False), cost_usd=(prod['cost_usd'] or 0)+cost1)
            _img_clean = json.loads(db.execute('SELECT image_urls_json FROM products WHERE id=?',(pid,)).fetchone()[0])
            _img_clean.pop('top_b64', None)
            _img_clean.pop('desc_b64', None)
            db.execute('UPDATE products SET image_urls_json=? WHERE id=?', (json.dumps(_img_clean, ensure_ascii=False), pid))
            db.commit()
            pstatus = 'vision_done'

        # ═══ STEP 3.5: STRATEGY AGENT (NEW — Opus 4.7 marketing positioning) ═══
        if pstatus == 'vision_done':
            _check_anthropic_budget()
            stage1 = json.loads(db.execute('SELECT stage1_json FROM products WHERE id=?',(pid,)).fetchone()[0])
            product = json.loads(db.execute('SELECT scrape_json FROM products WHERE id=?',(pid,)).fetchone()[0])
            print('  → Strategy (Opus 4.7)...')
            _strategy_user = (
                f"PRODUCT: {product['title']}\n"
                f"CATEGORY: {stage1.get('category','')}\n"
                f"TYPE: {stage1.get('product_type','simple')}\n"
                f"DESCRIPTION (scraped): {(product.get('description') or '')[:600]}\n\n"
                "VISION ANALYSIS:\n"
                f"{_build_vision_context(stage1)}\n\n"
                f"LANGUAGE: {LANGUAGE}\n"
                "OUTPUT: respond with ONLY compact JSON — no markdown fences, no prose, terse string values.\n"
            )
            try:
                _r_strat = await client_async.messages.create(
                    model=MODEL_STRATEGY, max_tokens=2500, timeout=120.0,
                    system=[{"type":"text","text":STRATEGY_SYSTEM_PROMPT,"cache_control":{"type":"ephemeral","ttl":"1h"}}],
                    messages=[{"role":"user","content":_strategy_user}],
                )
                _raw_strat = _r_strat.content[0].text.strip()
                if _raw_strat.startswith("```"):
                    _raw_strat = _raw_strat.split("\n",1)[-1].rsplit("\n",1)[0]
                try:
                    strategy = json.loads(_raw_strat)
                except json.JSONDecodeError:
                    _jm = re.search(r'\{.*\}', _raw_strat, re.DOTALL)
                    strategy = json.loads(_jm.group(0)) if _jm else {}
                # Opus 4.7 pricing: $15/$75 per 1M tokens
                _cost_strat = _r_strat.usage.input_tokens * 5/1e6 + _r_strat.usage.output_tokens * 25/1e6
                _track_anthropic_response(_r_strat, _cost_strat)
                _selling_idea = strategy.get('selling_idea', '')[:60]
                _voice = strategy.get('voice', '')
                print(f'    Strategy OK: ${_cost_strat:.4f} | voice={_voice} | idea: {_selling_idea}')
                cur_cost = db.execute('SELECT cost_usd FROM products WHERE id=?', (pid,)).fetchone()[0] or 0
                db_update_status(pid, 'strategy_done',
                                 strategy_json=json.dumps(strategy, ensure_ascii=False),
                                 cost_usd=cur_cost + _cost_strat)
                pstatus = 'strategy_done'
            except Exception as _e:
                print(f'    \u26a0 Strategy failed: {str(_e)[:120]} \u2014 falling back to vision-only mode')
                # Fallback: empty strategy, pipeline continues
                db_update_status(pid, 'strategy_done', strategy_json='{}')
                pstatus = 'strategy_done'

        # ═══ STEP 4: Designer (content + image briefs) ═══
        if pstatus == 'strategy_done':
            _check_anthropic_budget()
            product = json.loads(db.execute('SELECT scrape_json FROM products WHERE id=?',(pid,)).fetchone()[0])
            stage1 = json.loads(db.execute('SELECT stage1_json FROM products WHERE id=?',(pid,)).fetchone()[0])
            img_data = json.loads(db.execute('SELECT image_urls_json FROM products WHERE id=?',(pid,)).fetchone()[0])
            variants = json.loads(db.execute('SELECT scrape_json FROM products WHERE id=?',(pid,)).fetchone()[0] or '{}').get('variants', [])  # BUGFIX 2026-06-05: variants_json column was never populated -> read variants from scrape_json (source of truth)

            # FIX #6: top and desc image lists each restart their index at 1 — when
            # concatenated, vision's "index:5" matches whichever entry comes first in the
            # combined list (desc), which may not be what vision meant. Use a composite key
            # via the URL itself (vision saw the actual image bytes, but URL is unique).
            # Match by URL prefix from vision's 'desc' field if present, otherwise fall back
            # to first matching index in concatenated list (legacy behavior).
            all_avail_imgs = img_data.get('desc',[]) + img_data.get('top',[])
            img_ctx = ""
            _used_urls = set()
            for ii in stage1.get('images', []):
                # First try: if vision returned a URL hint in 'desc' field, match by URL substring
                url = None
                v_desc = (ii.get('desc') or '').lower()
                if v_desc:
                    for d in all_avail_imgs:
                        if d['url'] not in _used_urls:
                            # match if descriptive words from vision overlap with URL fragment
                            url_tail = d['url'].rsplit('/', 1)[-1].lower()
                            if any(w in url_tail for w in v_desc.split() if len(w) > 4):
                                url = d['url']; break
                # Fallback: pick by index, but skip already-used URLs to reduce collisions
                if not url:
                    for d in all_avail_imgs:
                        if d['index'] == ii['index'] and d['url'] not in _used_urls:
                            url = d['url']; break
                # Final fallback: just any unused image
                if not url:
                    for d in all_avail_imgs:
                        if d['url'] not in _used_urls:
                            url = d['url']; break
                if url: _used_urls.add(url)
                if not url or ii.get('role')=='skip': continue
                shows = ii.get('shows', '')
                shows_text = f"\n    SHOWS: {shows}" if shows else ''
                img_ctx += f"\n  [{ii.get('role','').upper()}] {url}\n    ALT: {ii.get('desc','')}{shows_text}"
            if not img_ctx.strip():
                for img in img_data.get('desc',[])[:6]:
                    img_ctx += f"\n  [DETAIL] {img['url']}\n    ALT: Product detail"
                for img in img_data.get('top',[])[:4]:
                    img_ctx += f"\n  [LIFESTYLE] {img['url']}\n    ALT: Product photo"

            safe_title = product['title']
            safe_desc = product.get('description','') or ''

            # Get the smart-shortlist (own + top related by keyword overlap) for interlinks[]
            # — used by Method 3 collection linking + still passed for backward compat.
            interlinks = get_smart_interlinks(pid, pname, ALL_SHOP_COLLECTIONS)
            # Broader catalog passed to writer: own collections + up to 25 of the most
            # plausibly-related ones, so the writer can pick 5-10 to reference inline.
            _own_handles = {l['handle'] for l in interlinks if l.get('relevance') == 'own'}
            # Filter: ONLY collections from THIS pipeline run (in local DB with
            # shopify_collection_id synced) — not the whole Shopify catalog of 100+
            # legacy collections from old runs/categories. This keeps the writer
            # focused on the new SEO collections we built for this batch.
            _run_handles = set()
            try:
                for _row in db.execute("SELECT seo_handle FROM collections WHERE shopify_collection_id IS NOT NULL").fetchall():
                    _h = _row['seo_handle'] if _row['seo_handle'] else ''
                    if _h: _run_handles.add(_h)
            except Exception:
                pass
            _candidates = []
            _name_words = {w for w in re.sub(r'[^a-z0-9 ]', ' ', pname.lower()).split()
                           if len(w) > 3 and w not in {'and','for','with','the','best','new','set','kit'}}
            for sc in ALL_SHOP_COLLECTIONS:
                h = sc.get('handle','')
                if not h: continue
                # Only collections from THIS run (skip legacy collections from old runs)
                if _run_handles and h not in _run_handles: continue
                t = sc.get('title','').lower()
                t_words = {w for w in re.sub(r'[^a-z0-9 ]', ' ', t).split() if len(w) > 3}
                overlap = len(_name_words & t_words)
                is_own = h in _own_handles
                _candidates.append((overlap, is_own, sc))
            _candidates.sort(key=lambda x: (-int(x[1]), -x[0]))
            _broad_collections = []
            for _, is_own, sc in _candidates[:30]:
                title = sc.get('title','')
                handle = sc.get('handle','')
                anchors = _get_collection_anchors(title)
                _broad_collections.append({
                    "handle": handle, "title": title,
                    "anchors": anchors, "relevance": "own" if is_own else "related",
                })
            interlink_block = ""
            if _broad_collections:
                interlink_block = "\n=== AVAILABLE COLLECTIONS (full catalog of relevant collections — pick 5-10 to reference inline + ALL of them in the footer wa-collections grid) ===\n"
                for lnk in _broad_collections:
                    tag = " *OWN" if lnk.get('relevance') == 'own' else ""
                    anchors = lnk.get('anchors', [])
                    anchor_str = ' | '.join(anchors[:3]) if anchors else lnk["title"]
                    interlink_block += f'  /collections/{lnk["handle"]}{tag}  →  anchors: {anchor_str}\n'
                interlink_block += '\n  USAGE:\n'
                interlink_block += '  1. Weave 5-10 inline <a class="wa-link" href="/collections/<handle>">anchor</a> links naturally throughout body paragraphs (3+ different paragraphs).\n'
                interlink_block += '  2. At the END of the page, include a wa-collections grid with EVERY handle from the list above.\n'
                interlink_block += '  3. Use the anchor variants from the "anchors:" column when present, otherwise the collection title.\n'
                interlink_block += '  4. Mark *OWN collections (current product belongs to them) with priority — those are highest-relevance.\n'
                interlink_block += '  5. Never invent /collections/ handles outside this list.\n'
                # Override interlinks[] for downstream Method 3 fallback (broader pool)
                interlinks = _broad_collections

            variant_ctx = ""
            if variants:
                variant_ctx = "\nPRODUCT VARIANTS:\n"
                for v in variants:
                    variant_ctx += f"  {v['option_name']}: {', '.join(v['values'][:10])}\n"

            specs = product.get('specs', {})
            specs_ctx = ""
            if specs:
                specs_ctx = "\nSPECS:\n"
                for k, v in list(specs.items())[:15]:
                    specs_ctx += f"  {k}: {v}\n"

            video_urls = product.get('video_urls', [])
            video_ctx = f"\nVIDEO: {video_urls[0]}\n" if video_urls else ""

            size_chart = product.get('size_chart', [])
            size_ctx = ""
            if size_chart and len(size_chart) >= 2:
                size_ctx = "\nSIZE CHART:\n"
                for row in size_chart[:10]:
                    size_ctx += "  " + " | ".join(row) + "\n"

            cross_sell_ctx = ""
            own_colls = get_product_collections(pid)
            if own_colls:
                sibling_products = []
                for coll_title, coll_url, coll_sid in own_colls:
                    if not coll_sid: continue
                    siblings = db.execute("""SELECT p.name, p.url_handle FROM products p
                        JOIN product_collections pc ON p.id = pc.product_id
                        WHERE pc.collection_id IN (SELECT id FROM collections WHERE shopify_collection_id=?)
                        AND p.id != ? AND p.shopify_product_id IS NOT NULL
                        LIMIT 6""", (coll_sid, pid)).fetchall()
                    for s in siblings:
                        if s['url_handle'] and s['name'] not in [x[0] for x in sibling_products]:
                            sibling_products.append((s['name'], s['url_handle']))
                if sibling_products:
                    cross_sell_ctx = "\nRELATED PRODUCTS:\n"
                    for name, handle in sibling_products[:4]:
                        cross_sell_ctx += f'  /products/{handle} — {name}\n'

            seo_kw_row = db.execute('SELECT seo_keywords FROM products WHERE id=?', (pid,)).fetchone()
            product_kws = seo_kw_row['seo_keywords'] if seo_kw_row and seo_kw_row['seo_keywords'] else ''

            all_winner_data = []
            own_colls_kw = get_product_collections(pid)
            _pw = set(re.sub(r'[^a-z0-9 ]', ' ', pname.lower()).split())
            _pw -= {'the','a','an','for','and','or','with','in','of','to','set','kit','pcs','pc','new','style','color','pack'}
            for coll_title, _, _ in own_colls_kw:
                rows_kw = db.execute(
                    'SELECT keyword, volume, kd, cpc FROM seo_keywords WHERE collection_name=? AND volume > 0',
                    (coll_title,)).fetchall()
                for rk in rows_kw:
                    kw_words = set(rk['keyword'].lower().split())
                    if _pw & kw_words or any(w in pname.lower() for w in kw_words if len(w) > 3):
                        all_winner_data.append({
                            'keyword': rk['keyword'], 'volume': rk['volume'],
                            'kd': rk['kd'], 'cpc': rk['cpc']
                        })

            t1_primary, t2_secondary, t3_support = tier_keywords(all_winner_data)

            kw_ctx = ""
            if product_kws or t1_primary or t2_secondary:
                kw_ctx = "\nSEO KEYWORDS (WANELO.com DA 60+):\n"
                if product_kws:
                    kw_ctx += f"  PRODUCT: {product_kws}\n"
                if t1_primary:
                    kw_ctx += "  PRIMARY (KD 0-20, use in H2 + first paragraphs):\n"
                    for kd in t1_primary[:6]:
                        kw_ctx += f"    {kd['keyword']} (vol:{kd['volume']:,} kd:{kd['kd']})\n"
                if t2_secondary:
                    kw_ctx += "  SECONDARY (KD 21-35, body + FAQ + alt):\n"
                    for kd in t2_secondary[:5]:
                        kw_ctx += f"    {kd['keyword']} (vol:{kd['volume']:,})\n"
                if t1_primary or t2_secondary:
                    print(f'    Keywords: {len(t1_primary)} primary + {len(t2_secondary)} secondary')

            print('  → Generating HTML + SEO...')

            # ═══ Taxonomy shortlist (v3.2) ═══
            _shortlist = get_cluster_shortlist(
                stage1.get('product_summary', ''),
                stage1.get('category', ''),
                top_k=TAXONOMY_SHORTLIST_K
            )
            _tax_block = format_shortlist_for_prompt(_shortlist)

            # Product context (path-agnostic) — used by both modular designer
            # and legacy writer.
            _photo_theme_str = _photo_theme(handle if handle else safe_title)
            _product_context = f"""Generate a product description page for this product:

PRODUCT: {safe_title}
CATEGORY: {stage1.get('category','')}
TYPE: {stage1.get('product_type','simple')}
SUMMARY: {stage1.get('product_summary','')}
DESCRIPTION: {safe_desc[:800]}
ACCENT: {stage1.get('accent','#C4B09A')}
ACCENT_SOFT: {stage1.get('accent_soft','#F5F0EA')}

PHOTO STYLING THEME (use this DISTINCT look for THIS product's photo_briefs so it differs from every other product; ALL backgrounds LIGHT): {_photo_theme_str}

VISION ANALYSIS:
{_build_vision_context(stage1)}

IMAGES — each line is a usable image URL, role, ALT, and what the image proves:
{img_ctx}

IMAGE PLACEMENT GUIDANCE: HERO=top of page, FEATURE=benefits/details, DETAIL=specs/closeup, LIFESTYLE=usage scenario, PACKAGING=whats included.
COPY THE FULL URL EXACTLY (starts with https://). Never shorten, modify, or invent URLs.
{interlink_block}
{variant_ctx}
{specs_ctx}
{video_ctx}
{size_ctx}
{cross_sell_ctx}
{kw_ctx}

DO NOT include shipping badges, trust badges, or delivery information — the store theme handles these.

LANGUAGE: {LANGUAGE}

{_tax_block}"""

            # === JSON-content path: designer outputs structured JSON for theme Liquid sections ===
            _designer_model = MODEL_DESIGNER
            _palette_hint = stage1.get("palette") or {
                "brand_1": stage1.get("accent","#ffe9d6"),
                "brand_2": stage1.get("accent_soft","#aac9f5"),
                "brand_3": stage1.get("accent","#a6d8be"),
                "brand_soft": stage1.get("accent_soft","#fdf6ee"),
                "brand_deep": stage1.get("accent","#ffc8b0"),
            }
            _designer_system = (
                "You are a senior product page copywriter and designer. You produce structured "
                "JSON content for a long-form product detail page. The visual design is fixed "
                "(Shopify Liquid sections); your only job is content + per-product palette.\n\n"
                "OUTPUT FORMAT — single JSON object. No prose, no markdown fences. Start with {.\n\n"
                "{\n"
                '  "hero":     {"kicker":"...","h1":"...","lead":"...","image_url":"https://...exact EPROLO URL...","image_alt":"..."},\n'
                '  "story":    {"head":{"kicker":"","h2":"","desc":""},\n'
                '               "chapters":[{"image_url":"https://...exact EPROLO URL...","image_alt":"","kicker":"CHAPTER 01","h3":"","p":""}]},  // 0-4 chapters (skip with [] for cheap utility products; 2-3 typical; 4 for transformations)\n'
                '  "features": {"head":{"kicker":"","h2":"","desc":""},\n'
                '               "items":[{"image_url":"https://...exact EPROLO photo URL, DISTINCT per feature...","image_alt":"","meta":"","h4":"","p":""}]},  // 2-4 items (3 typical). Oura-style: each feature leads with a square product/lifestyle photo. Pick a DIFFERENT EPROLO photo per feature (in-use, detail, packaging). Omit image_url only if no suitable photo.\n'
                '  "stats":    {"head":{"kicker":"","h2":"","desc":""},\n'
                '               "items":[{"kicker":"","value":"93<em>%</em>","label":""}]},  // 3-5 items (4 typical, fits grid; 3 for simple, 5 if rich)\n'
                '  "reviews":  {"head":{"kicker":"REAL VOICES","h2":"","desc":""},\n'
                '               "items":[{"stars":"\u2605\u2605\u2605\u2605\u2605","quote":"...with optional <em>...</em>","author":"Anna K.","meta":"Verified buyer"}]},  // 9 or 12 items\n'
                '  "faq":      {"head":{"kicker":"","h2":"","desc":""},\n'
                '               "items":[{"q":"","a":""}]},  // 4-7 items\n'
                '  "cta":      {"kicker":"","h2":"","p":"","button_label":"Add to cart"},\n'
                '  "palette":  {"brand_1":"#hex","brand_2":"#hex","brand_3":"#hex","brand_soft":"#hex","brand_deep":"#hex"},\n'
                '  "interlinks":{"head":{"kicker":"EXPLORE","h2":"Explore more"},\n'
                '               "items":[{"handle":"","anchor":""}]},  // ALL handles from AVAILABLE COLLECTIONS\n'
                '  // === Optional modules below — leave NULL or omit if not relevant for this product ===\n'
                '  "how_to":   {"head":{"kicker":"","h2":"","desc":""},\n'
                '               "items":[{"n":1,"h4":"","p":""}]},  // 3-7 steps. For: rituals, recipes, electronics setup, hobbies, supplements\n'
                '  "specs":   {"head":{"kicker":"SPECIFICATIONS","h2":"","desc":""},\n'
                '               "items":[{"label":"","value":""}]},  // 6-12 rows. For: electronics, kitchen appliances, jewelry materials, sports gear dimensions\n'
                '  "whats_included":{"head":{"kicker":"WHAT\u2019S IN THE BOX","h2":"","desc":""},\n'
                '               "items":[{"glyph":"✮","name":"","qty":"","note":""}]},  // 3-8 items. For: bundles, kits, gift sets, hobby starter packs\n'
                '  "ingredients":{"head":{"kicker":"KEY INGREDIENTS","h2":"","desc":""},\n'
                '               "items":[{"name":"","role":"","badge":""}]},  // 4-10 chips. For: skincare actives, food, supplements, candles\n'
                '  "timeline":{"head":{"kicker":"YOUR JOURNEY","h2":"","desc":""},\n'
                '               "items":[{"when":"DAY 1","h4":"","p":""}]},  // 3-5 milestones. For: skincare results, fitness journey, ritual progression\n'
                '  "trust":   {"head":{"kicker":"","h2":"","desc":""},\n'
                '               "certs":[{"label":"","sub":""}],   // 3-6 certifications\n'
                '               "press":[{"label":"","sub":""}]}, // 3-6 press mentions. For: regulated/premium/eco\n'
                '  "compare": {"head":{"kicker":"VS. ALTERNATIVES","h2":"","desc":""},\n'
                '               "columns":[{"name":"This","badge":"BEST"},{"name":"Generic","badge":""}], // 2-3 columns\n'
                '               "rows":[{"label":"","values":["✓","✗"]}]}, // 4-7 rows. For: research-buy, upgrade messaging\n'
                '  "size_guide":{"head":{"kicker":"FIND YOUR FIT","h2":"","desc":""},\n'
                '               "labels":["S","M","L"],\n'
                '               "rows":[{"label":"Chest","values":["86","92","98"],"unit":"cm"}],  // 2-5 rows\n'
                '               "note":"Measurements taken flat. Add 2cm for relaxed fit."}, // For clothing/jewelry/pet/baby\n'
                '  "care":    {"head":{"kicker":"CARE","h2":"","desc":""},\n'
                '               "items":[{"glyph":"🌊","label":"","note":""}]}, // 3-6 items. For: fashion/jewelry/textiles/leather/durables\n'
                '  "dimensions":{"head":{"kicker":"DIMENSIONS","h2":"","desc":""},\n'
                '               "items":[{"axis":"Height","value":"32 cm"}], // 3-5 axes\n'
                '               "scale_note":"Roughly the size of a hardcover book"}, // For decor/furniture/lighting/organization\n'
                '  "variants":{"head":{"kicker":"AVAILABLE IN","h2":"","desc":""},\n'
                '               "items":[{"name":"","color":"#hex","available":true}]}, // 2-8 swatches. For: fashion/accessories/decor\n'
                '  "gift_options":{"head":{"kicker":"GIFT-READY","h2":"","desc":""},\n'
                '               "wrap":{"available":true,"price":"$5","options":[""]},\n'
                '               "card":{"available":true,"max_chars":240,"examples":[""]},\n'
                '               "occasions":[""],\n'
                '               "ships_with":""}, // For gift intent + sections 2/5/11/10\n'
                # ─── Round 5 module schemas ───
                '  "safety":{"head":{"kicker":"BEFORE YOU START","h2":"","desc":""},\n'
                '             "age_rating":"",\n'
                '             "warnings":[{"label":"","detail":""}], // 2-5 warnings\n'
                '             "contraindications":[""], // 0-4 strings\n'
                '             "safety_certs":[{"label":"FDA Cleared","sub":""}]}, // FILL: beauty devices, supplements, kids toys, regulated\n'
                '  "protocol":{"head":{"kicker":"YOUR ROUTINE","h2":"","desc":""},\n'
                '             "schedule":[{"phase":"Week 1-2","frequency":"3× per week","duration":"10 min","intensity":"Low","note":""}], // 2-5 phases\n'
                '             "total_commitment":""}, // FILL: beauty devices, supplements, fitness, wellness rituals\n'
                '  "target_profile":{"head":{"kicker":"WHO IT IS FOR","h2":"","desc":""},\n'
                '             "ideal_for":[{"label":"","detail":""}], // 3-5 items\n'
                '             "not_ideal_for":[""], // 0-4 strings\n'
                '             "fitness_level":"", "skin_type":"", "age_range":""}, // FILL by category-fit\n'
                '  "video_demo":{"head":{"kicker":"SEE IT IN ACTION","h2":"","desc":""},\n'
                '             "video":{"source":"youtube|vimeo|direct","id_or_url":"","duration_seconds":0,"thumbnail_url":""},\n'
                '             "chapters":[{"time":"0:00","label":""}]}, // FILL: fitness/beauty devices/toys/tech demos\n'
                '  "compatibility":{"head":{"kicker":"WORKS WITH","h2":"","desc":""},\n'
                '             "platforms":[{"name":"iOS","version":"16.0+","logo_hint":"apple"}], // logo_hint: apple|android|applewatch|bluetooth\n'
                '             "accessories":[{"name":"","sku":"","fit_note":""}],\n'
                '             "incompatibility_warnings":[""]}, // FILL: smart home, wearables, tech accessories\n'
                '  "assembly":{"head":{"kicker":"SETUP","h2":"","desc":""},\n'
                '             "time_minutes":15,\n'
                '             "difficulty":"Easy|Moderate|Advanced",\n'
                '             "person_count":1,\n'
                '             "tools_required":[{"name":"","included":true,"optional":false}],\n'
                '             "warnings":[""]}, // FILL: furniture, fitness gear, construction toys\n'
                # ─── Round 6 module schemas ───
                '  "sustainability":{"head":{"kicker":"GOOD FOR THE PLANET","h2":"","desc":""},\n'
                '             "certs":[{"label":"Climate Neutral","since":"2024","verifier":"climateneutral.org"}], // 1-5 certs\n'
                '             "materials":[{"name":"","percent":80,"note":""}], // 1-4 materials\n'
                '             "carbon_footprint_g":0,\n'
                '             "recycling":""}, // FILL: eco/ethical positioning. SKIP: pure tech commodity\n'
                '  "brand_story":{"head":{"kicker":"WHO MADE THIS","h2":"","desc":""},\n'
                '             "founded":{"year":2018,"city":"","founder":""},\n'
                '             "mission":"",\n'
                '             "values":[{"label":"","detail":""}], // 2-4 values\n'
                '             "founder_quote":{"text":"","author":""},\n'
                '             "founder_image_url":""}, // FILL: premium positioning, indie/artisan, eco, family-business\n'
                '  "nutrition_facts":{"head":{"kicker":"WHAT IS IN A SERVING","h2":"","desc":""},\n'
                '             "serving_size":"1 scoop (32g)",\n'
                '             "servings_per_container":30,\n'
                '             "calories":0,\n'
                '             "macros":[{"label":"Protein","value":"24g","dv_percent":48}], // standard FDA macros\n'
                '             "highlights":[""], // No added sugar / Gluten-free / etc.\n'
                '             "allergens":[""]}, // FILL: food, beverages, supplements, protein, snacks\n'
                '  "app_showcase":{"head":{"kicker":"IN THE APP","h2":"","desc":""},\n'
                '             "app_name":"",\n'
                '             "store_links":[{"store":"App Store","url":""},{"store":"Google Play","url":""}],\n'
                '             "screens":[{"caption":"","image_url":""}], // 3 phone-screen mockups\n'
                '             "features":[""]}, // 3-5 app feature lines. FILL: smart devices with companion apps\n'
                '  "subscription_refill":{"head":{"kicker":"NEVER RUN OUT","h2":"","desc":""},\n'
                '             "lasts_days":60,\n'
                '             "default_refill_interval_days":60,\n'
                '             "discount_percent":15,\n'
                '             "perks":["Skip or cancel anytime","Free shipping always"], // 2-4 perks\n'
                '             "savings_per_year_usd":0,\n'
                '             "best_for_buyer_text":""}, // FILL: consumables (skincare, supplements, coffee, pet food)\n'
                '  "clinical_evidence":{"head":{"kicker":"THE SCIENCE","h2":"","desc":""},\n'
                '             "claims":[{"stat":"93%","claim":"saw firmer skin after 8 weeks","study_size":"n=42","study_type":"independent clinical","study_year":2024}], // 1-3 claims\n'
                '             "mechanism":"",\n'
                '             "disclaimer":"Results may vary. Not a treatment for any medical condition."}, // FILL: beauty devices, supplements, wellness with clinical claims\n'
                '  "room_placement":{"head":{"kicker":"IN YOUR SPACE","h2":"","desc":""},\n'
                '             "best_for_rooms":[""], // 2-4 room types\n'
                '             "scale_photos":[{"image_url":"","caption":""}], // 2 scale-reference photos\n'
                '             "pairs_well_with":[""]}, // FILL: furniture, lamps, wall decor, rugs, large appliances\n'
                '  "lifestyle_gallery":{"head":{"kicker":"IN THE WILD","h2":"","desc":""},\n'
                '             "items":[{"image_url":"","credit":"","caption":"","tall":false}], // 4-5 mood photos. tall:true=2-row grid span\n'
                '             "footer_text":""}, // FILL: fashion, accessories, candles, decor, wellness — anything aesthetic-driven\n'
                '  "visual_style":"ONE paragraph describing a DISTINCT light styling theme for THIS specific product, actively VARYING the surface/backdrop, props, palette accents and camera angle product-to-product so the catalog never looks templated or repetitive; the photo_pack ZIP — bg, light, palette, framing. Applies to ALL photo_briefs.",\n'
                '  "photo_briefs":[\n'
                '    {"id":"carousel/01","slot":"carousel-hero","source_index":1,"concept":"clean product on white seamless","edit_instructions":"remove bg, pure white seamless, soft shadow","slot_context":"first carousel impression — must answer \\"what IS it\\" in 1 second"},\n'
                '    ... 5-12 briefs total — Designer picks count based on product complexity. Each brief uses source_index from EPROLO PHOTOS list (1-N).\n'
                '    // slot enum: carousel-hero | carousel-lifestyle | carousel-detail | carousel-scale | carousel-in-use\n'
                '    //          | inline-hero | inline-story-1 | inline-story-2 | inline-story-3 | inline-feature | inline-cta\n'
                '  ],\n'
                '  "review_scenes":[ // EXACTLY 8 UGC phone-snapshot scene ideas for the review widget, matched to THIS product\n'
                '    {"title":"3-5 word scene name","scene":"ONE sentence: a candid iPhone shot of THIS product used or shown where a REAL owner would; derive place and activity from the product category (fishing rod to a riverbank with the catch; drill to mid-repair in a garage; perfume to a vanity before a night out). Real, lived-in, slightly cluttered. NO studio, NO white seamless.","setting":"2-4 word real place"},\n'
                '    ... 8 scenes total, ALL different places/activities\n'
                '  "meta":     {"short_description":"2-3 sentences","seo_meta":{"title":"55-60 chars","description":"140-155 chars","handle":"url-slug"},\n'
                '               "product_tags":["cluster:x","persona:y","intent:z","demo:female","demo:young-adult"]},\n'
                "}\n\n"
                "PALETTE — strict pastel rules (CRITICAL for visual consistency):\n"
                "- ALL 5 colors PASTEL (luminance 0.55-0.95) and DESATURATED (saturation < 50%).\n"
                "  Examples: #fdf6ee #f0e8d0 #e8efe5 #ffe9d6 #aac9f5 #a6d8be #f5e9c8 #e7e0d6 #ddd6e3 #fff5f0 #fae0d4 #d4e4d4\n"
                "- FORBIDDEN: dark colors (luminance < 0.55, e.g. #2c2c2c, #0e0e10), saturated/neon (#ff0066, #d4af37), pure black, near-black.\n"
                "- brand_soft = LIGHTEST (luminance 0.85-0.95) — used for soft section backgrounds (hero, CTA).\n"
                "- brand_deep = slightly DARKER (luminance 0.55-0.70) — used for emphasis. Still pastel.\n"
                "- brand_1, brand_2, brand_3 = mid-pastel accents (luminance 0.65-0.85).\n"
                "- All 5 share ONE tonal family. Pull from VISION palette but lighten/desaturate.\n\n"
                "GROUND EVERY SECTION IN THE STRATEGY \u2014 the STRATEGY JSON in the user message is a senior strategist's brief. USE it; do not treat it as background:\n"
                "- hero.h1 + hero.lead: express strategy.selling_idea in the product's own words. The H1 is the ONE promise \u2014 make it the transformation/outcome, not a spec.\n"
                "- story chapters: follow strategy.transformation \u2014 chapter 1 = the 'before' pain (strategy.pain_point / current_solutions), middle = the product as the turning point, final = the 'after' state.\n"
                "- features + stats: each should reinforce strategy.differentiator and pre-empt one of strategy.key_objections.\n"
                "- faq: answer strategy.key_objections head-on (price, longevity, side-effects, comparisons).\n"
                "- cta: fire on strategy.buying_trigger. The emotional register of ALL copy = strategy.key_emotion; the tone = strategy.voice.\n"
                "- Never contradict the strategy's positioning or persona. If a section cannot serve the strategy, SKIP it.\n"
                "\n"
                "COPY QUALITY:\n"
                "- Specific numbers > vague claims (\"93% improvement in 14 days\" not \"fast results\").\n"
                "- Editorial voice — magazine, not sales bot.\n"
                "- <em>...</em> sparingly (1-2 per heading) on emotional words. Powers italic Fraunces serif.\n"
                "- Pull facts from VISION ANALYSIS — never invent specs.\n"
                "- IMG_ALT 5-12 word descriptive (not \"image 1\").\n\n"
                "PRODUCT TITLE (meta.seo_meta.title) \u2014 THIS BECOMES THE STOREFRONT PRODUCT NAME shown in the catalog grid, on-site search, Google Shopping, the cart and the browser tab. It is the single highest-leverage sales + click-through element. Craft it like a marketplace merchandiser, NOT an SEO bot:\n"
                "- STRUCTURE: <recognizable product-type noun> + <strongest single benefit or differentiator>. The shopper must know WHAT it is in under 1 second, then WHY it is better. e.g. 'LED Light Therapy Wand \u2014 Firmer-Looking Skin in 8 Weeks'; 'Leg Compression Massager \u2014 Relief After Long Days on Your Feet'.\n"
                "- Lead with the category noun a real shopper would actually search ('Facial Steamer', 'Memory Foam Pillow') \u2014 never a brand buzzword or model code.\n"
                "- Front-load ONE concrete benefit/outcome. Title Case. Human and scannable.\n"
                "- 50-60 chars ideal, HARD cap 60 (Google truncates beyond ~60). Never pad to reach length.\n"
                "- FORBIDDEN (AliExpress/EPROLO tells that destroy CTR + trust): ALL-CAPS words, '2024/2025 New', 'Hot Sale', 'Free Shipping', emoji, spec dumps ('1500mAh 6-in-1 Multifunction'), keyword stuffing, comma-piles.\n"
                "- Include the primary search keyword naturally (helps Google Shopping + on-site search), but a HUMAN must want to click it.\n"
                "- SEO (this string is ALSO the <title> tag): put the PRIMARY KEYWORD in the first ~5 words (front-loaded ranking weight), keep every title UNIQUE across the catalog, and do NOT append the store/brand name \u2014 the storefront adds it automatically.\n"
                "META DESCRIPTION (meta.seo_meta.description) \u2014 the Google SERP snippet that wins or loses the click. Senior-SEO rules: 140-155 chars; put the PRIMARY KEYWORD + core benefit in the FIRST ~120 chars (mobile truncates ~120; Google bolds matched query terms = higher CTR); active voice; add one secondary keyword or proof hook; close with a soft pull ('See why...', 'Made for...'); UNIQUE per product; never restate the title verbatim; no price, no shipping, no double-quotes, no exclamation marks.\n"
                "SHORT DESCRIPTION (meta.short_description) \u2014 opens the body copy. Lead with the AFTER-STATE the buyer wants (the outcome/feeling), not the spec sheet. 2-3 editorial sentences.\n"
                "\n"
                "IMAGES (URL-based, picked from EPROLO source list):\n"
                "- For each photo slot in modules (hero.image_url, story.chapters[i].image_url): pick exact URL from EPROLO PHOTOS list. Copy-paste, never invent.\n"
                "- hero.image_url = the most hero-worthy EPROLO photo (clean product shot).\n"
                "- story chapters: pick supporting/lifestyle photos that match the chapter narrative.\n"
                "- image_alt: 5-12 word descriptive (not \"image 1\").\n\n"
                "REVIEWS — give 9 OR 12 items (3 per column × 3 columns × optional duplication).\n"
                "Each review: short reviewer-style sentence, mild skepticism beats over-the-top praise.\n"
                "Mix 4-star and 5-star ratings (\u2605\u2605\u2605\u2605\u2606 vs \u2605\u2605\u2605\u2605\u2605) for credibility.\n\n"
                "FAQ — 4-7 real-objection questions (price, longevity, side-effects, comparisons, etc.).\n\n"
                "VARIETY — KICKERS (CRITICAL anti-templating). The example kickers in the schema above are PLACEHOLDERS.\n"
                "For each section head.kicker, pick CONTEXTUALLY from the pools below — DO NOT use the same kicker across products:\n"
                "- hero:        kicker reflects product mood (no fixed pool — write a 2-3 word product-specific tease)\n"
                "- story:       THE SHIFT / HOW IT STARTED / FROM THERE TO HERE / OUR JOURNEY / THE STORY / BEHIND THIS\n"
                "- features:    WHY THIS ONE / THE ESSENTIALS / WHAT YOU GET / KEY POINTS / WHAT WORKS / WHAT MAKES IT\n"
                "- stats:       AT A GLANCE / BY THE NUMBERS / THE NUMBERS / QUICK FACTS / PROOF / THE FIGURES\n"
                "- reviews:     BUYERS SAY / REAL VOICES / FROM BUYERS / WHAT THEY SAY / IN THEIR WORDS / FIRST-HAND\n"
                "- faq:         QUICK ANSWERS / STRAIGHT TALK / BEFORE YOU BUY / HONEST FAQ / WHAT BUYERS ASK\n"
                "- cta:         READY TO COMMIT / YOUR CALL / NEXT MOVE / NOW THE FUN PART / WORTH A TRY\n"
                "- interlinks:  EXPLORE / KEEP LOOKING / ALSO CHECK / WHAT ELSE / SIMILAR FINDS\n"
                "- how_to:      GETTING STARTED / FIRST USE / SETUP / 3 MINUTES TO BEGIN / QUICK START\n"
                "- specs:       TECHNICAL / UNDER THE HOOD / THE BUILD / DETAILS / SPECS\n"
                "- whats_included: WHAT IS IN THE BOX / KIT CONTENTS / EVERYTHING YOU GET / IN THE BAG / THE PACKAGE\n"
                "- ingredients: KEY INGREDIENTS / WHAT IS INSIDE / THE FORMULA / SKIN BEST FRIENDS / THE BLEND\n"
                "- timeline:    YOUR JOURNEY / WHAT TO EXPECT / WEEK BY WEEK / THE TIMELINE / DAY ONE TO\n"
                "- trust:       TRUSTED BY / OUR CREDENTIALS / VERIFIED / CERTIFIED / THIRD-PARTY PROOF\n"
                "- compare:     VS ALTERNATIVES / SIDE BY SIDE / HEAD TO HEAD / WHY US / WHY THIS, NOT THAT\n"
                "- size_guide:  FIND YOUR FIT / SIZE CHART / YOUR SIZE / SIZING / GET THE FIT\n"
                "- care:        TAKING CARE / KEEP IT GOOD / CARE NOTES / MAINTENANCE / LOOKING AFTER IT\n"
                "- dimensions:  ACTUAL SIZE / IN YOUR SPACE / SCALE / DIMENSIONS / SIZE IN CONTEXT\n"
                "- variants:    PICK YOUR LOOK / COLOR OPTIONS / IN YOUR STYLE / SHADES / FINISHES\n"
                "- gift_options: GIFT-READY / FOR GIFTING / WRAP IT UP / GIFT NOTES / THE GIFT VERSION\n"
                "- safety:         BEFORE YOU START / SAFETY FIRST / IMPORTANT / READ THIS / FYI\n"
                "- protocol:       YOUR ROUTINE / THE REGIMEN / SCHEDULE / HOW OFTEN / THE RHYTHM\n"
                "- target_profile: WHO IT'S FOR / MADE FOR / IDEAL FOR / IF YOU / FIT CHECK\n"
                "- video_demo:     SEE IT IN ACTION / WATCH / 30-SECOND DEMO / IN MOTION / IN USE\n"
                "- compatibility:  WORKS WITH / PAIRS WITH / COMPATIBLE / IN YOUR ECOSYSTEM / SETUP SAVVY\n"
                "- assembly:       SETUP / OUT OF THE BOX / FROM FLATPACK / READY IN 15 / EASY BUILD\n"
                "- sustainability: GOOD FOR THE PLANET / MADE RESPONSIBLY / GREENPRINT / PLANET FIRST / OUR FOOTPRINT\n"
                "- brand_story:    WHO MADE THIS / OUR STORY / SINCE / THE MAKERS / FROM OUR TABLE\n"
                "- nutrition_facts: WHAT IS IN A SERVING / NUTRITION / FUEL FACTS / ON YOUR PLATE / PER SERVING\n"
                "- app_showcase:   IN THE APP / TRACK + TUNE / IN YOUR POCKET / CONNECTED / THE COMPANION\n"
                "- subscription_refill: NEVER RUN OUT / AUTO-REPLENISH / SUBSCRIBE & SAVE / SET IT, FORGET IT / ON A SCHEDULE\n"
                "- clinical_evidence: THE SCIENCE / PROVEN / TESTED VERIFIED / CLINICAL PROOF / EVIDENCE-BASED\n"
                "- room_placement: IN YOUR SPACE / MADE TO FIT / WHERE IT LIVES / IN CONTEXT / ROOM READY\n"
                "- lifestyle_gallery: IN THE WILD / ON REAL PEOPLE / OUT THERE / BY THE COMMUNITY / EVERYDAY\n"
                "RULE: pick the 2nd, 3rd, or 4th option from each pool more often than the 1st — over batches the\n"
                "variety distributes naturally. If voice is 'witty-irreverent' you may invent a kicker that\n"
                "fits the voice. The mood of the kicker should match strategy.voice.\n\n"
                "VARIETY — REVIEW NAME FORMATS. Mix at LEAST 3 different formats across your 9 or 12 reviews:\n"
                "  - Standard: 'Maya R.' or 'James K.'\n"
                "  - Handle:   '@maya_runs' or '@jkay' (lowercase)\n"
                "  - City:     'Maya (Toronto)' or 'James (San Diego)'\n"
                "  - Verified: 'Maya, verified buyer' or 'James, first-time buyer'\n"
                "  - Initials: 'M.R.' or 'J.K.'\n"
                "  - Age:      'Maya R., 30s' or 'James K., late 40s'\n"
                "Do NOT use 'firstname + lastinitial' for ALL reviews — that is the most recognised AI tell.\n\n"
                "VARIETY — CTA BUTTON LABEL. Pick by strategy.voice (NOT default 'Add to cart'):\n"
                "  - warm-confidant       → 'Take it home'\n"
                "  - witty-irreverent     → 'Grab one'  (or 'Yes please')\n"
                "  - clinical-precise     → 'Add to cart'\n"
                "  - editorial-thoughtful → 'Make it yours'\n"
                "  - aspirational-luxury  → 'Acquire this'\n"
                "  - down-to-earth-honest → 'Get this one'\n\n"
                "INTERLINKS:\n"
                "- Include EVERY handle from AVAILABLE COLLECTIONS in interlinks.items.\n"
                "- Anchor text: prefer the anchors[] entry over the title.\n\n"
                "OPTIONAL MODULES — fill ONLY when product fits, otherwise OMIT field or set null:\n"
                "- how_to: 3-7 numbered steps. FILL for: skincare/wellness rituals, food recipes, electronics setup,\n"
                "    hobby starter instructions, supplement protocols, fitness routines.\n"
                "    SKIP for: fashion accessories, decor, ready-to-eat snacks, non-action items.\n"
                "- specs: 6-12 row table {label, value}. FILL for: electronics, kitchen appliances, sports gear with\n"
                "    measurable specs, jewelry materials/dimensions, tools, lighting (lumens/wattage).\n"
                "    SKIP for: skincare/food/perfume (use ingredients instead), apparel (use size_guide-style notes in features).\n"
                "- whats_included: 3-8 items {glyph, name, qty, note}. FILL for: bundles, kits, gift sets, hobby starter packs,\n"
                "    multi-piece skincare/cosmetics sets, cookware sets.\n"
                "    SKIP for: single-piece products.\n"
                "- ingredients: 4-10 chips {name, role, badge}. FILL for: skincare actives (Hyaluronic Acid: hydrator, [HERO]),\n"
                "    food/supplements/candles. role describes function (\"hydrator\", \"antioxidant\", \"main note\").\n"
                "    badge: optional [HERO]/[ORGANIC]/[CLINICAL] tag.\n"
                "    SKIP for: electronics, accessories, decor, anything without notable composition.\n"
                "- timeline: 3-5 milestones {when, h4, p}. FILL for: products with measurable transformation over time —\n"
                "    skincare results, fitness/posture, supplements, hair growth.\n"
                "    SKIP for: instant-effect products, non-transformation categories.\n"
                "- trust: certs[] + press[]. FILL for: regulated categories (baby, health, food safety),\n"
                "    eco/sustainability claims, premium positioning, clinical/medical-grade. Use real-sounding labels:\n"
                "    \"FDA-registered\", \"USDA Organic\", \"Dermatologist-tested\", \"As seen in Vogue/Wired\".\n"
                "    SKIP for: ordinary commodity products without notable certifications.\n"
                "- compare: 2-3 columns × 4-7 rows. FILL for: premium positioning vs generic alternatives,\n"
                "    upgrade messaging vs basic version, research-buy categories where buyers compare specs.\n"
                "    SKIP for: low-stakes impulse purchases, gift items.\n"
                "- size_guide: 2-5 measurement rows × N size labels. FILL for: clothing/apparel, jewelry rings or chains\n"
                "    (sizes by inner diameter), pet collars/harnesses (by pet weight), baby/kids clothing, sports gear.\n"
                "    SKIP for: one-size-fits-all, electronics, decor, food.\n"
                "- care: 3-6 instruction chips {glyph, label, note}. FILL for: fashion (\"hand wash cold\"),\n"
                "    jewelry (\"avoid water\"), kitchen tools (\"hand wash, oil periodically\"), leather goods, textiles.\n"
                "    SKIP for: digital products, single-use items, electronics (use specs instead).\n"
                "- dimensions: 3-5 axes {axis, value} + optional scale_note. FILL for: wall decor (sizes for placement),\n"
                "    furniture, organization/storage, lighting (size in space), wall art, books.\n"
                "    SKIP for: skincare, food, supplements, anything where specs already covers measurements.\n"
                "- variants: 2-8 color/pattern swatches {name, color hex, available bool}. FILL for: fashion (multi-color\n"
                "    apparel), accessories (bag colors), decor (cushion colors), anything sold in multiple visual variants.\n"
                "    SKIP for: single-color products, food, supplements, specs-driven products.\n"
                "- gift_options: wrap{} + card{} + occasions[]. FILL for: section 2 (gifts-by-recipient), section 11\n"
                "    (specific events), section 10 (bundle-types), or anything tagged intent:gift. wrap.options 2-3,\n"
                "    card.examples 2-3 message templates, occasions 3-6 (Birthday, Anniversary, Mother\u2019s Day, etc.).\n"
                "    SKIP for: utility products, electronics not typically gifted, B2B-style items.\n"
                "- safety: warnings + age_rating + contraindications + safety_certs. FILL for: beauty devices (LED,\n"
                "    microcurrent, RF, IPL), supplements, kids toys (CPSIA / ASTM-F963 age ratings), kitchen appliances\n"
                "    with sharp/hot parts, fitness equipment with injury risk, anything FDA-regulated.\n"
                "    SKIP for: fashion accessories, decor, candles, jewelry, clothing.\n"
                "- protocol: schedule[] of usage phases. Distinct from how_to (one-time setup) and timeline\n"
                "    (transformation milestones). FILL for: beauty devices (treatment schedule), supplements (dosing\n"
                "    cycles), fitness (workout program), wellness rituals (daily/weekly rhythm).\n"
                "    SKIP for: one-time-use products, decor, accessories.\n"
                "- target_profile: ideal_for + not_ideal_for + category-tag (fitness_level/skin_type/age_range).\n"
                "    FILL for: fitness (skill level), beauty (skin type/age), supplements (lifestyle goal),\n"
                "    toys (age range), pet products (species/size).\n"
                "    SKIP for: broad-appeal commodity, decor, food.\n"
                "- video_demo: embedded product video (youtube/vimeo/direct).\n"
                "    FILL for: fitness (form demos), beauty devices (application demos), toys (play patterns),\n"
                "    tech (UI walkthroughs), kitchen tools (cooking demos).\n"
                "    SKIP for: apparel, accessories, decor, supplements.\n"
                "- compatibility: platforms[] + accessories[] + incompatibility_warnings[]. Distinct from\n"
                "    app_showcase (which is COMPANION-APP). FILL for: phone accessories (iOS/Android), smart home\n"
                "    (HomeKit/Alexa/Google), fitness wearables, beauty replaceable heads.\n"
                "    SKIP for: standalone non-tech products.\n"
                "- assembly: time + tools + difficulty for one-time setup. Distinct from how_to (ongoing usage).\n"
                "    FILL for: furniture (beds, desks), fitness equipment (foldable bikes, racks), kids construction\n"
                "    toys, decor requiring mounting.\n"
                "    SKIP for: ready-to-use products.\n"
                "- sustainability: certs[] + materials[] + carbon + recycling. Distinct from trust (positive\n"
                "    press/awards) and safety (cautionary warnings). FILL for: skincare/food with eco claims,\n"
                "    fashion (recycled fabric, Fair Trade), decor (FSC wood), pet (sustainably sourced).\n"
                "    SKIP for: pure tech, generic commodity.\n"
                "- brand_story: COMPANY narrative — founded + mission + values + founder_quote. Distinct from\n"
                "    story (PRODUCT problem→solution narrative). FILL for: premium positioning, indie/artisan\n"
                "    products, eco brands, family-business positioning.\n"
                "    SKIP for: commodity, dropship, unbranded.\n"
                "- nutrition_facts: FDA-style label box — serving + macros + highlights + allergens.\n"
                "    FILL for: food, beverages, supplements, protein powders, snacks.\n"
                "    SKIP for: anything non-edible / non-ingestible.\n"
                "- app_showcase: app_name + store_links + screens + features. Distinct from compatibility\n"
                "    (platform support). FILL for: smart home, fitness wearables, beauty devices with apps,\n"
                "    connected kitchen/sleep/wellness products.\n"
                "    SKIP for: non-connected products.\n"
                "- subscription_refill: VISUAL Subscribe & Save messaging — lasts_days + discount + perks.\n"
                "    Actual subscription commerce logic is operator (Recharge/Skio/Shopify Subscriptions); this\n"
                "    module just communicates value. FILL for: skincare/supplements/coffee/food/pet food/contact\n"
                "    lenses — anything refilled.\n"
                "    SKIP for: one-time-buy products.\n"
                "- clinical_evidence: scientific claims + mechanism. Distinct from trust (press/awards) and\n"
                "    safety (warnings). FILL for: beauty devices, supplements, wellness products with clinical\n"
                "    claims.\n"
                "    SKIP for: generic / commodity.\n"
                "- room_placement: visual scale + room-fit. Distinct from dimensions (numbers only).\n"
                "    FILL for: furniture, lamps, wall decor, mirrors, rugs, large kitchen appliances.\n"
                "    SKIP for: small / portable items.\n"
                "- lifestyle_gallery: UGC-mood photography grid. Distinct from story chapter images\n"
                "    (narrative-anchored). FILL for: fashion, accessories, candles, decor, wellness — anything\n"
                "    where aesthetic = part of the value.\n"
                "    SKIP for: utility products.\n\n"
                "RULE: never fabricate certs/specs/ingredients. Pull only what VISION ANALYSIS confirmed (packaging text,\n"
                "claims, ingredients_visible, certifications fields). If vision found nothing — skip the module.\n\n"
                "RULE (SHIPPING / RETURNS / REFUND): NEVER mention shipping cost, delivery times, return policy,\n"
                "refund terms, 'secure checkout', money-back guarantees, or warranty terms anywhere — not in FAQ,\n"
                "not in features, not in trust, not in hero/lead, not in CTA copy. These concerns are handled by a\n"
                "separate Shopify checkout-page module. Mentioning them on the PDP creates contradictions with the\n"
                "actual stored policy and kills buyer trust on inspection.\n\n"
                "RULE (COMPARE MODULE — anti-hallucination): The compare matrix is high-risk because Designer has\n"
                "no source-of-truth for competitor numbers. For row values, prefer Yes/No/Partial cells over\n"
                "specific numbers (battery hours, dB, prices, weights). If you must include a number for the OWN\n"
                "column, ONLY use values that appeared verbatim in VISION ANALYSIS or scrape facts. For competitor\n"
                "columns, NEVER invent specific numbers — leave as Yes/No or omit the row. A buyer who fact-checks\n"
                "an invented 'Handheld fan: 40-50dB' competitor stat finds nothing and trust collapses. Better to\n"
                "skip the compare module entirely than to fill it with plausible-sounding fakes.\n\n"
                "RULE (SPEC SOURCE-TRACING): Every spec row value must be traceable to a specific evidence source.\n"
                "Allowed sources, in order of priority: (1) scrape specs/variants dict, (2) VISION packaging_text\n"
                "claims/ingredients/instructions, (3) VISION physical key_features. If a candidate value isn't in\n"
                "any of those, OMIT THE ROW. Prefer a 4-row honest specs table over a 12-row half-invented one.\n\n"
                "TAG RULES:\n"
                "- meta.product_tags from AVAILABLE CLUSTERS / PERSONAS / INTENTS / DEMOS lists ONLY.\n"
                "- 2-4 cluster + 1-2 persona + 1-2 intent + 1 demo:gender + 1-2 demo:age. Total 6-9. Never invent.\n\n"
                "PHOTO_BRIEFS — for the photo_pack ZIP that ships with each product:\n"
                "- TOTAL CAP: 7-8 briefs MAX per product. ChatGPT/Ideogram batch limit is ~8 photos.\n"
                "  Going over means the operator has to split into more batches, painful.\n"
                "- BUDGET split: 5 carousel + 2-3 inline. Carousel is the conversion driver\n"
                "  (every buyer sees thumbnails; most never scroll to metafield-inline photos).\n"
                "  Spending the budget on more inline at the expense of carousel = wasted effort.\n"
                "- Carousel: emit EXACTLY 5 briefs (the full funnel: hero, lifestyle, in-use, detail, scale).\n"
                "  Drop one ONLY if the product genuinely has no usable source photo for that slot.\n"
                "- Inline: emit AT MOST 3 briefs. Prioritise inline-hero (page hero block), then\n"
                "  inline-story-1 (most-visible story chapter), then ONE more if it adds real value.\n"
                "  Skip inline-story-2, inline-story-3, inline-feature, inline-cta unless absolutely needed.\n"
                "- Each brief shape: id, slot, source_index (1-N from EPROLO PHOTOS list), concept (one sentence what the edited photo shows), edit_instructions (concrete edit ask), slot_context (ONE sentence: what buyer-question this photo answers OR what page text it sits next to).\n"
                "- NO CLONES: inline briefs MUST use a DIFFERENT source_index than EVERY carousel brief whenever the EPROLO list allows. Reuse a source_index across carousel and inline ONLY with a DRAMATICALLY different concept AND edit_instructions (different scene, crop, people, angle, context) — same source_index + similar concept = duplicate-looking photos, FORBIDDEN. Carousel = clean studio-style product shots; inline = lifestyle / narrative / in-use scenes. They must look clearly different.\n"
                "- Skip a slot entirely rather than duplicate weakly. 5 carousel + 2 strong inline > 12 weak briefs.\n\n"
                "IMAGE LOOK \u2014 LIGHT BACKGROUNDS ONLY (operator rule, NON-NEGOTIABLE \u2014 same as video):\n"
                "- The store is LIGHT. Every generated/edited photo AND the visual_style paragraph that governs them MUST use a bright white, pastel, or airy-daylight background. ABSOLUTELY NO black, dark, charcoal, moody or 'dramatic studio' background \u2014 any slot, any brief.\n"
                "- visual_style: describe ONE consistent light set \u2014 soft diffused daylight, gentle natural shadows, clean white/cream/pastel seamless OR a tasteful light lifestyle surface (marble, pale wood, linen). Pull 1-2 soft accent colors from the product palette. Editorial e-commerce quality, uncluttered.\n"
                "- Every brief's edit_instructions MUST name the background explicitly (e.g. 'pure white seamless', 'soft blush-cream gradient', 'bright sunlit kitchen counter') so no photo ever defaults to a dark background.\n\n"

                "CAROUSEL CONVERSION FUNNEL — 5 slots in this order, each answers one buyer question in <2s:\n"
                "  1. carousel-hero      → \"What IS it?\" Clean product hero placed ON the product's PHOTO STYLING THEME surface/backdrop (NOT generic white seamless — use the theme's surface, light mood and camera angle so EVERY product's hero looks visibly different). Buyer recognises product type instantly. MANDATORY for ALL carousel + inline briefs: build each brief's concept AND edit_instructions FROM the PHOTO STYLING THEME in the user prompt (its exact surface, props, palette, light, angle, composition). NEVER reuse 'white seamless / clean product on white' across products — two different products must not share the same background or composition.\n"
                "  2. carousel-lifestyle → \"Who uses it / when?\" Real scene with target persona, natural light. Buyer projects into situation.\n"
                "  3. carousel-in-use    → \"How does it work?\" Hand applying / dispensing / mid-action. Removes \"don\u2019t know how to use it\" objection.\n"
                "  4. carousel-detail    → \"Is it quality?\" Macro of texture / finish / key feature / ingredient label. Builds trust.\n"
                "  5. carousel-scale     → \"How big? Where does it fit?\" Size next to hand / counter / packaged with bottle/box. Removes \"is it tiny/huge\" objection.\n"
                "  GOAL: by the end of these 5 the buyer should want to buy with no remaining questions. Each photo earns its slot or gets cut.\n\n"
                "INLINE SLOT SEMANTICS — match the placement context inside the page narrative:\n"
                "  - inline-hero       → goes into the hero CONTENT block. It MUST be a LIFESTYLE / in-context / emotional scene that reinforces the H1 — NEVER a clean product-on-background shot, NEVER the same source_index or look as carousel-hero. If carousel-hero is the clean product, inline-hero shows the product BEING USED in a real setting.\n"
                "  - inline-story-1    → first story chapter — usually the BEFORE/PROBLEM state. Show the pain or the failing alternative.\n"
                "  - inline-story-2    → second story chapter — usually the DISCOVERY/SOLUTION moment. Show the product entering the scene.\n"
                "  - inline-story-3    → third story chapter — usually the TRANSFORMATION/RESULT state. Show life with product.\n"
                "  - inline-feature    → next to features grid; should illustrate ONE specific feature mentioned in features.items[].h4.\n"
                "  - inline-cta        → near final CTA; aspirational close-up that reinforces purchase decision.\n"
                "  - INLINE = ZERO TEXT: every inline brief is editorial in-page imagery. Its concept AND edit_instructions MUST NOT request overlay text, callouts, infographics, badges, specs, step numbers, labels, arrows or annotations (those belong to CAROUSEL only). A brief that requests 'infographic / callout / overlay / step / badge' for an inline slot is INVALID; rewrite it as a pure mood / lifestyle scene.\n"
                "  RULE: each inline brief\u2019s slot_context MUST quote the actual H2/h4/heading from the corresponding section so the editor sees what TEXT the photo accompanies.\n\n"
                "REVIEW_SCENES (meta-level, EXACTLY 8 items) — UGC phone-snapshot scene ideas for the Judge.me/Loox review widget. NOT studio: imitate a real owner's candid iPhone shot. Ground EVERY scene in where and how a real owner of THIS product would use or show it; derive place and activity from the product category, never a generic template: fishing rod -> riverbank with the catch / casting at dawn; cordless drill -> mid-repair on a workbench / install at home; perfume -> vanity before a night out / in the handbag / gifted; kids toy -> child mid-play on the rug / birthday. All 8 DISTINCT places, real and lived-in, slightly cluttered, NO white seamless, NO studio light.\n\n"
                "============================================================\n"
                "POLICY — FILL EXISTING FIRST, EMIT NEW ONLY IF TRULY NEEDED\n"
                "============================================================\n"
                "1. FIRST PASS: go through every section in the schema above\n"
                "   (hero, story, features, how_to, protocol, timeline, safety,\n"
                "   target_profile, brand_story, lifestyle_gallery, ingredients,\n"
                "   clinical_evidence, compare, gift_options, subscription_refill,\n"
                "   sustainability, app_showcase, unboxing_journey,\n"
                "   progress_milestones, mistake_warnings, use_scenarios, etc).\n"
                "   For EACH section: decide FILL or SKIP based on whether the\n"
                "   product category + scrape data + strategy genuinely support it.\n"
                "   AIM HIGH — most beauty/skincare products realistically support\n"
                "   18-25 of the 40+ defined sections. Don't be conservative.\n"
                "\n"
                "2. NEVER DUPLICATE: if how_to already covers ritual steps, do not\n"
                "   emit a parallel custom_ritual or my_steps key. If protocol\n"
                "   covers the schedule, do not invent a routine_breakdown key.\n"
                "   Re-read the existing schemas and check coverage before emitting.\n"
                "\n"
                "3. NEW KEYS — VERY RARE EXCEPTION:\n"
                "   Only emit a new top-level key when ALL three are true:\n"
                "     a) The content adds REAL buyer value (would directly help\n"
                "        conversion or reduce returns)\n"
                "     b) NONE of the existing 40+ schemas can hold this content\n"
                "        without distortion\n"
                "     c) The content is FACTUAL (from scrape/strategy/vision),\n"
                "        not invented to justify a new section\n"
                "   Typical run: ZERO new keys per product. If you find yourself\n"
                "   emitting more than 1 new key, you are probably duplicating an\n"
                "   existing section. Re-read the schema list.\n"
                "\n"
                "4. SHAPE FOR NEW KEYS (REQUIRED — the storefront auto-renderer\n"
                "   only handles this exact shape):\n"
                "     {\n"
                "       \"head\": {\"kicker\": \"SHORT UPPERCASE TAG\", \"h2\": \"Section title\"},\n"
                "       \"<plural_array_name>\": [\n"
                "         {\"<first_field>\": \"main title/name\",\n"
                "          \"<other_field>\": \"detail/value\",\n"
                "          ...}\n"
                "       ]\n"
                "     }\n"
                "   Conventions for new keys:\n"
                "     - key name: lowercase_snake_case, descriptive plural\n"
                "       (e.g. body_routine, packaging_breakdown, fit_finder)\n"
                "     - head.kicker: 1-3 words ALL CAPS\n"
                "     - head.h2: 4-10 words, sentence case\n"
                "     - array name: descriptive plural (steps, phases, chapters,\n"
                "       items, warnings, scenarios, options, layers)\n"
                "     - first field in each array item = title/name (rendered as h3)\n"
                "     - if a step-number is needed, name it 'step_number' or 'order'\n"
                "       (renderer recognises these as numeric badges)\n"
                "     - 3-6 items per array (under 3 = section feels thin, over 6 = noise)\n"
                "\n"
                "5. DO NOT emit keys for: rating, trust_strip, videos,\n"
                "   research_refs, palette — those are filled by other agents\n"
                "   (Opus Strategy, manual review, etc) outside Designer.\n"
                "\n"
                "============================================================\n"
                "\n"
                "OUTPUT: single valid JSON. Begin with {. End with }. NO markdown fences.\n"
                "CRITICAL: the // comments in the schema example above are FOR YOUR REFERENCE ONLY. "
                "Your actual output MUST be strictly valid JSON — no // comments, no trailing commas, "
                "no JS-style annotations. The pipeline will reject malformed JSON."
                + HTML_CLEAN_OUTPUT_RULES
            )
            # Pull strategy from DB (saved by Step 3.5)
            _strategy = json.loads(db.execute('SELECT strategy_json FROM products WHERE id=?', (pid,)).fetchone()[0] or '{}')
            # Build EPROLO photo source list — designer picks source_image_index per brief
            _img_data_for_designer = json.loads(db.execute('SELECT image_urls_json FROM products WHERE id=?',(pid,)).fetchone()[0])
            _stage1_for_designer   = json.loads(db.execute('SELECT stage1_json FROM products WHERE id=?',(pid,)).fetchone()[0] or '{}')
            _all_eprolo_d = (_img_data_for_designer.get('top') or []) + (_img_data_for_designer.get('desc') or [])
            if len(_all_eprolo_d) > 14:
                print(f'    \u26a0 EPROLO has {len(_all_eprolo_d)} photos but Designer sees first 14 (token-budget cap). Photos 15-{len(_all_eprolo_d)} ignored for source_index selection.')
            _vision_imgs_d = {ii.get('index'): ii for ii in _stage1_for_designer.get('images', [])}
            _eprolo_list = []
            for _i, _img in enumerate(_all_eprolo_d[:14]):
                _vi = _vision_imgs_d.get(_img.get('index'), {})
                _eprolo_list.append({
                    "index": _i + 1,
                    "url": _img.get('url',''),
                    "role": _vi.get('role','unknown'),
                    "shows": _vi.get('shows',''),
                })
            _designer_user = _product_context + (
                "\n\n=== MARKETING STRATEGY (from Strategy Agent — frames ALL copy) ===\n"
                + json.dumps(_strategy, ensure_ascii=False, indent=2) +
                "\n\n=== EPROLO SOURCE PHOTOS (pick source_image_index 1-N per brief) ===\n"
                + json.dumps(_eprolo_list, ensure_ascii=False, indent=2) +
                "\n\n=== PALETTE HINT (from vision — refine to pastel range, MUST match strategy.voice mood) ===\n"
                + json.dumps(_palette_hint) +
                "\n\nINSTRUCTIONS:\n"
                "- hero.h1 = strategy.selling_idea (or close paraphrase). hero.lead echoes the transformation.\n"
                "- story.chapters: each chapter is one beat from strategy.narrative_arc.\n"
                "- features.items: 3 features that prove strategy.differentiator. Each feature.image_url = a DISTINCT EPROLO photo (use the EPROLO SOURCE PHOTOS list; pick in-use / detail / packaging shots so the 3 cards are visually different, Oura-style). Reuse the hero photo for none of them.\n"
                "- stats.items: 4 measurable proof points addressing strategy.key_objections.\n"
                "- reviews.items: 9-12 reviews demonstrating the transformation. Voice = strategy.voice.\n"
                "- faq.items: 4-7 questions addressing strategy.key_objections directly.\n"
                "- cta: end-of-funnel resolution beat from strategy.narrative_arc.\n"
                "- Voice in ALL copy = strategy.voice. Don’t mix tones.\n"
                "- For each photo slot in modules: pick exact image_url from EPROLO PHOTOS list above (use the .url field of an entry)."
            )
            # ── Inject dynamic-schema awareness via USER message ──
            # CRITICAL: append to user message NOT system, otherwise
            # cache_control:ephemeral on system breaks (every product would
            # become a fresh cache write = +$0.30/call regression).
            if _DYNAMIC_DESIGNER_SCHEMAS:
                _dyn_block = [
                    "",
                    "============================================================",
                    "ADDITIONAL KEYS AVAILABLE (auto-created by earlier products)",
                    "============================================================",
                    "These keys ALREADY EXIST as Shopify metafield definitions.",
                    "You SHOULD use them if the current product genuinely supports",
                    "the same content shape — keeps the storefront consistent.",
                    "MUST match the cached SHAPE exactly (same top-level fields,",
                    "same array name) — drift creates rendering inconsistency.",
                    "",
                ]
                # Cap at 25 to keep token budget reasonable; trim sample to 600 chars each
                for _dk, _ds in list(_DYNAMIC_DESIGNER_SCHEMAS.items())[:25]:
                    _sample_str = json.dumps(_ds, ensure_ascii=False, indent=2)
                    if len(_sample_str) > 600:
                        _sample_str = _sample_str[:600] + '...'
                    _dyn_block.append(f'  "{_dk}": {_sample_str}')
                    _dyn_block.append("")
                _designer_user = _designer_user + "\n" + "\n".join(_dyn_block)
                print(f'    Injecting {len(_DYNAMIC_DESIGNER_SCHEMAS)} dynamic-schema key(s) into Designer prompt (user content, cache-preserving)')

            # Stream the Designer response — token-by-token, no 10-min read
            # timeout that kills long-generation calls. With 16K output @
            # 30 tok/sec the call can legitimately take 8-10 min; chunks
            # arrive every ~50ms so the connection stays alive forever.
            if DESIGNER_PROVIDER == 'deepseek':
                raw2, r2 = await _deepseek_designer(_designer_system, _designer_user, 16000)
                cost2 = r2.usage.input_tokens * 0.435 / 1e6 + r2.usage.output_tokens * 0.87 / 1e6  # DeepSeek V4 Pro
            else:
                async with client_async.messages.stream(
                    model=_designer_model, max_tokens=16000, timeout=1800.0,
                    system=[{"type":"text","text":_designer_system,"cache_control":{"type":"ephemeral","ttl":"1h"}}],
                    messages=[{"role":"user","content":_designer_user}]) as _stream:
                    _parts = []
                    async for _chunk in _stream.text_stream:
                        _parts.append(_chunk)
                    raw2 = ''.join(_parts).strip()
                    r2 = await _stream.get_final_message()
                cost2 = r2.usage.input_tokens * 3 / 1e6 + r2.usage.output_tokens * 15 / 1e6  # Sonnet 4.6 pricing
            _track_anthropic_response(r2, cost2)

            # Parse JSON (strip code-fence + comments — designer schema has //-style
            # examples that the model sometimes mimics in output, breaking strict json.loads).
            if raw2.startswith("```"):
                raw2 = raw2.split("\n", 1)[-1].rsplit("\n", 1)[0]

            def _strip_jsonc_comments(s: str) -> str:
                # Remove `// rest of line` comments. Ignore `//` inside string literals
                # by tracking a simple in-string state.
                out, i, n = [], 0, len(s)
                in_str = False; esc = False
                while i < n:
                    ch = s[i]
                    if esc:
                        out.append(ch); esc = False; i += 1; continue
                    if ch == '\\' and in_str:
                        out.append(ch); esc = True; i += 1; continue
                    if ch == '"':
                        in_str = not in_str
                        out.append(ch); i += 1; continue
                    if (not in_str) and ch == '/' and i + 1 < n and s[i+1] == '/':
                        # skip until newline
                        nl = s.find('\n', i)
                        if nl == -1: break
                        i = nl
                        continue
                    out.append(ch); i += 1
                return ''.join(out)

            def _parse_designer(raw: str):
                # Try strict first, then comment-stripped, then comment-stripped+regex-extracted.
                for attempt in ("strict", "stripped", "stripped+regex"):
                    try:
                        if attempt == "strict":
                            return json.loads(raw)
                        if attempt == "stripped":
                            return json.loads(_strip_jsonc_comments(raw))
                        if attempt == "stripped+regex":
                            cleaned = _strip_jsonc_comments(raw)
                            _jm = re.search(r'\{.*\}', cleaned, re.DOTALL)
                            if _jm:
                                return json.loads(_jm.group(0))
                    except json.JSONDecodeError:
                        continue
                return {}

            designer_resp = _parse_designer(raw2)

            # Retry once with error context if parse + 3-attempt fallback all yielded
            # an empty/incomplete response (no hero — the most reliable always-on field).
            if not designer_resp or not designer_resp.get('hero'):
                print('    \u26a0 Designer JSON invalid — retrying once with error context...')
                _retry_user = _designer_user + (
                    "\n\n### PREVIOUS ATTEMPT FAILED — TRY AGAIN ###\n"
                    "Your previous response was not valid JSON. Common errors to avoid:\n"
                    "- // comments inside JSON (FORBIDDEN — the schema example shows them as references only)\n"
                    "- trailing commas before } or ]\n"
                    "- unescaped quotes inside string values\n"
                    "- truncated output (incomplete JSON object)\n\n"
                    "First 500 chars of broken response:\n"
                    + (raw2 or '')[:500] + "\n\n"
                    "RETURN ONLY VALID JSON. NO COMMENTS. NO MARKDOWN FENCES. Begin with { end with }."
                )
                try:
                    # Streaming retry — same no-timeout protection as main call.
                    if DESIGNER_PROVIDER == 'deepseek':
                        _raw2_retry, _r2_retry = await _deepseek_designer(_designer_system, _retry_user, 16000)
                        _retry_cost_d = _r2_retry.usage.input_tokens * 0.435 / 1e6 + _r2_retry.usage.output_tokens * 0.87 / 1e6
                    else:
                        async with client_async.messages.stream(
                            model=_designer_model, max_tokens=16000, timeout=1800.0,
                            system=[{"type":"text","text":_designer_system,"cache_control":{"type":"ephemeral","ttl":"1h"}}],
                            messages=[{"role":"user","content":_retry_user}]) as _stream2:
                            _parts2 = []
                            async for _chunk2 in _stream2.text_stream:
                                _parts2.append(_chunk2)
                            _raw2_retry = ''.join(_parts2).strip()
                            _r2_retry = await _stream2.get_final_message()
                        _retry_cost_d = _r2_retry.usage.input_tokens * 3 / 1e6 + _r2_retry.usage.output_tokens * 15 / 1e6
                    if _raw2_retry.startswith("```"):
                        _raw2_retry = _raw2_retry.split("\n", 1)[-1].rsplit("\n", 1)[0]
                    cost2 += _retry_cost_d
                    _track_anthropic_response(_r2_retry, _retry_cost_d)
                    designer_resp = _parse_designer(_raw2_retry)
                    if designer_resp.get('hero'):
                        print('    \u2713 Retry succeeded')
                    else:
                        print('    \u2717 Retry also failed \u2014 product will save with empty content')
                except Exception as _e_retry:
                    print(f'    \u2717 Retry call failed: {str(_e_retry)[:120]}')

            # ── Capture any NEW keys Designer emitted (not in hardcoded schema)
            # 1. Add to sections dict → they get pushed to Shopify in Step 5
            # 2. Cache shape into _DYNAMIC_DESIGNER_SCHEMAS (with shape lock!)
            # 3. Persist + audit-log every new-key event
            _new_keys_emitted = []
            _drifted_keys = []
            for _nk, _nv in (designer_resp or {}).items():
                if _nk in _DESIGNER_HARDCODED_KEYS:
                    continue
                if not isinstance(_nv, (dict, list)) or not _nv:
                    continue
                _new_fp = _shape_fingerprint(_nv)
                if _nk in _DYNAMIC_DESIGNER_SCHEMAS:
                    # Existing dynamic key — verify shape matches
                    _cached_fp = _shape_fingerprint(_DYNAMIC_DESIGNER_SCHEMAS[_nk])
                    if _new_fp != _cached_fp:
                        _drifted_keys.append((_nk, _cached_fp, _new_fp))
                        # KEEP cached version as canonical; current value still
                        # gets pushed but renderer will treat it consistently
                        _audit_log({
                            'event': 'shape_drift', 'product_id': pid,
                            'product_title': product.get('title','')[:80],
                            'key': _nk,
                            'cached_shape': str(_cached_fp),
                            'new_shape': str(_new_fp),
                        })
                    # Still update sections so this product's data is pushed
                else:
                    # First emission of this key — set as canonical
                    _DYNAMIC_DESIGNER_SCHEMAS[_nk] = _nv
                    _new_keys_emitted.append(_nk)
                    _audit_log({
                        'event': 'new_key', 'product_id': pid,
                        'product_title': product.get('title','')[:80],
                        'key': _nk,
                        'shape': str(_new_fp),
                        'sample': json.dumps(_nv, ensure_ascii=False)[:500],
                    })
            if _new_keys_emitted:
                print(f'    Designer emitted NEW keys: {_new_keys_emitted} (cached + audit-logged)')
            if _drifted_keys:
                print(f'    ⚠ SHAPE DRIFT detected on {len(_drifted_keys)} key(s): {[k for k,_,_ in _drifted_keys]} — see audit log')

            # Persist dynamic schemas after each product (atomic, idempotent)
            if _new_keys_emitted:
                _save_dynamic_schemas(_DYNAMIC_DESIGNER_SCHEMAS)

            # Extract sections (each becomes a separate JSON metafield)
            sections = {
                'hero':           designer_resp.get('hero')           or {},
                'story':          designer_resp.get('story')          or {},
                'features':       designer_resp.get('features')       or {},
                'stats':          designer_resp.get('stats')          or {},
                'reviews':        designer_resp.get('reviews')        or {},
                'faq':            designer_resp.get('faq')            or {},
                'cta':            designer_resp.get('cta')            or {},
                'palette':        designer_resp.get('palette')        or _palette_hint,
                # Override Designer's interlinks — он галлюцинирует handles
                # ('makeup-basics', 'style-y2k') которых нет в магазине.
                # Принудительно берём реальный список из get_smart_interlinks
                # (`interlinks` var выше). Designer's `head` сохраняем.
                'interlinks':     {
                    'head': (designer_resp.get('interlinks') or {}).get('head') or {'kicker': 'EXPLORE MORE', 'h2': 'Explore more'},
                    'items': [
                        {'handle': l['handle'],
                         'anchor': (l.get('anchors')[0] if isinstance(l.get('anchors'), list) and l.get('anchors') else (l.get('anchor') or l.get('title', '')))}
                        for l in (interlinks or [])
                    ],
                },
                # Optional modules — designer fills only when relevant. Empty {} → snippet skips.
                'how_to':         designer_resp.get('how_to')         or {},
                'specs':          designer_resp.get('specs')          or {},
                'whats_included': designer_resp.get('whats_included') or {},
                'ingredients':    designer_resp.get('ingredients')    or {},
                'timeline':       designer_resp.get('timeline')       or {},
                'trust':          designer_resp.get('trust')          or {},
                'compare':        designer_resp.get('compare')        or {},
                'size_guide':     designer_resp.get('size_guide')     or {},
                'care':           designer_resp.get('care')           or {},
                'dimensions':     designer_resp.get('dimensions')     or {},
                'variants':       designer_resp.get('variants')       or {},
                'gift_options':   designer_resp.get('gift_options')   or {},
            }
            # Include EVERY key Designer emitted that isn't in the hardcoded
            # sections dict (incl. both FIRST-time auto-created AND REPEAT
            # emissions of dynamic keys from previous products). Without
            # this, product N+1's value for an existing dynamic key gets
            # dropped instead of pushed to Shopify.
            for _ek, _ev in (designer_resp or {}).items():
                if _ek in _DESIGNER_HARDCODED_KEYS:
                    continue
                if not isinstance(_ev, (dict, list)) or not _ev:
                    continue
                sections[_ek] = _ev

            meta = designer_resp.get('meta') or {}
            # Tuck visual_style into meta so STEP 4.5 (photo pack) can read it from final_html.meta
            meta['visual_style'] = (designer_resp.get('visual_style') or '').strip()
            # photo_briefs[] — Designer picks 5-12 briefs for the photo_pack ZIP. Stored in meta.
            meta['photo_briefs'] = designer_resp.get('photo_briefs') or []
            meta['review_scenes'] = designer_resp.get('review_scenes') or []
            seo_m = meta.get('seo_meta', {})
            # ── #2 quality: prefer premium Strategy (Opus 4.8) for the highest-leverage
            # customer-facing strings (title / meta description / hero); DeepSeek fills the rest. ──
            if _strategy.get('seo_title'):       seo_m['title'] = _strategy['seo_title']
            if _strategy.get('seo_description'): seo_m['description'] = _strategy['seo_description']
            _strat_hero = _strategy.get('hero') or {}
            if _strat_hero and isinstance(sections.get('hero'), dict):
                for _hk in ('kicker', 'h1', 'lead'):
                    if _strat_hero.get(_hk):
                        sections['hero'][_hk] = _strat_hero[_hk]

            # Quick sanity check for visibility (count items per section)
            _hero_ok = bool(sections['hero'].get('h1'))
            _story_n = len((sections['story'] or {}).get('chapters', []))
            _feat_n  = len((sections['features'] or {}).get('items', []))
            _stat_n  = len((sections['stats'] or {}).get('items', []))
            _rev_n   = len((sections['reviews'] or {}).get('items', []))
            _faq_n   = len((sections['faq'] or {}).get('items', []))
            print(f'    JSON OK: hero={_hero_ok} story={_story_n} feat={_feat_n} stat={_stat_n} rev={_rev_n} faq={_faq_n} | ${cost2:.4f}')

            # Tags validation (against taxonomy)
            _raw_tags = meta.get('product_tags', [])
            _validated_tags, _dropped_tags = validate_tags(_raw_tags)
            if _dropped_tags:
                print(f'    \u26a0 Dropped {len(_dropped_tags)} hallucinated tags: {_dropped_tags[:3]}{"..." if len(_dropped_tags)>3 else ""}')
            if len(_validated_tags) < TAGS_PER_PRODUCT_MIN:
                print(f'    \u26a0 Only {len(_validated_tags)} valid tags (min {TAGS_PER_PRODUCT_MIN}) \u2014 designer underused taxonomy')
            print(f'    Tags: {len(_validated_tags)} valid')

            # Persist sections as JSON in DB final_html column (we reuse the column).
            # Keep stage2_json for meta/seo. Each section is written as a separate
            # metafield in Stage 5 below via batch metafieldsSet.
            _all_content = json.dumps({"sections": sections, "meta": meta}, ensure_ascii=False)
            cur_cost = db.execute('SELECT cost_usd FROM products WHERE id=?', (pid,)).fetchone()[0] or 0
            db_update_status(
                pid, 'html_ready',
                final_html=_all_content,
                stage2_json=json.dumps(meta, ensure_ascii=False),
                seo_title=seo_m.get('title',''), seo_description=seo_m.get('description',''),
                url_handle=seo_m.get('handle',''), product_tags=json.dumps(_validated_tags),
                cost_usd=cur_cost + cost2,
            )
            # Save a debug HTML preview to disk too (for manual inspection)
            _fname = re.sub(r'[^a-zA-Z0-9]', '_', product['title'][:50]).strip('_').lower() or 'product'
            with open(f'{OUTPUT_DIR}/{_fname}.json', 'w', encoding='utf-8') as f:
                f.write(_all_content)
            pstatus = 'html_ready'

        # ═══ STEP 4.5: PHOTO PACK — bundle EPROLO photos + prompt.txt + README.md + manifest.json into ZIP ═══
        # Upload to Shopify Files; URL stored in admin-only custom.photo_pack metafield.
        # Operator downloads ZIP from Admin → uses prompt.txt as model context in
        # ChatGPT-UI → uploads edited photos back to Shopify product.
        if pstatus == 'html_ready':
            # _PhotoPackSkipped is now module-level (Cell 2).
            try:
                import zipfile as _zf
                import io as _io_pp
                from datetime import datetime as _dt

                # Reload product from DB — Step 4.5 may run on a resumed pipeline
                # where Step 4 (which used to set 'product' in-loop) was skipped.
                # Without this, prompt.txt/README/manifest get product.get('title')
                # against an undefined name → NameError swallowed by outer except,
                # ZIP silently dropped for resumed products.
                product = json.loads(db.execute('SELECT scrape_json FROM products WHERE id=?',(pid,)).fetchone()[0])
                _img_data_pp = json.loads(db.execute('SELECT image_urls_json FROM products WHERE id=?',(pid,)).fetchone()[0])
                _content_pp = json.loads(db.execute('SELECT final_html FROM products WHERE id=?',(pid,)).fetchone()[0] or '{}')
                _meta_pp = (_content_pp.get('meta') if isinstance(_content_pp, dict) else {}) or {}

                # ── Visual style: Designer-emitted preferred; else build a PERSONALIZED
                # fallback from product palette + strategy.voice. Generic-editorial
                # fallback removed — it conflicted with pastel section palettes and
                # produced visually inconsistent ZIP edits vs the rest of the page.
                _visual_style_pp = (_meta_pp.get('visual_style') if isinstance(_meta_pp, dict) else '') or ''
                if not _visual_style_pp:
                    _resp_check = _content_pp.get('sections', _content_pp) if isinstance(_content_pp, dict) else {}
                    _visual_style_pp = (_resp_check.get('visual_style') if isinstance(_resp_check, dict) else '') or ''
                if not _visual_style_pp:
                    _palette_pp = (_content_pp.get('sections', {}) or {}).get('palette', {}) or {}
                    _strat_row = db.execute('SELECT strategy_json FROM products WHERE id=?', (pid,)).fetchone()
                    _strategy_pp = json.loads(_strat_row[0]) if (_strat_row and _strat_row[0]) else {}
                    _voice_pp = _strategy_pp.get('voice', '') if isinstance(_strategy_pp, dict) else ''
                    _palette_hex = [v for v in (_palette_pp.get(k, '') for k in ('brand_1', 'brand_2', 'brand_3', 'brand_soft', 'brand_deep')) if v]
                    if _palette_hex or _voice_pp:
                        _voice_anchor = {
                            'warm-confidant':       'warm, intimate framing — feels like a friend showing the product.',
                            'witty-irreverent':     'playful surprising angles with confident wit.',
                            'clinical-precise':     'sterile precision, minimal styling, technical clarity.',
                            'editorial-thoughtful': 'magazine editorial — considered light, calm composition.',
                            'aspirational-luxury':  'soft luxurious materials, golden-hour glow, premium textures.',
                            'down-to-earth-honest': 'natural daylight, real-world settings, unstyled.',
                        }.get(_voice_pp, 'editorial product photography with considered composition.')
                        _palette_desc = ', '.join(_palette_hex[:5]) if _palette_hex else 'soft pastel tones'
                        _visual_style_pp = (
                            f'{_voice_anchor} '
                            f'Palette anchored on {_palette_desc} — match the product\'s own tonal family. '
                            f'Soft natural light, subtle warm shadows, no harsh highlights. '
                            f'Pastel aesthetic — never saturated or neon. '
                            f'For CAROUSEL slots: white / very light pastel background, product '
                            f'centered, overlay text in clean sans-serif (Inter / Helvetica / SF Pro), '
                            f'thin connector lines for callouts, accent color from the product palette, '
                            f'Amazon-style infographic density (3-5 elements per image). '
                            f'For INLINE slots: real-world environment background (counter, vanity, '
                            f'nightstand), product in natural light, no graphic elements, no overlay text. '
                            f'Do PRESERVE all existing packaging text (brand, ingredients, warnings) '
                            f'exactly as printed on the product in BOTH styles. '
                            f"Preserve the product's actual appearance, materials, and proportions."
                        )

                _all_eprolo_pp = (_img_data_pp.get('top') or []) + (_img_data_pp.get('desc') or [])
                _photo_briefs_pp = _meta_pp.get('photo_briefs') if isinstance(_meta_pp, dict) else None
                if not _visual_style_pp:
                    print('  → Photo pack: skip (no visual_style + no palette/voice fallback available)')
                elif not _photo_briefs_pp:
                    print('  → Photo pack: skip (designer emitted no photo_briefs)')
                elif not _all_eprolo_pp:
                    print('  → Photo pack: skip (no EPROLO source photos)')
                else:
                    # Resolve each brief to (brief, source_url). out-of-range → skip+warn.
                    _resolved = []
                    _n_avail = len(_all_eprolo_pp)
                    for _brief_pp in _photo_briefs_pp:
                        try:
                            _raw_idx = int(_brief_pp.get('source_index', 0) or 0)
                        except (TypeError, ValueError):
                            _raw_idx = 0
                        _src_idx_pp = _raw_idx - 1  # designer uses 1-based
                        if _src_idx_pp < 0 or _src_idx_pp >= _n_avail:
                            print(f'    \u26a0 brief {_brief_pp.get("id","?")}: source_index={_raw_idx} out of range (1-{_n_avail}) — skipped')
                            continue
                        _src_url_pp = _all_eprolo_pp[_src_idx_pp].get('url', '')
                        if _src_url_pp.startswith('http'):
                            _resolved.append((_brief_pp, _src_url_pp))

                    # Sort by carousel conversion funnel (module-level _PHOTO_FUNNEL_IDX).
                    # Filenames `01-...`, `02-...` reflect the order the operator should
                    # drop into Shopify carousel. Unknown slots go after (key=999).
                    _resolved.sort(key=lambda rb: _PHOTO_FUNNEL_IDX.get(rb[0].get('slot', '') or '', 999))

                    if not _resolved:
                        print('  → Photo pack: skip (no resolvable source photos for briefs)')
                    else:
                        print(f'  → Photo pack: bundling {len(_resolved)} briefs + prompt.txt + README.md + manifest.json...')
                        _photo_files = []
                        async with httpx.AsyncClient(timeout=30, follow_redirects=True) as _c_dl_pp:
                            for _bi_pp, (_brief_pp, _src_url_pp) in enumerate(_resolved[:30]):
                                # 3 attempts with exp backoff (1s, 2s) — EPROLO CDN can
                                # blip on big batches, silent-skip used to drop briefs.
                                _r_pp = None
                                for _att in range(3):
                                    try:
                                        _r_pp = await _c_dl_pp.get(_src_url_pp)
                                        if _r_pp.status_code == 200:
                                            break
                                        _r_pp = None
                                    except Exception:
                                        _r_pp = None
                                    if _att < 2:
                                        await asyncio.sleep(2 ** _att)
                                if _r_pp is None:
                                    print(f'    \u26a0 brief {_brief_pp.get("id","?")}: download failed after 3 attempts')
                                    continue
                                _ct_pp = _r_pp.headers.get('content-type', '')
                                _ext_pp = 'jpg' if 'jpeg' in _ct_pp else ('png' if 'png' in _ct_pp else ('webp' if 'webp' in _ct_pp else 'jpg'))
                                _photo_files.append((_bi_pp, _brief_pp, _r_pp.content, _ext_pp, _src_url_pp))

                        def _safe_slot(s):
                            return ''.join(ch if (ch.isalnum() or ch == '-') else '-' for ch in (s or 'photo'))[:40]

                        # Pre-compute filenames (funnel-sorted positions already in _bi_pp).
                        _files_with_names = []
                        for _bi_pp, _brief_pp, _bytes_pp, _ext_pp, _src_url_pp in _photo_files:
                            _slot_pp = _safe_slot(_brief_pp.get('slot', 'photo'))
                            _fname_pp = f"{_bi_pp+1:02d}-{_slot_pp}.{_ext_pp}"
                            _files_with_names.append((_bi_pp, _brief_pp, _bytes_pp, _src_url_pp, _fname_pp))

                        # If every download failed (3-attempt retry exhausted), don't ship
                        # a docs-only ZIP — operator would download a useless archive.
                        if not _files_with_names:
                            print('  → Photo pack: skip (all photo downloads failed after retries)')
                            raise _PhotoPackSkipped()

                        # ── prompt files — TWO separate files per scope:
                        #   prompt_carousel.txt → marketplace/Amazon-style (carousel-* briefs)
                        #   prompt_inline.txt   → editorial in-page (inline-* briefs)
                        # User uploads each batch to ChatGPT/Ideogram separately — styles
                        # are too different to combine in one prompt and confuse the model.
                        _carousel_pairs = [e for e in _files_with_names
                                           if e[1].get('slot', '').startswith('carousel-')]
                        _inline_pairs   = [e for e in _files_with_names
                                           if e[1].get('slot', '').startswith('inline-')]

                        # Slot → theme display-aspect map (from wanelo.css). Used in
                        # per-brief format hints for inline slots so editor produces
                        # the right aspect (page CSS would crop otherwise). Carousel
                        # always uses 1:1 (Amazon standard).
                        _INLINE_ASPECT = {
                            'inline-hero':           '16:10 WIDE landscape  (1600x1000 px)',
                            'inline-story-1':        '4:5 TALL portrait      (1200x1500 px)',
                            'inline-story-2':        '4:5 TALL portrait      (1200x1500 px)',
                            'inline-story-3':        '4:5 TALL portrait      (1200x1500 px)',
                            'inline-lifestyle':      '1:1 square or 4:5 tall (grid auto-fit)',
                            'inline-features':       '1:1 square             (1200x1200 px)',
                            'inline-video-demo':     '16:9 landscape         (1920x1080 px)',
                            'inline-brand-story':    '1:1 square avatar      (600x600 px circle)',
                            'inline-app-showcase':   '9:19 phone portrait    (1080x2280 px)',
                            'inline-room-placement': '4:3 landscape          (1600x1200 px)',
                        }

                        def _build_scope_prompt(scope, pairs):
                            '''Build one prompt file for one scope: carousel or inline.'''
                            if not pairs:
                                return ''
                            n = len(pairs)
                            if scope == 'carousel':
                                header = [
                                    f"# Photo Edit Pack — CAROUSEL (Amazon / marketplace-style)",
                                    f"# Product: {product.get('title','')}",
                                    "",
                                    f"Upload this file + the {n} carousel photos to ChatGPT / Ideogram",
                                    "in ONE batch. Save returned photos with the SAME filenames.",
                                    "",
                                    "============================================================================",
                                    "STYLE — Amazon / marketplace product listing image",
                                    "",
                                    "INFORMATIVE & DENSE. Like top Amazon Best-Seller listings: hero shot",
                                    "+ feature callouts + benefit infographic + before/after + usage diagram.",
                                    "Each photo answers ONE buyer question at a glance.",
                                    "",
                                    "FORMAT (CRITICAL)",
                                    "  - Aspect ratio: 3:4 PORTRAIT (matches homepage card cells)",
                                    "  - Target resolution: 1500 x 2000 px (3:4) — retina-safe",
                                    "  - Background: white / very light pastel anchored on product tones",
                                    "  - 12-18% padding (vertical breathing room for portrait crop)",
                                    "  - Product should sit centered, slightly upper-third for visual",
                                    "    balance — text overlay rooms read better below the product",
                                    "",
                                    "TEXT OVERLAYS — REQUIRED for carousel",
                                    "  + Feature callouts with thin connector lines pointing to parts",
                                    "    (e.g. 'Diamond tip', '5-needle micro-infusion')",
                                    "  + Benefit claims as bold headers (e.g. '47% wrinkle reduction in 8 weeks')",
                                    "  + Specs / certifications as small badges (e.g. '30 ml | FDA | Vegan')",
                                    "  + Before/After labels, Step numbers (Step 1 -> 2 -> 3)",
                                    "  + ALL TEXT IN ENGLISH. Replace any Chinese characters from source.",
                                    "  Typography: clean sans-serif (Inter / Helvetica / SF Pro), bold for",
                                    "  headers, ONE accent color from product palette, thin connector lines.",
                                    "",
                                    "TOOL RECOMMENDATION (important)",
                                    "  ChatGPT/DALL-E 3 is WEAK at long English text in images. STRONGLY prefer:",
                                    "    * Ideogram.ai (best-in-class for text-in-image, free + paid)",
                                    "    * GPT-4o image gen (Pro plan; much better English typography)",
                                    "    * Adobe Firefly Generative Fill (overlay text on source photo)",
                                    "    * Manual Canva / Figma (most reliable for text quality)",
                                    "============================================================================",
                                ]
                            else:  # inline
                                header = [
                                    f"# Photo Edit Pack — INLINE / METAFIELD (Editorial in-page)",
                                    f"# Product: {product.get('title','')}",
                                    "",
                                    f"Upload this file + the {n} inline photos to ChatGPT in ONE batch.",
                                    "Save returned photos with the SAME filenames.",
                                    "",
                                    "============================================================================",
                                    "STYLE — Editorial in-page imagery",
                                    "",
                                    "These photos sit INSIDE the product description, next to paragraphs of",
                                    "body copy on the page. Like beauty-magazine editorial — the COPY on the",
                                    "page does the talking, the photo provides the MOOD.",
                                    "",
                                    "NO OVERLAY TEXT. None.",
                                    "  - No callouts, claims, specs, badges, step numbers, comparison labels",
                                    "  - No arrows / circles / annotations",
                                    "  - Page already has H1/H2/paragraph copy next to the photo — adding",
                                    "    overlay text duplicates the page copy and breaks editorial feel.",
                                    "",
                                    "Density: 1-2 elements only. Just product + ambient context.",
                                    "",
                                    "FORMAT — DIFFERENT ASPECT RATIO PER SLOT",
                                    "  Each slot has a fixed display container in the theme. Match the",
                                    "  aspect or page CSS will crop. See per-brief edits below for exact",
                                    "  aspect each photo needs.",
                                    "",
                                    "TOOL RECOMMENDATION",
                                    "  ChatGPT (DALL-E 3) works WELL here — no overlay text to garble.",
                                    "  Upload all photos + this file in one go.",
                                    "============================================================================",
                                ]
                            shared = [
                                "",
                                "============================================================================",
                                "CRITICAL — OUTPUT FORMAT (READ FIRST)",
                                "============================================================================",
                                "",
                                "ONE EDITED IMAGE PER SOURCE PHOTO. SEPARATE FILES.",
                                "",
                                "DO NOT:",
                                "  X  Combine the photos into a collage / grid / mosaic / 2x2 layout",
                                "  X  Place multiple photos side-by-side in one canvas",
                                "  X  Return a single image that contains all source photos",
                                "  X  Stitch photos into a comparison sheet or contact sheet",
                                "  X  Show all photos as thumbnails inside one big image",
                                "",
                                "DO:",
                                "  >  Edit each numbered source photo INDEPENDENTLY",
                                "  >  Return ONE finished image per source photo, in its OWN message",
                                "  >  Photo 01 -> one output. Photo 02 -> another output. etc.",
                                "  >  Each output keeps the SAME filename as the source",
                                "  >  Generate them one at a time if needed for separation",
                                "",
                                "How to phrase the ChatGPT request (copy-paste):",
                                "  'Edit each numbered photo from this batch one at a time, output as",
                                "   SEPARATE individual images, NOT a collage. Start with photo 01.",
                                "   After you finish 01, generate 02 separately. Keep going through",
                                "   all photos. Each output = ONE image file, same filename as source.'",
                                "",
                                "If ChatGPT still combines them, send: \"split the result into",
                                "individual images, one per source photo, separate messages\".",
                                "============================================================================",
                                "",
                                "PACKAGING TEXT — PRESERVE EXACTLY (both styles)",
                                "",
                                "Text PHYSICALLY printed on the product / box / tube / bottle / sticker",
                                "must remain readable and IDENTICAL to source:",
                                "  + Brand name, product name, ingredient/INCI list, dosage, warnings,",
                                "    volume/weight, certifications (FDA/CE/vegan/cruelty-free),",
                                "    batch numbers, expiry dates, barcodes.",
                                "  > Reason: customers compare label on arrival = dispute/return if different.",
                                "  > Never erase, blur, or substitute packaging text.",
                                "  > If unreadable in source (blur/glare): leave as-is, do NOT invent.",
                                "",
                                "NEVER INVENT",
                                "  - Made-up ingredients / fake numeric claims / fake certifications",
                                "  - Fake brand names or competitor names / made-up clinical citations",
                                "  When unsure: use generic supportable claims, not specific numbers.",
                                "",
                                "## Unified visual style (apply to ALL photos in this batch)",
                                "",
                                _visual_style_pp,
                                "",
                                f"## Per-brief edits ({n} photos)",
                                "",
                            ]
                            lines = header + shared
                            for _bi_pp, _brief_pp, _bytes_pp, _src_url_pp, _fname_pp in pairs:
                                _slot_pp = _brief_pp.get('slot', '?')
                                _ctx_pp  = _brief_pp.get('slot_context', '')
                                _entry = [f"### `{_fname_pp}` — slot: `{_slot_pp}`"]
                                if scope == 'inline':
                                    _asp = _INLINE_ASPECT.get(_slot_pp, '4:5 portrait or 1:1 square (theme container varies)')
                                    _entry.append(f"- ASPECT: {_asp}")
                                else:
                                    _entry.append("- ASPECT: 3:4 PORTRAIT (1500x2000 px) — matches homepage card aspect")
                                if _ctx_pp:
                                    _entry.append(f"- Slot context: {_ctx_pp}")
                                _entry.extend([
                                    f"- Concept: {_brief_pp.get('concept','')}",
                                    f"- Edit instructions: {_brief_pp.get('edit_instructions','')}",
                                ])
                                if scope == 'inline':
                                    _ov_blob = (str(_brief_pp.get('concept','')) + ' ' + str(_brief_pp.get('edit_instructions',''))).lower()
                                    _OV_KW = ('infographic', 'callout', 'call-out', 'overlay', 'badge', 'step ', 'step-', 'annotat', 'connector', 'caption', 'headline', 'text label', 'arrow')
                                    if any(_k in _ov_blob for _k in _OV_KW):
                                        _entry.append("- \u26a0 EDITORIAL OVERRIDE: ignore any infographic / overlay / callout / step / badge / label / arrow wording above. Render NO text or graphics on this image \u2014 product + ambient mood ONLY (page copy sits beside it).")
                                    if _slot_pp == 'inline-hero':
                                        _entry.append("- \u26a0 inline-hero must be a LIFESTYLE / in-use / emotional scene (a person or a real lived-in environment), NOT a product-on-surface still-life and NOT a repeat of carousel-hero's backdrop.")
                                _entry.append("")
                                lines.extend(_entry)
                            return "\n".join(lines)

                        _prompt_carousel_txt = _build_scope_prompt('carousel', _carousel_pairs)
                        _prompt_inline_txt   = _build_scope_prompt('inline',   _inline_pairs)

                        # designer_prompt admin metafield = both prompts combined with
                        # a delimiter, for archival/copy-paste from Admin without ZIP.
                        _combined_prompt = "\n\n".join(part for part in [
                            ("=== CAROUSEL prompt ===\n\n" + _prompt_carousel_txt) if _prompt_carousel_txt else "",
                            ("=== INLINE prompt ===\n\n" + _prompt_inline_txt) if _prompt_inline_txt else "",
                        ] if part)
                        _content_pp.setdefault('sections', {})['designer_prompt'] = _combined_prompt

                        # ── README.md — operator-facing workflow. Operator should NOT
                        # upload this to the model (the model only sees prompt.txt + photos).
                        _readme_lines = [
                            "# Photo Pack — Operator Workflow",
                            "",
                            f"Product: {product.get('title','')}",
                            "",
                            "## ZIP contents",
                            "",
                            f"  /photos/             — {len(_files_with_names)} EPROLO photos, sorted by funnel order",
                            "  prompt_carousel.txt  — instructions for the CAROUSEL batch (if any)",
                            "  prompt_inline.txt    — instructions for the INLINE batch (if any)",
                            "  prompt_pinterest.txt — 3 Pinterest pin variations (if Strategy emitted them)",
                            "  prompt_reviews.txt   — 8 UGC review-photo prompts (Judge.me/Loox/Stamped)",
                            "  manifest.json        — audit trail (operator only, do NOT upload to model)",
                            "  README.md            — this file (operator only)",
                            "",
                            "## Workflow — TWO separate batches",
                            "",
                            "Photos split by slot prefix. Edit each batch as a SEPARATE session.",
                            "",
                            "### Batch 1 — CAROUSEL (carousel-* photos, Amazon-style)",
                            "1. Open ChatGPT (preferably GPT-4o image) OR Ideogram.ai.",
                            "2. Upload prompt_carousel.txt + all 01-05 carousel photos in ONE message.",
                            "3. Tell ChatGPT EXACTLY: 'Edit each numbered photo from this batch one",
                            "   at a time, output as SEPARATE individual images, NOT a collage.",
                            "   Start with 01. After you finish 01, generate 02 separately. Keep",
                            "   going through all photos. Same filename as source.'",
                            "4. Save returned photos (still the SAME filenames as source).",
                            "5. If ChatGPT made a collage instead: reply 'split the result into",
                            "   individual images, one per source photo, separate messages'.",
                            "",
                            "### Batch 2 — INLINE (inline-* photos, editorial)",
                            "1. Open ChatGPT (DALL-E is fine — no overlay text needed).",
                            "2. Upload prompt_inline.txt + all 06+ inline photos in ONE message.",
                            "3. Tell ChatGPT EXACTLY: 'Edit each numbered photo from this batch one",
                            "   at a time, output as SEPARATE individual images, NOT a collage.",
                            "   Start with 06. After you finish 06, generate 07 separately. Keep",
                            "   going through all photos. Same filename as source.'",
                            "4. IMPORTANT: each inline slot has a SPECIFIC aspect ratio (see prompt_inline.txt).",
                            "5. Save with the SAME filenames.",
                            "",
                            "### Then push back",
                            "",
                            "    python pipeline/tools/upload_edited_photos.py --folder <your-folder>",
                            "",
                            "  carousel-* → REPLACE Shopify product carousel (delete old, attach new)",
                            "  inline-*   → upload to Shopify Files + swap URLs in metafield JSONs",
                            "",
                            "## How filenames are ordered",
                            "",
                            "Filenames `NN-slot.ext` are sorted by carousel conversion funnel:",
                            "  01. carousel-hero       (\"What IS it?\")",
                            "  02. carousel-lifestyle  (\"Who uses it?\")",
                            "  03. carousel-in-use     (\"How does it work?\")",
                            "  04. carousel-detail     (\"Is it quality?\")",
                            "  05. carousel-scale      (\"How big?\")",
                            "  06-N. inline-* in narrative order",
                            "  N+. any other slots, in Designer's emission order",
                            "",
                            "If Designer skipped a slot, the numbering closes the gap automatically.",
                        ]
                        _readme_txt = "\n".join(_readme_lines)

                        # ── manifest.json — audit-only map of filename → EPROLO source URL.
                        # Lets the operator verify which original photo each brief used.
                        # Never seen by the model (operator must NOT upload).
                        _manifest = {
                            'product_title': product.get('title', ''),
                            'created_at':    _dt.utcnow().isoformat() + 'Z',
                            'visual_style':  _visual_style_pp,
                            'briefs': [
                                {
                                    'filename':          _fname_pp,
                                    'slot':              _brief_pp.get('slot', ''),
                                    'source_index':      _brief_pp.get('source_index'),
                                    'source_url':        _src_url_pp,
                                    'concept':           _brief_pp.get('concept', ''),
                                    'edit_instructions': _brief_pp.get('edit_instructions', ''),
                                    'slot_context':      _brief_pp.get('slot_context', ''),
                                }
                                for _bi_pp, _brief_pp, _bytes_pp, _src_url_pp, _fname_pp in _files_with_names
                            ],
                        }
                        _manifest_txt = json.dumps(_manifest, ensure_ascii=False, indent=2)

                        # Hard byte cap — drop briefs (in order) once cumulative size hits cap
                        # so a few huge EPROLO photos don't blow up Shopify staged upload.
                        _MAX_ZIP_BYTES = 50 * 1024 * 1024
                        _files_capped = []
                        _running = 0
                        for _entry_pp in _files_with_names:
                            _bytes_size = len(_entry_pp[2])
                            if _running + _bytes_size > _MAX_ZIP_BYTES:
                                print(f'    \u26a0 ZIP cap (50 MB) reached \u2014 dropping brief {_entry_pp[1].get("id","?")} (+{_bytes_size//1024} KB)')
                                continue
                            _files_capped.append(_entry_pp)
                            _running += _bytes_size
                        _files_with_names = _files_capped

                        _buf_pp = _io_pp.BytesIO()
                        with _zf.ZipFile(_buf_pp, 'w', _zf.ZIP_DEFLATED) as _zip:
                            if _prompt_carousel_txt:
                                _zip.writestr('prompt_carousel.txt', _prompt_carousel_txt)
                            if _prompt_inline_txt:
                                _zip.writestr('prompt_inline.txt', _prompt_inline_txt)
                            # ── Pinterest pin prompts — 3 variations from Opus strategy.pinterest_pins[]
                            # Each variation tests a different hypothesis (evidence-based /
                            # personal-insider / visual-aesthetic). Operator generates all 3
                            # via Ideogram/Midjourney/Canva and A/B tests which one pulls
                            # Pinterest traffic best. Backward-compat: legacy singular
                            # pinterest_pin field is wrapped as [pin] for unified handling.
                            try:
                                _strat_row_pin = db.execute(
                                    'SELECT strategy_json FROM products WHERE id=?', (pid,)
                                ).fetchone()
                                _strategy_pin = json.loads(_strat_row_pin[0]) if (_strat_row_pin and _strat_row_pin[0]) else {}
                                _pins_list = (_strategy_pin or {}).get('pinterest_pins') or []
                                if not _pins_list and (_strategy_pin or {}).get('pinterest_pin'):
                                    _pins_list = [(_strategy_pin or {}).get('pinterest_pin')]
                                _pins_list = [p for p in _pins_list if p and p.get('hook_headline')]

                                if _pins_list:
                                    _pin_lines = [
                                        f"# Pinterest Pin Prompts — {product.get('title','')}",
                                        f"# {len(_pins_list)} variations to A/B test. Each tests a different hypothesis.",
                                        "",
                                        "Pinterest doesn't tell you what will pull traffic until you ship.",
                                        "Generate ALL variations. Schedule them on different boards / dates.",
                                        "Track saves + outbound clicks per pin in Pinterest Analytics.",
                                        "Double down on the format that wins.",
                                        "",
                                        "============================================================",
                                        "HOW TO USE",
                                        "============================================================",
                                        "Each VARIATION block below is a standalone prompt. Copy the full",
                                        "block (from 'VARIATION X' to the next '###' line) and paste into",
                                        "Ideogram / Midjourney / GPT-4o / Canva — ONE TOOL CALL PER VARIATION.",
                                        "DO NOT paste all 3 variations together — that gives you a collage.",
                                        "Output per pin: 1 vertical 2:3 Pinterest pin (1000x1500 px min).",
                                        "Repeat 3 times (once per variation) to get 3 separate pins.",
                                        "",
                                    ]
                                    for _pi, _pin_obj in enumerate(_pins_list[:3]):
                                        _vid = _pin_obj.get('variation_id', chr(ord('A') + _pi))
                                        _pin_lines.extend([
                                            "",
                                            "############################################################",
                                            f"VARIATION {_vid}  —  {_pin_obj.get('variation_intent', '')}",
                                            "############################################################",
                                            "",
                                            f"  Category: {_pin_obj.get('category_profile', 'other')}",
                                            f"  Format:   {_pin_obj.get('format', 'flatlay_editorial')}",
                                            f"  Voice:    {_pin_obj.get('pin_voice', 'magazine_editorial')}",
                                            "",
                                            "  HEADLINE",
                                            f"    Main:    {_pin_obj.get('hook_headline', '')}",
                                            f"    Subhead: {_pin_obj.get('hook_subhead', '')}",
                                            "",
                                            "  DEMONSTRATION (what the product DOES)",
                                            f"    Type: {_pin_obj.get('demonstration', {}).get('type', '')}",
                                            "",
                                            "    Visual instructions:",
                                            f"    {_pin_obj.get('demonstration', {}).get('details_for_image', '')}",
                                            "",
                                            "    Overlay labels:",
                                        ])
                                        for _lbl in (_pin_obj.get('demonstration') or {}).get('labels', []) or []:
                                            _pin_lines.append(f"      - {_lbl}")

                                        _pin_lines.extend([
                                            "",
                                            "  STATS (verified only — NEVER invent)",
                                        ])
                                        _stats_list = _pin_obj.get('stats_to_use') or []
                                        if not _stats_list:
                                            _pin_lines.append("    (no verified stats — use generic phrasing)")
                                        else:
                                            for _s in _stats_list:
                                                _src = f" — {_s.get('source')}" if _s.get('source') else ''
                                                _pin_lines.append(
                                                    f"    - {_s.get('value', '')} {_s.get('label', '')} "
                                                    f"({_s.get('context', '')}){_src}"
                                                )

                                        _pin_lines.extend([
                                            "",
                                            "  MOOD PALETTE",
                                            f"    Colors (max 3): {', '.join(_pin_obj.get('mood_palette') or [])}",
                                            "",
                                            "  BRAND CTA (bottom 5-8%, whispered)",
                                            f"    {_pin_obj.get('brand_attribution', '')}",
                                            "",
                                        ])

                                    _pin_lines.extend([
                                        "",
                                        "############################################################",
                                        "UNIVERSAL RULES (apply to all variations)",
                                        "############################################################",
                                        "",
                                        "  PRODUCT PRESERVATION:",
                                        "    If the product appears in the pin (recommended: small reference",
                                        "    photo bottom corner), preserve packaging text EXACTLY. Do not",
                                        "    recolor or redesign the label.",
                                        "",
                                        "  STYLE GUARDRAILS (Pinterest 2026):",
                                        "    AVOID:",
                                        "      - Pure white background (looks like Amazon)",
                                        "      - 'BUY NOW' / 'SHOP' buttons",
                                        "      - '50% OFF' badges",
                                        "      - Stock-photo handshakes / fake smiling models",
                                        "      - Generic Canva-default templates",
                                        "",
                                        "    AIM FOR:",
                                        "      - Magazine-page energy (Vogue Beauty / Allure / NYT Style)",
                                        "      - Subtle paper grain texture on background",
                                        "      - Serif typography with personality (Playfair, Cormorant)",
                                        "      - Demonstration over description — show, don't tell",
                                        "      - Save-worthy through INFORMATION, not aesthetics alone",
                                        "",
                                        "  TOOL RECOMMENDATION (Pinterest pin text quality):",
                                        "      1. Ideogram.ai — best AI for English text in images",
                                        "      2. Midjourney v6+ with --style raw for editorial flatlay",
                                        "      3. GPT-4o image (ChatGPT Pro) — newer typography",
                                        "      4. Canva Pro — manual layout when AI text gets garbled",
                                        "      AVOID DALL-E 3 — long English text gets garbled.",
                                    ])
                                    _prompt_pinterest_txt = "\n".join(_pin_lines)
                                    _zip.writestr('prompt_pinterest.txt', _prompt_pinterest_txt)
                            except Exception as _e_pin:
                                print(f'    ⚠ Pinterest prompt build failed: {str(_e_pin)[:120]}')

                            # ── UGC REVIEW PHOTOS — 8 authentic phone-snapshot
                            # prompts for review widgets (Judge.me / Loox / Stamped),
                            # social proof carousels, "as seen on Insta" sections.
                            # NOT studio. NOT polished. iPhone candid energy.
                            # Generic per-product template — instructions baked in,
                            # only the product title + 1-line use-context plug in.
                            try:
                                _strat_row_rev = db.execute(
                                    'SELECT strategy_json FROM products WHERE id=?', (pid,)
                                ).fetchone()
                                _strategy_rev = json.loads(_strat_row_rev[0]) if (_strat_row_rev and _strat_row_rev[0]) else {}
                                _hero_obj = (_strategy_rev or {}).get('hero') or {}
                                _use_context = _hero_obj.get('lead') or _hero_obj.get('h1') or ''
                                _use_context = (_use_context or '')[:200]

                                # Per-product review scenes. PRIMARY = Designer meta.review_scenes (product-
                                # specific, e.g. fishing rod -> riverbank). FALLBACK = category-aware pool keyed
                                # off the product title/tags. Both seed-shuffled so two products never share the
                                # same set/order; the product palette is injected so shots read in its tonal family.
                                import hashlib as _hl_rev
                                _seed_src = (product.get('url_handle') or product.get('handle') or product.get('title') or 'x')
                                _seed_rev = int(_hl_rev.md5(_seed_src.encode('utf-8', 'ignore')).hexdigest(), 16)
                                def _rot(_lst, _k, _salt=0):
                                    if not _lst:
                                        return []
                                    _n = len(_lst)
                                    _order = sorted(range(_n), key=lambda _j: _hl_rev.md5(('%d-%d-%d' % (_seed_rev, _salt, _j)).encode()).hexdigest())
                                    return [_lst[_order[_i % _n]] for _i in range(_k)]
                                try:
                                    _meta_for_rev = _meta_pp if isinstance(_meta_pp, dict) else {}
                                except NameError:
                                    _meta_for_rev = {}
                                _palette_for_rev = (_visual_style_pp or '').strip()
                                if len(_palette_for_rev) > 600:
                                    _palette_for_rev = _palette_for_rev[:600].rsplit(' ', 1)[0] + ' …'
                                _scenes_src = []
                                for _s in (_meta_for_rev.get('review_scenes') or []):
                                    if not isinstance(_s, dict):
                                        continue
                                    _t = (_s.get('title') or '').strip()
                                    _sc = (_s.get('scene') or '').strip()
                                    _set = (_s.get('setting') or '').strip()
                                    if not (_t and _sc):
                                        continue
                                    _bd = [_sc]
                                    if _set:
                                        _bd.append('Setting: %s.' % _set)
                                    _scenes_src.append((_t, _bd))
                                _used_designer = len(_scenes_src) >= 6
                                _n_designer = len(_scenes_src)
                                # tier-2: derive product-specific scenes from the product's OWN photo_briefs
                                # already in the archive/meta (FREE — no API call, no re-parse).
                                if len(_scenes_src) < 6:
                                    _SCENE_SLOTS = ('carousel-lifestyle', 'carousel-in-use', 'inline-hero', 'inline-story-1', 'inline-story-2', 'inline-story-3', 'inline-cta', 'inline-lifestyle', 'inline-room-placement')
                                    _SLOT_TITLE = {'carousel-lifestyle': 'In real life', 'carousel-in-use': 'Mid-use moment', 'inline-hero': 'In the moment', 'inline-story-1': 'Real-life story', 'inline-story-2': 'A few weeks in', 'inline-story-3': 'The payoff', 'inline-cta': 'Why it stays out', 'inline-lifestyle': 'Everyday use', 'inline-room-placement': 'In its place'}
                                    _SKIP_KW = ('diagram', 'infographic', 'chart', 'illustration', 'anatomical', 'schematic', 'callout', 'overlay', 'before/after', 'comparison', 'white seamless')
                                    _seen_t = set(_t.lower() for _t, _ in _scenes_src)
                                    for _b in (_meta_for_rev.get('photo_briefs') or []):
                                        if not isinstance(_b, dict):
                                            continue
                                        _sl = _b.get('slot', '')
                                        if _sl not in _SCENE_SLOTS:
                                            continue
                                        _c = (_b.get('concept') or '').strip()
                                        if not _c or any(_k in _c.lower() for _k in _SKIP_KW):
                                            continue
                                        _ti = _SLOT_TITLE.get(_sl, 'In real life')
                                        while _ti.lower() in _seen_t:
                                            _ti = _ti + ' +'
                                        _seen_t.add(_ti.lower())
                                        _scenes_src.append((_ti, ['Candid iPhone-snapshot version of this real product moment: ' + _c[:170], 'Real lived-in setting, a little clutter, phone-camera look — NOT studio, NO overlay text.']))
                                _n_archive = len(_scenes_src) - _n_designer
                                _GENERIC_FB = [
                                    ("Just arrived / unboxing", ["Product fresh out of the shipping mailer on a table,", "cardboard and packing slip still in frame."]),
                                    ("Among everyday things", ["Product among the owner's clutter — keys, mug, phone —", "low-angle candid phone shot."]),
                                    ("Gifted to someone", ["Handed over with a simple ribbon or sticky note,", "two people's hands in frame."]),
                                    ("In the bag / on the go", ["Tucked into a tote or backpack with everyday carry,", "shot in a car or on a cafe table."]),
                                    ("Close-up in hand", ["Macro-ish close-up held between thumb and finger,", "background out of focus."]),
                                    ("In its real spot", ["Where the owner actually keeps it, slightly messy", "real room, natural window light."]),
                                ]
                                _CAT_FB = {
                                    "fragrance": [
                                        ("Before a night out", ["At the vanity mid-routine, mirror lights, a spritz", "to the wrist before heading out."]),
                                        ("On the dresser", ["Bottle by a watch, jewelry dish and keys on a dresser,", "morning window light."]),
                                        ("In the handbag", ["Bottle tucked in an open handbag with phone and lipstick,", "candid grab shot."]),
                                        ("Gifted with a ribbon", ["Boxed bottle with a ribbon just handed over,", "two hands in frame, warm light."]),
                                        ("Spritz on the wrist", ["Wrist mid-spritz, fine mist in the air, hallway backdrop,", "slight motion blur."]),
                                    ],
                                    "beauty": [
                                        ("Morning routine at the sink", ["On the bathroom sink mid-routine, towel and toothbrush", "in frame, mirror slightly steamy."]),
                                        ("On the vanity", ["Among makeup and a mirror on the vanity, soft daylight,", "a little clutter."]),
                                        ("Mid-application", ["Hand applying it, caught in motion with slight blur,", "real bathroom backdrop."]),
                                        ("Shelfie", ["Lined up with other bottles on a bathroom shelf,", "candid phone shot."]),
                                        ("Just arrived", ["Fresh out of the mailer on the counter, packing slip in frame."]),
                                    ],
                                    "tools": [
                                        ("Mid-repair on the bench", ["In use on a cluttered workbench, screws and sawdust around,", "garage light."]),
                                        ("On the pegboard", ["Resting on a garage pegboard among other tools,", "candid phone shot."]),
                                        ("On the jobsite", ["Pulled from a tool bag on a worksite floor,", "dust and offcuts around."]),
                                        ("Mid-install at home", ["Owner mid-install with the tool in hand,", "real room, slightly messy."]),
                                        ("Just unboxed", ["Fresh out of the box on the garage bench, packaging beside it."]),
                                    ],
                                    "car": [
                                        ("Mounted in the car", ["Installed in the car interior, dashboard and wheel in frame,", "daylight through the windshield."]),
                                        ("Garage install", ["Owner fitting it by the open car door in a garage,", "tools nearby."]),
                                        ("On a road trip", ["In use during a drive, road and dashboard visible,", "casual phone snap."]),
                                        ("In the trunk", ["In an open trunk with everyday car clutter,", "parking-lot light."]),
                                        ("Just arrived", ["Fresh out of the box on the driver's seat, packaging beside it."]),
                                    ],
                                    "electronics": [
                                        ("On the desk in use", ["In use on a cluttered desk with a laptop and mug,", "candid shot."]),
                                        ("Charging on the nightstand", ["On the bedside table charging overnight, soft lamp light."]),
                                        ("On the commute", ["In use on a train or in a car, device in hand,", "real commute backdrop."]),
                                        ("On the couch", ["Used while relaxing on the sofa, TV or window behind,", "evening light."]),
                                        ("Just unboxed", ["Fresh out of the box on a table, cable and manual in frame."]),
                                    ],
                                    "apparel": [
                                        ("Outfit mirror selfie", ["Wearing or holding it in a mirror selfie, phone in hand,", "bedroom or hallway."]),
                                        ("Flat-lay on the bed", ["Laid out on the bed with accessories, top-down phone shot."]),
                                        ("Out and about", ["Worn or carried outdoors — street, cafe or park —", "candid shot."]),
                                        ("In the closet", ["Hung among other clothes in the closet, phone snap."]),
                                        ("Trying it on", ["Fresh out of the poly mailer, mid try-on, mirror in frame."]),
                                    ],
                                    "jewelry": [
                                        ("Worn close-up", ["On the wrist, neck or hand, skin and natural light,", "slightly soft focus."]),
                                        ("In the gift box", ["In its little box with a ribbon, just opened, two hands."]),
                                        ("On the vanity dish", ["Resting in a trinket dish with other jewelry,", "morning light."]),
                                        ("Dressed up", ["Worn with an outfit in a mirror selfie before going out."]),
                                        ("Just arrived", ["Fresh out of the pouch on a hand, packaging beside it."]),
                                    ],
                                    "kids": [
                                        ("Child mid-play", ["Child playing with it on the living-room rug, toys around,", "candid parent phone shot."]),
                                        ("Birthday moment", ["Just unwrapped at a small party, ribbon and box in frame."]),
                                        ("In the nursery", ["In its spot in the kid's room, soft daylight, slightly messy."]),
                                        ("On the go", ["In the stroller or diaper bag out and about, candid grab shot."]),
                                        ("Just arrived", ["Fresh out of the box on the play mat, packaging beside it."]),
                                    ],
                                    "sports": [
                                        ("Out in use", ["Outdoors — riverbank, trail, court or gym — owner mid-", "activity, natural light."]),
                                        ("Gear laid out", ["Laid out with the rest of the kit before heading out,", "entryway or garage floor."]),
                                        ("Post-activity", ["After use, a bit of dirt or water on it, candid shot", "at the car or trailhead."]),
                                        ("In the gear bag", ["Packed in the duffel or tackle bag with other gear,", "candid shot."]),
                                        ("Ready to try", ["Fresh out of the box at home, ready to take out,", "packaging beside it."]),
                                    ],
                                    "home": [
                                        ("In use on the counter", ["In use on the kitchen counter mid-task, cookware and", "ingredients around."]),
                                        ("In its spot", ["Where it lives — shelf, corner, counter — slightly", "lived-in room, daylight."]),
                                        ("Mid-task at home", ["Owner using it mid-chore, candid phone shot."]),
                                        ("On the shelf", ["Among everyday household items on a shelf, phone snap."]),
                                        ("Just unboxed", ["Fresh out of the box on the table, packaging in frame."]),
                                    ],
                                }
                                _CAT_KW = [
                                    ("fragrance", ("perfume", "cologne", "fragrance", "eau de", "parfum", " scent")),
                                    ("beauty", ("serum", "cream", "skin", "facial", "makeup", "lipstick", " mask", "lotion", "cosmetic", "shampoo", "nail", "lash", "wrinkle", "moistur")),
                                    ("tools", ("drill", " saw", "wrench", "screwdriver", " tool", "sander", "hammer", "plier", "hardware", "grinder")),
                                    ("car", ("car ", " auto", "vehicle", "motorcycle", "dashboard", "windshield", " tire", "engine", "automotive")),
                                    ("electronics", ("phone", "charger", "earbud", "headphone", "speaker", "camera", " cable", "laptop", "tablet", "bluetooth", "wireless", " led", "gadget", "power bank")),
                                    ("apparel", ("shirt", "dress", "jacket", "hoodie", "pants", "jeans", " sock", "legging", "sweater", " coat", " shoe", "sneaker", " boot", "backpack", "wallet", " bag")),
                                    ("jewelry", (" ring", "necklace", "bracelet", "earring", "pendant", "jewelry", " bangle", "anklet")),
                                    ("kids", (" toy", " baby", " kids", " child", "nursery", "stroller", "infant", "plush", "toddler")),
                                    ("sports", ("fishing", " rod", " reel", "tackle", "camping", " tent", "hiking", "cycling", " bike", "fitness", " yoga", " gym", "dumbbell", "swim", " ski", "hunting", " golf", "running")),
                                    ("home", ("kitchen", " cook", " pan", " pot", " knife", " decor", " lamp", "storage", "cleaning", " pet ", "garden", " towel", "curtain", "pillow", "organizer", "household")),
                                ]
                                _hay = ((product.get('title') or '') + ' ' + ' '.join(str(_x) for _x in (_meta_for_rev.get('product_tags') or []))).lower()
                                _bucket = ''
                                for _bk, _kws in _CAT_KW:
                                    if any(_kw in _hay for _kw in _kws):
                                        _bucket = _bk
                                        break
                                if len(_scenes_src) < 8:
                                    _fb_pool = _CAT_FB.get(_bucket, []) + _GENERIC_FB
                                    _seen = set(_t.lower() for _t, _ in _scenes_src)
                                    for _t, _bd in _rot(_fb_pool, len(_fb_pool), 77):
                                        if len(_scenes_src) >= 8:
                                            break
                                        if _t.lower() in _seen:
                                            continue
                                        _seen.add(_t.lower())
                                        _scenes_src.append((_t, _bd))
                                _picked = _rot(_scenes_src, min(8, len(_scenes_src)), 55)
                                if _n_designer >= 6:
                                    _scene_source = 'Designer (product-specific)'
                                elif _n_archive > 0:
                                    _scene_source = 'archive photo_briefs (%d) + category top-up [%s]' % (_n_archive, _bucket or 'generic')
                                else:
                                    _scene_source = 'category fallback: %s' % (_bucket or 'generic')
                                _rev_lines = [
                                    f"# UGC Review Photos — {product.get('title','')}",
                                    "",
                                    "8 authentic iPhone-snapshot prompts for review widgets",
                                    "(Judge.me / Loox / Stamped) and 'as seen on Insta' carousels.",
                                    "",
                                    "============================================================",
                                    "HOW TO USE",
                                    "============================================================",
                                    "1. Open ChatGPT-UI (GPT-4o image gen) OR Midjourney v6+",
                                    "   --style raw. AVOID DALL-E 3 — too clean / studio-y.",
                                    "2. Upload this file + 1-2 reference photos of the product",
                                    "   (the carousel hero from /photos/01-carousel-hero.* works).",
                                    "3. Tell ChatGPT EXACTLY: 'Generate 8 UGC review photos one at",
                                    "   a time, output as SEPARATE individual images, NOT a collage",
                                    "   or grid. Generate review_01 first, then in the NEXT message",
                                    "   generate review_02, then review_03, etc. Each output = ONE",
                                    "   image file. Use the reference for product likeness but make",
                                    "   it look like real candid phone snapshots.'",
                                    "4. Save outputs as review_01.jpg ... review_08.jpg",
                                    "5. If ChatGPT made a collage: reply 'split that result into 8",
                                    "   separate individual images, one per output message'.",
                                    "5. Upload to your review widget (Judge.me bulk import,",
                                    "   Loox Photos tab, Stamped UGC, etc).",
                                    "",
                                    "============================================================",
                                    "PRODUCT CONTEXT",
                                    "============================================================",
                                    f"  Product: {product.get('title','')}",
                                    f"  Use:     {_use_context}",
                                    f"  Scenes:  {_scene_source}",
                                    "",
                                    "============================================================",
                                    "PRODUCT PALETTE & MOOD (carry THIS product's tonal family)",
                                    "============================================================",
                                    "  These are candid phone snapshots, but the product and its",
                                    "  surroundings should read in the product's own colour world so",
                                    "  the set looks like THIS product, not a generic template:",
                                    f"    {_palette_for_rev or 'soft natural tones drawn from the product packaging'}",
                                    "",
                                    "============================================================",
                                    "UNIVERSAL VISUAL DNA (apply to ALL 8 photos)",
                                    "============================================================",
                                    "",
                                    "  CAMERA",
                                    "    - Shot on iPhone 13-15, default camera app, no filters",
                                    "    - 4:5 OR 3:4 vertical aspect (Instagram / phone-screen native)",
                                    "    - Slight handshake — NOT tripod-stable",
                                    "    - Auto-focus moment captured — sometimes slightly soft",
                                    "    - Auto-exposure imperfection: occasional mild over/under-exposure",
                                    "",
                                    "  LIGHT",
                                    "    - Mixed: warm tungsten + cool window light = realistic temperature",
                                    "    - No ring-light, no softbox, no rim-light",
                                    "    - Shadows are visible, not bounced out",
                                    "    - Occasional lens flare or window glare allowed",
                                    "",
                                    "  COMPOSITION",
                                    "    - Slightly off-center subject — not perfectly framed",
                                    "    - Real, lived-in background — NOT a clean white studio",
                                    "    - Everyday clutter appropriate to the scene's place",
                                    "",
                                    "  POST",
                                    "    - NO retouching, NO skin smoothing",
                                    "    - Pore-level skin texture (not airbrushed)",
                                    "    - JPEG compression artifacts subtly visible",
                                    "    - Optional: tiny EXIF-style timestamp overlay in corner",
                                    "",
                                    "  PEOPLE (when included)",
                                    "    - Diverse ages, skin tones, body types — DO NOT default",
                                    "      to one demographic. Mix across the 8 photos.",
                                    "    - NO model-perfect faces. Normal pores, hair flyaways,",
                                    "      slight asymmetry, occasional fingernail polish chip.",
                                    "    - NO posed smiling — neutral or mid-expression",
                                    "",
                                    "  PRODUCT PRESERVATION",
                                    "    - Brand text on packaging must remain READABLE and",
                                    "      unchanged. Don't redesign labels.",
                                    "",
                                    "============================================================",
                                    "8 SHOT CONCEPTS — distinct scenes, matched to THIS product",
                                    "============================================================",
                                    "",
                                ]
                                for _ri, (_rtitle, _rbody) in enumerate(_picked, 1):
                                    _rev_lines.append("### review_%02d — '%s'" % (_ri, _rtitle))
                                    for _bl in _rbody:
                                        _rev_lines.append("    " + _bl)
                                    _rev_lines.append("")
                                _rev_lines.extend([
                                    "============================================================",
                                    "OUTPUT CHECKLIST",
                                    "============================================================",
                                    "  □ 8 distinct images — different places/activities, not one room",
                                    "  □ Every scene fits where a real owner of THIS product would shoot it",
                                    "  □ NO white seamless backdrops anywhere",
                                    "  □ NO studio lighting setups visible",
                                    "  □ Product packaging text remains readable + unchanged",
                                    "  □ Faces (when present) are demographically varied",
                                    "  □ At least 2 photos include NO person at all",
                                    "    (product-only candid moments)",
                                ])
                                _prompt_reviews_txt = "\n".join(_rev_lines)
                                _zip.writestr('prompt_reviews.txt', _prompt_reviews_txt)
                            except Exception as _e_rev:
                                print(f'    ⚠ Reviews prompt build failed: {str(_e_rev)[:120]}')

                            # ── Variant swatch photos → photos/variants/ + prompt_variants.txt ──
                            # Operator edits these; upload_edited_photos.py reassigns
                            # each to its variant (image switches on colour select).
                            try:
                                _vphotos_map = {}
                                for _opt in (product.get('variants') or []):
                                    for _vn, _vurl in (_opt.get('_value_photos') or {}).items():
                                        if _vn and _vurl and _vn not in _vphotos_map:
                                            _vphotos_map[_vn] = _vurl
                                if _vphotos_map:
                                    import httpx as _httpx_v
                                    _vi = 0
                                    _vnames_ordered = []
                                    with _httpx_v.Client(timeout=30, follow_redirects=True) as _vc:
                                        for _vn, _vurl in _vphotos_map.items():
                                            try:
                                                _vr = _vc.get(_vurl.split('?')[0])
                                                if _vr.status_code != 200 or len(_vr.content) < 1000:
                                                    continue
                                                _vi += 1
                                                _ext_v = 'png' if _vr.content[:8] == b'\x89PNG\r\n\x1a\n' else 'jpg'
                                                _safe_vn = ''.join(ch if ch.isalnum() else '-' for ch in _vn).strip('-').lower()[:30]
                                                _vfname = f'variants/{_vi:02d}-{_safe_vn}.{_ext_v}'
                                                _zip.writestr(f'photos/{_vfname}', _vr.content)
                                                _vnames_ordered.append((_vi, _vn))
                                            except Exception:
                                                continue
                                    if _vnames_ordered:
                                        _vp_lines = [
                                            f"# Variant Swatch Photos — {product.get('title','')}",
                                            "",
                                            f"{len(_vnames_ordered)} variant photos in photos/variants/.",
                                            "Edit each so the swatch looks premium — same product, clean",
                                            "background, consistent lighting across all variants.",
                                            "",
                                            "============================================================",
                                            "CRITICAL — KEEP ORDER + ONE IMAGE PER VARIANT",
                                            "============================================================",
                                            "  - Edit each numbered swatch SEPARATELY (NOT a collage).",
                                            "  - Keep them in photos/variants/ with ANY filename — the",
                                            "    uploader maps by save-order to the variant list below.",
                                            "  - Output ONE edited image per swatch, same order.",
                                            "  - 3:4 portrait, 1500x2000, consistent style across all.",
                                            "  - The product colour/finish of EACH swatch MUST stay true",
                                            "    (don't recolour a pink swatch to red — it maps to that",
                                            "    exact variant the customer selects).",
                                            "",
                                            "VARIANT ORDER (edited file N → this variant):",
                                        ]
                                        for _idx, _vn in _vnames_ordered:
                                            _vp_lines.append(f"  {_idx:02d}. {_vn}")
                                        _vp_lines += [
                                            "",
                                            "After editing, the operator runs:",
                                            "  python pipeline/tools/upload_edited_photos.py --folder <pack>",
                                            "and photos/variants/ are reassigned to each variant.",
                                        ]
                                        _zip.writestr('prompt_variants.txt', "\n".join(_vp_lines))
                            except Exception as _e_vp:
                                print(f'    \u26a0 variant photo pack failed: {str(_e_vp)[:100]}')

                            _zip.writestr('README.md',     _readme_txt)
                            _zip.writestr('manifest.json', _manifest_txt)
                            for _bi_pp, _brief_pp, _bytes_pp, _src_url_pp, _fname_pp in _files_with_names:
                                _zip.writestr(f'photos/{_fname_pp}', _bytes_pp)
                        _zip_bytes_pp = _buf_pp.getvalue()

                        # Upload to Shopify Files
                        _url_handle_row = db.execute('SELECT url_handle FROM products WHERE id=?', (pid,)).fetchone()
                        _safe_handle = ((_url_handle_row[0] if _url_handle_row else None) or f'product-{pid}')[:60].replace('/', '_')
                        _zip_filename = f'photo-pack-{_safe_handle}.zip'
                        _upload_res = await shopify_upload_file(_zip_bytes_pp, _zip_filename, 'application/zip',
                                                                alt=f'Photo pack for {product.get("title","")}')
                        if _upload_res and _upload_res.get('url'):
                            # photo_pack metafield is type 'url' — store JUST the URL string.
                            # Shopify Admin renders it as a one-click download button.
                            _content_pp.setdefault('sections', {})['photo_pack'] = _upload_res['url']
                            db.execute('UPDATE products SET final_html=? WHERE id=?',
                                       (json.dumps(_content_pp, ensure_ascii=False), pid))
                            db.commit()
                            print(f'    Photo pack OK: {_upload_res["size_kb"]:.0f} KB → {_upload_res["url"][:60]}...')
                        else:
                            print('    \u26a0 Photo pack upload failed (Shopify Files API)')
            except _PhotoPackSkipped:
                pass  # graceful skip — reason already printed inline
            except Exception as _e_pp:
                print(f'    \u26a0 Photo pack failed: {str(_e_pp)[:120]}')
            # STEP 4.5 always advances status — even on failure or skip — so STEP 5
            # runs next and the product still publishes (photo_pack is optional).
            db_update_status(pid, 'pack_done')
            pstatus = 'pack_done'

        # ═══ STEP 5: Shopify (v9: + variants) ═══
        if pstatus == 'pack_done':
            row = db.execute('SELECT * FROM products WHERE id=?',(pid,)).fetchone()
            product = json.loads(row['scrape_json'])
            stage1 = json.loads(row['stage1_json'] or '{}')
            full_html = row['final_html']
            meta = json.loads(row['stage2_json'] or '{}')
            seo_t = row['seo_title'] or product['title'][:60]
            seo_d = row['seo_description'] or ''
            # Hard-truncate meta description to 155 chars (Google limit).
            # Models overshoot the prompt rule; enforce it in assembly.
            if len(seo_d) > 155:
                seo_d = seo_d[:152].rsplit(' ', 1)[0] + '...'
            handle = row['url_handle'] or ''
            tags = json.loads(row['product_tags'] or '[]')
            img_data = json.loads(row['image_urls_json'])
            variants = json.loads(row['variants_json'] or '[]')

            # Sanitize: when Designer's meta.short_description is empty we fall back
            # to the raw EPROLO description, which can contain <img>, <a>, embedded
            # styling, EPROLO URLs, etc. Stripping tags + collapsing whitespace
            # keeps the body_html clean and prevents EPROLO links leaking onto
            # storefront markup.
            _raw_short = meta.get('short_description') or (product.get('description') or '')
            short_desc = re.sub(r'<[^>]+>', '', _raw_short)
            short_desc = re.sub(r'\s+', ' ', short_desc).strip()[:200]
            body_parts = [f'<p>{short_desc}</p>']
            hero = stage1.get('conversion',{}).get('hero_claim','')
            if hero: body_parts.append(f'<p><strong>{hero}</strong></p>')
            pain_pts = stage1.get('conversion',{}).get('pain_points_solved',[])
            if pain_pts: body_parts.append('<ul>' + ''.join(f'<li>{p}</li>' for p in pain_pts[:4]) + '</ul>')
            body_html = '\n'.join(body_parts)

            print('  → Shopify...')
            existing_id = await shopify_find_product(product['title'])
            if not existing_id and handle:
                _hq = f'handle:{handle}'
                r_handle = await shopify_gql('query($q:String!){products(first:1,query:$q){nodes{id}}}', {"q": _hq})
                nodes = r_handle.get('data',{}).get('products',{}).get('nodes',[]) if r_handle else []
                if nodes: existing_id = nodes[0]['id']
            # SOURCE_URL dedup (prevents duplicates when same EPROLO product is processed
            # in multiple chunk runs with different generated titles / handles).
            if not existing_id:
                _src_url = re.sub(r'(--\d+)-\d+$', r'\1', (product.get('url') or '').strip().split('?')[0].rstrip('/'))
                if _src_url:
                    _src_q = (
                        'query($ns:String!,$k:String!,$v:String!){'
                        'products(first:1,query:$v){nodes{id title}}}'
                    )
                    _src_q2 = 'query($v:String!){products(first:2,query:$v){nodes{id title}}}'
                    _src_r = await shopify_gql(_src_q2, {"v": f'metafield:custom.source_url:{_src_url}'})
                    _src_nodes = (_src_r or {}).get('data',{}).get('products',{}).get('nodes',[]) if _src_r else []
                    if _src_nodes:
                        existing_id = _src_nodes[0]['id']
                        print(f'    [dedup] Found by source_url: {existing_id} ({_src_nodes[0]["title"][:40]})')
            if existing_id:
                # Also update product title (old EPROLO title was stuck on update)
                _title_update = seo_t or product['title'][:60]; _title_update = (_title_update[:60].rsplit(' ',1)[0] if len(_title_update) > 62 else _title_update)
                await shopify_gql(
                    'mutation($id:ID!,$t:String!){productUpdate(input:{id:$id,title:$t}){product{id}}}',
                    {'id': existing_id, 't': _title_update})
                shop_pid = await shopify_update_product(existing_id, body_html, seo_t, seo_d, handle)
                print(f'    Updated: {existing_id}')
            else:
                p_type = stage1.get('category', '')
                # Use the clean SEO title as the product title — the raw EPROLO
                # scrape name is keyword-stuffed junk (shows in cart/tab/Google
                # H1). Raw name is preserved in custom.source for provenance.
                _clean_title = (seo_t or product['title']); _clean_title = (_clean_title[:60].rsplit(' ',1)[0] if len(_clean_title) > 62 else _clean_title)[:255]
                shop_pid = await shopify_create_product(_clean_title, body_html, handle, seo_t, seo_d, tags,
                    product_type=p_type, vendor=STORE_VENDOR)
                print(f'    Created: {shop_pid} (title: {_clean_title[:50]})')
            if shop_pid:
                # Persist sid IMMEDIATELY — even if next steps crash, we keep the link DB↔Shopify
                # for resumable retries / backfill scripts.
                db.execute('UPDATE products SET shopify_product_id=? WHERE id=?', (shop_pid, pid))
                db.commit()
                # Write 9 JSON metafields in one batch metafieldsSet call.
                # full_html stores {"sections": {...}, "meta": {...}} from Stage 4.
                if full_html:
                    try:
                        _content = json.loads(full_html)
                    except Exception:
                        _content = {}
                    _sections = _content.get('sections', {}) if isinstance(_content, dict) else {}
                    # Inject EPROLO source provenance into admin-only custom.source metafield.
                    # Customer never sees this — it's purely so the operator can audit
                    # the original product page from Shopify Admin.
                    try:
                        _src_meta = {
                            'platform':       'EPROLO',
                            'url':            purl or '',
                            'scraped_at':     str(row['updated_at']) if 'updated_at' in row.keys() else '',
                            'title':          (product.get('title') or '')[:200],
                            'description':    (product.get('description') or '')[:5000],
                            'cost_price_usd': product.get('cost_price', 0),
                            'image_urls': {
                                'top':  list(product.get('top_image_urls')  or [])[:50],
                                'desc': list(product.get('desc_image_urls') or [])[:50],
                            },
                        }
                        # Surface silent truncation — operator can decide whether to
                        # widen the [:50] cap. EPROLO occasionally returns 60-80 desc
                        # image URLs; rest are dropped from the audit record.
                        _src_top_n  = len(product.get('top_image_urls')  or [])
                        _src_desc_n = len(product.get('desc_image_urls') or [])
                        if _src_top_n > 50 or _src_desc_n > 50:
                            print(f'    \u26a0 custom.source image_urls truncated (top {_src_top_n}\u219250, desc {_src_desc_n}\u219250)')
                        _sections['source'] = _src_meta
                        # Mirror the URL as a standalone clickable button so the
                        # operator can open the EPROLO source page from Admin in
                        # one click without expanding the source JSON blob.
                        if purl:
                            _sections['source_url'] = re.sub(r'(--\d+)-\d+$', r'\1', (purl or '').strip().split('?')[0].rstrip('/'))
                        # Inject auto-generated research_refs from Strategy
                        # (Opus 4.7 emits per-category peer-reviewed sources).
                        _strat_row_rr = db.execute(
                            'SELECT strategy_json FROM products WHERE id=?', (pid,)
                        ).fetchone()
                        _strategy_rr = json.loads(_strat_row_rr[0]) if (_strat_row_rr and _strat_row_rr[0]) else {}
                        _rr_obj = (_strategy_rr or {}).get('research_refs') or {}
                        if _rr_obj and (_rr_obj.get('items') or []):
                            _sections['research_refs'] = _rr_obj
                    except Exception as _e_src:
                        print(f'    ⚠ source metafield build failed: {str(_e_src)[:80]}')
                    # Build batch metafieldsSet input — up to 23 JSON metafields per product.
                    # Optional modules (how_to/specs/etc) skip if designer left them blank.
                    # Round-6 order — matches sections/wanelo-product-page.liquid render
                    # sequence. 35 storefront + 2 admin-only = 37 keys total.
                    _wanelo_keys = [
                        # Hero + positioning
                        'hero', 'videos', 'target_profile', 'brand_story',
                        # Narrative + media
                        'story', 'features', 'video_demo', 'lifestyle_gallery',
                        # Ingredients + usage
                        'ingredients', 'nutrition_facts', 'how_to', 'protocol',
                        'timeline', 'clinical_evidence',
                        # Specs + tech
                        'stats', 'specs', 'compatibility', 'app_showcase',
                        # Fit + form
                        'size_guide', 'dimensions', 'room_placement', 'variants',
                        # Care + box
                        'care', 'whats_included', 'assembly',
                        # Conversion
                        'gift_options', 'subscription_refill', 'compare',
                        # Credentials
                        'trust', 'sustainability', 'safety',
                        # Social + close
                        'reviews', 'faq', 'cta', 'interlinks', 'research_refs',
                        # Conversion-focused universal sections (added 2026-05-27)
                        'unboxing_journey', 'progress_milestones', 'mistake_warnings', 'use_scenarios',
                        # Theme + admin
                        'palette',
                        'photo_pack', 'source',
                        # Operator-convenience clickable URL + plain-text designer
                        # prompt. Both admin-only (visible_to_storefront=False at
                        # definition time). source_url duplicates source.url; 
                        # designer_prompt duplicates prompt.txt inside the ZIP.
                        'source_url', 'designer_prompt',
                    ]
                    # photo_pack + source_url are type 'url' (admin clickable button);
                    # designer_prompt is type 'multi_line_text_field' (admin text area).
                    # All others are json. Special-cased so values are not JSON-encoded.
                    _mf_types = {
                        'photo_pack':       'url',
                        'source_url':      'url',
                        'designer_prompt': 'multi_line_text_field',
                    }
                    # Build _mf_inputs from BOTH _wanelo_keys (canonical order)
                    # AND any extra keys Designer emitted that we don't yet
                    # have a metafield definition for. Auto-create those defs
                    # on-the-fly via shopify_ensure_metafield_definition.
                    _curated = globals().get("_KNOWN_CUSTOM_KEYS") or set()
                    _emitted_extra = [
                        k for k, v in (_sections or {}).items()
                        if k not in _wanelo_keys and isinstance(v, (dict, list)) and v
                    ]
                    _all_keys = list(_wanelo_keys) + _emitted_extra

                    _mf_inputs = []
                    for _k in _all_keys:
                        _v = _sections.get(_k)
                        _t = _mf_types.get(_k, 'json')
                        if _t == 'url':
                            if not isinstance(_v, str) or not _v.startswith('http'):
                                continue
                            _val = _v
                        elif _t == 'multi_line_text_field':
                            if not isinstance(_v, str) or not _v.strip():
                                continue
                            _val = _v
                        else:
                            _val = json.dumps(_v or {}, ensure_ascii=False)
                        _mf_inputs.append({
                            "ownerId": shop_pid,
                            "namespace": "custom",
                            "key": _k,
                            "type": _t,
                            "value": _val,
                        })

                    # ════ AUTO-CREATE definitions for ANY emitted key that
                    # ════ doesn't yet have one in Shopify. Idempotent —
                    # ════ TAKEN errors are silently swallowed.
                    _to_create = [
                        m["key"] for m in _mf_inputs
                        if m["value"] not in ("{}", "", None)
                        and m["key"] not in _curated
                    ]
                    if _to_create:
                        # NOTE: _curated already prevents redundant calls —
                        # successful ensure adds key to _curated below, so
                        # the original `_to_create` filter (m["key"] not in
                        # _curated) skips already-known keys. No second
                        # cache needed; a separate "attempted" cache would
                        # block legitimate retries after network failures.
                        print(f"    auto-create definitions: {_to_create}")
                        for _ack in _to_create:
                            # Friendly name: my_section_name -> 'Wanelo · My Section Name'
                            _friendly = "Wanelo · " + " ".join(
                                w.capitalize() for w in _ack.replace('-', '_').split('_')
                            )
                            _desc = f"Auto-created by pipeline on first emission of '{_ack}'. Operator can rename in Shopify Admin."
                            try:
                                _new_id = await shopify_ensure_metafield_definition(
                                    name=_friendly,
                                    namespace="custom",
                                    key=_ack,
                                    type_name=_mf_types.get(_ack, 'json'),
                                    description=_desc,
                                    pin=False,
                                    visible_to_storefront=(_ack not in ('source', 'photo_pack', 'source_url', 'designer_prompt')),
                                )
                                # Refresh allowlist in-memory so filter below accepts it
                                _curated.add(_ack)
                                globals()["_KNOWN_CUSTOM_KEYS"] = _curated
                                print(f"      + {_ack:25s} ({_mf_types.get(_ack, 'json')})")
                            except Exception as _e_ac:
                                print(f"      ⚠ failed to create '{_ack}': {str(_e_ac)[:100]}")

                    # Filter: drop empty {} (Designer SKIPped) + drop keys
                    # we still don't have definition for (auto-create failed).
                    _before = len(_mf_inputs)
                    _mf_inputs = [
                        m for m in _mf_inputs
                        if m.get("value") not in ("{}", "", None)
                        and (m.get("namespace") != "custom"
                             or m.get("key") in _curated)
                    ]
                    _drop = _before - len(_mf_inputs)
                    if _drop:
                        print(f"    filtered {_drop} metafields (empty/non-curated)")
                    _mq = "mutation($m:[MetafieldsSetInput!]!){metafieldsSet(metafields:$m){metafields{id namespace key} userErrors{field message code}}}"
                    # Shopify metafieldsSet hard limit: 25 inputs per call.
                    # We can have 35+ definitions, so chunk to stay under.
                    # LESS_THAN_OR_EQUAL_TO error otherwise rejects the WHOLE batch (Bug B).
                    _MAX_MF_CHUNK = 25
                    _ok_mf = 0
                    _err_mf = []
                    _top_errs_all = []
                    _total_size = sum(len(m['value']) for m in _mf_inputs)
                    _n_chunks = (len(_mf_inputs) + _MAX_MF_CHUNK - 1) // _MAX_MF_CHUNK
                    for _ci in range(0, len(_mf_inputs), _MAX_MF_CHUNK):
                        _chunk = _mf_inputs[_ci:_ci + _MAX_MF_CHUNK]
                        _r_mf = await shopify_gql(_mq, {"m": _chunk})
                        _top_err = (_r_mf or {}).get('errors') if isinstance(_r_mf, dict) else None
                        if _top_err:
                            if isinstance(_top_err, list):
                                _top_errs_all.extend(_top_err)
                            else:
                                _top_errs_all.append(_top_err)
                        _data_mf = ((_r_mf or {}).get('data', {}) or {}).get('metafieldsSet', {}) or {}
                        _ok_mf += len(_data_mf.get('metafields') or [])
                        _err_mf.extend(_data_mf.get('userErrors') or [])
                    if _top_errs_all:
                        print(f'    \u26a0 GraphQL top-level errors: {_top_errs_all}')
                    print(f'    Metafields: {_ok_mf}/{len(_mf_inputs)} written ({_total_size} chars total, {_n_chunks} chunks of \u2264{_MAX_MF_CHUNK})')
                    if _ok_mf == 0 and _mf_inputs:
                        print(f'    \u26a0 ZERO metafields written. Inputs sent:')
                        for _mi in _mf_inputs:
                            _vlen = len(_mi.get('value') or '')
                            print(f'       - {_mi["key"]:<22} type={_mi["type"]:<6} value_len={_vlen}')
                    if _err_mf:
                        print(f'    \u26a0 userErrors ({len(_err_mf)} total):')
                        for _eu in _err_mf[:10]:
                            print(f'       - field={_eu.get("field")} code={_eu.get("code")} message={_eu.get("message")}')
                else:
                    print(f'    \u26a0 Metafields SKIPPED: no JSON content for product {pid}')
                # Sort gallery by Vision-assigned role so 'hero' goes first in
                # Shopify carousel — provides editorial first-impression even
                # before operator runs the photo_pack edit workflow. Order:
                # hero → lifestyle → in_use → detail → packaging → other.
                _ROLE_PRIORITY = {'hero':0, 'lifestyle':1, 'in_use':2, 'feature':3,
                                  'detail':4, 'ingredient':5, 'packaging':6, 'before_after':7}
                _role_by_index = {ii.get('index'): (ii.get('role','') or 'other')
                                  for ii in stage1.get('images', [])}
                _gallery_raw = [img for img in img_data.get('top',[]) if img.get('url','').startswith('http')]
                _gallery_raw.sort(key=lambda im: _ROLE_PRIORITY.get(
                    _role_by_index.get(im.get('index'), 'other'), 99))
                gallery = [img['url'] for img in _gallery_raw]
                _alt_map = {}
                for img_info in stage1.get('images', []):
                    idx = img_info.get('index')
                    desc = img_info.get('desc', '')
                    if idx and desc: _alt_map[idx] = desc
                og_img = None
                if gallery:
                    og_img = gallery[0]
                    n = await shopify_attach_media(shop_pid, gallery, alt_map=_alt_map)
                    print(f'    Gallery: {n}/{len(gallery)} (EPROLO)')
                if og_img:
                    await shopify_set_metafield(shop_pid, 'global', 'og_image', og_img, mtype='url')
                # Attach EPROLO product video(s) to the carousel (after images)
                _vid_urls = product.get('video_urls', []) or []
                _n_vid_ok = 0
                for _vu in _vid_urls[:2]:  # cap 2 videos per product
                    if await shopify_attach_video_from_url(shop_pid, _vu,
                                                           alt=f"{product.get('title','')[:60]} video"):
                        _n_vid_ok += 1
                if _vid_urls:
                    print(f'    Videos: {_n_vid_ok}/{min(len(_vid_urls),2)} attached to carousel')
                weight_g = product.get('weight_g', 0)
                if weight_g > 0:
                    ok = await shopify_set_weight(shop_pid, weight_g)
                    print(f'    Weight: {weight_g:.0f}g {"OK" if ok else "FAIL"}')
                g_cat = stage1.get('category', '')
                if g_cat:
                    ok = await shopify_set_google_shopping(shop_pid, g_cat)
                    if ok: print(f'    Google Shopping: {g_cat}')
                n_merchant = await shopify_set_merchant_extended(shop_pid, stage1, variants)
                if n_merchant: print(f'    Merchant: {n_merchant} fields')
                if variants:
                    if product.get('variant_prices') and variants:
                        variants[0]['_variant_prices'] = product['variant_prices']
                    nv = await shopify_create_variants(shop_pid, variants, product.get('cost_price', 0))
                    print(f'    Variants: {nv} created')
                elif product.get('cost_price', 0) > 0:
                    cost = product['cost_price']
                    bundle_n = detect_bundle(product.get('title',''))
                    if bundle_n > 1:
                        p = round(calc_price(cost) * min(bundle_n * 0.8, 3.5), 2)
                        p = round(round(p) - 0.10, 2)
                        cp = round(calc_compare_price(cost) * bundle_n)
                        print(f'    Bundle: {bundle_n} pcs ${p:.2f}')
                    else:
                        p, cp = calc_price(cost), calc_compare_price(cost)
                    ok = await shopify_set_price(shop_pid, p, cp)
                    print(f'    Price: ${p:.2f} (was ${cp:.0f}) {"OK" if ok else "FAIL"}')
                # Always-in-stock LAST so it covers ALL variants (default +
                # any options created by shopify_create_variants above).
                # Sets tracked=False + inventoryPolicy=CONTINUE per variant.
                inv_count = await shopify_set_inventory(shop_pid, DEFAULT_INVENTORY)
                if inv_count: print(f'    Inventory: {inv_count} variants untracked + CONTINUE')
                # Publish to storefront channels (Online Store + Shop).
                # productCreate(status=ACTIVE) alone does NOT attach to
                # any sales channel — storefront returns 404. Idempotent.
                _pub_n = await shopify_publish_to_storefronts(shop_pid)
                if _pub_n: print(f'    Published: {_pub_n} channels (Online Store + Shop)')
                # Link to collections (primary + any from junction table)
                coll_ids = set()
                # Method 1: From junction table
                for c in get_product_collections(pid):
                    if len(c) > 2 and c[2]: coll_ids.add(c[2])
                # Method 2: Direct from primary_collection field (more reliable)
                # Defensive: column may not exist in legacy DBs (it's added via ALTER in cell #2).
                try:
                    pc_row = db.execute('SELECT primary_collection FROM products WHERE id=?', (pid,)).fetchone()
                except Exception:
                    pc_row = None
                if pc_row and pc_row['primary_collection']:
                    pc_title = pc_row['primary_collection']
                    # Find Shopify collection ID by seo_title
                    cr = db.execute('SELECT shopify_collection_id FROM collections WHERE seo_title=?', (pc_title,)).fetchone()
                    if cr and cr['shopify_collection_id']:
                        coll_ids.add(cr['shopify_collection_id'])
                    else:
                        # Fallback: search by handle in ALL_SHOP_COLLECTIONS
                        cr2 = db.execute('SELECT seo_handle FROM collections WHERE seo_title=?', (pc_title,)).fetchone()
                        if cr2:
                            for sc in ALL_SHOP_COLLECTIONS:
                                if sc.get('handle') == cr2['seo_handle']:
                                    coll_ids.add(sc['id'])
                                    break
                # Method 3: fallback — DB пуста или primary_collection не сматчился.
                # Берём stage1.category + название товара, fuzzy-match против Shopify
                # коллекций (ALL_SHOP_COLLECTIONS уже скачаны в начале Cell 6).
                if not coll_ids and ALL_SHOP_COLLECTIONS:
                    _cat = (stage1.get('category') or '').strip()
                    _src = (_cat + ' ' + pname).lower()
                    _src_words = set(re.sub(r'[^a-z0-9 ]', ' ', _src).split())
                    _stop = {'and','or','for','the','a','an','of','to','with','in','on','&','-','set','kit','pack','pcs','pc','best','new'}
                    _src_words -= _stop
                    _src_words = {w for w in _src_words if len(w) > 2}
                    if _src_words:
                        _scored = []
                        for sc in ALL_SHOP_COLLECTIONS:
                            _t = (sc.get('title','') + ' ' + sc.get('handle','').replace('-',' ')).lower()
                            _t_words = set(re.sub(r'[^a-z0-9 ]', ' ', _t).split())
                            _score = len(_src_words & _t_words)
                            if _score >= 2:  # минимум 2 общих слова чтобы исключить шум
                                _scored.append((_score, sc.get('id'), sc.get('title','')))
                        _scored.sort(reverse=True)
                        # Берём топ-2 совпадения (primary + 1 related)
                        for _score, _cid, _ctitle in _scored[:2]:
                            if _cid:
                                coll_ids.add(_cid)
                                print(f'    Method 3: matched (score={_score}) → {_ctitle[:45]}')
                if coll_ids:
                    await shopify_add_to_collections(shop_pid, list(coll_ids))
                    print(f'    Collections: {len(coll_ids)}')
                else:
                    print(f'    ⚠ No collections found for this product (DB+vision both empty)')
                _cost = product.get('cost_price', 0)
                _sell = calc_price(_cost) if _cost > 0 else 0
                if _sell > 0 and _cost > 0:
                    _fees = _sell * 0.029 + 0.30 + _sell * 0.02
                    _net = _sell - _cost - _fees
                    _margin_pct = (_net / _sell) * 100
                    if _margin_pct < 40:
                        print(f'    MARGIN: ${_sell:.2f} - ${_cost:.2f} - ${_fees:.2f} = ${_net:.2f} ({_margin_pct:.0f}%)')
                db_update_status(pid, 'done', shopify_product_id=shop_pid)
            else:
                db_update_status(pid, 'done')
            print(f'  DONE!')
            shutil.copy(DB_LOCAL, DB_PATH)

    except Exception as e:
        import traceback; traceback.print_exc()
        db_update_status(pid, 'error', error_msg=str(e)[:500])
        print(f'  ERROR: {e}')


# ── Dispatcher: serial (BATCH_CONCURRENCY=1) or concurrent (>1) ──
def _print_cost_checkpoint(idx, total_n):
    """Periodic cost checkpoint so 10K-batch operators see spend live,
    not only at end-of-batch. Triggers every 25 finished products."""
    if (idx + 1) % 25 == 0 or (idx + 1) == total_n:
        _spent = _anthropic_cost_tracker['spent_usd']
        _proj  = (_spent / (idx + 1)) * total_n if (idx + 1) > 0 else 0
        print(f'\n  ── Checkpoint [{idx+1}/{total_n}] ── '
              f'${_spent:.2f} spent · ${_proj:.2f} projected · '
              f'{_anthropic_cost_tracker["calls"]} AI calls ──')

if BATCH_CONCURRENCY <= 1:
    # Serial path — preserves natural log streaming, zero overhead.
    for _pi, _prod in enumerate(pending):
        await _process_one(_pi, _prod)
        _print_cost_checkpoint(_pi, len(pending))
else:
    # Concurrent path — Semaphore caps in-flight tasks; each task buffers
    # its stdout so per-product log blocks stay intact in mixed output.
    _enable_task_buffering()
    _sem = asyncio.Semaphore(BATCH_CONCURRENCY)
    async def _slot(_pi, _prod):
        async with _sem:
            _buf = _io_buf.StringIO()
            _tok = _TASK_LOG_BUF.set(_buf)
            try:
                await _process_one(_pi, _prod)
            except Exception as _e_top:
                _buf.write(f'\n  TOP-LEVEL TASK ERROR: {str(_e_top)[:200]}\n')
            finally:
                _TASK_LOG_BUF.reset(_tok)
                _real_stdout.write(_buf.getvalue())
                _real_stdout.flush()
    # Wrap _slot to print checkpoint after each finishes (still under sem).
    _finished_count = {'n': 0}
    async def _slot_with_checkpoint(_pi, _prod):
        await _slot(_pi, _prod)
        _finished_count['n'] += 1
        _print_cost_checkpoint(_finished_count['n'] - 1, len(pending))
    try:
        await asyncio.gather(*[_slot_with_checkpoint(_pi, _p) for _pi, _p in enumerate(pending)])
    finally:
        _disable_task_buffering()

print(f'\n{"="*60}')
# Per-status counts so user sees exactly where products got stuck.
# Status order matches pipeline flow — anything left of 'done' is incomplete.
_status_order = ['pending', 'scraped', 'images_uploaded', 'vision_done',
                 'strategy_done', 'html_ready', 'pack_done', 'done', 'error']
_counts = {}
for _st in _status_order:
    _counts[_st] = db.execute("SELECT COUNT(*) FROM products WHERE status=?", (_st,)).fetchone()[0]
d = _counts['done']
e = _counts['error']
cost = db.execute("SELECT SUM(cost_usd) FROM products").fetchone()[0] or 0
s = db.execute("SELECT COUNT(*) FROM products WHERE shopify_product_id IS NOT NULL").fetchone()[0]
print(f'  Done: {d}/{total} | Shopify pushed: {s} | Errors: {e} | Cost: ${cost:.4f}')

# ── Anthropic spend + cache hit verification ──
# Cache should account for ≥40% of input tokens after the first 3-4 products
# of a batch warm the cache. <10% means cache_control is broken or TTL
# expired (5 min). >80% means cache is working perfectly.
_load_usage_once()  # merge prior restart-segments so the ratio is JOB-WIDE, not just this segment
_cct = _anthropic_cost_tracker
_total_in = _cct['fresh_in'] + _cct['cache_read_in'] + _cct['cache_create_in']
_cache_ratio = (_cct['cache_read_in'] / _total_in * 100) if _total_in else 0.0
print(f"  Anthropic: {_cct['calls']} calls | spent ${_cct['spent_usd']:.2f} of ${MAX_ANTHROPIC_BUDGET:.2f} cap")
print(f"    Input tokens: {_total_in:,} | cache hit: {_cache_ratio:.1f}% "
      f"({_cct['cache_read_in']:,} read / {_cct['cache_create_in']:,} written / "
      f"{_cct['fresh_in']:,} fresh)")
if _cache_ratio < 15 and _total_in > 50000:
    print(f"    \u26a0 blended cache hit low ({_cache_ratio:.1f}%) — usually CHEAP Haiku image tokens, not a leak; check per-model below")
for _m, _mv in sorted(_anthropic_cost_tracker.get('by_model', {}).items(), key=lambda kv: -kv[1].get('spent', 0)):
    _mt = _mv['fresh'] + _mv['read'] + _mv['create']
    _mr = (_mv['read'] / _mt * 100) if _mt else 0.0
    print(f"      {_m:<26} {_mv['calls']:>4} calls | cache {_mr:4.1f}% "
          f"({_mv['read']:,}r/{_mv['create']:,}w/{_mv['fresh']:,}f) | ${_mv['spent']:.2f}")
# Per-status breakdown — only show non-zero stuck states
_stuck = {st: n for st, n in _counts.items() if st not in ('done', 'pending') and n > 0}
if _stuck:
    print(f'  ⚠ {sum(_stuck.values())} products NOT done — re-run Cell 6 to resume:')
    for _st, _n in _stuck.items():
        if _st == 'error':
            # Pull a sample error message
            _samples = db.execute("SELECT name, error_msg FROM products WHERE status='error' LIMIT 3").fetchall()
            _samp_str = ' | '.join(f'{(r[0] or "?")[:30]}: {((r[1] or "no msg"))[:60]}' for r in _samples)
            print(f'    {_st:18} {_n:5}  e.g. {_samp_str}')
        else:
            print(f'    {_st:18} {_n:5}  (will resume from this step on next Cell 6 run)')
    # Hint for re-running errors
    if _counts['error']:
        print(f'\n  To retry errored products: db.execute("UPDATE products SET status=\'pending\', error_msg=NULL WHERE status=\'error\'"); db.commit(); then ►Cell 6')
print(f'{"="*60}')
shutil.copy(DB_LOCAL, DB_PATH)
# Tear down the shared Playwright browser before post-loop work — it's
# no longer needed and holding a Chromium process steals memory in Colab.
await close_shared_browser_ctx()

if d > 0:
    ok = ping_google_sitemap(STORE_DOMAIN)
    print(f'  Google sitemap ping: {"OK" if ok else "FAIL"}')


Shopify collections for interlinking: 100

  26 total | 0 done | 26 remaining

────────────────────────────────────────────────────────────
[1/26] Transparent Crystal Grape Soap Gift Box Set, Handmade Grape 
  Status: pending
  → Scraping...
  Scrape: 10 top + 6 desc | $1.81 | variants: none | 7 specs
    Title: Transparent Crystal Grape Soap Gift Box Set, Handmade Grape 
  → Images...
    CDN: 16/16
  → Vision...
    OK $0.0158 | decorative | 5 imgs | claims:5 ingr:9
    Hero: Handmade decorative soap that's beautifu
  → Generating HTML + SEO...
    Removed bad collection: natural-skincare-essentials
    Removed bad collection: decorative-soap-sets
    Removed bad collection: natural-skincare-essentials
    Removed bad collection: luxury-bath-gift-sets
    Removed bad collection: luxury-bath-gift-sets
    Removed bad collection: decorative-soap-sets
    Removed bad collection: natural-skincare-essentials
    HTML: 27885 chars | $0.0988 | QA:WARN
  → Shopify...
    Created: gid://shopi

Traceback (most recent call last):
  File "/tmp/ipykernel_2552/2687877196.py", line 633, in <cell line: 1>
    pc_row = db.execute('SELECT primary_collection FROM products WHERE id=?', (pid,)).fetchone()
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
sqlite3.OperationalError: no such column: primary_collection


  Scrape: 5 top + 3 desc | $1.81 | variants: none | 5 specs, size chart
    Title: Style Solid Color Water-Absorbing Hair Drying Cap Thickened 
  → Images...
    CDN: 8/8
  → Vision...
    OK $0.0088 | simple | 5 imgs | claims:4 ingr:0
    Hero: Super-absorbent microfiber dries hair in
  → Generating HTML + SEO...
    Removed bad collection: hair-care-accessories
    Removed bad collection: bath-towels
    Removed bad collection: bathroom-essentials
    HTML issues: no inline body links found (all links in footer only)
    Meta issues: meta description too long: 173 chars
    HTML: 24138 chars | $0.0819 | QA:WARN
  → Shopify...
    Created: gid://shopify/Product/8790649897138
    Metafield: OK (24138 chars)
    Gallery: 5/5
    Google Shopping: Bath & Towels / Hair Care Accessories
    Merchant: 5 fields
    Price: $8.90 (was $15) OK
  ERROR: no such column: primary_collection

────────────────────────────────────────────────────────────
[3/26] Baby Silicone Shampoo Scalp Massage Body 

Traceback (most recent call last):
  File "/tmp/ipykernel_2552/2687877196.py", line 633, in <cell line: 1>
    pc_row = db.execute('SELECT primary_collection FROM products WHERE id=?', (pid,)).fetchone()
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
sqlite3.OperationalError: no such column: primary_collection


  Scrape: 6 top + 0 desc | $1.81 | variants: none | 6 specs
    Title: Baby Silicone Shampoo Scalp Massage Body Clean Scrub Feel Ba
  → Images...


## Cell 7 — Status

In [ ]:
for s in ['done','html_ready','content_ready','vision_done','images_uploaded','scraped','pending','error']:
    n = db.execute("SELECT COUNT(*) FROM products WHERE status=?", (s,)).fetchone()[0]
    if n > 0: print(f'  {s:>18}: {n}')
cost = db.execute("SELECT SUM(cost_usd) FROM products").fetchone()[0] or 0
s_ok = db.execute("SELECT COUNT(*) FROM products WHERE shopify_product_id IS NOT NULL").fetchone()[0]
print(f'  Shopify: {s_ok} | Cost: ${cost:.4f}')
# ── Smart Retry (resume from last successful step) ──
# db.execute("UPDATE products SET status='scraped' WHERE status='error' AND scrape_json IS NOT NULL AND image_urls_json IS NULL"); db.commit()
# db.execute("UPDATE products SET status='images_uploaded' WHERE status='error' AND image_urls_json IS NOT NULL AND stage1_json IS NULL"); db.commit()
# db.execute("UPDATE products SET status='vision_done' WHERE status='error' AND stage1_json IS NOT NULL AND final_html IS NULL"); db.commit()
# db.execute("UPDATE products SET status='html_ready' WHERE status='error' AND final_html IS NOT NULL AND shopify_product_id IS NULL"); db.commit()
# db.execute("UPDATE products SET status='pending' WHERE status='error' AND scrape_json IS NULL"); db.commit()  # Full re-scrape only if no data

# ── Batch Publish (after reviewing DRAFT products) ──
# Uncomment to activate all DRAFT products at once:
# import httpx, asyncio
# async def batch_publish():
#     rows = db.execute("SELECT shopify_product_id FROM products WHERE shopify_product_id IS NOT NULL").fetchall()
#     for r in rows:
#         pid = r['shopify_product_id']
#         await shopify_gql("mutation($p:ProductUpdateInput!){productUpdate(product:$p){product{id}userErrors{field message}}}",
#             {"p":{"id":pid,"status":"ACTIVE"}})
#         await asyncio.sleep(0.3)
#     print(f"Published {len(rows)} products")
# await batch_publish()
